# Key idea + main steps (what the whole script is doing)

Goal: train a learner in a 1D grid world where another agent (“teacher”) competes for food. The learner has access to social observation (sees the other agent position) and you optionally add an observation reward shaping phase.

Main pipeline

Define environment (SocialFoodEnv): two agents + one food on a 1D grid.

Train teacher using standard DQN without social info (teacher sees only [self_pos, food_pos]).

Train baseline learner using DQN with social observation in the input (learner sees [self_pos, other_pos, food_pos]), but no special observation reward.

Train observation learner:

Phase A (observation reward shaping): teacher acts greedily; learner explores; learner gets extra reward when it “observes” near teacher’s eat event.

Phase B (execution RL): learner continues standard DQN on task reward only.

Evaluate baseline vs observation learner across many episodes and many random seeds.

> Note: `main.py` is the canonical runnable implementation for this repository.\n> This notebook is kept as archived exploratory analysis with outputs cleared.\n

## diagram

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, FancyArrowPatch
import matplotlib as mpl

def plot_fig_ab_social_lick_task(
    grid_size=15,
    window_steps=3,
    teacher_period=60,
    lick_duration_steps=1,
    example_lick_t=3.0,            # for schematic Panel B (seconds)
    example_obs_bouts=((2.0, 2.25), (2.8, 3.05), (3.55, 3.70)),
    example_eat_t=4.2,             # eat attempt marker inside window
    titleA="A",
    titleB="B",
):
    """
    Schematic Figure A–B for your paradigm:
      - A: task diagram (teacher licking; learner observes + goes to patch + eats)
      - B: timing contingency (lick -> 3s reward window; observe bouts; eat)
    """

    mpl.rcParams["font.size"] = 10

    fig = plt.figure(figsize=(14, 5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.65, 1.0], wspace=0.20)

    # -------------------------
    # Panel A
    # -------------------------
    axA = fig.add_subplot(gs[0, 0])
    axA.set_axis_off()
    axA.set_xlim(0, 1)
    axA.set_ylim(0, 1)

    # Panel label
    axA.text(0.01, 0.98, titleA, fontsize=16, fontweight="bold", va="top")

    # Agent boxes
    # Teacher box
    teacher_box = Rectangle((0.05, 0.55), 0.34, 0.36, fill=False, linewidth=1.8)
    axA.add_patch(teacher_box)
    axA.text(0.07, 0.88, "Teacher (demonstrator)", fontweight="bold")
    axA.text(0.07, 0.83, "Policy: fixed schedule", fontsize=9)
    axA.text(0.07, 0.78, f"Lick bout: {lick_duration_steps} step", fontsize=9)
    axA.text(0.07, 0.73, f"Period: ~{teacher_period} steps", fontsize=9)
    axA.text(0.07, 0.66, "Output:", fontweight="bold", fontsize=9)
    axA.text(0.15, 0.66, "Lick event", fontsize=9)

    # Learner box
    learner_box = Rectangle((0.61, 0.55), 0.34, 0.36, fill=False, linewidth=1.8)
    axA.add_patch(learner_box)
    axA.text(0.63, 0.88, "Learner (student)", fontweight="bold")
    axA.text(0.63, 0.83, "Policy: DQN", fontsize=9)

    axA.text(0.63, 0.78, "Observations:", fontweight="bold", fontsize=9)
    axA.text(0.65, 0.74, "• self position", fontsize=9)
    axA.text(0.65, 0.70, "• own food patch position", fontsize=9)
    axA.text(0.65, 0.66, "• lick/window signal (if observing)", fontsize=9)
    axA.text(0.65, 0.62, "• attention / cooldown (optional)", fontsize=9)

    axA.text(0.63, 0.56, "Actions:", fontweight="bold", fontsize=9)
    axA.text(0.65, 0.52, "move ◀ / stay / move ▶", fontsize=9)
    axA.text(0.65, 0.48, "eat", fontsize=9)
    axA.text(0.65, 0.44, "observe (0/1)", fontsize=9)

    # Environment box (middle)
    env_box = Rectangle((0.41, 0.55), 0.18, 0.36, fill=False, linewidth=1.8)
    axA.add_patch(env_box)
    axA.text(0.43, 0.88, "Environment", fontweight="bold")

    # Draw 1D track inside env box
    x0, y0 = 0.43, 0.70
    x1 = 0.56
    axA.plot([x0, x1], [y0, y0], linewidth=3)

    # Tick marks for grid cells (schematic)
    for k in range(6):
        xx = x0 + (x1 - x0) * k / 5
        axA.plot([xx, xx], [y0 - 0.01, y0 + 0.01], linewidth=1)

    # Teacher & learner icons + patches (simple shapes)
    teacher_pos = (x0 + 0.22 * (x1 - x0), y0)
    learner_pos = (x0 + 0.78 * (x1 - x0), y0)
    teacher_patch = (x0 + 0.18 * (x1 - x0), y0)
    learner_patch = (x0 + 0.82 * (x1 - x0), y0)

    axA.add_patch(Circle(teacher_pos, 0.012, color="black"))
    axA.text(teacher_pos[0] - 0.02, teacher_pos[1] + 0.05, "Teacher", fontsize=9)

    axA.add_patch(Circle(learner_pos, 0.012, color="black"))
    axA.text(learner_pos[0] - 0.02, learner_pos[1] + 0.05, "Learner", fontsize=9)

    # Food patches (triangles via marker)
    axA.scatter([teacher_patch[0]], [teacher_patch[1]], marker="^", s=90)
    axA.scatter([learner_patch[0]], [learner_patch[1]], marker="^", s=90)
    axA.text(teacher_patch[0] - 0.05, teacher_patch[1] - 0.07, "Teacher patch", fontsize=8)
    axA.text(learner_patch[0] - 0.05, learner_patch[1] - 0.07, "Learner patch", fontsize=8)

    # Arrows: teacher lick -> learner observation
    axA.add_patch(FancyArrowPatch((0.39, 0.73), (0.61, 0.73), arrowstyle="->", mutation_scale=12, linewidth=1.5))
    axA.text(0.43, 0.76, "lick timing info\n(visible if observe=1)", fontsize=8)

    # Reward rule text
    axA.text(
        0.05, 0.08,
        f"Reward (learner): must be at learner patch AND eat within {window_steps} steps after teacher lick\n"
        f"AND must have detected the lick (via observing / leak).",
        fontsize=10
    )

    # -------------------------
    # Panel B
    # -------------------------
    axB = fig.add_subplot(gs[0, 1])
    axB.set_xlim(0, 10)
    axB.set_ylim(0, 1)
    axB.spines["top"].set_visible(False)
    axB.spines["right"].set_visible(False)
    axB.spines["left"].set_visible(False)
    axB.set_yticks([])
    axB.set_xlabel("time (seconds; schematic)")
    axB.set_title(f"{titleB}  Lick → reward window (3s) + observe bouts", loc="left", fontweight="bold")

    # Lick event (1s)
    lick_t = example_lick_t
    axB.axvline(lick_t, linestyle="--", linewidth=1.5)
    axB.text(lick_t + 0.1, 0.86, "lick", fontsize=10)
    axB.plot([lick_t, lick_t + 0.8], [0.80, 0.80], linewidth=6)  # lick duration bar
    axB.text(lick_t + 0.85, 0.78, f"{lick_duration_steps}s bout", fontsize=9)

    # Reward window shading: [lick_t, lick_t+window_steps)
    axB.axvspan(lick_t, lick_t + window_steps, alpha=0.18)
    axB.text(lick_t + 0.15, 0.62, f"reward window\n({window_steps}s after last lick)", fontsize=9)

    # Observe bouts (short, frequent)
    y_obs = 0.30
    for (a, b) in example_obs_bouts:
        axB.plot([a, b], [y_obs, y_obs], linewidth=7)
    axB.text(0.25, y_obs + 0.03, "observe bouts", fontsize=9)

    # Eat attempt marker (needs to be at patch too; indicated by note)
    axB.scatter([example_eat_t], [0.45], marker="*", s=200)
    axB.text(example_eat_t + 0.2, 0.43, "eat @ patch", fontsize=10)

    # Annotation: must observe + go to patch
    axB.text(0.2, 0.05, "Learner strategy: short peeks (observe) → move to patch → eat within window", fontsize=9)

    plt.tight_layout()
    plt.show()


# Example call:
plot_fig_ab_social_lick_task()


## multiple control

In [ ]:
"""
=====================================================================
Two-mice social timing task (1D grid): observe lick + go to patch + eat
=====================================================================

YOUR CONTINGENCY (implemented):
- Teacher emits a lick event lasting 1 second (1 step) at roughly every 60s (period),
  with jitter (unpredictability).
- Each lick opens a 3-second reward window AFTER that lick:
    window_steps = 3  -> valid at dt = 0,1,2 steps after lick
- Learner must:
    (A) go to its own food patch (learner_food_pos)
    (B) learn timing by observing teacher lick
    (C) press EAT within the 3-step window to get reward
- AND: learner must have "registered" the lick (i.e., detected it) for reward to count.

Observation model (information leak):
- Learner has an observe flag (0/1) each step, independent of movement/eat.
- When observe=1, learner gets accurate teacher lick / window info.
- When observe=0, learner may still detect lick/window *probabilistically* IF it has
  built up "attention" by observing frequently in recent time ("observe=1 gradually").
  This makes timing info leak more with sustained observation, as you requested.

Baselines:
1) SOCIAL DQN: has social signal + leak mechanism
2) CONTROL DQN: identical but NO social signal and NO leak (teacher info always hidden)
   -> should stay low, not learn
3) RANDOM: sanity check

What to check (alignment):
- SOCIAL: success ↑, observe_at_lick ↑, eat_in_window ↑, eat_valid ↑
- CONTROL: stays near 0 success, flat
- RANDOM: low

Run:
  python this_file.py
"""

import numpy as np
import random
from collections import deque, defaultdict

import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# Reproducibility
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# Action encoding: base_action (4) × observe (2) = 8 actions
#   base_action:
#     0 left, 1 stay, 2 right, 3 eat
#   observe:
#     0/1
# ============================================================
def decode_action8(a: int):
    base = a % 4
    obs = a // 4
    return base, obs


def encode_action8(base4: int, obs01: int) -> int:
    return int(base4) + 4 * int(obs01)


def clamp_int(x: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, x))


# ============================================================
# Environment: spatial + temporal + observation-dependent reward
# ============================================================
class SocialLickTimingEnv1D:
    """
    State design:
      learner sees its own position and its own food patch always.
      social timing info (lick + window) is obtained via observe=1 (accurate),
      or via "leak" when observe=0 but attention is high.

    Reward condition (SUCCESS):
      learner attempts EAT
      AND learner is at learner_food_pos
      AND within true window since last teacher lick (dt < window_steps)
      AND learner has registered the lick (detected it sometime at/after lick)
        (registration does NOT extend the true window)

    Why registration matters:
      It enforces "must observe teacher lick to eat", not just time by phase.
    """

    def __init__(
        self,
        grid_size=15,
        max_steps=240,            # 4 minutes if 1 step = 1 sec
        teacher_period=60,        # ~1 minute
        teacher_jitter=8,         # unpredictability
        window_steps=3,           # 3 seconds after last lick
        eat_cooldown=8,           # prevents spamming eat
        observe_cost=0.002,       # small cost to discourage always-observe
        eat_cost=0.01,            # cost per eat attempt
        attend_bonus=0.01,        # small shaping: observing exactly on lick helps learn
        allow_social_signal=True, # False for control baseline
        leak_base=0.0,            # base leak prob when not observing
        leak_gain=0.45,           # additional leak prob scaled by attention
        window_leak_gain=0.35,    # chance to "sense window" even without observe (if attention high)
        attention_inc=0.08,       # attention increases with observe=1
        attention_decay=0.96,     # attention decays when not observing
    ):
        self.grid_size = int(grid_size)
        self.max_steps = int(max_steps)
        self.teacher_period = int(teacher_period)
        self.teacher_jitter = int(teacher_jitter)
        self.window_steps = int(window_steps)

        self.eat_cooldown = int(eat_cooldown)
        self.observe_cost = float(observe_cost)
        self.eat_cost = float(eat_cost)
        self.attend_bonus = float(attend_bonus)

        self.allow_social_signal = bool(allow_social_signal)
        self.leak_base = float(leak_base)
        self.leak_gain = float(leak_gain)
        self.window_leak_gain = float(window_leak_gain)

        self.attention_inc = float(attention_inc)
        self.attention_decay = float(attention_decay)

        assert self.grid_size >= 5
        assert self.teacher_period > self.window_steps
        assert self.window_steps >= 1

        self.reset()

    def reset(self):
        # Teacher is stationary on its own patch (teacher patch not used for learner reward)
        self.teacher_pos = int(np.random.randint(0, self.grid_size))

        # Learner patch different from teacher patch
        candidates = [i for i in range(self.grid_size) if i != self.teacher_pos]
        self.learner_food_pos = int(np.random.choice(candidates))

        # Learner starts somewhere (can be far)
        self.learner_pos = int(np.random.randint(0, self.grid_size))

        self.t = 0
        self.eat_cd = 0

        # Social timing hidden state
        self.last_lick_step = None
        self.registered_lick_step = None  # when learner detected the lick
        self.attention = 0.0              # builds up with observe=1 gradually

        # Build lick schedule (one lick per period, with jitter)
        lick_steps = []
        k = 1
        while True:
            nominal = k * self.teacher_period
            if nominal >= self.max_steps:
                break
            jitter = int(np.random.randint(-self.teacher_jitter, self.teacher_jitter + 1))
            s = clamp_int(nominal + jitter, 1, self.max_steps - 1)
            lick_steps.append(s)
            k += 1
        lick_steps = sorted(list(set(lick_steps)))
        self.lick_steps = lick_steps

        # For fast lookup
        self.lick_set = set(self.lick_steps)

        return self._obs(observe_flag=1)

    def _teacher_lick_now(self, t: int) -> int:
        return 1 if t in self.lick_set else 0

    def _true_window_open(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        return 1 if (0 <= dt < self.window_steps) else 0

    def _true_window_remaining(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        rem = self.window_steps - dt
        return int(max(0, rem))

    def _phase_to_next_nominal(self, t: int) -> int:
        # Not the true signal; just an internal periodic expectation (still ambiguous due to jitter).
        mod = t % self.teacher_period
        return self.teacher_period - mod

    def _obs(self, observe_flag: int):
        """
        Observation:
          [ learner_pos_norm,
            learner_food_pos_norm,
            lick_signal (0/1)            # only if observe=1 OR leaked
            window_remaining_norm (0..1) # only if observe=1 OR leaked
            phase_to_next_nominal_norm,
            attention,
            eat_cd_norm
          ]
        """
        lp = float(self.learner_pos) / (self.grid_size - 1)
        lf = float(self.learner_food_pos) / (self.grid_size - 1)

        phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period)
        attn = float(self.attention)
        cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown))

        # By default: no social info
        lick_sig = 0.0
        win_rem = 0.0

        if self.allow_social_signal:
            # If observing: accurate social info
            if observe_flag == 1:
                # lick signal = 1 only at lick time
                lick_sig = float(self._teacher_lick_now(self.t))
                # window remaining is informative only after lick
                win_rem = float(self._true_window_remaining(self.t)) / float(self.window_steps)
            else:
                # Not observing: partial leak depends on attention built up by observing gradually
                # Leak of lick detection: only possible on the lick step itself
                if self._teacher_lick_now(self.t) == 1:
                    p = min(1.0, max(0.0, self.leak_base + self.leak_gain * self.attention))
                    if np.random.rand() < p:
                        lick_sig = 1.0

                # Leak of window remaining during the open window (weaker than lick detection)
                if self._true_window_open(self.t) == 1:
                    p2 = min(1.0, max(0.0, self.window_leak_gain * self.attention))
                    if np.random.rand() < p2:
                        # provide a noisy estimate of remaining window
                        true_rem = self._true_window_remaining(self.t)
                        noise = np.random.randint(-1, 2)  # -1,0,1
                        est = int(clamp_int(true_rem + noise, 0, self.window_steps))
                        win_rem = float(est) / float(self.window_steps)

        return np.array([lp, lf, lick_sig, win_rem, phase, attn, cdn], dtype=np.float32)

    def step(self, action8: int):
        self.t += 1
        base, observe = decode_action8(int(action8))

        # Update teacher lick / last lick
        teacher_lick = self._teacher_lick_now(self.t)
        if teacher_lick == 1:
            self.last_lick_step = self.t

        # Update attention (your "observe gradually" requirement)
        # observe=1 increases attention; otherwise decays
        self.attention = float(self.attention * self.attention_decay + self.attention_inc * float(observe))
        self.attention = float(np.clip(self.attention, 0.0, 1.0))

        # Decrement eat cooldown
        if self.eat_cd > 0:
            self.eat_cd -= 1

        # Move/eat
        did_eat = 0
        if base == 0:
            self.learner_pos -= 1
        elif base == 2:
            self.learner_pos += 1
        elif base == 3:
            did_eat = 1
        # base==1 -> stay

        self.learner_pos = clamp_int(self.learner_pos, 0, self.grid_size - 1)

        # Determine what learner "sensed" this step (affects registration)
        obs_now = self._obs(observe_flag=observe)
        lick_signal = int(obs_now[2] > 0.5)  # whether lick was detected via observe or leak

        # Register lick if detected at lick time
        if teacher_lick == 1 and lick_signal == 1:
            self.registered_lick_step = self.t

        # True window based on true lick time (physics), not when learner noticed
        window_open = self._true_window_open(self.t)

        # Registration valid if learner detected the lick and it's still within the true window
        reg_valid = 0
        if self.registered_lick_step is not None and self.last_lick_step is not None:
            # must have registered at/after last lick (in practice same step)
            if self.registered_lick_step >= self.last_lick_step:
                reg_valid = 1 if window_open == 1 else 0

        # Reward
        reward = 0.0
        reward -= self.observe_cost * float(observe)
        reward -= self.eat_cost * float(did_eat)

        # Shaping: observing exactly at lick time helps (only when social signal allowed)
        if self.allow_social_signal and observe == 1 and teacher_lick == 1:
            reward += self.attend_bonus

        # Eat cooldown applies to any eat attempt
        if did_eat == 1 and self.eat_cd == 0:
            self.eat_cd = self.eat_cooldown

        # SUCCESS condition: must be at patch + eat attempt + cooldown ok + within true window + registered lick
        eat_valid = (
            did_eat == 1
            and self.eat_cd == self.eat_cooldown  # indicates cooldown was set now (i.e., attempt was allowed)
            and (self.learner_pos == self.learner_food_pos)
            and (window_open == 1)
            and (reg_valid == 1)
        )

        success = 1 if eat_valid else 0
        if success:
            # Optional: reward more if attention is high (your "observe gradually -> more reward" request)
            # This does NOT make success easier; it only increases payoff when you truly used observation.
            reward += 1.0 + 0.4 * float(self.attention)  # bonus up to +0.4
        done = bool(success or (self.t >= self.max_steps))

        info = {
            "t": self.t,
            "teacher_lick": teacher_lick,
            "window_open": window_open,
            "observe": observe,
            "base_action": base,
            "did_eat": did_eat,
            "eat_valid": int(eat_valid),
            "learner_pos": self.learner_pos,
            "learner_food_pos": self.learner_food_pos,
            "eat_cd": self.eat_cd,
            "attention": self.attention,
            "lick_steps": self.lick_steps,
            "registered_lick_step": self.registered_lick_step,
            "last_lick_step": self.last_lick_step,
        }

        return self._obs(observe_flag=observe), reward, done, info


# ============================================================
# DQN (dueling + double + huber + soft target)
# ============================================================
class DuelingQNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.V = nn.Linear(hidden, 1)
        self.A = nn.Linear(hidden, action_dim)

    def forward(self, x):
        h = self.backbone(x)
        v = self.V(h)
        a = self.A(h)
        return v + (a - a.mean(dim=1, keepdim=True))


class ReplayBuffer:
    def __init__(self, capacity=200000):
        self.buf = deque(maxlen=int(capacity))

    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return np.array(s), np.array(a), np.array(r), np.array(s2), np.array(d)

    def __len__(self):
        return len(self.buf)


class DQNAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        batch_size=128,
        buffer_size=200000,
        eps_start=1.0,
        eps_end=0.05,
        eps_decay=0.9995,
        tau=0.01,
        grad_clip=10.0,
        device=None,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.batch_size = int(batch_size)
        self.tau = float(tau)
        self.grad_clip = float(grad_clip)

        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.q = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt.load_state_dict(self.q.state_dict())

        self.opt = optim.Adam(self.q.parameters(), lr=float(lr))
        self.loss_fn = nn.SmoothL1Loss()
        self.rb = ReplayBuffer(buffer_size)

    def act(self, s):
        if np.random.rand() < self.eps:
            return np.random.randint(self.action_dim)
        return self.act_greedy(s)

    def act_greedy(self, s):
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            qv = self.q(st)
        return int(torch.argmax(qv, dim=1).item())

    def push(self, s, a, r, s2, done):
        self.rb.push(s, int(a), float(r), s2, float(done))

    def update(self):
        if len(self.rb) < self.batch_size:
            return None

        s, a, r, s2, d = self.rb.sample(self.batch_size)
        s = torch.tensor(s, dtype=torch.float32, device=self.device)
        a = torch.tensor(a, dtype=torch.int64, device=self.device).unsqueeze(1)
        r = torch.tensor(r, dtype=torch.float32, device=self.device)
        s2 = torch.tensor(s2, dtype=torch.float32, device=self.device)
        d = torch.tensor(d, dtype=torch.float32, device=self.device)

        q_sa = self.q(s).gather(1, a).squeeze(1)

        with torch.no_grad():
            a_star = torch.argmax(self.q(s2), dim=1, keepdim=True)
            q_next = self.qt(s2).gather(1, a_star).squeeze(1)
            target = r + self.gamma * q_next * (1.0 - d)

        loss = self.loss_fn(q_sa, target)

        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), self.grad_clip)
        self.opt.step()

        with torch.no_grad():
            for p, pt in zip(self.q.parameters(), self.qt.parameters()):
                pt.data.mul_(1.0 - self.tau).add_(self.tau * p.data)

        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        return float(loss.item())


# ============================================================
# Random policy baseline (optional sanity)
# ============================================================
class RandomPolicy:
    def __init__(self, p_observe=0.5, p_eat=0.15):
        self.p_observe = float(p_observe)
        self.p_eat = float(p_eat)

    def act(self, s):
        obs = 1 if np.random.rand() < self.p_observe else 0
        if np.random.rand() < self.p_eat:
            base = 3
        else:
            base = np.random.randint(3)  # move/stay
        return encode_action8(base, obs)


# ============================================================
# Training with alignment metrics
# ============================================================
def train_agent(env, agent, episodes=2000, log_every=100):
    logs = []
    for ep in range(1, episodes + 1):
        s = env.reset()
        done = False

        steps = 0
        ep_reward = 0.0
        success = 0

        obs_steps = 0
        obs_at_lick = 0
        obs_in_window = 0

        eat_steps = 0
        eat_in_window = 0
        eat_valid = 0

        while not done:
            a = agent.act(s) if hasattr(agent, "act") else agent(s)
            s2, r, done, info = env.step(a)

            steps += 1
            ep_reward += r

            if info["observe"] == 1:
                obs_steps += 1
                if info["teacher_lick"] == 1:
                    obs_at_lick += 1
                if info["window_open"] == 1:
                    obs_in_window += 1

            if info["did_eat"] == 1:
                eat_steps += 1
                if info["window_open"] == 1:
                    eat_in_window += 1
                if info["eat_valid"] == 1:
                    eat_valid += 1
                    success = 1

            if isinstance(agent, DQNAgent):
                agent.push(s, a, r, s2, done)
                agent.update()

            s = s2

        logs.append({
            "ep": ep,
            "success": float(success),
            "mean_reward": float(ep_reward),
            "observe_rate": float(obs_steps / max(1, steps)),
            "obs_at_lick_rate": float(obs_at_lick / max(1, steps)),
            "obs_in_window_rate": float(obs_in_window / max(1, steps)),
            "eat_rate": float(eat_steps / max(1, steps)),
            "eat_in_window_rate": float(eat_in_window / max(1, steps)),
            "eat_valid": float(eat_valid),
            "eps": float(agent.eps) if isinstance(agent, DQNAgent) else np.nan,
        })

        if ep % log_every == 0:
            recent = logs[-log_every:]
            print(
                f"[ep {ep:4d}] "
                f"succ={np.mean([x['success'] for x in recent]):.3f}  "
                f"obs={np.mean([x['observe_rate'] for x in recent]):.3f}  "
                f"obs@lick={np.mean([x['obs_at_lick_rate'] for x in recent]):.3f}  "
                f"eatWin={np.mean([x['eat_in_window_rate'] for x in recent]):.3f}  "
                f"R={np.mean([x['mean_reward'] for x in recent]):.3f}  "
                f"eps={(agent.eps if isinstance(agent, DQNAgent) else 0):.3f}"
            )

    return logs


def moving_average(x, w=50):
    x = np.asarray(x, dtype=np.float32)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w) / w, mode="valid")


def plot_curves(logs_dict, window=50):
    import matplotlib.pyplot as plt

    def plot_key(key, title, ylabel):
        plt.figure()
        for name, logs in logs_dict.items():
            arr = [d[key] for d in logs]
            plt.plot(moving_average(arr, window), label=name)
        plt.title(f"{title} (window={window})")
        plt.xlabel("episode")
        plt.ylabel(ylabel)
        plt.legend()
        plt.show()

    plot_key("success", "Rolling success rate", "success")
    plot_key("observe_rate", "Rolling observe rate", "observe_rate")
    plot_key("obs_at_lick_rate", "Rolling observe-at-lick rate", "obs@lick")
    plot_key("eat_in_window_rate", "Rolling eat-in-window rate", "eat_in_window")


# ============================================================
# Trajectory visualizer
# ============================================================
def rollout_episode(env, policy_fn):
    s = env.reset()
    done = False
    traj = defaultdict(list)

    while not done:
        a = policy_fn(s)
        s2, r, done, info = env.step(a)

        traj["t"].append(info["t"])
        traj["learner_pos"].append(info["learner_pos"])
        traj["learner_food_pos"].append(info["learner_food_pos"])
        traj["teacher_lick"].append(info["teacher_lick"])
        traj["window_open"].append(info["window_open"])
        traj["observe"].append(info["observe"])
        traj["did_eat"].append(info["did_eat"])
        traj["eat_valid"].append(info["eat_valid"])
        traj["attention"].append(info["attention"])
        traj["reward"].append(r)

        s = s2

    return traj


def plot_trajectory(traj, grid_size, title="Trajectory"):
    import matplotlib.pyplot as plt

    t = np.array(traj["t"], dtype=int)
    lp = np.array(traj["learner_pos"], dtype=float)
    lf = np.array(traj["learner_food_pos"], dtype=float)

    lick = np.array(traj["teacher_lick"], dtype=int)
    win = np.array(traj["window_open"], dtype=int)
    obs = np.array(traj["observe"], dtype=int)
    eat = np.array(traj["did_eat"], dtype=int)
    succ = np.array(traj["eat_valid"], dtype=int)

    plt.figure()

    # shade open window
    idx = np.where(win == 1)[0]
    if len(idx) > 0:
        start = idx[0]
        for i in range(1, len(idx)):
            if idx[i] != idx[i - 1] + 1:
                plt.axvspan(t[start], t[idx[i - 1]] + 1, alpha=0.12, label="reward window")
                start = idx[i]
        plt.axvspan(t[start], t[idx[-1]] + 1, alpha=0.12, label=None)

    plt.plot(t, lp, label="learner_pos")
    plt.plot(t, lf, label="learner_food_pos")

    # lick marker
    lidx = np.where(lick == 1)[0]
    if len(lidx) > 0:
        plt.scatter(t[lidx], lp[lidx], marker="x", label="teacher lick")

    # observe marker
    oidx = np.where(obs == 1)[0]
    if len(oidx) > 0:
        plt.scatter(t[oidx], lp[oidx], marker="o", label="observe=1")

    # eat attempts
    eidx = np.where(eat == 1)[0]
    if len(eidx) > 0:
        plt.scatter(t[eidx], lp[eidx], marker="+", label="eat attempt")

    # success
    sidx = np.where(succ == 1)[0]
    if len(sidx) > 0:
        plt.scatter(t[sidx], lp[sidx], marker="*", s=160, label="SUCCESS")

    plt.ylim(-1, grid_size)
    plt.xlabel("time step")
    plt.ylabel("position")
    plt.title(title)
    plt.legend()
    plt.show()


# ============================================================
# Experiment: Social vs Control vs Random
# ============================================================
def run_experiment(seed=42, episodes=2000):
    set_seed(seed)

    # SOCIAL environment: teacher timing info can be obtained (observe + leak)
    env_social = SocialLickTimingEnv1D(
        grid_size=15,
        max_steps=240,
        teacher_period=60,
        teacher_jitter=8,     # makes timing uncertain -> observation needed
        window_steps=3,
        eat_cooldown=8,
        observe_cost=0.002,
        eat_cost=0.01,
        attend_bonus=0.01,
        allow_social_signal=True,
        leak_base=0.0,
        leak_gain=0.45,
        window_leak_gain=0.35,
        attention_inc=0.08,
        attention_decay=0.96,
    )

    # CONTROL environment: NO teacher signal and NO leak -> should stay low and not learn
    env_control = SocialLickTimingEnv1D(
        grid_size=15,
        max_steps=240,
        teacher_period=60,
        teacher_jitter=8,
        window_steps=3,
        eat_cooldown=8,
        observe_cost=0.002,
        eat_cost=0.01,
        attend_bonus=0.0,
        allow_social_signal=False,  # key: social channels always zero
        leak_base=0.0,
        leak_gain=0.0,
        window_leak_gain=0.0,
        attention_inc=0.08,
        attention_decay=0.96,
    )

    state_dim = 7
    action_dim = 8

    social_agent = DQNAgent(state_dim, action_dim, lr=3e-4)
    control_agent = DQNAgent(state_dim, action_dim, lr=3e-4)
    rand_policy = RandomPolicy(p_observe=0.5, p_eat=0.12)

    log_every = max(50, episodes // 20)

    print("\nTraining CONTROL DQN (no social signal) — should remain low/flat")
    logs_control = train_agent(env_control, control_agent, episodes=episodes, log_every=log_every)

    print("\nTraining SOCIAL DQN — should improve and align observe@lick + eat-in-window with success")
    logs_social = train_agent(env_social, social_agent, episodes=episodes, log_every=log_every)

    print("\nRunning RANDOM baseline (no learning) — sanity check, should be low")
    # random baseline is not trained; just generate logs by interacting
    class _RandWrap:
        def act(self, s): return rand_policy.act(s)
    logs_random = train_agent(env_social, _RandWrap(), episodes=episodes, log_every=log_every)

    plot_curves(
        {"control_no_social": logs_control, "social": logs_social, "random": logs_random},
        window=50
    )

    # Trajectory debug: greedy social agent
    old_eps = social_agent.eps
    social_agent.eps = 0.0
    traj = rollout_episode(env_social, social_agent.act_greedy)
    plot_trajectory(traj, env_social.grid_size, title="SOCIAL greedy trajectory (debug)")
    social_agent.eps = old_eps

    return logs_social, logs_control, logs_random


if __name__ == "__main__":
    run_experiment(seed=42, episodes=2000)


## Final

In [ ]:
"""
FAST SOCIAL LEARNING DEMO (1D)
================================

Main idea (matches your animal logic):
1) Teacher performs lick BOUTS with duration distribution (mean ~1s).
2) Each bout ends with a last lick. After the last lick, a reward window opens for 3s.
3) Learner has its own food patch. To earn reward it must:
   - OBSERVE at least once DURING the bout (>=0.5s is enough) to "register" that bout,
   - be at its food patch,
   - EAT within the 3s window after the last lick.

Key: OBSERVE costs TIME (no movement/eat that step).
So the learner should learn to observe briefly (often 0.5s) at the right time,
then move/eat quickly. Observing too long delays eating -> misses window.

We train:
- SOCIAL DQN: gets social signal only when it takes OBSERVE action.
- CONTROL: random policy (no training), should remain low.

Outputs:
- EMA learning curves (bout success rate, observe rate, eat-in-window rate)
- peri-lick alignment curves
- raster trajectories for debugging

"""

import numpy as np
import random
from collections import deque, defaultdict

import torch
import torch.nn as nn
import torch.optim as optim


# -------------------------
# Seed
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -------------------------
# Action space (5 actions)
# 0 left, 1 stay, 2 right, 3 OBSERVE, 4 EAT
# OBSERVE consumes time (no movement/eat)
# -------------------------
A_LEFT = 0
A_STAY = 1
A_RIGHT = 2
A_OBS = 3
A_EAT = 4


def clamp_int(x: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, x))


# ============================================================
# Environment
# ============================================================
class SocialLickEnv1D:
    """
    dt_s = seconds per step (default 0.5s) for speed.
    Lick bout duration is sampled in seconds (Gamma or LogNormal) with mean ~ 1s,
    then discretized into steps.

    Reward window: 3s after the LAST lick step of the bout.
      window_steps = round(3.0 / dt_s)

    Learner must observe at least once DURING the bout to "register" that bout.
    (0.5s observation is enough if it occurs while teacher_lick==1.)

    IMPORTANT:
      Social cues are only revealed on OBSERVE steps. Otherwise they are hidden (0).
      This makes observation truly necessary, and also costly in time.
    """

    def __init__(
        self,
        grid_size=15,
        dt_s=0.5,
        max_time_s=120.0,           # shorter = faster
        teacher_period_s=30.0,      # more bouts/episode = better signal
        teacher_jitter_s=6.0,

        lick_mean_s=1.0,
        lick_dist="gamma",          # "gamma" or "lognormal"
        lick_cv=0.8,                # for lognormal only

        window_s=3.0,
        eat_cooldown_s=2.0,

        # small costs to encourage efficient behavior
        observe_cost=0.002,
        eat_cost=0.005,

        # optional shaping: tiny bonus for observing during lick (helps learning)
        attend_bonus=0.005,
    ):
        self.grid_size = int(grid_size)

        self.dt_s = float(dt_s)
        self.max_steps = int(round(max_time_s / self.dt_s))

        self.teacher_period_steps = int(round(teacher_period_s / self.dt_s))
        self.teacher_jitter_steps = int(round(teacher_jitter_s / self.dt_s))

        self.lick_mean_s = float(lick_mean_s)
        self.lick_dist = str(lick_dist)
        self.lick_cv = float(lick_cv)

        self.window_steps = int(round(window_s / self.dt_s))
        self.eat_cooldown_steps = int(round(eat_cooldown_s / self.dt_s))

        self.observe_cost = float(observe_cost)
        self.eat_cost = float(eat_cost)
        self.attend_bonus = float(attend_bonus)

        assert self.teacher_period_steps > self.window_steps + 1
        assert self.max_steps > self.teacher_period_steps + 5

        self.reset()

    # ---- sample lick duration distribution (mean ~1s) ----
    def _sample_lick_duration_steps(self):
        if self.lick_dist == "gamma":
            # non-degenerate distribution with fixed mean:
            # choose shape=2, scale=mean/shape
            shape = 2.0
            scale = self.lick_mean_s / shape
            dur_s = float(np.random.gamma(shape, scale))
        elif self.lick_dist == "lognormal":
            cv = max(1e-6, self.lick_cv)
            sigma2 = np.log(1.0 + cv**2)
            sigma = np.sqrt(sigma2)
            mu = np.log(self.lick_mean_s) - 0.5 * sigma2
            dur_s = float(np.random.lognormal(mean=mu, sigma=sigma))
        else:
            raise ValueError(f"Unknown lick_dist={self.lick_dist}")

        steps = int(np.round(dur_s / self.dt_s))
        steps = max(1, steps)
        return steps, dur_s

    def reset(self):
        self.t = 0

        # learner patch + positions
        self.teacher_pos = int(np.random.randint(0, self.grid_size))
        candidates = [i for i in range(self.grid_size) if i != self.teacher_pos]
        self.learner_food_pos = int(np.random.choice(candidates))
        self.learner_pos = int(np.random.randint(0, self.grid_size))

        # cooldown
        self.eat_cd = 0

        # lick timing
        self.last_lick_step = None

        # bout identity:
        # active_bout_id: current bout if teacher licking now
        # detected_bout_id: last bout that learner OBSERVED during lick
        # rewarded_bout_ids: set of bouts already rewarded (1 reward/bout)
        self.active_bout_id = None
        self.detected_bout_id = None
        self.rewarded_bout_ids = set()

        # build teacher lick timeline for this episode
        self.bout_id_at_step = -np.ones(self.max_steps + 1, dtype=np.int32)
        self.bout_start_steps = []
        self.bout_end_steps = []
        self.bout_dur_s_list = []

        bout_id = 0
        k = 1
        while True:
            nominal = k * self.teacher_period_steps
            if nominal >= self.max_steps:
                break
            jitter = int(np.random.randint(-self.teacher_jitter_steps, self.teacher_jitter_steps + 1))
            start = clamp_int(nominal + jitter, 1, self.max_steps - 2)

            dur_steps, dur_s = self._sample_lick_duration_steps()
            end = clamp_int(start + dur_steps - 1, start, self.max_steps - 1)

            self.bout_id_at_step[start:end + 1] = bout_id
            self.bout_start_steps.append(start)
            self.bout_end_steps.append(end)
            self.bout_dur_s_list.append(dur_s)

            bout_id += 1
            k += 1

        self.n_bouts = bout_id

        # returned obs at reset has no revealed social signal (no observe yet)
        return self._get_obs(lick_sig=0.0, win_rem=0.0)

    def _teacher_lick_now(self, t: int):
        bid = int(self.bout_id_at_step[t]) if (0 <= t <= self.max_steps) else -1
        return (1, bid) if bid >= 0 else (0, -1)

    def _window_open(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        return 1 if (0 <= dt < self.window_steps) else 0

    def _window_remaining(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        rem = self.window_steps - dt
        return int(max(0, rem))

    def _phase_to_next_nominal(self, t: int) -> int:
        mod = t % self.teacher_period_steps
        return self.teacher_period_steps - mod

    def _get_obs(self, lick_sig: float, win_rem: float):
        """
        State features (dim=6):
          [learner_pos, food_pos, lick_sig, win_rem, phase_to_next, eat_cd]
        lick_sig/win_rem are ONLY non-zero when the learner chooses OBSERVE.
        """
        lp = float(self.learner_pos) / float(self.grid_size - 1)
        lf = float(self.learner_food_pos) / float(self.grid_size - 1)
        phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period_steps)
        cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown_steps))
        return np.array([lp, lf, float(lick_sig), float(win_rem), phase, cdn], dtype=np.float32)

    def step(self, action: int):
        self.t += 1

        # teacher lick state at this step
        teacher_lick, bout_id = self._teacher_lick_now(self.t)
        if teacher_lick == 1:
            self.last_lick_step = self.t
            self.active_bout_id = int(bout_id)
        else:
            self.active_bout_id = None

        window_open = self._window_open(self.t)
        win_rem_true = self._window_remaining(self.t)

        # cooldown update
        if self.eat_cd > 0:
            self.eat_cd -= 1

        # default: no social info revealed unless OBSERVE
        lick_sig = 0.0
        win_rem = 0.0

        did_observe = 0
        did_eat = 0
        moved = 0

        # action effects (OBSERVE consumes time)
        if action == A_LEFT:
            self.learner_pos -= 1
            moved = 1
        elif action == A_RIGHT:
            self.learner_pos += 1
            moved = 1
        elif action == A_OBS:
            did_observe = 1
            # reveal true lick signal + window remaining
            lick_sig = float(teacher_lick)
            win_rem = float(win_rem_true) / float(max(1, self.window_steps))

            # if observe during licking, register this bout
            if teacher_lick == 1 and bout_id >= 0:
                self.detected_bout_id = int(bout_id)
        elif action == A_EAT:
            did_eat = 1
        # A_STAY does nothing

        self.learner_pos = clamp_int(self.learner_pos, 0, self.grid_size - 1)

        # costs
        reward = 0.0
        reward -= self.observe_cost * float(did_observe)
        reward -= self.eat_cost * float(did_eat)

        # tiny shaping: observing during lick
        if did_observe == 1 and teacher_lick == 1:
            reward += self.attend_bonus

        # eat attempt allowed?
        eat_allowed = (did_eat == 1 and self.eat_cd == 0)
        if eat_allowed:
            self.eat_cd = self.eat_cooldown_steps

        # SUCCESS: must be at patch, eat within window, and have detected THIS bout,
        # and only one reward per bout.
        eat_valid = 0
        if (
            eat_allowed
            and (self.learner_pos == self.learner_food_pos)
            and (window_open == 1)
            and (self.last_lick_step is not None)
        ):
            # which bout produced the most recent lick? We approximate as the bout active at last lick step.
            last_bid = int(self.bout_id_at_step[self.last_lick_step]) if self.last_lick_step <= self.max_steps else -1

            if (last_bid >= 0) and (self.detected_bout_id == last_bid) and (last_bid not in self.rewarded_bout_ids):
                eat_valid = 1
                self.rewarded_bout_ids.add(last_bid)
                reward += 1.0

        done = bool(self.t >= self.max_steps)  # fixed horizon => stable curves

        info = {
            "t": self.t,
            "teacher_lick": int(teacher_lick),
            "bout_id": int(bout_id),
            "window_open": int(window_open),
            "win_rem_true": int(win_rem_true),
            "action": int(action),
            "observe": int(did_observe),
            "eat": int(did_eat),
            "eat_allowed": int(eat_allowed),
            "eat_valid": int(eat_valid),
            "learner_pos": int(self.learner_pos),
            "food_pos": int(self.learner_food_pos),
            "detected_bout_id": -1 if self.detected_bout_id is None else int(self.detected_bout_id),
            "rewarded_bouts": len(self.rewarded_bout_ids),
            "n_bouts": int(self.n_bouts),
            "dt_s": float(self.dt_s),
            "window_steps": int(self.window_steps),
            "lick_dur_mean_s": float(np.mean(self.bout_dur_s_list) if len(self.bout_dur_s_list) else 0.0),
        }

        obs = self._get_obs(lick_sig=lick_sig, win_rem=win_rem)
        return obs, reward, done, info


# ============================================================
# DQN (Dueling + Double + Huber + soft target)
# ============================================================
class DuelingQNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.V = nn.Linear(hidden, 1)
        self.A = nn.Linear(hidden, action_dim)

    def forward(self, x):
        h = self.backbone(x)
        v = self.V(h)
        a = self.A(h)
        return v + (a - a.mean(dim=1, keepdim=True))


class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.buf = deque(maxlen=int(capacity))

    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return np.array(s), np.array(a), np.array(r), np.array(s2), np.array(d)

    def __len__(self):
        return len(self.buf)


class DQNAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        batch_size=64,
        buffer_size=100000,
        eps_start=1.0,
        eps_end=0.05,
        eps_decay=0.9995,   # a bit faster learning
        tau=0.02,
        grad_clip=10.0,
        device=None,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.batch_size = int(batch_size)
        self.tau = float(tau)
        self.grad_clip = float(grad_clip)

        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.q = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt.load_state_dict(self.q.state_dict())

        self.opt = optim.Adam(self.q.parameters(), lr=float(lr))
        self.loss_fn = nn.SmoothL1Loss()
        self.rb = ReplayBuffer(buffer_size)

    def act(self, s):
        if np.random.rand() < self.eps:
            return np.random.randint(self.action_dim)
        return self.act_greedy(s)

    def act_greedy(self, s):
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            qv = self.q(st)
        return int(torch.argmax(qv, dim=1).item())

    def push(self, s, a, r, s2, done):
        self.rb.push(s, int(a), float(r), s2, float(done))

    def update(self, updates_per_step=1):
        if len(self.rb) < self.batch_size:
            return None

        last_loss = None
        for _ in range(int(updates_per_step)):
            if len(self.rb) < self.batch_size:
                break

            s, a, r, s2, d = self.rb.sample(self.batch_size)
            s = torch.tensor(s, dtype=torch.float32, device=self.device)
            a = torch.tensor(a, dtype=torch.int64, device=self.device).unsqueeze(1)
            r = torch.tensor(r, dtype=torch.float32, device=self.device)
            s2 = torch.tensor(s2, dtype=torch.float32, device=self.device)
            d = torch.tensor(d, dtype=torch.float32, device=self.device)

            q_sa = self.q(s).gather(1, a).squeeze(1)

            with torch.no_grad():
                a_star = torch.argmax(self.q(s2), dim=1, keepdim=True)  # Double DQN
                q_next = self.qt(s2).gather(1, a_star).squeeze(1)
                target = r + self.gamma * q_next * (1.0 - d)

            loss = self.loss_fn(q_sa, target)

            self.opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.q.parameters(), self.grad_clip)
            self.opt.step()

            # soft target update
            with torch.no_grad():
                for p, pt in zip(self.q.parameters(), self.qt.parameters()):
                    pt.data.mul_(1.0 - self.tau).add_(self.tau * p.data)

            last_loss = float(loss.item())

        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        return last_loss


# ============================================================
# Random control policy (no training)
# ============================================================
class RandomPolicy:
    """
    Random control: does not learn. Low success expected because:
      - must observe DURING lick AND reach patch AND eat in window.
    """
    def __init__(self, p_obs=0.08, p_eat=0.08):
        self.p_obs = float(p_obs)
        self.p_eat = float(p_eat)

    def act(self, s):
        if np.random.rand() < self.p_obs:
            return A_OBS
        if np.random.rand() < self.p_eat:
            return A_EAT
        return np.random.choice([A_LEFT, A_STAY, A_RIGHT])


# ============================================================
# Metrics + plots
# ============================================================
def ema(x, alpha=0.02):
    x = np.asarray(x, dtype=np.float32)
    y = np.zeros_like(x)
    m = 0.0
    for i in range(len(x)):
        m = (1 - alpha) * m + alpha * x[i]
        y[i] = m
    return y


def observe_run_stats(obs_arr):
    """
    Not "bout metrics" — just consecutive OBSERVE run lengths.
    This captures your idea: optimal strategy tends to reduce long runs.
    """
    obs = np.asarray(obs_arr, dtype=np.int32)
    runs = []
    cur = 0
    for v in obs:
        if v == 1:
            cur += 1
        else:
            if cur > 0:
                runs.append(cur)
            cur = 0
    if cur > 0:
        runs.append(cur)
    if len(runs) == 0:
        return 0.0, 0.0
    return float(np.mean(runs)), float(np.median(runs))


def train_social(env, agent, episodes=1500, log_every=100, keep_traces=30, updates_per_step=1):
    logs = []
    traces = deque(maxlen=int(keep_traces))

    for ep in range(1, episodes + 1):
        s = env.reset()
        done = False

        ep_reward = 0.0
        steps = 0

        # successes counted as rewarded bouts in this episode
        succ_bouts = 0
        n_bouts = max(1, env.n_bouts)

        # alignment behavior
        obs_steps = 0
        obs_at_lick = 0
        eat_steps = 0
        eat_in_window = 0

        tr = defaultdict(list)

        while not done:
            a = agent.act(s)
            s2, r, done, info = env.step(a)

            ep_reward += r
            steps += 1

            if info["observe"] == 1:
                obs_steps += 1
                if info["teacher_lick"] == 1:
                    obs_at_lick += 1

            if info["eat"] == 1:
                eat_steps += 1
                if info["window_open"] == 1:
                    eat_in_window += 1

            if info["eat_valid"] == 1:
                succ_bouts += 1

            # trace
            tr["t"].append(info["t"])
            tr["learner_pos"].append(info["learner_pos"])
            tr["food_pos"].append(info["food_pos"])
            tr["teacher_lick"].append(info["teacher_lick"])
            tr["window_open"].append(info["window_open"])
            tr["observe"].append(info["observe"])
            tr["eat"].append(info["eat"])
            tr["eat_valid"].append(info["eat_valid"])
            tr["reward"].append(r)

            agent.push(s, a, r, s2, done)
            agent.update(updates_per_step=updates_per_step)

            s = s2

        traces.append(tr)

        obs_mean_run, obs_med_run = observe_run_stats(tr["observe"])

        logs.append({
            "ep": ep,
            "bout_success_rate": float(succ_bouts / n_bouts),
            "mean_reward": float(ep_reward),

            "observe_rate": float(obs_steps / max(1, steps)),
            "obs_at_lick_rate": float(obs_at_lick / max(1, steps)),
            "eat_rate": float(eat_steps / max(1, steps)),
            "eat_in_window_rate": float(eat_in_window / max(1, steps)),

            "obs_run_mean_steps": obs_mean_run,
            "obs_run_median_steps": obs_med_run,

            "eps": float(agent.eps),
            "lick_dur_mean_s": float(np.mean(env.bout_dur_s_list) if len(env.bout_dur_s_list) else 0.0),
            "dt_s": float(env.dt_s),
        })

        if ep % log_every == 0:
            recent = logs[-log_every:]
            print(
                f"[SOC ep {ep:4d}] "
                f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f}  "
                f"obs={np.mean([x['observe_rate'] for x in recent]):.3f}  "
                f"obs@lick={np.mean([x['obs_at_lick_rate'] for x in recent]):.3f}  "
                f"eatWin={np.mean([x['eat_in_window_rate'] for x in recent]):.3f}  "
                f"obsRunMean={np.mean([x['obs_run_mean_steps'] for x in recent]):.2f}steps  "
                f"R={np.mean([x['mean_reward'] for x in recent]):.3f}  "
                f"eps={agent.eps:.3f}  "
                f"lickDurMean={np.mean([x['lick_dur_mean_s'] for x in recent]):.2f}s"
            )

    return logs, list(traces)


def run_random_control(env, policy, episodes=1500, log_every=100):
    logs = []
    for ep in range(1, episodes + 1):
        s = env.reset()
        done = False

        succ_bouts = 0
        n_bouts = max(1, env.n_bouts)
        ep_reward = 0.0

        while not done:
            a = policy.act(s)
            s, r, done, info = env.step(a)
            ep_reward += r
            if info["eat_valid"] == 1:
                succ_bouts += 1

        logs.append({
            "ep": ep,
            "bout_success_rate": float(succ_bouts / n_bouts),
            "mean_reward": float(ep_reward),
        })

        if ep % log_every == 0:
            recent = logs[-log_every:]
            print(
                f"[RND ep {ep:4d}] "
                f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f}  "
                f"R={np.mean([x['mean_reward'] for x in recent]):.3f}"
            )

    return logs


def plot_learning(logs_social, logs_random=None):
    import matplotlib.pyplot as plt

    def get(logs, key):
        return np.array([x[key] for x in logs], dtype=np.float32)

    plt.figure(figsize=(12, 9))

    ax = plt.subplot(2, 2, 1)
    ax.plot(ema(get(logs_social, "bout_success_rate")), label="social")
    if logs_random is not None:
        ax.plot(ema(get(logs_random, "bout_success_rate")), label="random control")
    ax.set_title("EMA bout success rate")
    ax.set_xlabel("episode")
    ax.set_ylabel("rate")
    ax.legend()

    ax = plt.subplot(2, 2, 2)
    ax.plot(ema(get(logs_social, "observe_rate")), label="observe_rate")
    ax.plot(ema(get(logs_social, "obs_at_lick_rate")), label="observe@lick")
    ax.set_title("EMA observation alignment")
    ax.set_xlabel("episode")
    ax.set_ylabel("rate")
    ax.legend()

    ax = plt.subplot(2, 2, 3)
    ax.plot(ema(get(logs_social, "eat_in_window_rate")), label="eat_in_window")
    ax.plot(ema(get(logs_social, "eat_rate")), label="eat_rate")
    ax.set_title("EMA eating alignment")
    ax.set_xlabel("episode")
    ax.set_ylabel("rate")
    ax.legend()

    ax = plt.subplot(2, 2, 4)
    ax.plot(ema(get(logs_social, "obs_run_mean_steps")), label="mean consecutive OBS steps")
    ax.plot(ema(get(logs_social, "lick_dur_mean_s")), label="teacher mean lick dur (s)")
    ax.set_title("Efficiency: OBS run length should shrink (0.5s often enough)")
    ax.set_xlabel("episode")
    ax.legend()

    plt.tight_layout()
    plt.show()


def peri_lick_alignment(traces, max_dt_steps=20):
    """
    Returns peri-lick probabilities for:
      - P(OBSERVE | dt)
      - P(EAT | dt)
      - P(at_patch | dt)
    dt in STEPS relative to lick step (dt=0 at lick).
    """
    dt_range = np.arange(-max_dt_steps, max_dt_steps + 1)
    obs_counts = np.zeros_like(dt_range, dtype=np.float32)
    eat_counts = np.zeros_like(dt_range, dtype=np.float32)
    patch_counts = np.zeros_like(dt_range, dtype=np.float32)
    denom = np.zeros_like(dt_range, dtype=np.float32)

    for tr in traces:
        t = np.array(tr["t"], dtype=int)
        lick = np.array(tr["teacher_lick"], dtype=int)
        obs = np.array(tr["observe"], dtype=int)
        eat = np.array(tr["eat"], dtype=int)
        lp = np.array(tr["learner_pos"], dtype=int)
        fp = np.array(tr["food_pos"], dtype=int)

        lick_ts = t[lick == 1]
        for L in lick_ts:
            for i, dt in enumerate(dt_range):
                tt = L + dt
                idx = np.where(t == tt)[0]
                if len(idx) == 0:
                    continue
                j = idx[0]
                denom[i] += 1.0
                obs_counts[i] += float(obs[j] == 1)
                eat_counts[i] += float(eat[j] == 1)
                patch_counts[i] += float(lp[j] == fp[j])

    denom = np.maximum(denom, 1.0)
    return dt_range, obs_counts / denom, eat_counts / denom, patch_counts / denom


def plot_peri_lick(traces, dt_s, max_dt_steps=20):
    import matplotlib.pyplot as plt
    dt, p_obs, p_eat, p_patch = peri_lick_alignment(traces, max_dt_steps=max_dt_steps)
    dt_sec = dt * float(dt_s)

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(dt_sec, p_obs)
    plt.axvline(0, linestyle="--")
    plt.title("P(OBSERVE) aligned to lick")
    plt.xlabel("dt (s)")
    plt.ylabel("prob")

    plt.subplot(1, 3, 2)
    plt.plot(dt_sec, p_eat)
    plt.axvline(0, linestyle="--")
    plt.title("P(EAT) aligned to lick")
    plt.xlabel("dt (s)")
    plt.ylabel("prob")

    plt.subplot(1, 3, 3)
    plt.plot(dt_sec, p_patch)
    plt.axvline(0, linestyle="--")
    plt.title("P(at patch) aligned to lick")
    plt.xlabel("dt (s)")
    plt.ylabel("prob")

    plt.tight_layout()
    plt.show()


def plot_raster(tr, dt_s, title="Episode raster"):
    import matplotlib.pyplot as plt
    t = np.array(tr["t"], dtype=int)
    tt = t * float(dt_s)
    lp = np.array(tr["learner_pos"], dtype=float)
    fp = np.array(tr["food_pos"], dtype=float)
    lick = np.array(tr["teacher_lick"], dtype=int)
    win = np.array(tr["window_open"], dtype=int)
    obs = np.array(tr["observe"], dtype=int)
    eat = np.array(tr["eat"], dtype=int)
    succ = np.array(tr["eat_valid"], dtype=int)

    plt.figure(figsize=(11, 4))

    # shade window open
    idx = np.where(win == 1)[0]
    if len(idx) > 0:
        start = idx[0]
        for i in range(1, len(idx)):
            if idx[i] != idx[i - 1] + 1:
                plt.axvspan(tt[start], tt[idx[i - 1]] + dt_s, alpha=0.12)
                start = idx[i]
        plt.axvspan(tt[start], tt[idx[-1]] + dt_s, alpha=0.12)

    plt.plot(tt, lp, label="learner_pos")
    plt.plot(tt, fp, label="food_pos")

    lidx = np.where(lick == 1)[0]
    if len(lidx) > 0:
        plt.scatter(tt[lidx], lp[lidx], marker="x", label="teacher lick")

    oidx = np.where(obs == 1)[0]
    if len(oidx) > 0:
        plt.scatter(tt[oidx], lp[oidx], marker="o", label="OBSERVE")

    eidx = np.where(eat == 1)[0]
    if len(eidx) > 0:
        plt.scatter(tt[eidx], lp[eidx], marker="+", label="EAT")

    sidx = np.where(succ == 1)[0]
    if len(sidx) > 0:
        plt.scatter(tt[sidx], lp[sidx], marker="*", s=150, label="SUCCESS")

    plt.title(title)
    plt.xlabel("time (s)")
    plt.ylabel("position")
    plt.legend()
    plt.tight_layout()
    plt.show()


# ============================================================
# Run experiment
# ============================================================
def run_experiment(seed=42, episodes=1500):
    set_seed(seed)

    env = SocialLickEnv1D(
        grid_size=15,
        dt_s=0.5,                # FAST (0.5s resolution)
        max_time_s=120.0,        # FAST horizon
        teacher_period_s=30.0,   # more bouts per episode => more learning signal
        teacher_jitter_s=6.0,

        lick_mean_s=1.0,
        lick_dist="gamma",       # distributed durations, mean ~1s
        # lick_dist="lognormal", lick_cv=0.8,

        window_s=3.0,            # your 3s window
        eat_cooldown_s=2.0,

        observe_cost=0.002,
        eat_cost=0.005,
        attend_bonus=0.005,
    )

    agent = DQNAgent(state_dim=6, action_dim=5, lr=3e-4, batch_size=64)

    logs_social, traces = train_social(
        env, agent, episodes=episodes,
        log_every=max(100, episodes // 15),
        keep_traces=30,
        updates_per_step=1,      # set to 2 if you want faster learning (more compute)
    )

    rnd = RandomPolicy(p_obs=0.08, p_eat=0.08)
    logs_random = run_random_control(
        env, rnd, episodes=episodes,
        log_every=max(100, episodes // 15),
    )

    plot_learning(logs_social, logs_random)
    plot_peri_lick(traces, dt_s=env.dt_s, max_dt_steps=20)

    # show a few recent episodes to debug "observe briefly then eat quickly"
    for i in range(min(3, len(traces))):
        plot_raster(traces[-(i + 1)], dt_s=env.dt_s, title=f"Recent episode raster #{i+1}")

    return logs_social, logs_random


if __name__ == "__main__":
    run_experiment(seed=42, episodes=1500)


## Active reward

In [ ]:
# """
# FAST SOCIAL LEARNING DEMO (1D) + FULL BEHAVIOR BREAKDOWN + DA SIM
# =================================================================

# Main task (your animal logic):
# 1) Teacher produces LICK BOUTS with duration distribution (mean ~1s).
# 2) Each bout ends with LAST LICK -> opens a 3s reward window (window_steps).
# 3) Learner has its OWN food patch. Reward requires ALL:
#    - OBSERVE at least once DURING the bout (>= 0.5s is enough) to "register" that bout,
#    - be at its food patch,
#    - EAT within the 3s window after the last lick,
#    - only 1 reward per bout.

# Observation is costly (consumes a step). So the learner must learn:
# observe briefly near licking -> then move/eat quickly in window.

# Control:
# - RANDOM policy (no training). Should remain low.

# Added (your new request):
# - Eat probabilities split by:
#   * active lick bout vs window vs passive
#   * window&detected vs window&not-detected
#   * bout-level P(reward|detected) vs P(reward|not detected)
#   * EAT-run start timing: starts during lick (before lick end) vs in window (after lick end) vs passive
#   Only “window & detected” should rise strongly with social learning.

# Also includes:
# - learning curves over training episodes
# - peri-lick alignment curves
# - raster trajectory debug plot
# - simulated DA/photometry under several hypotheses (optional)

# Deps: numpy, torch, matplotlib
# """

# import numpy as np
# import random
# from collections import deque, defaultdict

# import torch
# import torch.nn as nn
# import torch.optim as optim


# # -------------------------
# # Seed
# # -------------------------
# def set_seed(seed: int):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False


# def clamp_int(x: int, lo: int, hi: int) -> int:
#     return max(lo, min(hi, x))


# # -------------------------
# # Action space (5 actions)
# # 0 left, 1 stay, 2 right, 3 OBSERVE, 4 EAT
# # OBSERVE consumes time (no movement/eat)
# # -------------------------
# A_LEFT = 0
# A_STAY = 1
# A_RIGHT = 2
# A_OBS = 3
# A_EAT = 4


# # ============================================================
# # Environment
# # ============================================================
# class SocialLickEnv1D:
#     """
#     dt_s = seconds per step (default 0.5s) for speed.
#     Teacher lick bout duration is sampled (Gamma or LogNormal) with mean ~ 1s, then discretized.

#     Reward window: 3s after the LAST lick step of the bout.
#       window_steps = round(3.0 / dt_s)

#     Learner must observe >= 1 step DURING licking to "register" that bout.
#     Social cues are only revealed on OBSERVE steps.

#     Success requires BOTH:
#       - social registration of the bout (detected_bout_id == last_lick_bout_id)
#       - at patch
#       - EAT within window

#     Episode ends at fixed horizon (NOT at first reward) so windows exist.
#     """

#     def __init__(
#         self,
#         grid_size=15,
#         dt_s=0.5,

#         # --- speed/learning signal ---
#         max_time_s=80.0,            # shorter = faster
#         teacher_period_s=16.0,      # smaller = more bouts/episode
#         teacher_jitter_s=3.0,

#         # --- lick bout distribution ---
#         lick_mean_s=1.0,
#         lick_dist="gamma",          # "gamma" or "lognormal"
#         lick_cv=0.8,                # lognormal only

#         # --- reward window ---
#         window_s=3.0,
#         eat_cooldown_s=2.0,

#         # --- costs ---
#         observe_cost=0.002,
#         eat_cost=0.005,

#         # --- shaping (small) ---
#         attend_bonus=0.003,

#         # --- imperfect perception (more realistic) ---
#         p_detect_per_obs=0.85,      # 1 OBS step (0.5s) often enough
#         win_rem_noise_steps=1,
#     ):
#         self.grid_size = int(grid_size)
#         self.dt_s = float(dt_s)
#         self.max_steps = int(round(max_time_s / self.dt_s))

#         self.teacher_period_steps = int(round(teacher_period_s / self.dt_s))
#         self.teacher_jitter_steps = int(round(teacher_jitter_s / self.dt_s))

#         self.lick_mean_s = float(lick_mean_s)
#         self.lick_dist = str(lick_dist)
#         self.lick_cv = float(lick_cv)

#         self.window_steps = int(round(window_s / self.dt_s))
#         self.eat_cooldown_steps = int(round(eat_cooldown_s / self.dt_s))

#         self.observe_cost = float(observe_cost)
#         self.eat_cost = float(eat_cost)
#         self.attend_bonus = float(attend_bonus)

#         self.p_detect_per_obs = float(p_detect_per_obs)
#         self.win_rem_noise_steps = int(win_rem_noise_steps)

#         assert self.teacher_period_steps > self.window_steps + 1
#         assert self.max_steps > self.teacher_period_steps + 5

#         self.reset()

#     def _sample_lick_duration_steps(self):
#         if self.lick_dist == "gamma":
#             shape = 2.0
#             scale = self.lick_mean_s / shape
#             dur_s = float(np.random.gamma(shape, scale))
#         elif self.lick_dist == "lognormal":
#             cv = max(1e-6, self.lick_cv)
#             sigma2 = np.log(1.0 + cv**2)
#             sigma = np.sqrt(sigma2)
#             mu = np.log(self.lick_mean_s) - 0.5 * sigma2
#             dur_s = float(np.random.lognormal(mean=mu, sigma=sigma))
#         else:
#             raise ValueError(f"Unknown lick_dist={self.lick_dist}")

#         steps = int(np.round(dur_s / self.dt_s))
#         steps = max(1, steps)
#         return steps, dur_s

#     def reset(self):
#         self.t = 0

#         # positions
#         self.teacher_pos = int(np.random.randint(0, self.grid_size))
#         candidates = [i for i in range(self.grid_size) if i != self.teacher_pos]
#         self.learner_food_pos = int(np.random.choice(candidates))
#         self.learner_pos = int(np.random.randint(0, self.grid_size))

#         # cooldown
#         self.eat_cd = 0

#         # lick timing
#         self.last_lick_step = None

#         # social registration
#         self.detected_bout_id = None
#         self.rewarded_bout_ids = set()

#         # teacher lick timeline
#         self.bout_id_at_step = -np.ones(self.max_steps + 1, dtype=np.int32)
#         self.bout_start_steps = []
#         self.bout_end_steps = []
#         self.bout_dur_s_list = []

#         bout_id = 0
#         k = 1
#         while True:
#             nominal = k * self.teacher_period_steps
#             if nominal >= self.max_steps:
#                 break

#             jitter = int(np.random.randint(-self.teacher_jitter_steps, self.teacher_jitter_steps + 1))
#             start = clamp_int(nominal + jitter, 1, self.max_steps - 2)

#             dur_steps, dur_s = self._sample_lick_duration_steps()
#             end = clamp_int(start + dur_steps - 1, start, self.max_steps - 1)

#             self.bout_id_at_step[start:end + 1] = bout_id
#             self.bout_start_steps.append(start)
#             self.bout_end_steps.append(end)
#             self.bout_dur_s_list.append(dur_s)

#             bout_id += 1
#             k += 1

#         self.n_bouts = int(bout_id)

#         return self._get_obs(lick_sig=0.0, win_rem=0.0)

#     def _teacher_lick_now(self, t: int):
#         bid = int(self.bout_id_at_step[t]) if (0 <= t <= self.max_steps) else -1
#         return (1, bid) if bid >= 0 else (0, -1)

#     def _window_open(self, t: int) -> int:
#         if self.last_lick_step is None:
#             return 0
#         dt = t - self.last_lick_step
#         return 1 if (0 <= dt < self.window_steps) else 0

#     def _window_remaining(self, t: int) -> int:
#         if self.last_lick_step is None:
#             return 0
#         dt = t - self.last_lick_step
#         rem = self.window_steps - dt
#         return int(max(0, rem))

#     def _phase_to_next_nominal(self, t: int) -> int:
#         mod = t % self.teacher_period_steps
#         return self.teacher_period_steps - mod

#     def _get_obs(self, lick_sig: float, win_rem: float):
#         """
#         State features (dim=6):
#           [learner_pos, food_pos, lick_sig, win_rem, phase_to_next, eat_cd]
#         lick_sig/win_rem are ONLY non-zero when action==OBSERVE.
#         """
#         lp = float(self.learner_pos) / float(self.grid_size - 1)
#         lf = float(self.learner_food_pos) / float(self.grid_size - 1)
#         phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period_steps)
#         cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown_steps))
#         return np.array([lp, lf, float(lick_sig), float(win_rem), phase, cdn], dtype=np.float32)

#     def step(self, action: int):
#         self.t += 1

#         teacher_lick, bout_id = self._teacher_lick_now(self.t)
#         if teacher_lick == 1:
#             self.last_lick_step = self.t

#         window_open = self._window_open(self.t)
#         win_rem_true = self._window_remaining(self.t)

#         # identify which bout produced the most recent lick
#         if self.last_lick_step is None:
#             last_lick_bout_id = -1
#             dt_from_last_lick = 10**9
#         else:
#             last_lick_bout_id = int(self.bout_id_at_step[self.last_lick_step])
#             dt_from_last_lick = int(self.t - self.last_lick_step)

#         if self.eat_cd > 0:
#             self.eat_cd -= 1

#         lick_sig = 0.0
#         win_rem = 0.0

#         did_observe = 0
#         did_eat = 0
#         seen_lick = 0

#         if action == A_LEFT:
#             self.learner_pos -= 1
#         elif action == A_RIGHT:
#             self.learner_pos += 1
#         elif action == A_OBS:
#             did_observe = 1

#             if teacher_lick == 1 and bout_id >= 0 and (np.random.rand() < self.p_detect_per_obs):
#                 seen_lick = 1
#                 self.detected_bout_id = int(bout_id)

#             lick_sig = float(seen_lick)

#             if win_rem_true > 0:
#                 noise = int(np.random.randint(-self.win_rem_noise_steps, self.win_rem_noise_steps + 1))
#                 est = clamp_int(win_rem_true + noise, 0, self.window_steps)
#                 win_rem = float(est) / float(max(1, self.window_steps))
#             else:
#                 win_rem = 0.0

#         elif action == A_EAT:
#             did_eat = 1

#         self.learner_pos = clamp_int(self.learner_pos, 0, self.grid_size - 1)

#         reward = 0.0
#         reward -= self.observe_cost * float(did_observe)
#         reward -= self.eat_cost * float(did_eat)

#         # small bonus: “trying to attend” during lick even if not detected
#         if did_observe == 1 and teacher_lick == 1 and bout_id >= 0 and seen_lick == 0:
#             reward += self.attend_bonus

#         eat_allowed = (did_eat == 1 and self.eat_cd == 0)
#         if eat_allowed:
#             self.eat_cd = self.eat_cooldown_steps

#         eat_valid = 0
#         if (
#             eat_allowed
#             and (self.learner_pos == self.learner_food_pos)
#             and (window_open == 1)
#             and (last_lick_bout_id >= 0)
#         ):
#             det = -1 if self.detected_bout_id is None else int(self.detected_bout_id)
#             if (det == last_lick_bout_id) and (last_lick_bout_id not in self.rewarded_bout_ids):
#                 eat_valid = 1
#                 self.rewarded_bout_ids.add(last_lick_bout_id)
#                 reward += 1.0

#         done = bool(self.t >= self.max_steps)

#         info = {
#             "t": self.t,
#             "teacher_lick": int(teacher_lick),
#             "bout_id": int(bout_id),

#             "window_open": int(window_open),
#             "win_rem_true": int(win_rem_true),

#             "last_lick_bout_id": int(last_lick_bout_id),
#             "dt_from_last_lick": int(dt_from_last_lick),

#             "action": int(action),
#             "observe": int(did_observe),
#             "seen_lick": int(seen_lick),
#             "eat": int(did_eat),
#             "eat_allowed": int(eat_allowed),
#             "eat_valid": int(eat_valid),

#             "learner_pos": int(self.learner_pos),
#             "food_pos": int(self.learner_food_pos),

#             "detected_bout_id": -1 if self.detected_bout_id is None else int(self.detected_bout_id),
#             "rewarded_bouts": len(self.rewarded_bout_ids),
#             "n_bouts": int(self.n_bouts),

#             "dt_s": float(self.dt_s),
#             "window_steps": int(self.window_steps),
#             "lick_dur_mean_s": float(np.mean(self.bout_dur_s_list) if len(self.bout_dur_s_list) else 0.0),
#         }

#         obs = self._get_obs(lick_sig=lick_sig, win_rem=win_rem)
#         return obs, reward, done, info


# # ============================================================
# # DQN (Dueling + Double + Huber + soft target)
# # ============================================================
# class DuelingQNet(nn.Module):
#     def __init__(self, state_dim, action_dim, hidden=128):
#         super().__init__()
#         self.backbone = nn.Sequential(
#             nn.Linear(state_dim, hidden),
#             nn.ReLU(),
#             nn.Linear(hidden, hidden),
#             nn.ReLU(),
#         )
#         self.V = nn.Linear(hidden, 1)
#         self.A = nn.Linear(hidden, action_dim)

#     def forward(self, x):
#         h = self.backbone(x)
#         v = self.V(h)
#         a = self.A(h)
#         return v + (a - a.mean(dim=1, keepdim=True))


# class ReplayBuffer:
#     def __init__(self, capacity=100000):
#         self.buf = deque(maxlen=int(capacity))

#     def push(self, s, a, r, s2, done):
#         self.buf.append((s, a, r, s2, done))

#     def sample(self, batch_size):
#         batch = random.sample(self.buf, batch_size)
#         s, a, r, s2, d = zip(*batch)
#         return np.array(s), np.array(a), np.array(r), np.array(s2), np.array(d)

#     def __len__(self):
#         return len(self.buf)


# class DQNAgent:
#     def __init__(
#         self,
#         state_dim,
#         action_dim,
#         lr=3e-4,
#         gamma=0.99,
#         batch_size=64,
#         buffer_size=150000,
#         eps_start=1.0,
#         eps_end=0.05,
#         eps_decay=0.9995,
#         tau=0.02,
#         grad_clip=10.0,
#         device=None,
#     ):
#         self.state_dim = int(state_dim)
#         self.action_dim = int(action_dim)
#         self.gamma = float(gamma)
#         self.batch_size = int(batch_size)
#         self.tau = float(tau)
#         self.grad_clip = float(grad_clip)

#         self.eps = float(eps_start)
#         self.eps_end = float(eps_end)
#         self.eps_decay = float(eps_decay)

#         self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

#         self.q = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
#         self.qt = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
#         self.qt.load_state_dict(self.q.state_dict())

#         self.opt = optim.Adam(self.q.parameters(), lr=float(lr))
#         self.loss_fn = nn.SmoothL1Loss()
#         self.rb = ReplayBuffer(buffer_size)

#     def act(self, s):
#         if np.random.rand() < self.eps:
#             return np.random.randint(self.action_dim)
#         return self.act_greedy(s)

#     def act_greedy(self, s):
#         st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
#         with torch.no_grad():
#             qv = self.q(st)
#         return int(torch.argmax(qv, dim=1).item())

#     def V_batch(self, states_np: np.ndarray) -> np.ndarray:
#         st = torch.tensor(states_np, dtype=torch.float32, device=self.device)
#         with torch.no_grad():
#             qv = self.q(st)
#             v = torch.max(qv, dim=1).values
#         return v.detach().cpu().numpy().astype(np.float32)

#     def push(self, s, a, r, s2, done):
#         self.rb.push(s, int(a), float(r), s2, float(done))

#     def update(self, updates_per_step=1):
#         if len(self.rb) < self.batch_size:
#             return None

#         last_loss = None
#         for _ in range(int(updates_per_step)):
#             if len(self.rb) < self.batch_size:
#                 break

#             s, a, r, s2, d = self.rb.sample(self.batch_size)
#             s = torch.tensor(s, dtype=torch.float32, device=self.device)
#             a = torch.tensor(a, dtype=torch.int64, device=self.device).unsqueeze(1)
#             r = torch.tensor(r, dtype=torch.float32, device=self.device)
#             s2 = torch.tensor(s2, dtype=torch.float32, device=self.device)
#             d = torch.tensor(d, dtype=torch.float32, device=self.device)

#             q_sa = self.q(s).gather(1, a).squeeze(1)

#             with torch.no_grad():
#                 a_star = torch.argmax(self.q(s2), dim=1, keepdim=True)  # Double DQN
#                 q_next = self.qt(s2).gather(1, a_star).squeeze(1)
#                 target = r + self.gamma * q_next * (1.0 - d)

#             loss = self.loss_fn(q_sa, target)

#             self.opt.zero_grad()
#             loss.backward()
#             nn.utils.clip_grad_norm_(self.q.parameters(), self.grad_clip)
#             self.opt.step()

#             with torch.no_grad():
#                 for p, pt in zip(self.q.parameters(), self.qt.parameters()):
#                     pt.data.mul_(1.0 - self.tau).add_(self.tau * p.data)

#             last_loss = float(loss.item())

#         self.eps = max(self.eps_end, self.eps * self.eps_decay)
#         return last_loss


# # ============================================================
# # Random control (no training)
# # ============================================================
# class RandomPolicy:
#     def __init__(self, p_obs=0.06, p_eat=0.06):
#         self.p_obs = float(p_obs)
#         self.p_eat = float(p_eat)

#     def act(self, s):
#         if np.random.rand() < self.p_obs:
#             return A_OBS
#         if np.random.rand() < self.p_eat:
#             return A_EAT
#         return np.random.choice([A_LEFT, A_STAY, A_RIGHT])


# # ============================================================
# # Helpers
# # ============================================================
# def ema(x, alpha=0.03):
#     x = np.asarray(x, dtype=np.float32)
#     y = np.zeros_like(x)
#     m = 0.0
#     for i in range(len(x)):
#         m = (1 - alpha) * m + alpha * x[i]
#         y[i] = m
#     return y


# def observe_run_stats(obs_arr):
#     obs = np.asarray(obs_arr, dtype=np.int32)
#     runs = []
#     cur = 0
#     for v in obs:
#         if v == 1:
#             cur += 1
#         else:
#             if cur > 0:
#                 runs.append(cur)
#             cur = 0
#     if cur > 0:
#         runs.append(cur)
#     if len(runs) == 0:
#         return 0.0, 0.0
#     return float(np.mean(runs)), float(np.median(runs))


# # ============================================================
# # Training with full behavior breakdown
# # ============================================================
# def train_social_with_traces(
#     env,
#     agent,
#     episodes=1200,
#     log_every=100,
#     updates_per_step=2,   # <-- SPEED knob: 1 is stable; 2-4 usually faster learning
#     trace_stride=4,       # store 1/4 episodes for plots & DA sim (saves memory, still enough)
# ):
#     logs = []
#     traces = []

#     for ep in range(1, episodes + 1):
#         s = env.reset()
#         done = False

#         steps = 0
#         ep_reward = 0.0

#         # ===== step-context denominators =====
#         n_active = 0     # teacher_lick==1
#         n_window = 0     # window_open==1 & teacher_lick==0
#         n_passive = 0    # else

#         # ===== eat numerators by context =====
#         eat_active = 0
#         eat_window = 0
#         eat_passive = 0

#         # ===== window split by detected/not =====
#         n_win_det = 0
#         n_win_nodet = 0
#         eat_win_det = 0
#         eat_win_nodet = 0

#         # ===== observation alignment =====
#         obs_steps = 0
#         obs_at_lick = 0
#         obs_seen = 0

#         # ===== eat success =====
#         succ_bouts = 0
#         n_bouts = max(1, env.n_bouts)

#         # ===== EAT-run start timing =====
#         prev_eat = 0
#         eat_run_start_active = 0   # starts during lick
#         eat_run_start_window = 0   # starts after lick (in window)
#         eat_run_start_passive = 0
#         eat_run_starts = 0

#         store_trace = ((ep % trace_stride) == 0)
#         tr = defaultdict(list)

#         # for bout-level conditional probabilities
#         bout_detect = defaultdict(int)   # bout_id -> detected?
#         bout_reward = defaultdict(int)   # bout_id -> rewarded?

#         while not done:
#             a = agent.act(s)
#             s2, r, done, info = env.step(a)

#             steps += 1
#             ep_reward += r

#             teacher_lick = info["teacher_lick"]
#             window_open = info["window_open"]
#             last_bid = info["last_lick_bout_id"]
#             det_bid = info["detected_bout_id"]

#             # ---- step context classification ----
#             if teacher_lick == 1:
#                 n_active += 1
#             elif window_open == 1:
#                 n_window += 1
#             else:
#                 n_passive += 1

#             # ---- observation stats ----
#             if info["observe"] == 1:
#                 obs_steps += 1
#                 if teacher_lick == 1:
#                     obs_at_lick += 1
#                 if info["seen_lick"] == 1:
#                     obs_seen += 1
#                     # seen_lick only happens during lick; bout_id is valid
#                     if info["bout_id"] >= 0:
#                         bout_detect[int(info["bout_id"])] = 1

#             # ---- EAT by context ----
#             did_eat = info["eat"]
#             if did_eat == 1:
#                 if teacher_lick == 1:
#                     eat_active += 1
#                 elif window_open == 1:
#                     eat_window += 1
#                 else:
#                     eat_passive += 1

#             # ---- window detected vs not-detected ----
#             if (teacher_lick == 0) and (window_open == 1) and (last_bid >= 0):
#                 if det_bid == last_bid:
#                     n_win_det += 1
#                     if did_eat == 1:
#                         eat_win_det += 1
#                 else:
#                     n_win_nodet += 1
#                     if did_eat == 1:
#                         eat_win_nodet += 1

#             # ---- reward / bout success ----
#             if info["eat_valid"] == 1:
#                 succ_bouts += 1
#                 if last_bid >= 0:
#                     bout_reward[int(last_bid)] = 1

#             # ---- EAT-run start timing (start of consecutive EAT steps) ----
#             if (did_eat == 1) and (prev_eat == 0):
#                 eat_run_starts += 1
#                 if teacher_lick == 1:
#                     eat_run_start_active += 1     # starts before lick-end (during lick)
#                 elif window_open == 1:
#                     eat_run_start_window += 1     # starts after lick-end (window)
#                 else:
#                     eat_run_start_passive += 1
#             prev_eat = did_eat

#             # ---- trace storage ----
#             if store_trace:
#                 tr["t"].append(info["t"])
#                 tr["learner_pos"].append(info["learner_pos"])
#                 tr["food_pos"].append(info["food_pos"])
#                 tr["teacher_lick"].append(info["teacher_lick"])
#                 tr["window_open"].append(info["window_open"])
#                 tr["observe"].append(info["observe"])
#                 tr["seen_lick"].append(info["seen_lick"])
#                 tr["eat"].append(info["eat"])
#                 tr["eat_valid"].append(info["eat_valid"])

#             agent.push(s, a, r, s2, done)
#             agent.update(updates_per_step=updates_per_step)

#             s = s2

#         if store_trace:
#             for k in list(tr.keys()):
#                 tr[k] = np.array(tr[k])
#             traces.append(tr)

#             obs_mean_run, obs_med_run = observe_run_stats(tr["observe"])
#         else:
#             obs_mean_run, obs_med_run = 0.0, 0.0

#         # ===== bout-level conditionals =====
#         # P(reward | detected) and P(reward | not detected)
#         det_list = []
#         rew_list = []
#         for bid in range(env.n_bouts):
#             det_list.append(bout_detect.get(bid, 0))
#             rew_list.append(bout_reward.get(bid, 0))

#         det_list = np.array(det_list, dtype=np.int32)
#         rew_list = np.array(rew_list, dtype=np.int32)

#         eps_smooth = 1e-6
#         n_det = int(det_list.sum())
#         n_nodet = int((1 - det_list).sum())

#         p_rew_given_det = float(rew_list[det_list == 1].mean()) if n_det > 0 else 0.0
#         p_rew_given_nodet = float(rew_list[det_list == 0].mean()) if n_nodet > 0 else 0.0

#         logs.append({
#             "ep": ep,
#             "bout_success_rate": float(succ_bouts / n_bouts),
#             "mean_reward": float(ep_reward),

#             # observation
#             "observe_rate": float(obs_steps / max(1, steps)),
#             "obs_at_lick_rate": float(obs_at_lick / max(1, steps)),
#             "obs_seen_lick_rate": float(obs_seen / max(1, steps)),

#             # EAT probabilities by context (per-step conditional)
#             "p_eat_active": float(eat_active / max(1, n_active)),
#             "p_eat_window": float(eat_window / max(1, n_window)),
#             "p_eat_passive": float(eat_passive / max(1, n_passive)),

#             # window split (this is your “active social learning” signature)
#             "p_eat_win_detected": float(eat_win_det / max(1, n_win_det)),
#             "p_eat_win_notdet": float(eat_win_nodet / max(1, n_win_nodet)),

#             # bout-level conditional rewards
#             "p_rew_given_det": float(p_rew_given_det),
#             "p_rew_given_nodet": float(p_rew_given_nodet),

#             # EAT-run start timing
#             "frac_eat_run_start_active": float(eat_run_start_active / max(1, eat_run_starts)),
#             "frac_eat_run_start_window": float(eat_run_start_window / max(1, eat_run_starts)),
#             "frac_eat_run_start_passive": float(eat_run_start_passive / max(1, eat_run_starts)),

#             "obs_run_mean_steps": float(obs_mean_run),
#             "obs_run_median_steps": float(obs_med_run),

#             "eps": float(agent.eps),
#             "lick_dur_mean_s": float(np.mean(env.bout_dur_s_list) if len(env.bout_dur_s_list) else 0.0),
#             "dt_s": float(env.dt_s),
#         })

#         if ep % log_every == 0:
#             recent = logs[-log_every:]
#             print(
#                 f"[SOC ep {ep:4d}] "
#                 f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f}  "
#                 f"eatWin(det)={np.mean([x['p_eat_win_detected'] for x in recent]):.3f}  "
#                 f"eatWin(!det)={np.mean([x['p_eat_win_notdet'] for x in recent]):.3f}  "
#                 f"P(rew|det)={np.mean([x['p_rew_given_det'] for x in recent]):.3f}  "
#                 f"P(rew|!det)={np.mean([x['p_rew_given_nodet'] for x in recent]):.3f}  "
#                 f"obs={np.mean([x['observe_rate'] for x in recent]):.3f}  "
#                 f"eps={agent.eps:.3f}"
#             )

#     return logs, traces


# def run_random_control(env, policy, episodes=1200, log_every=100):
#     logs = []
#     for ep in range(1, episodes + 1):
#         s = env.reset()
#         done = False
#         succ_bouts = 0
#         n_bouts = max(1, env.n_bouts)
#         ep_reward = 0.0

#         while not done:
#             a = policy.act(s)
#             s, r, done, info = env.step(a)
#             ep_reward += r
#             if info["eat_valid"] == 1:
#                 succ_bouts += 1

#         logs.append({
#             "ep": ep,
#             "bout_success_rate": float(succ_bouts / n_bouts),
#             "mean_reward": float(ep_reward),
#         })

#         if ep % log_every == 0:
#             recent = logs[-log_every:]
#             print(
#                 f"[RND ep {ep:4d}] "
#                 f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f}  "
#                 f"R={np.mean([x['mean_reward'] for x in recent]):.3f}"
#             )

#     return logs


# # ============================================================
# # Visualization
# # ============================================================
# def plot_learning(logs_social, logs_random=None, ema_alpha=0.03):
#     import matplotlib.pyplot as plt

#     def get(logs, key):
#         return np.array([x[key] for x in logs], dtype=np.float32)

#     plt.figure(figsize=(13, 10))

#     ax = plt.subplot(2, 2, 1)
#     ax.plot(ema(get(logs_social, "bout_success_rate"), alpha=ema_alpha), label="social")
#     if logs_random is not None:
#         ax.plot(ema(get(logs_random, "bout_success_rate"), alpha=ema_alpha), label="random control")
#     ax.set_title("EMA bout success rate (rewarded bouts / total bouts)")
#     ax.set_xlabel("episode")
#     ax.set_ylabel("rate")
#     ax.legend()

#     ax = plt.subplot(2, 2, 2)
#     ax.plot(ema(get(logs_social, "observe_rate"), alpha=ema_alpha), label="observe_rate")
#     ax.plot(ema(get(logs_social, "obs_at_lick_rate"), alpha=ema_alpha), label="obs@lick")
#     ax.plot(ema(get(logs_social, "obs_seen_lick_rate"), alpha=ema_alpha), label="obs detects lick")
#     ax.set_title("EMA observation alignment")
#     ax.set_xlabel("episode")
#     ax.set_ylabel("rate")
#     ax.legend()

#     ax = plt.subplot(2, 2, 3)
#     ax.plot(ema(get(logs_social, "p_eat_active"), alpha=ema_alpha), label="P(EAT | active lick)")
#     ax.plot(ema(get(logs_social, "p_eat_window"), alpha=ema_alpha), label="P(EAT | window)")
#     ax.plot(ema(get(logs_social, "p_eat_passive"), alpha=ema_alpha), label="P(EAT | passive)")
#     ax.set_title("Eat probability by state (per-step conditional)")
#     ax.set_xlabel("episode")
#     ax.set_ylabel("prob")
#     ax.legend()

#     ax = plt.subplot(2, 2, 4)
#     ax.plot(ema(get(logs_social, "p_eat_win_detected"), alpha=ema_alpha), label="P(EAT | window & detected)")
#     ax.plot(ema(get(logs_social, "p_eat_win_notdet"), alpha=ema_alpha), label="P(EAT | window & NOT detected)")
#     ax.plot(ema(get(logs_social, "p_rew_given_det"), alpha=ema_alpha), label="P(reward | detected bout)")
#     ax.plot(ema(get(logs_social, "p_rew_given_nodet"), alpha=ema_alpha), label="P(reward | NOT detected bout)")
#     ax.set_title("The social-learning signature (should separate)")
#     ax.set_xlabel("episode")
#     ax.set_ylabel("prob")
#     ax.legend()

#     plt.tight_layout()
#     plt.show()


# def plot_eat_run_starts(logs_social, ema_alpha=0.03):
#     import matplotlib.pyplot as plt
#     def get(key):
#         return np.array([x[key] for x in logs_social], dtype=np.float32)

#     plt.figure(figsize=(10, 4))
#     plt.plot(ema(get("frac_eat_run_start_active"), alpha=ema_alpha), label="EAT-run starts during lick (before lick end)")
#     plt.plot(ema(get("frac_eat_run_start_window"), alpha=ema_alpha), label="EAT-run starts in window (after lick end)")
#     plt.plot(ema(get("frac_eat_run_start_passive"), alpha=ema_alpha), label="EAT-run starts passive")
#     plt.title("When does the learner *start* an eating attempt sequence?")
#     plt.xlabel("episode")
#     plt.ylabel("fraction")
#     plt.legend()
#     plt.tight_layout()
#     plt.show()


# def peri_lick_alignment(traces, max_dt_steps=20):
#     dt_range = np.arange(-max_dt_steps, max_dt_steps + 1)
#     obs_counts = np.zeros_like(dt_range, dtype=np.float32)
#     eat_counts = np.zeros_like(dt_range, dtype=np.float32)
#     patch_counts = np.zeros_like(dt_range, dtype=np.float32)
#     denom = np.zeros_like(dt_range, dtype=np.float32)

#     for tr in traces:
#         t = np.array(tr["t"], dtype=int)
#         lick = np.array(tr["teacher_lick"], dtype=int)
#         obs = np.array(tr["observe"], dtype=int)
#         eat = np.array(tr["eat"], dtype=int)
#         lp = np.array(tr["learner_pos"], dtype=int)
#         fp = np.array(tr["food_pos"], dtype=int)

#         lick_ts = t[lick == 1]
#         for L in lick_ts:
#             for i, dt in enumerate(dt_range):
#                 tt = L + dt
#                 idx = np.where(t == tt)[0]
#                 if len(idx) == 0:
#                     continue
#                 j = idx[0]
#                 denom[i] += 1.0
#                 obs_counts[i] += float(obs[j] == 1)
#                 eat_counts[i] += float(eat[j] == 1)
#                 patch_counts[i] += float(lp[j] == fp[j])

#     denom = np.maximum(denom, 1.0)
#     return dt_range, obs_counts / denom, eat_counts / denom, patch_counts / denom


# def plot_peri_lick(traces, dt_s, max_dt_steps=20):
#     import matplotlib.pyplot as plt
#     dt, p_obs, p_eat, p_patch = peri_lick_alignment(traces, max_dt_steps=max_dt_steps)
#     dt_sec = dt * float(dt_s)

#     plt.figure(figsize=(12, 4))
#     plt.subplot(1, 3, 1)
#     plt.plot(dt_sec, p_obs)
#     plt.axvline(0, linestyle="--")
#     plt.title("P(OBSERVE) aligned to lick")
#     plt.xlabel("dt (s)")
#     plt.ylabel("prob")

#     plt.subplot(1, 3, 2)
#     plt.plot(dt_sec, p_eat)
#     plt.axvline(0, linestyle="--")
#     plt.title("P(EAT) aligned to lick")
#     plt.xlabel("dt (s)")
#     plt.ylabel("prob")

#     plt.subplot(1, 3, 3)
#     plt.plot(dt_sec, p_patch)
#     plt.axvline(0, linestyle="--")
#     plt.title("P(at patch) aligned to lick")
#     plt.xlabel("dt (s)")
#     plt.ylabel("prob")

#     plt.tight_layout()
#     plt.show()


# def plot_raster(tr, dt_s, title="Episode raster"):
#     import matplotlib.pyplot as plt
#     t = np.array(tr["t"], dtype=int)
#     tt = t * float(dt_s)

#     lp = np.array(tr["learner_pos"], dtype=float)
#     fp = np.array(tr["food_pos"], dtype=float)
#     lick = np.array(tr["teacher_lick"], dtype=int)
#     win = np.array(tr["window_open"], dtype=int)
#     obs = np.array(tr["observe"], dtype=int)
#     eat = np.array(tr["eat"], dtype=int)
#     succ = np.array(tr["eat_valid"], dtype=int)

#     plt.figure(figsize=(11, 4))

#     idx = np.where(win == 1)[0]
#     if len(idx) > 0:
#         start = idx[0]
#         for i in range(1, len(idx)):
#             if idx[i] != idx[i - 1] + 1:
#                 plt.axvspan(tt[start], tt[idx[i - 1]] + dt_s, alpha=0.12)
#                 start = idx[i]
#         plt.axvspan(tt[start], tt[idx[-1]] + dt_s, alpha=0.12)

#     plt.plot(tt, lp, label="learner_pos")
#     plt.plot(tt, fp, label="food_pos")

#     lidx = np.where(lick == 1)[0]
#     if len(lidx) > 0:
#         plt.scatter(tt[lidx], lp[lidx], marker="x", label="teacher lick")

#     oidx = np.where(obs == 1)[0]
#     if len(oidx) > 0:
#         plt.scatter(tt[oidx], lp[oidx], marker="o", label="OBSERVE")

#     eidx = np.where(eat == 1)[0]
#     if len(eidx) > 0:
#         plt.scatter(tt[eidx], lp[eidx], marker="+", label="EAT")

#     sidx = np.where(succ == 1)[0]
#     if len(sidx) > 0:
#         plt.scatter(tt[sidx], lp[sidx], marker="*", s=150, label="SUCCESS")

#     plt.title(title)
#     plt.xlabel("time (s)")
#     plt.ylabel("position")
#     plt.legend()
#     plt.tight_layout()
#     plt.show()


# # ============================================================
# # Run experiment
# # ============================================================
# def run_experiment(seed=42, episodes=1200):
#     set_seed(seed)

#     env = SocialLickEnv1D(
#         grid_size=15,
#         dt_s=0.5,

#         # SPEED / learning signal
#         max_time_s=80.0,
#         teacher_period_s=16.0,
#         teacher_jitter_s=3.0,

#         lick_mean_s=1.0,
#         lick_dist="gamma",   # distributed durations

#         window_s=3.0,
#         eat_cooldown_s=2.0,

#         observe_cost=0.002,
#         eat_cost=0.005,
#         attend_bonus=0.003,

#         p_detect_per_obs=0.85,
#         win_rem_noise_steps=1,
#     )

#     agent = DQNAgent(state_dim=6, action_dim=5, lr=3e-4, batch_size=64)

#     logs_social, traces = train_social_with_traces(
#         env, agent,
#         episodes=episodes,
#         log_every=max(100, episodes // 12),
#         updates_per_step=2,    # faster than 1
#         trace_stride=4,
#     )

#     rnd = RandomPolicy(p_obs=0.06, p_eat=0.06)
#     logs_random = run_random_control(
#         env, rnd,
#         episodes=episodes,
#         log_every=max(100, episodes // 12),
#     )

#     plot_learning(logs_social, logs_random, ema_alpha=0.03)
#     plot_eat_run_starts(logs_social, ema_alpha=0.03)

#     if len(traces) > 0:
#         plot_peri_lick(traces[-10:], dt_s=env.dt_s, max_dt_steps=20)
#         for i in range(min(3, len(traces))):
#             plot_raster(traces[-(i + 1)], dt_s=env.dt_s, title=f"Recent episode raster #{i+1}")

#     return logs_social, logs_random, traces


# if __name__ == "__main__":
#     run_experiment(seed=42, episodes=1200)


In [ ]:
"""
FAST SOCIAL LEARNING DEMO (1D) — MAPPO(PPO) + CURRICULUM FIX (LEARNABLE)
======================================================================

Fixes for "not learning":
1) Curriculum: start easy (no costs, high detect prob, shorter cooldown) -> anneal to target.
2) Light shaping: tiny positive reward for OBSERVE during lick, and for successful detect.
3) Add learnable flag to obs: revealed_det (shows 1 on the step AFTER a successful detect).
4) Keep no information leakage: cues appear only after OBSERVE and in next obs.

Deps: numpy, torch, matplotlib
"""

import numpy as np
import random
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical


# -------------------------
# Seed
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def clamp_int(x: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, x))


# -------------------------
# Actions
# -------------------------
A_LEFT = 0
A_STAY = 1
A_RIGHT = 2
A_OBS = 3
A_EAT = 4
N_ACTIONS = 5


# ============================================================
# Environment
# ============================================================
class SocialLickEnv1D_MA:
    """
    Multi-agent API:
      obs = reset() -> {"teacher": obs_T, "learner": obs_L}
      obs, rewards, done, info = step({"teacher": aT, "learner": aL})

    Teacher: scripted lick bouts.
    Learner: must OBSERVE during lick bout to register, then EAT at patch during reward window.

    Key: cues appear only after OBSERVE and in NEXT obs (no leakage).
    """

    def __init__(
        self,
        grid_size=15,
        dt_s=0.5,

        # episode / teacher bouts
        max_time_s=80.0,
        teacher_period_s=10.0,
        teacher_jitter_s=2.0,

        # lick bout distribution
        lick_mean_s=1.0,
        lick_dist="gamma",
        lick_cv=0.8,

        # reward window
        window_s=3.0,

        # --- target params (final difficulty) ---
        eat_cooldown_s_final=2.0,
        observe_cost_final=0.002,
        eat_cost_final=0.005,
        p_detect_final=0.85,

        # --- curriculum start (easy) ---
        eat_cooldown_s_easy=0.5,   # 1 step
        observe_cost_easy=0.0,
        eat_cost_easy=0.0,
        p_detect_easy=1.0,

        # observe integration requirement
        tau_obs_s=0.5,

        # reveal noise
        win_rem_noise_steps=1,

        # shaping (tiny, only to help learning)
        obs_lick_bonus=0.01,     # OBSERVE during lick
        obs_detect_bonus=0.01,   # OBSERVE and detect lick
        attend_bonus=0.0,        # optional legacy, default 0
    ):
        self.grid_size = int(grid_size)
        self.dt_s = float(dt_s)
        self.max_steps = int(round(max_time_s / self.dt_s))

        self.teacher_period_s = float(teacher_period_s)
        self.teacher_jitter_s = float(teacher_jitter_s)

        self.teacher_period_steps = int(round(self.teacher_period_s / self.dt_s))
        self.teacher_jitter_steps = int(round(self.teacher_jitter_s / self.dt_s))

        self.lick_mean_s = float(lick_mean_s)
        self.lick_dist = str(lick_dist)
        self.lick_cv = float(lick_cv)

        self.window_s = float(window_s)
        self.window_steps = int(round(self.window_s / self.dt_s))

        # curriculum endpoints
        self.eat_cooldown_s_final = float(eat_cooldown_s_final)
        self.observe_cost_final = float(observe_cost_final)
        self.eat_cost_final = float(eat_cost_final)
        self.p_detect_final = float(p_detect_final)

        self.eat_cooldown_s_easy = float(eat_cooldown_s_easy)
        self.observe_cost_easy = float(observe_cost_easy)
        self.eat_cost_easy = float(eat_cost_easy)
        self.p_detect_easy = float(p_detect_easy)

        # current (will be set by curriculum)
        self.eat_cooldown_steps = int(round(self.eat_cooldown_s_easy / self.dt_s))
        self.observe_cost = self.observe_cost_easy
        self.eat_cost = self.eat_cost_easy
        self.p_detect_per_obs = self.p_detect_easy

        self.tau_obs_s = float(tau_obs_s)
        self.req_obs_steps = int(np.ceil(self.tau_obs_s / self.dt_s))

        self.win_rem_noise_steps = int(win_rem_noise_steps)

        self.obs_lick_bonus = float(obs_lick_bonus)
        self.obs_detect_bonus = float(obs_detect_bonus)
        self.attend_bonus = float(attend_bonus)

        assert self.teacher_period_steps > self.window_steps + 1
        assert self.max_steps > self.teacher_period_steps + 5

        self.teacher_id = "teacher"
        self.learner_id = "learner"

        self.reset()

    def set_curriculum(self, frac: float):
        """
        frac in [0,1]: 0 = easiest, 1 = final target.
        """
        f = float(np.clip(frac, 0.0, 1.0))

        def lerp(a, b, t):
            return a + (b - a) * t

        self.observe_cost = float(lerp(self.observe_cost_easy, self.observe_cost_final, f))
        self.eat_cost = float(lerp(self.eat_cost_easy, self.eat_cost_final, f))
        self.p_detect_per_obs = float(lerp(self.p_detect_easy, self.p_detect_final, f))

        cd_s = float(lerp(self.eat_cooldown_s_easy, self.eat_cooldown_s_final, f))
        self.eat_cooldown_steps = int(max(1, round(cd_s / self.dt_s)))

    def _sample_lick_duration_steps(self):
        if self.lick_dist == "gamma":
            shape = 2.0
            scale = self.lick_mean_s / shape
            dur_s = float(np.random.gamma(shape, scale))
        elif self.lick_dist == "lognormal":
            cv = max(1e-6, self.lick_cv)
            sigma2 = np.log(1.0 + cv**2)
            sigma = np.sqrt(sigma2)
            mu = np.log(self.lick_mean_s) - 0.5 * sigma2
            dur_s = float(np.random.lognormal(mean=mu, sigma=sigma))
        else:
            raise ValueError(f"Unknown lick_dist={self.lick_dist}")

        steps = int(np.round(dur_s / self.dt_s))
        return max(1, steps), dur_s

    def _teacher_lick_now(self, t: int):
        bid = int(self.bout_id_at_step[t]) if (0 <= t <= self.max_steps) else -1
        return (1, bid) if bid >= 0 else (0, -1)

    def _window_open(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        return 1 if (0 <= dt < self.window_steps) else 0

    def _window_remaining(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        rem = self.window_steps - dt
        return int(max(0, rem))

    def _phase_to_next_nominal(self, t: int) -> int:
        mod = t % self.teacher_period_steps
        return self.teacher_period_steps - mod

    def _central_state(self):
        """
        Central state for critic (CTDE). 12 dims.
        """
        lp = float(self.learner_pos) / float(self.grid_size - 1)
        lf = float(self.learner_food_pos) / float(self.grid_size - 1)
        tp = float(self.teacher_pos) / float(self.grid_size - 1)

        t_norm = float(self.t) / float(max(1, self.max_steps))
        phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period_steps)

        teacher_lick = float(self._teacher_lick_cache)
        window_open = float(self._window_open_cache)
        win_rem_true = float(self._win_rem_true_cache) / float(max(1, self.window_steps))

        last_bid = float(max(-1, self._last_lick_bout_id_cache) + 1) / float(max(1, self.n_bouts + 1))
        det_bid_int = -1 if self.detected_bout_id is None else int(self.detected_bout_id)
        det_bid = float(max(-1, det_bid_int) + 1) / float(max(1, self.n_bouts + 1))

        cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown_steps))
        dtll = float(min(self._dt_from_last_lick_cache, self.window_steps * 3)) / float(max(1, self.window_steps * 3))

        return np.array(
            [lp, lf, tp, t_norm, phase, teacher_lick, window_open, win_rem_true, last_bid, det_bid, cdn, dtll],
            dtype=np.float32,
        )

    def _get_obs(self):
        """
        Learner obs (7 dims):
          [learner_pos, food_pos, revealed_lick_sig, revealed_win_rem, phase_to_next, eat_cd, revealed_det]
        Teacher obs (3 dims): [teacher_pos, phase, teacher_lick]
        """
        lp = float(self.learner_pos) / float(self.grid_size - 1)
        lf = float(self.learner_food_pos) / float(self.grid_size - 1)
        phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period_steps)
        cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown_steps))

        obs_L = np.array(
            [lp, lf, float(self.revealed_lick_sig), float(self.revealed_win_rem), phase, cdn, float(self.revealed_det)],
            dtype=np.float32,
        )

        tp = float(self.teacher_pos) / float(self.grid_size - 1)
        obs_T = np.array([tp, phase, float(self._teacher_lick_cache)], dtype=np.float32)

        return {self.teacher_id: obs_T, self.learner_id: obs_L}

    def reset(self):
        self.t = 0

        self.teacher_pos = int(np.random.randint(0, self.grid_size))
        candidates = [i for i in range(self.grid_size) if i != self.teacher_pos]
        self.learner_food_pos = int(np.random.choice(candidates))
        self.learner_pos = int(np.random.randint(0, self.grid_size))

        self.eat_cd = 0
        self.last_lick_step = None

        self.detected_bout_id = None
        self.rewarded_bout_ids = set()
        self.obs_steps_in_bout = defaultdict(int)

        # schedule bouts
        self.bout_id_at_step = -np.ones(self.max_steps + 1, dtype=np.int32)
        self.bout_start_steps = []
        self.bout_end_steps = []
        self.bout_dur_s_list = []

        bout_id = 0
        k = 1
        while True:
            nominal = k * self.teacher_period_steps
            if nominal >= self.max_steps:
                break

            jitter = int(np.random.randint(-self.teacher_jitter_steps, self.teacher_jitter_steps + 1))
            start = clamp_int(nominal + jitter, 1, self.max_steps - 2)

            dur_steps, dur_s = self._sample_lick_duration_steps()
            end = clamp_int(start + dur_steps - 1, start, self.max_steps - 1)

            self.bout_id_at_step[start : end + 1] = bout_id
            self.bout_start_steps.append(start)
            self.bout_end_steps.append(end)
            self.bout_dur_s_list.append(dur_s)

            bout_id += 1
            k += 1

        self.n_bouts = int(bout_id)

        # delayed reveals (appear in NEXT obs)
        self.revealed_lick_sig = 0.0
        self.revealed_win_rem = 0.0
        self.revealed_det = 0.0

        # caches for central state
        self._teacher_lick_cache = 0
        self._window_open_cache = 0
        self._win_rem_true_cache = 0
        self._last_lick_bout_id_cache = -1
        self._dt_from_last_lick_cache = 10**9

        return self._get_obs()

    def step(self, actions: dict):
        aL = int(actions.get(self.learner_id, A_STAY))

        self.t += 1

        teacher_lick, bout_id = self._teacher_lick_now(self.t)
        if teacher_lick == 1:
            self.last_lick_step = self.t

        window_open = self._window_open(self.t)
        win_rem_true = self._window_remaining(self.t)

        if self.last_lick_step is None:
            last_lick_bout_id = -1
            dt_from_last_lick = 10**9
        else:
            last_lick_bout_id = int(self.bout_id_at_step[self.last_lick_step])
            dt_from_last_lick = int(self.t - self.last_lick_step)

        if self.eat_cd > 0:
            self.eat_cd -= 1

        # update caches
        self._teacher_lick_cache = int(teacher_lick)
        self._window_open_cache = int(window_open)
        self._win_rem_true_cache = int(win_rem_true)
        self._last_lick_bout_id_cache = int(last_lick_bout_id)
        self._dt_from_last_lick_cache = int(dt_from_last_lick)

        # reveal for next obs (default 0 unless OBSERVE)
        next_reveal_lick_sig = 0.0
        next_reveal_win_rem = 0.0
        next_reveal_det = 0.0

        did_observe = 0
        did_eat = 0
        seen_lick = 0
        det_success = 0

        # movement / action
        if aL == A_LEFT:
            self.learner_pos -= 1
        elif aL == A_RIGHT:
            self.learner_pos += 1
        elif aL == A_OBS:
            did_observe = 1
            if teacher_lick == 1 and bout_id >= 0:
                # integrate observation time in bout
                self.obs_steps_in_bout[int(bout_id)] += 1

                # detection is probabilistic and tied to "seen_lick"
                if np.random.rand() < self.p_detect_per_obs:
                    seen_lick = 1
                    det_success = 1

                # registration requires enough observe time AND at least one successful detect
                # (this makes "detected" learnable)
                if det_success == 1 and (self.obs_steps_in_bout[int(bout_id)] >= self.req_obs_steps):
                    self.detected_bout_id = int(bout_id)

            next_reveal_lick_sig = float(seen_lick)
            next_reveal_det = float(det_success)

            # window remaining estimate (noisy)
            if win_rem_true > 0:
                noise = int(np.random.randint(-self.win_rem_noise_steps, self.win_rem_noise_steps + 1))
                est = clamp_int(win_rem_true + noise, 0, self.window_steps)
                next_reveal_win_rem = float(est) / float(max(1, self.window_steps))
            else:
                next_reveal_win_rem = 0.0

        elif aL == A_EAT:
            did_eat = 1

        self.learner_pos = clamp_int(self.learner_pos, 0, self.grid_size - 1)

        # rewards
        reward_L = 0.0
        reward_T = 0.0

        # costs (curriculum-annealed)
        reward_L -= self.observe_cost * float(did_observe)
        reward_L -= self.eat_cost * float(did_eat)

        # shaping (tiny)
        if did_observe == 1 and teacher_lick == 1 and bout_id >= 0:
            reward_L += self.obs_lick_bonus
        if did_observe == 1 and det_success == 1:
            reward_L += self.obs_detect_bonus

        # optional legacy attend bonus (default 0)
        if did_observe == 1 and teacher_lick == 1 and bout_id >= 0 and seen_lick == 0:
            reward_L += self.attend_bonus

        # eating
        eat_allowed = (did_eat == 1 and self.eat_cd == 0)
        if eat_allowed:
            self.eat_cd = self.eat_cooldown_steps

        eat_valid = 0
        if (
            eat_allowed
            and (self.learner_pos == self.learner_food_pos)
            and (window_open == 1)
            and (last_lick_bout_id >= 0)
        ):
            det = -1 if self.detected_bout_id is None else int(self.detected_bout_id)
            if (det == last_lick_bout_id) and (last_lick_bout_id not in self.rewarded_bout_ids):
                eat_valid = 1
                self.rewarded_bout_ids.add(last_lick_bout_id)
                reward_L += 1.0

        done = bool(self.t >= self.max_steps)

        # commit reveals to NEXT obs
        self.revealed_lick_sig = float(next_reveal_lick_sig)
        self.revealed_win_rem = float(next_reveal_win_rem)
        self.revealed_det = float(next_reveal_det)

        info = {
            "t": self.t,
            "teacher_lick": int(teacher_lick),
            "bout_id": int(bout_id),
            "window_open": int(window_open),
            "win_rem_true": int(win_rem_true),
            "last_lick_bout_id": int(last_lick_bout_id),
            "dt_from_last_lick": int(dt_from_last_lick),
            "action": int(aL),
            "observe": int(did_observe),
            "seen_lick": int(seen_lick),
            "det_success": int(det_success),
            "eat": int(did_eat),
            "eat_allowed": int(eat_allowed),
            "eat_valid": int(eat_valid),
            "learner_pos": int(self.learner_pos),
            "food_pos": int(self.learner_food_pos),
            "detected_bout_id": -1 if self.detected_bout_id is None else int(self.detected_bout_id),
            "rewarded_bouts": len(self.rewarded_bout_ids),
            "n_bouts": int(self.n_bouts),
            "central_state": self._central_state(),
            "dt_s": float(self.dt_s),
            "window_steps": int(self.window_steps),
        }

        obs = self._get_obs()
        rewards = {self.teacher_id: float(reward_T), self.learner_id: float(reward_L)}
        return obs, rewards, done, info


# ============================================================
# MAPPO-style PPO (CTDE): GRU Actor + Central Critic
# ============================================================
class GRUActor(nn.Module):
    def __init__(self, obs_dim: int, action_dim: int, hidden=128):
        super().__init__()
        self.obs_dim = int(obs_dim)
        self.action_dim = int(action_dim)
        self.hidden_dim = int(hidden)

        self.fc = nn.Sequential(nn.Linear(self.obs_dim, self.hidden_dim), nn.ReLU())
        self.gru = nn.GRUCell(self.hidden_dim, self.hidden_dim)
        self.head = nn.Linear(self.hidden_dim, self.action_dim)

    def forward_step(self, obs_t: torch.Tensor, h: torch.Tensor):
        x = self.fc(obs_t)
        h2 = self.gru(x, h)
        logits = self.head(h2)
        return logits, h2

    def init_hidden(self, device):
        return torch.zeros(1, self.hidden_dim, device=device)


class CentralCritic(nn.Module):
    def __init__(self, state_dim: int, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, s: torch.Tensor):
        return self.net(s).squeeze(-1)


class MAPPOTrainer:
    def __init__(
        self,
        obs_dim=7,
        action_dim=5,
        state_dim=12,
        lr=2e-4,
        gamma=0.99,
        lam=0.95,
        clip_ratio=0.2,
        entropy_coef=0.005,
        value_coef=0.5,
        max_grad_norm=0.5,
        device=None,
    ):
        self.obs_dim = int(obs_dim)
        self.action_dim = int(action_dim)
        self.state_dim = int(state_dim)

        self.gamma = float(gamma)
        self.lam = float(lam)
        self.clip_ratio = float(clip_ratio)
        self.entropy_coef = float(entropy_coef)
        self.value_coef = float(value_coef)
        self.max_grad_norm = float(max_grad_norm)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.actor = GRUActor(self.obs_dim, self.action_dim, hidden=128).to(self.device)
        self.critic = CentralCritic(self.state_dim, hidden=256).to(self.device)

        self.opt = optim.Adam(list(self.actor.parameters()) + list(self.critic.parameters()), lr=float(lr))

    @torch.no_grad()
    def act(self, obs_np: np.ndarray, h: torch.Tensor, central_state_np: np.ndarray):
        obs_t = torch.tensor(obs_np, dtype=torch.float32, device=self.device).unsqueeze(0)
        logits, h2 = self.actor.forward_step(obs_t, h)
        dist = Categorical(logits=logits)
        a = dist.sample()
        logp = dist.log_prob(a)

        s_t = torch.tensor(central_state_np, dtype=torch.float32, device=self.device).unsqueeze(0)
        v = self.critic(s_t)
        return int(a.item()), float(logp.item()), float(v.item()), h2

    def _compute_gae(self, rewards: np.ndarray, values: np.ndarray, dones: np.ndarray):
        T = len(rewards)
        adv = np.zeros(T, dtype=np.float32)
        last_gae = 0.0
        for t in reversed(range(T)):
            nonterminal = 1.0 - float(dones[t])
            v_next = values[t + 1] if (t + 1) < T else 0.0
            delta = rewards[t] + self.gamma * v_next * nonterminal - values[t]
            last_gae = delta + self.gamma * self.lam * nonterminal * last_gae
            adv[t] = last_gae
        ret = adv + values
        return adv, ret

    def update(self, batch_eps, ppo_epochs=4):
        all_adv = []
        for ep in batch_eps:
            adv, ret = self._compute_gae(ep["rew"], ep["val"], ep["done"])
            ep["adv"] = adv.astype(np.float32)
            ep["ret"] = ret.astype(np.float32)
            all_adv.append(ep["adv"])

        adv_cat = np.concatenate(all_adv, axis=0)
        adv_mean = float(adv_cat.mean())
        adv_std = float(adv_cat.std() + 1e-8)
        for ep in batch_eps:
            ep["adv"] = (ep["adv"] - adv_mean) / adv_std

        for _ in range(int(ppo_epochs)):
            random.shuffle(batch_eps)
            self.opt.zero_grad()

            for ep in batch_eps:
                obs = torch.tensor(ep["obs"], dtype=torch.float32, device=self.device)                # (T, obs_dim)
                act = torch.tensor(ep["act"], dtype=torch.int64, device=self.device).view(-1)        # (T,)
                logp_old = torch.tensor(ep["logp"], dtype=torch.float32, device=self.device).view(-1)# (T,)
                state = torch.tensor(ep["state"], dtype=torch.float32, device=self.device)           # (T, state_dim)
                adv = torch.tensor(ep["adv"], dtype=torch.float32, device=self.device).view(-1)      # (T,)
                ret = torch.tensor(ep["ret"], dtype=torch.float32, device=self.device).view(-1)      # (T,)

                T = obs.shape[0]
                assert act.shape[0] == T and logp_old.shape[0] == T and adv.shape[0] == T and ret.shape[0] == T

                h = torch.tensor(ep["h0"], dtype=torch.float32, device=self.device)  # (1,H)
                logp_new_list, ent_list = [], []

                for t in range(T):
                    logits, h = self.actor.forward_step(obs[t].unsqueeze(0), h)
                    dist = Categorical(logits=logits)
                    logp_new_list.append(dist.log_prob(act[t].unsqueeze(0)).squeeze(0))
                    ent_list.append(dist.entropy().squeeze(0))

                logp_new = torch.stack(logp_new_list, dim=0).view(-1)
                ent = torch.stack(ent_list, dim=0).view(-1)

                ratio = torch.exp(logp_new - logp_old)
                clipped = torch.clamp(ratio, 1.0 - self.clip_ratio, 1.0 + self.clip_ratio)
                pi_loss = -(torch.min(ratio * adv, clipped * adv)).mean()

                v_pred = self.critic(state).view(-1)
                v_loss = ((v_pred - ret) ** 2).mean()

                ent_loss = -ent.mean()
                loss = pi_loss + self.value_coef * v_loss + self.entropy_coef * ent_loss
                loss.backward()

            nn.utils.clip_grad_norm_(list(self.actor.parameters()) + list(self.critic.parameters()), self.max_grad_norm)
            self.opt.step()


# ============================================================
# Random control
# ============================================================
class RandomPolicy:
    def __init__(self, p_obs=0.06, p_eat=0.06):
        self.p_obs = float(p_obs)
        self.p_eat = float(p_eat)

    def act(self, _obs):
        if np.random.rand() < self.p_obs:
            return A_OBS
        if np.random.rand() < self.p_eat:
            return A_EAT
        return np.random.choice([A_LEFT, A_STAY, A_RIGHT])


# ============================================================
# Helpers / plots
# ============================================================
def ema(x, alpha=0.03):
    x = np.asarray(x, dtype=np.float32)
    y = np.zeros_like(x)
    m = 0.0
    for i in range(len(x)):
        m = (1 - alpha) * m + alpha * x[i]
        y[i] = m
    return y


def plot_rewardrate_only(logs_social, logs_random=None, ema_alpha=0.03):
    import matplotlib.pyplot as plt

    def get(logs, key):
        return np.array([x[key] for x in logs], dtype=np.float32)

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(ema(get(logs_social, "bout_success_rate"), alpha=ema_alpha), label="MAPPO social")
    if logs_random is not None:
        plt.plot(ema(get(logs_random, "bout_success_rate"), alpha=ema_alpha), label="random")
    plt.title("EMA bout success rate")
    plt.xlabel("episode")
    plt.ylabel("rate")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(ema(get(logs_social, "observe_rate"), alpha=ema_alpha), label="observe_rate")
    plt.plot(ema(get(logs_social, "obs_at_lick_rate"), alpha=ema_alpha), label="obs@lick")
    plt.plot(ema(get(logs_social, "obs_detect_rate"), alpha=ema_alpha), label="obs detects")
    plt.title("EMA observation alignment")
    plt.xlabel("episode")
    plt.ylabel("rate")
    plt.legend()

    plt.tight_layout()
    plt.show()


# ============================================================
# Training (fixed)
# ============================================================
def train_mappo_fixed(
    env,
    trainer,
    episodes=1200,
    batch_episodes=16,
    ppo_epochs=4,
    curriculum_episodes=400,   # <= key
    log_every=100,
):
    logs = []
    batch_buf = []

    for ep in range(1, episodes + 1):
        # curriculum: from easy -> final
        frac = min(1.0, ep / float(max(1, curriculum_episodes)))
        env.set_curriculum(frac)

        obs = env.reset()
        done = False

        h = trainer.actor.init_hidden(device=trainer.device)
        h0 = h.detach().cpu().numpy()

        obs_list, act_list, logp_list, val_list = [], [], [], []
        rew_list, done_list, state_list = [], [], []

        steps = 0
        ep_reward = 0.0
        succ_bouts = 0
        n_bouts = max(1, env.n_bouts)

        obs_steps = 0
        obs_at_lick = 0
        obs_detect = 0

        while not done:
            steps += 1
            obs_L = obs["learner"]
            central = env._central_state()

            a, logp, v, h = trainer.act(obs_L, h, central)

            obs2, rewards, done, info = env.step({"teacher": 0, "learner": a})
            rL = float(rewards["learner"])

            ep_reward += rL
            if info["eat_valid"] == 1:
                succ_bouts += 1

            # observation stats
            if info["observe"] == 1:
                obs_steps += 1
                if info["teacher_lick"] == 1:
                    obs_at_lick += 1
                if info["det_success"] == 1:
                    obs_detect += 1

            # store PPO data
            obs_list.append(obs_L.copy())
            act_list.append(int(a))
            logp_list.append(float(logp))
            val_list.append(float(v))
            rew_list.append(float(rL))
            done_list.append(float(done))
            state_list.append(info["central_state"].copy())

            obs = obs2

        logs.append({
            "ep": ep,
            "mean_reward": float(ep_reward),
            "bout_success_rate": float(succ_bouts / n_bouts),
            "observe_rate": float(obs_steps / max(1, steps)),
            "obs_at_lick_rate": float(obs_at_lick / max(1, steps)),
            "obs_detect_rate": float(obs_detect / max(1, steps)),
            "curriculum_frac": float(frac),
            "p_detect": float(env.p_detect_per_obs),
            "observe_cost": float(env.observe_cost),
            "eat_cost": float(env.eat_cost),
            "eat_cd_steps": int(env.eat_cooldown_steps),
        })

        ep_data = {
            "obs": np.asarray(obs_list, dtype=np.float32),
            "act": np.asarray(act_list, dtype=np.int64),
            "logp": np.asarray(logp_list, dtype=np.float32),
            "val": np.asarray(val_list, dtype=np.float32),
            "rew": np.asarray(rew_list, dtype=np.float32),
            "done": np.asarray(done_list, dtype=np.float32),
            "state": np.asarray(state_list, dtype=np.float32),
            "h0": np.asarray(h0, dtype=np.float32),
        }
        batch_buf.append(ep_data)

        if len(batch_buf) >= batch_episodes:
            trainer.update(batch_buf, ppo_epochs=ppo_epochs)
            batch_buf = []

        if ep % log_every == 0:
            recent = logs[-log_every:]
            print(
                f"[ep {ep:4d}] "
                f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.4f}  "
                f"obs={np.mean([x['observe_rate'] for x in recent]):.4f}  "
                f"obs@lick={np.mean([x['obs_at_lick_rate'] for x in recent]):.4f}  "
                f"det={np.mean([x['obs_detect_rate'] for x in recent]):.4f}  "
                f"curr={recent[-1]['curriculum_frac']:.2f}  "
                f"pdet={recent[-1]['p_detect']:.2f} cd={recent[-1]['eat_cd_steps']}"
            )

    return logs


def run_random_control(env, policy, episodes=1200, log_every=100):
    logs = []
    for ep in range(1, episodes + 1):
        obs = env.reset()
        done = False
        succ_bouts = 0
        n_bouts = max(1, env.n_bouts)

        obs_steps = 0
        obs_at_lick = 0
        obs_detect = 0

        while not done:
            a = policy.act(obs["learner"])
            obs, rewards, done, info = env.step({"teacher": 0, "learner": a})
            if info["eat_valid"] == 1:
                succ_bouts += 1
            if info["observe"] == 1:
                obs_steps += 1
                if info["teacher_lick"] == 1:
                    obs_at_lick += 1
                if info["det_success"] == 1:
                    obs_detect += 1

        logs.append({
            "ep": ep,
            "bout_success_rate": float(succ_bouts / n_bouts),
            "observe_rate": float(obs_steps / max(1, env.max_steps)),
            "obs_at_lick_rate": float(obs_at_lick / max(1, env.max_steps)),
            "obs_detect_rate": float(obs_detect / max(1, env.max_steps)),
        })

        if ep % log_every == 0:
            recent = logs[-log_every:]
            print(
                f"[RND ep {ep:4d}] "
                f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.4f}"
            )

    return logs


# ============================================================
# Run
# ============================================================
def run_experiment(seed=42, episodes=1200):
    set_seed(seed)

    env = SocialLickEnv1D_MA(
        grid_size=15,
        dt_s=0.5,
        max_time_s=80.0,
        teacher_period_s=10.0,
        teacher_jitter_s=2.0,
        lick_mean_s=1.0,
        lick_dist="gamma",
        window_s=3.0,

        # final difficulty (your target-ish)
        eat_cooldown_s_final=2.0,
        observe_cost_final=0.002,
        eat_cost_final=0.005,
        p_detect_final=0.85,

        # easy start
        eat_cooldown_s_easy=0.5,
        observe_cost_easy=0.0,
        eat_cost_easy=0.0,
        p_detect_easy=1.0,

        tau_obs_s=0.5,
        win_rem_noise_steps=1,

        # shaping (tiny but crucial)
        obs_lick_bonus=0.01,
        obs_detect_bonus=0.01,
        attend_bonus=0.0,
    )

    trainer = MAPPOTrainer(obs_dim=7, action_dim=5, state_dim=12, lr=2e-4)

    logs_social = train_mappo_fixed(
        env, trainer,
        episodes=episodes,
        batch_episodes=16,
        ppo_epochs=4,
        curriculum_episodes=400,
        log_every=max(100, episodes // 12),
    )

    # random baseline (use EASY settings too, so it's a fair baseline early)
    env.set_curriculum(1.0)  # evaluate baseline at final difficulty
    rnd = RandomPolicy(p_obs=0.06, p_eat=0.06)
    logs_random = run_random_control(env, rnd, episodes=episodes, log_every=max(100, episodes // 12))

    plot_rewardrate_only(logs_social, logs_random, ema_alpha=0.03)
    return logs_social, logs_random


if __name__ == "__main__":
    run_experiment(seed=42, episodes=1200)


## DA signalling

In [ ]:
"""
SIMULATE “DA photometry” under multiple social-learning hypotheses
=================================================================

You asked: “simulate how dopamine activity should look like” under different models,
so later you can match fiber photometry DA against simulated signals across learning.

This file contains:
1) The social observational-eating task (fast; dt=0.5s) with lick-bout duration distribution (mean ~1s)
2) A stable DQN learner (dueling + double + huber + soft target)
3) A training loop that logs step-wise events + value/RPE latents
4) Multiple DA models (reward RPE, social cue RPE, social value, action salience, info-gain)
5) Photometry-like filtering + plots (early vs late learning)

Key contingency (your biology):
- Teacher licking bout lasts ~1s on average (distributed), and the last lick opens a 3s window.
- Learner must (i) OBSERVE during the lick bout (to “register” that bout),
             (ii) be at its own food patch,
             (iii) EAT within the 3s window after the last lick,
             (iv) 1 reward per bout.
- OBSERVE costs time (can’t move/eat that step). So the learner should learn: brief observe (~1s),
  then move/eat fast.

IMPORTANT ABOUT “DA SIMULATION”:
- These are *candidate* computational hypotheses that generate a latent “DA impulse train”.
- Then we convert impulses → photometry-like signal with a calcium filter (rise/decay).
- Later you can fit/compare these to real DA photometry via GLM / peri-event averages / cross-val.

"""

import numpy as np
import random
from collections import deque, defaultdict

import torch
import torch.nn as nn
import torch.optim as optim


# -------------------------
# Seed
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def clamp_int(x: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, x))


# -------------------------
# Actions
# -------------------------
A_LEFT = 0
A_STAY = 1
A_RIGHT = 2
A_OBS = 3     # consumes time; reveals social timing info stochastically
A_EAT = 4


# ============================================================
# Environment
# ============================================================
class SocialLickEnv1D:
    """
    Fast but detailed:
      dt_s = 0.5s/step
      teacher licking bouts: duration distribution (mean ~ 1s)
      reward window: 3s after LAST lick of a bout

    Observation gating:
      - social signal (lick + window remaining estimate) only appears when action==OBS
      - detection during lick is probabilistic per OBS step (p_detect_per_obs < 1)

    This creates the behavioral tradeoff you want:
      - OBS too long delays reaching patch and eating in window
      - but OBS is needed to correctly time the window
    """

    def __init__(
        self,
        grid_size=15,
        dt_s=0.5,
        max_time_s=120.0,

        teacher_period_s=30.0,
        teacher_jitter_s=6.0,

        lick_mean_s=1.0,
        lick_dist="gamma",     # "gamma" or "lognormal"
        lick_cv=0.8,           # for lognormal

        window_s=3.0,
        eat_cooldown_s=2.0,

        observe_cost=0.002,
        eat_cost=0.005,

        # shaping: small bonus if OBS during lick and bout not yet detected
        attend_bonus=0.004,

        # imperfect detection on OBS steps during lick
        p_detect_per_obs=0.70,

        # OBS reveals window remaining with small noise
        win_rem_noise_steps=1,
    ):
        self.grid_size = int(grid_size)
        self.dt_s = float(dt_s)
        self.max_steps = int(round(max_time_s / self.dt_s))

        self.teacher_period_steps = int(round(teacher_period_s / self.dt_s))
        self.teacher_jitter_steps = int(round(teacher_jitter_s / self.dt_s))

        self.lick_mean_s = float(lick_mean_s)
        self.lick_dist = str(lick_dist)
        self.lick_cv = float(lick_cv)

        self.window_steps = int(round(window_s / self.dt_s))
        self.eat_cooldown_steps = int(round(eat_cooldown_s / self.dt_s))

        self.observe_cost = float(observe_cost)
        self.eat_cost = float(eat_cost)
        self.attend_bonus = float(attend_bonus)

        self.p_detect_per_obs = float(p_detect_per_obs)
        self.win_rem_noise_steps = int(win_rem_noise_steps)

        assert self.teacher_period_steps > self.window_steps + 1
        assert self.max_steps > self.teacher_period_steps + 5
        self.reset()

    # ---- lick duration distribution (mean ~1s) ----
    def _sample_lick_duration_steps(self):
        if self.lick_dist == "gamma":
            # shape=2, scale=mean/shape → mean fixed, non-degenerate
            shape = 2.0
            scale = self.lick_mean_s / shape
            dur_s = float(np.random.gamma(shape, scale))
        elif self.lick_dist == "lognormal":
            cv = max(1e-6, self.lick_cv)
            sigma2 = np.log(1.0 + cv**2)
            sigma = np.sqrt(sigma2)
            mu = np.log(self.lick_mean_s) - 0.5 * sigma2
            dur_s = float(np.random.lognormal(mean=mu, sigma=sigma))
        else:
            raise ValueError(f"Unknown lick_dist={self.lick_dist}")

        steps = int(np.round(dur_s / self.dt_s))
        steps = max(1, steps)
        return steps, dur_s

    def reset(self):
        self.t = 0

        # positions
        self.teacher_pos = int(np.random.randint(0, self.grid_size))
        candidates = [i for i in range(self.grid_size) if i != self.teacher_pos]
        self.learner_food_pos = int(np.random.choice(candidates))
        self.learner_pos = int(np.random.randint(0, self.grid_size))

        # internal timing
        self.eat_cd = 0
        self.last_lick_step = None

        # bout bookkeeping
        self.detected_bout_id = None
        self.rewarded_bout_ids = set()

        # build lick timeline
        self.bout_id_at_step = -np.ones(self.max_steps + 1, dtype=np.int32)
        self.bout_start_steps = []
        self.bout_end_steps = []
        self.bout_dur_s_list = []

        bout_id = 0
        k = 1
        while True:
            nominal = k * self.teacher_period_steps
            if nominal >= self.max_steps:
                break
            jitter = int(np.random.randint(-self.teacher_jitter_steps, self.teacher_jitter_steps + 1))
            start = clamp_int(nominal + jitter, 1, self.max_steps - 2)

            dur_steps, dur_s = self._sample_lick_duration_steps()
            end = clamp_int(start + dur_steps - 1, start, self.max_steps - 1)

            self.bout_id_at_step[start:end + 1] = bout_id
            self.bout_start_steps.append(start)
            self.bout_end_steps.append(end)
            self.bout_dur_s_list.append(dur_s)

            bout_id += 1
            k += 1

        self.n_bouts = int(bout_id)
        return self._get_obs(lick_sig=0.0, win_rem=0.0)

    def _teacher_lick_now(self, t: int):
        bid = int(self.bout_id_at_step[t]) if (0 <= t <= self.max_steps) else -1
        return (1, bid) if bid >= 0 else (0, -1)

    def _window_open(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        return 1 if (0 <= dt < self.window_steps) else 0

    def _window_remaining(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        rem = self.window_steps - dt
        return int(max(0, rem))

    def _phase_to_next_nominal(self, t: int) -> int:
        # steps until the next nominal bout time (periodic clock)
        mod = t % self.teacher_period_steps
        return self.teacher_period_steps - mod

    def _get_obs(self, lick_sig: float, win_rem: float):
        """
        State (dim=6):
          [learner_pos, food_pos, lick_sig, win_rem, phase_to_next, eat_cd]

        lick_sig & win_rem are only non-zero if the learner chose OBSERVE in that step.
        phase_to_next gives a coarse clock (helps learned anticipation).
        """
        lp = float(self.learner_pos) / float(self.grid_size - 1)
        lf = float(self.learner_food_pos) / float(self.grid_size - 1)
        phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period_steps)
        cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown_steps))
        return np.array([lp, lf, float(lick_sig), float(win_rem), phase, cdn], dtype=np.float32)

    def step(self, action: int):
        self.t += 1

        teacher_lick, bout_id = self._teacher_lick_now(self.t)
        if teacher_lick == 1:
            self.last_lick_step = self.t

        window_open = self._window_open(self.t)
        win_rem_true = self._window_remaining(self.t)

        # cooldown update
        if self.eat_cd > 0:
            self.eat_cd -= 1

        lick_sig = 0.0
        win_rem = 0.0
        did_observe = 0
        did_eat = 0
        seen_lick = 0  # what the learner "saw" on this OBS step (0/1)

        # movement/eat/obs
        if action == A_LEFT:
            self.learner_pos -= 1
        elif action == A_RIGHT:
            self.learner_pos += 1
        elif action == A_OBS:
            did_observe = 1

            # imperfect detection of licking
            if teacher_lick == 1 and bout_id >= 0 and (np.random.rand() < self.p_detect_per_obs):
                seen_lick = 1
                self.detected_bout_id = int(bout_id)

            lick_sig = float(seen_lick)

            # noisy window remaining estimate (normalized)
            if win_rem_true > 0:
                noise = int(np.random.randint(-self.win_rem_noise_steps, self.win_rem_noise_steps + 1))
                est = clamp_int(win_rem_true + noise, 0, self.window_steps)
                win_rem = float(est) / float(max(1, self.window_steps))
            else:
                win_rem = 0.0

        elif action == A_EAT:
            did_eat = 1
        # A_STAY does nothing

        self.learner_pos = clamp_int(self.learner_pos, 0, self.grid_size - 1)

        # costs
        reward = 0.0
        reward -= self.observe_cost * float(did_observe)
        reward -= self.eat_cost * float(did_eat)

        # shaping: reward OBS during lick only if this bout is not yet detected
        if did_observe == 1 and teacher_lick == 1 and bout_id >= 0:
            if self.detected_bout_id != int(bout_id):
                reward += self.attend_bonus

        # eat allowed?
        eat_allowed = (did_eat == 1 and self.eat_cd == 0)
        if eat_allowed:
            self.eat_cd = self.eat_cooldown_steps

        # success logic (1 reward per bout)
        eat_valid = 0
        if eat_allowed and (self.learner_pos == self.learner_food_pos) and (window_open == 1) and (self.last_lick_step is not None):
            last_bid = int(self.bout_id_at_step[self.last_lick_step]) if self.last_lick_step <= self.max_steps else -1
            if (last_bid >= 0) and (self.detected_bout_id == last_bid) and (last_bid not in self.rewarded_bout_ids):
                eat_valid = 1
                self.rewarded_bout_ids.add(last_bid)
                reward += 1.0

        done = bool(self.t >= self.max_steps)

        info = {
            "t": self.t,
            "teacher_lick": int(teacher_lick),
            "bout_id": int(bout_id),
            "window_open": int(window_open),
            "win_rem_true": int(win_rem_true),

            "action": int(action),
            "observe": int(did_observe),
            "eat": int(did_eat),
            "eat_allowed": int(eat_allowed),
            "eat_valid": int(eat_valid),

            "learner_pos": int(self.learner_pos),
            "food_pos": int(self.learner_food_pos),

            "seen_lick": int(seen_lick),
            "win_rem_est": float(win_rem),

            "detected_bout_id": -1 if self.detected_bout_id is None else int(self.detected_bout_id),
            "rewarded_bouts": len(self.rewarded_bout_ids),
            "n_bouts": int(self.n_bouts),

            "dt_s": float(self.dt_s),
            "window_steps": int(self.window_steps),
        }

        obs = self._get_obs(lick_sig=lick_sig, win_rem=win_rem)
        return obs, reward, done, info


# ============================================================
# DQN (dueling + double + huber + soft target)
# ============================================================
class DuelingQNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.V = nn.Linear(hidden, 1)
        self.A = nn.Linear(hidden, action_dim)

    def forward(self, x):
        h = self.backbone(x)
        v = self.V(h)
        a = self.A(h)
        return v + (a - a.mean(dim=1, keepdim=True))


class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.buf = deque(maxlen=int(capacity))

    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return np.array(s), np.array(a), np.array(r), np.array(s2), np.array(d)

    def __len__(self):
        return len(self.buf)


class DQNAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        batch_size=64,
        buffer_size=100000,
        eps_start=1.0,
        eps_end=0.05,
        eps_decay=0.9995,
        tau=0.02,
        grad_clip=10.0,
        device=None,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.batch_size = int(batch_size)
        self.tau = float(tau)
        self.grad_clip = float(grad_clip)

        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.q = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt.load_state_dict(self.q.state_dict())

        self.opt = optim.Adam(self.q.parameters(), lr=float(lr))
        self.loss_fn = nn.SmoothL1Loss()
        self.rb = ReplayBuffer(buffer_size)

    def act(self, s):
        if np.random.rand() < self.eps:
            return np.random.randint(self.action_dim)
        return self.act_greedy(s)

    def act_greedy(self, s):
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            qv = self.q(st)
        return int(torch.argmax(qv, dim=1).item())

    def V(self, s):
        """State value proxy V(s)=max_a Q(s,a)."""
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            qv = self.q(st).squeeze(0)
        return float(torch.max(qv).item())

    def push(self, s, a, r, s2, done):
        self.rb.push(s, int(a), float(r), s2, float(done))

    def update(self, updates_per_step=1):
        if len(self.rb) < self.batch_size:
            return None

        last_loss = None
        for _ in range(int(updates_per_step)):
            if len(self.rb) < self.batch_size:
                break

            s, a, r, s2, d = self.rb.sample(self.batch_size)
            s = torch.tensor(s, dtype=torch.float32, device=self.device)
            a = torch.tensor(a, dtype=torch.int64, device=self.device).unsqueeze(1)
            r = torch.tensor(r, dtype=torch.float32, device=self.device)
            s2 = torch.tensor(s2, dtype=torch.float32, device=self.device)
            d = torch.tensor(d, dtype=torch.float32, device=self.device)

            q_sa = self.q(s).gather(1, a).squeeze(1)

            with torch.no_grad():
                a_star = torch.argmax(self.q(s2), dim=1, keepdim=True)
                q_next = self.qt(s2).gather(1, a_star).squeeze(1)
                target = r + self.gamma * q_next * (1.0 - d)

            loss = self.loss_fn(q_sa, target)

            self.opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.q.parameters(), self.grad_clip)
            self.opt.step()

            with torch.no_grad():
                for p, pt in zip(self.q.parameters(), self.qt.parameters()):
                    pt.data.mul_(1.0 - self.tau).add_(self.tau * p.data)

            last_loss = float(loss.item())

        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        return last_loss


# ============================================================
# DA simulation utilities
# ============================================================
def calcium_filter(impulses, dt_s, tau_rise_s=0.2, tau_decay_s=1.2):
    """
    Convert impulse train to photometry-like signal using a simple bi-exponential kernel.
    (Discrete causal filter; not meant to be biophysically perfect—just a reasonable calcium-like smoother.)
    """
    impulses = np.asarray(impulses, dtype=np.float32)
    n = len(impulses)

    # kernel length: ~8 decay constants
    L = int(np.ceil(8 * tau_decay_s / dt_s))
    t = np.arange(L, dtype=np.float32) * dt_s

    # normalized biexponential (rise then decay)
    k = (1 - np.exp(-t / max(1e-6, tau_rise_s))) * np.exp(-t / max(1e-6, tau_decay_s))
    k = k / (k.sum() + 1e-9)

    y = np.convolve(impulses, k, mode="full")[:n]
    return y.astype(np.float32)


def make_event_indices(tr):
    """Extract key event indices from an episode trace."""
    lick = np.array(tr["teacher_lick"], dtype=int)
    obs = np.array(tr["observe"], dtype=int)
    seen = np.array(tr["seen_lick"], dtype=int) if "seen_lick" in tr else None
    reward = np.array(tr["eat_valid"], dtype=int)

    # last lick of bout ≈ lick falling edge (1->0)
    lick_end = np.where((lick[:-1] == 1) & (lick[1:] == 0))[0] + 1

    # observe steps, and observe-with-lick-seen
    obs_idx = np.where(obs == 1)[0]
    if seen is not None:
        obs_seen_lick_idx = np.where((obs == 1) & (seen == 1))[0]
    else:
        obs_seen_lick_idx = np.array([], dtype=int)

    rew_idx = np.where(reward == 1)[0]

    return {
        "lick_end": lick_end,
        "obs": obs_idx,
        "obs_seen_lick": obs_seen_lick_idx,
        "reward": rew_idx,
    }


def peri_event_average(signal, event_idx, pre_steps=10, post_steps=20):
    """Mean peri-event trace for one episode (can average later across episodes)."""
    signal = np.asarray(signal, dtype=np.float32)
    out = []
    for e in event_idx:
        a = e - pre_steps
        b = e + post_steps + 1
        if a < 0 or b > len(signal):
            continue
        out.append(signal[a:b])
    if len(out) == 0:
        return None
    return np.mean(np.stack(out, axis=0), axis=0)


def simulate_da_models_for_episode(tr, agent, gamma=0.99, ep_frac=0.0):
    """
    Build several candidate DA impulse trains from one episode trajectory.

    Models (impulse-level):
      1) reward_RPE:  δ_t = r_t + γ V(s_{t+1}) - V(s_t)
      2) social_cue_RPE: on OBS steps, ΔV due to social features being revealed (V_with - V_without)
      3) social_value: V_with - V_without (continuous but only "available" if OBS; here we gate by OBS)
      4) action_salience: spikes at OBS onset and EAT onset (vigor/initiations)
      5) info_gain_surprise: spikes when OBS sees lick, scaled by surprise under a timing prior

    Notes:
      - These are *hypotheses*. Later you can compare which best matches DA photometry.
      - We keep the models “cheap” and interpretable.
    """
    # arrays
    r = np.array(tr["reward"], dtype=np.float32)        # step reward (includes costs + success)
    obs = np.array(tr["observe"], dtype=int)
    eat = np.array(tr["eat"], dtype=int)
    seen_lick = np.array(tr["seen_lick"], dtype=int)
    phase = np.array(tr["phase"], dtype=np.float32)    # from state (0..1), smaller ≈ closer to expected lick time

    # states
    S = np.array(tr["state"], dtype=np.float32)
    S2 = np.array(tr["next_state"], dtype=np.float32)

    T = len(r)

    # helper to compute V(s)=max_a Q(s,a) quickly in batch
    def V_batch(states):
        st = torch.tensor(states, dtype=torch.float32, device=agent.device)
        with torch.no_grad():
            qv = agent.q(st)
            v = torch.max(qv, dim=1).values
        return v.detach().cpu().numpy().astype(np.float32)

    V_s = V_batch(S)
    V_s2 = V_batch(S2)

    done = np.array(tr["done"], dtype=np.float32)

    # ---- Model 1: reward RPE ----
    rpe = r + gamma * V_s2 * (1.0 - done) - V_s

    # ---- Model 2: social cue RPE (value jump on OBS because social features appear) ----
    # Use next_state as the "post-observation" state (it contains lick_sig & win_rem when OBS),
    # and a "prior" version with social features zeroed out.
    S2_prior = S2.copy()
    S2_prior[:, 2] = 0.0  # lick_sig
    S2_prior[:, 3] = 0.0  # win_rem
    V_post = V_s2
    V_prior = V_batch(S2_prior)

    cue_deltaV = (V_post - V_prior) * obs.astype(np.float32)

    # ---- Model 3: social value (gated by OBS availability) ----
    # Could also be continuous if you assume the animal has continuous social channel.
    social_value = (V_post - V_prior) * obs.astype(np.float32)

    # ---- Model 4: action salience (OBS/EAT initiation) ----
    # Spikes at action onset; you can tune weights later.
    sal = np.zeros(T, dtype=np.float32)
    sal += 1.0 * (obs == 1).astype(np.float32)
    sal += 0.6 * (eat == 1).astype(np.float32)

    # ---- Model 5: information-gain / surprise on observing lick ----
    # prior probability of lick given phase (smaller phase => closer to predicted lick)
    # We also shrink sigma with learning (ep_frac), imitating better internal timing prediction.
    sigma = 0.25 * (1.0 - ep_frac) + 0.10 * ep_frac  # early broad, late sharper
    p = np.exp(-(phase / max(1e-6, sigma)) ** 2)      # 0..1
    surprise = -np.log(p + 1e-6)
    info_gain = surprise * ((obs == 1) & (seen_lick == 1)).astype(np.float32)

    # Scale impulses to comparable magnitude
    # (DA amplitude is arbitrary in simulation; later you fit scaling to real data.)
    impulses = {
        "reward_RPE": 0.8 * rpe,
        "social_cue_RPE": 1.2 * cue_deltaV,
        "social_value": 0.6 * social_value,
        "action_salience": 0.3 * sal,
        "info_gain": 0.25 * info_gain,
    }
    return impulses


# ============================================================
# Training + logging (stores state transitions for DA sims)
# ============================================================
def train_social_with_latents(env, agent, episodes=1500, log_every=100, updates_per_step=1):
    """
    Train agent and record per-step data needed for DA simulations.
    Returns:
      logs: per-episode summary metrics
      traces: list of dict traces, each trace stores:
        state, next_state, reward, done, events, plus a few extras (phase)
    """
    logs = []
    traces = []

    for ep in range(1, episodes + 1):
        s = env.reset()
        done = False

        ep_reward = 0.0
        succ_bouts = 0
        n_bouts = max(1, env.n_bouts)

        obs_steps = 0
        obs_seen = 0
        eat_in_win = 0

        tr = defaultdict(list)

        while not done:
            a = agent.act(s)
            s2, r, done, info = env.step(a)

            ep_reward += r
            if info["eat_valid"] == 1:
                succ_bouts += 1
            if info["observe"] == 1:
                obs_steps += 1
                if info["seen_lick"] == 1:
                    obs_seen += 1
            if info["eat"] == 1 and info["window_open"] == 1:
                eat_in_win += 1

            # store transition for DA simulation
            tr["state"].append(s.copy())
            tr["next_state"].append(s2.copy())
            tr["reward"].append(float(r))
            tr["done"].append(float(done))

            # store events
            tr["teacher_lick"].append(int(info["teacher_lick"]))
            tr["window_open"].append(int(info["window_open"]))
            tr["observe"].append(int(info["observe"]))
            tr["eat"].append(int(info["eat"]))
            tr["eat_valid"].append(int(info["eat_valid"]))
            tr["seen_lick"].append(int(info["seen_lick"]))
            tr["win_rem_est"].append(float(info["win_rem_est"]))

            # store phase (from state feature)
            tr["phase"].append(float(s[4]))

            # learning
            agent.push(s, a, r, s2, done)
            agent.update(updates_per_step=updates_per_step)

            s = s2

        # finalize arrays
        for k in list(tr.keys()):
            tr[k] = np.array(tr[k])

        traces.append(tr)

        logs.append({
            "ep": ep,
            "bout_success_rate": float(succ_bouts / n_bouts),
            "mean_reward": float(ep_reward),
            "obs_rate": float(obs_steps / max(1, env.max_steps)),
            "obs_seen_lick_rate": float(obs_seen / max(1, env.max_steps)),
            "eat_in_window_rate": float(eat_in_win / max(1, env.max_steps)),
            "eps": float(agent.eps),
        })

        if ep % log_every == 0:
            recent = logs[-log_every:]
            print(
                f"[SOC ep {ep:4d}] "
                f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f}  "
                f"obs={np.mean([x['obs_rate'] for x in recent]):.3f}  "
                f"obsSeen={np.mean([x['obs_seen_lick_rate'] for x in recent]):.3f}  "
                f"eatWin={np.mean([x['eat_in_window_rate'] for x in recent]):.3f}  "
                f"R={np.mean([x['mean_reward'] for x in recent]):.3f}  "
                f"eps={agent.eps:.3f}"
            )

    return logs, traces


# ============================================================
# Plotting helpers
# ============================================================
def ema(x, alpha=0.02):
    x = np.asarray(x, dtype=np.float32)
    y = np.zeros_like(x)
    m = 0.0
    for i in range(len(x)):
        m = (1 - alpha) * m + alpha * x[i]
        y[i] = m
    return y


def plot_learning_curves(logs):
    import matplotlib.pyplot as plt
    ep = np.array([x["ep"] for x in logs])
    succ = ema([x["bout_success_rate"] for x in logs])
    obs = ema([x["obs_rate"] for x in logs])
    obsSeen = ema([x["obs_seen_lick_rate"] for x in logs])
    eatWin = ema([x["eat_in_window_rate"] for x in logs])

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(ep, succ)
    plt.title("EMA bout success rate")
    plt.xlabel("episode"); plt.ylabel("rate")

    plt.subplot(1, 3, 2)
    plt.plot(ep, obs, label="observe")
    plt.plot(ep, obsSeen, label="obs sees lick")
    plt.title("EMA observation metrics")
    plt.xlabel("episode"); plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(ep, eatWin)
    plt.title("EMA eat-in-window rate")
    plt.xlabel("episode"); plt.ylabel("rate")
    plt.tight_layout()
    plt.show()


def plot_da_peri_event(da_by_ep, traces, dt_s, model_name, event_key="reward", pre_s=5, post_s=8):
    """
    Compare early vs late learning peri-event DA traces.
    """
    import matplotlib.pyplot as plt

    pre = int(round(pre_s / dt_s))
    post = int(round(post_s / dt_s))
    x = (np.arange(pre + post + 1) - pre) * dt_s

    n = len(traces)
    early_idx = range(0, max(1, n // 5))               # first 20%
    late_idx = range(max(0, n - n // 5), n)            # last 20%

    def mean_peri(idxs):
        mats = []
        for i in idxs:
            ev = make_event_indices(traces[i])[event_key]
            m = peri_event_average(da_by_ep[i], ev, pre_steps=pre, post_steps=post)
            if m is not None:
                mats.append(m)
        if len(mats) == 0:
            return None
        return np.mean(np.stack(mats, axis=0), axis=0)

    e = mean_peri(early_idx)
    l = mean_peri(late_idx)

    plt.figure(figsize=(6, 4))
    if e is not None:
        plt.plot(x, e, label="early (first 20%)")
    if l is not None:
        plt.plot(x, l, label="late (last 20%)")
    plt.axvline(0, linestyle="--")
    plt.title(f"{model_name}: peri-{event_key} DA")
    plt.xlabel("time (s)")
    plt.ylabel("simulated photometry")
    plt.legend()
    plt.tight_layout()
    plt.show()


def simulate_and_plot_da(traces, agent, dt_s, gamma=0.99):
    """
    Simulate DA for each episode trace under multiple models, filter to photometry,
    and plot peri-event averages around lick_end and reward.
    """
    # 1) build impulses per episode
    impulses_by_model = defaultdict(list)

    n = len(traces)
    for i, tr in enumerate(traces):
        ep_frac = i / max(1, n - 1)
        imp = simulate_da_models_for_episode(tr, agent, gamma=gamma, ep_frac=ep_frac)
        for k, v in imp.items():
            impulses_by_model[k].append(v)

    # 2) filter to photometry-like signals
    phot_by_model = {}
    for k, imps in impulses_by_model.items():
        phot_by_model[k] = [calcium_filter(x, dt_s=dt_s, tau_rise_s=0.2, tau_decay_s=1.2) for x in imps]

    # 3) plot peri-event for each model
    for k, da_eps in phot_by_model.items():
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="lick_end", pre_s=6, post_s=10)
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="reward", pre_s=6, post_s=10)
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="obs_seen_lick", pre_s=6, post_s=10)

    return phot_by_model


# ============================================================
# Run experiment
# ============================================================
def run_experiment(seed=42, episodes=1200):
    set_seed(seed)

    env = SocialLickEnv1D(
        grid_size=15,
        dt_s=0.5,
        max_time_s=120.0,
        teacher_period_s=30.0,
        teacher_jitter_s=6.0,

        lick_mean_s=1.0,
        lick_dist="gamma",      # distribution, mean ~1s
        # lick_dist="lognormal", lick_cv=0.8,

        window_s=3.0,
        eat_cooldown_s=2.0,

        observe_cost=0.002,
        eat_cost=0.005,
        attend_bonus=0.004,

        p_detect_per_obs=0.70,
        win_rem_noise_steps=1,
    )

    agent = DQNAgent(state_dim=6, action_dim=5)

    logs, traces = train_social_with_latents(
        env, agent,
        episodes=episodes,
        log_every=max(100, episodes // 12),
        updates_per_step=1,
    )

    plot_learning_curves(logs)

    # Simulate DA under multiple models and plot early vs late peri-event signals
    phot_by_model = simulate_and_plot_da(traces, agent, dt_s=env.dt_s, gamma=agent.gamma)

    return logs, traces, phot_by_model


if __name__ == "__main__":
    run_experiment(seed=42, episodes=1200)


## Latest

In [ ]:
import numpy as np
import random
from collections import deque, defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import argparse
import os

# -------------------------
# Seed
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def clamp_int(x: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, x))

# -------------------------
# Actions
# -------------------------
A_LEFT = 0
A_STAY = 1
A_RIGHT = 2
A_OBS = 3 # consumes time; reveals social timing info stochastically
A_EAT = 4

# ============================================================
# Environment
# ============================================================
class SocialLickEnv1D:
    """
    Fast but detailed:
      dt_s = 0.5s/step
      teacher licking bouts: duration distribution (mean ~ 1s)
      reward window: 3s after LAST lick of a bout
    Observation gating:
      - social signal (lick + window remaining estimate) only appears when action==OBS
      - detection during lick is probabilistic per OBS step (p_detect_per_obs < 1)
    This creates the behavioral tradeoff you want:
      - OBS too long delays reaching patch and eating in window
      - but OBS is needed to correctly time the window
    """
    def __init__(
        self,
        grid_size=15,
        dt_s=0.5,
        max_time_s=120.0,
        teacher_period_s=30.0,
        teacher_jitter_s=6.0,
        lick_mean_s=1.0,
        lick_dist="gamma", # "gamma" or "lognormal"
        lick_cv=0.8, # for lognormal
        window_s=3.0,
        eat_cooldown_s=2.0,
        observe_cost=0.002,
        eat_cost=0.005,
        # shaping: small bonus if OBS during lick and bout not yet detected
        attend_bonus=0.004,
        # imperfect detection on OBS steps during lick
        p_detect_per_obs=0.70,
        # OBS reveals window remaining with small noise
        win_rem_noise_steps=1,
    ):
        self.grid_size = int(grid_size)
        self.dt_s = float(dt_s)
        self.max_steps = int(round(max_time_s / self.dt_s))
        self.teacher_period_steps = int(round(teacher_period_s / self.dt_s))
        self.teacher_jitter_steps = int(round(teacher_jitter_s / self.dt_s))
        self.lick_mean_s = float(lick_mean_s)
        self.lick_dist = str(lick_dist)
        self.lick_cv = float(lick_cv)
        self.window_steps = int(round(window_s / self.dt_s))
        self.eat_cooldown_steps = int(round(eat_cooldown_s / self.dt_s))
        self.observe_cost = float(observe_cost)
        self.eat_cost = float(eat_cost)
        self.attend_bonus = float(attend_bonus)
        self.p_detect_per_obs = float(p_detect_per_obs)
        self.win_rem_noise_steps = int(win_rem_noise_steps)
        assert self.teacher_period_steps > self.window_steps + 1
        assert self.max_steps > self.teacher_period_steps + 5
        self.reset()
    # ---- lick duration distribution (mean ~1s) ----
    def _sample_lick_duration_steps(self):
        if self.lick_dist == "gamma":
            # shape=2, scale=mean/shape → mean fixed, non-degenerate
            shape = 2.0
            scale = self.lick_mean_s / shape
            dur_s = float(np.random.gamma(shape, scale))
        elif self.lick_dist == "lognormal":
            cv = max(1e-6, self.lick_cv)
            sigma2 = np.log(1.0 + cv**2)
            sigma = np.sqrt(sigma2)
            mu = np.log(self.lick_mean_s) - 0.5 * sigma2
            dur_s = float(np.random.lognormal(mean=mu, sigma=sigma))
        else:
            raise ValueError(f"Unknown lick_dist={self.lick_dist}")
        steps = int(np.round(dur_s / self.dt_s))
        steps = max(1, steps)
        return steps, dur_s
    def reset(self):
        self.t = 0
        # positions
        self.teacher_pos = int(np.random.randint(0, self.grid_size))
        candidates = [i for i in range(self.grid_size) if i != self.teacher_pos]
        self.learner_food_pos = int(np.random.choice(candidates))
        self.learner_pos = int(np.random.randint(0, self.grid_size))
        # internal timing
        self.eat_cd = 0
        self.last_lick_step = None
        # bout bookkeeping
        self.detected_bout_id = None
        self.rewarded_bout_ids = set()
        # build lick timeline
        self.bout_id_at_step = -np.ones(self.max_steps + 1, dtype=np.int32)
        self.bout_start_steps = []
        self.bout_end_steps = []
        self.bout_dur_s_list = []
        bout_id = 0
        k = 1
        while True:
            nominal = k * self.teacher_period_steps
            if nominal >= self.max_steps:
                break
            jitter = int(np.random.randint(-self.teacher_jitter_steps, self.teacher_jitter_steps + 1))
            start = clamp_int(nominal + jitter, 1, self.max_steps - 2)
            dur_steps, dur_s = self._sample_lick_duration_steps()
            end = clamp_int(start + dur_steps - 1, start, self.max_steps - 1)
            self.bout_id_at_step[start:end + 1] = bout_id
            self.bout_start_steps.append(start)
            self.bout_end_steps.append(end)
            self.bout_dur_s_list.append(dur_s)
            bout_id += 1
            k += 1
        self.n_bouts = int(bout_id)
        return self._get_obs(lick_sig=0.0, win_rem=0.0)
    def _teacher_lick_now(self, t: int):
        bid = int(self.bout_id_at_step[t]) if (0 <= t <= self.max_steps) else -1
        return (1, bid) if bid >= 0 else (0, -1)
    def _window_open(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        return 1 if (0 <= dt < self.window_steps) else 0
    def _window_remaining(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        rem = self.window_steps - dt
        return int(max(0, rem))
    def _phase_to_next_nominal(self, t: int) -> int:
        # steps until the next nominal bout time (periodic clock)
        mod = t % self.teacher_period_steps
        return self.teacher_period_steps - mod
    def _get_obs(self, lick_sig: float, win_rem: float):
        """
        State (dim=6):
          [learner_pos, food_pos, lick_sig, win_rem, phase_to_next, eat_cd]
        lick_sig & win_rem are only non-zero if the learner chose OBSERVE in that step.
        phase_to_next gives a coarse clock (helps learned anticipation).
        """
        lp = float(self.learner_pos) / float(self.grid_size - 1)
        lf = float(self.learner_food_pos) / float(self.grid_size - 1)
        phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period_steps)
        cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown_steps))
        return np.array([lp, lf, float(lick_sig), float(win_rem), phase, cdn], dtype=np.float32)
    def step(self, action: int):
        self.t += 1
        teacher_lick, bout_id = self._teacher_lick_now(self.t)
        if teacher_lick == 1:
            self.last_lick_step = self.t
        window_open = self._window_open(self.t)
        win_rem_true = self._window_remaining(self.t)
        # cooldown update
        if self.eat_cd > 0:
            self.eat_cd -= 1
        lick_sig = 0.0
        win_rem = 0.0
        did_observe = 0
        did_eat = 0
        seen_lick = 0 # what the learner "saw" on this OBS step (0/1)
        # movement/eat/obs
        if action == A_LEFT:
            self.learner_pos -= 1
        elif action == A_RIGHT:
            self.learner_pos += 1
        elif action == A_OBS:
            did_observe = 1
            # imperfect detection of licking
            if teacher_lick == 1 and bout_id >= 0 and (np.random.rand() < self.p_detect_per_obs):
                seen_lick = 1
                self.detected_bout_id = int(bout_id)
            lick_sig = float(seen_lick)
            # noisy window remaining estimate (normalized)
            if win_rem_true > 0:
                noise = int(np.random.randint(-self.win_rem_noise_steps, self.win_rem_noise_steps + 1))
                est = clamp_int(win_rem_true + noise, 0, self.window_steps)
                win_rem = float(est) / float(max(1, self.window_steps))
            else:
                win_rem = 0.0
        elif action == A_EAT:
            did_eat = 1
        # A_STAY does nothing
        self.learner_pos = clamp_int(self.learner_pos, 0, self.grid_size - 1)
        # costs
        reward = 0.0
        reward -= self.observe_cost * float(did_observe)
        reward -= self.eat_cost * float(did_eat)
        # shaping: reward OBS during lick only if this bout is not yet detected
        if did_observe == 1 and teacher_lick == 1 and bout_id >= 0:
            if self.detected_bout_id != int(bout_id):
                reward += self.attend_bonus
        # eat allowed?
        eat_allowed = (did_eat == 1 and self.eat_cd == 0)
        if eat_allowed:
            self.eat_cd = self.eat_cooldown_steps
        # success logic (1 reward per bout)
        eat_valid = 0
        if eat_allowed and (self.learner_pos == self.learner_food_pos) and (window_open == 1) and (self.last_lick_step is not None):
            last_bid = int(self.bout_id_at_step[self.last_lick_step]) if self.last_lick_step <= self.max_steps else -1
            if (last_bid >= 0) and (self.detected_bout_id == last_bid) and (last_bid not in self.rewarded_bout_ids):
                eat_valid = 1
                self.rewarded_bout_ids.add(last_bid)
                reward += 1.0
        done = bool(self.t >= self.max_steps)
        info = {
            "t": self.t,
            "teacher_lick": int(teacher_lick),
            "bout_id": int(bout_id),
            "window_open": int(window_open),
            "win_rem_true": int(win_rem_true),
            "action": int(action),
            "observe": int(did_observe),
            "eat": int(did_eat),
            "eat_allowed": int(eat_allowed),
            "eat_valid": int(eat_valid),
            "learner_pos": int(self.learner_pos),
            "food_pos": int(self.learner_food_pos),
            "seen_lick": int(seen_lick),
            "win_rem_est": float(win_rem),
            "detected_bout_id": -1 if self.detected_bout_id is None else int(self.detected_bout_id),
            "rewarded_bouts": len(self.rewarded_bout_ids),
            "n_bouts": int(self.n_bouts),
            "dt_s": float(self.dt_s),
            "window_steps": int(self.window_steps),
        }
        obs = self._get_obs(lick_sig=lick_sig, win_rem=win_rem)
        return obs, reward, done, info

# ============================================================
# DQN (dueling + double + huber + soft target)
# ============================================================
class DuelingQNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.V = nn.Linear(hidden, 1)
        self.A = nn.Linear(hidden, action_dim)
    def forward(self, x):
        h = self.backbone(x)
        v = self.V(h)
        a = self.A(h)
        return v + (a - a.mean(dim=1, keepdim=True))

class TorchReplayBuffer:
    def __init__(self, capacity, state_dim, device):
        self.capacity = int(capacity)
        self.device = device
        self.state = torch.zeros((capacity, state_dim), dtype=torch.float32, device=device)
        self.action = torch.zeros(capacity, dtype=torch.int64, device=device)
        self.reward = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.next_state = torch.zeros((capacity, state_dim), dtype=torch.float32, device=device)
        self.done = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.idx = 0
        self.size = 0

    def push(self, s, a, r, s2, done):
        self.state[self.idx] = torch.tensor(s, dtype=torch.float32, device=self.device)
        self.action[self.idx] = int(a)
        self.reward[self.idx] = float(r)
        self.next_state[self.idx] = torch.tensor(s2, dtype=torch.float32, device=self.device)
        self.done[self.idx] = float(done)
        self.idx = (self.idx + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        indices = torch.randint(0, self.size, (batch_size,), device=self.device)
        return (
            self.state[indices],
            self.action[indices].unsqueeze(1),
            self.reward[indices],
            self.next_state[indices],
            self.done[indices],
        )

    def __len__(self):
        return self.size

class DQNAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        batch_size=256,
        buffer_size=50000,
        eps_start=1.0,
        eps_end=0.05,
        eps_decay=0.9995,
        tau=0.02,
        grad_clip=10.0,
        device=None,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.batch_size = int(batch_size)
        self.tau = float(tau)
        self.grad_clip = float(grad_clip)
        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.q = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        if hasattr(torch, "compile"):
            self.q = torch.compile(self.q)
        self.qt = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt.load_state_dict(self.q.state_dict())
        self.opt = optim.Adam(self.q.parameters(), lr=float(lr))
        self.loss_fn = nn.SmoothL1Loss()
        self.rb = TorchReplayBuffer(buffer_size, state_dim, self.device)
        self.scaler = GradScaler(enabled=self.device.type == 'cuda')

    def act(self, s):
        if np.random.rand() < self.eps:
            return np.random.randint(self.action_dim)
        return self.act_greedy(s)

    def act_greedy(self, s):
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            qv = self.q(st)
        return int(torch.argmax(qv, dim=1).item())

    def V(self, s):
        """State value proxy V(s)=max_a Q(s,a)."""
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            qv = self.q(st).squeeze(0)
        return float(torch.max(qv).item())

    def push(self, s, a, r, s2, done):
        self.rb.push(s, a, r, s2, done)

    def update(self, updates_per_step=1):
        if len(self.rb) < self.batch_size:
            return None
        last_loss = None
        for _ in range(int(updates_per_step)):
            if len(self.rb) < self.batch_size:
                break
            s, a, r, s2, d = self.rb.sample(self.batch_size)
            with autocast(enabled=self.device.type == 'cuda'):
                q_sa = self.q(s).gather(1, a).squeeze(1)
                with torch.no_grad():
                    a_star = torch.argmax(self.q(s2), dim=1, keepdim=True)
                    q_next = self.qt(s2).gather(1, a_star).squeeze(1)
                    target = r + self.gamma * q_next * (1.0 - d)
                loss = self.loss_fn(q_sa, target)
            self.opt.zero_grad()
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.opt)
            nn.utils.clip_grad_norm_(self.q.parameters(), self.grad_clip)
            self.scaler.step(self.opt)
            self.scaler.update()
            with torch.no_grad():
                for p, pt in zip(self.q.parameters(), self.qt.parameters()):
                    pt.data.mul_(1.0 - self.tau).add_(self.tau * p.data)
            last_loss = float(loss.item())
        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        return last_loss

# ============================================================
# DA simulation utilities
# ============================================================
def calcium_filter(impulses, dt_s, tau_rise_s=0.2, tau_decay_s=1.2):
    """
    Convert impulse train to photometry-like signal using a simple bi-exponential kernel.
    (Discrete causal filter; not meant to be biophysically perfect—just a reasonable calcium-like smoother.)
    """
    impulses = np.asarray(impulses, dtype=np.float32)
    n = len(impulses)
    # kernel length: ~8 decay constants
    L = int(np.ceil(8 * tau_decay_s / dt_s))
    t = np.arange(L, dtype=np.float32) * dt_s
    # normalized biexponential (rise then decay)
    k = (1 - np.exp(-t / max(1e-6, tau_rise_s))) * np.exp(-t / max(1e-6, tau_decay_s))
    k = k / (k.sum() + 1e-9)
    y = np.convolve(impulses, k, mode="full")[:n]
    return y.astype(np.float32)

def make_event_indices(tr):
    """Extract key event indices from an episode trace."""
    lick = np.array(tr["teacher_lick"], dtype=int)
    obs = np.array(tr["observe"], dtype=int)
    seen = np.array(tr["seen_lick"], dtype=int) if "seen_lick" in tr else None
    reward = np.array(tr["eat_valid"], dtype=int)
    # last lick of bout ≈ lick falling edge (1->0)
    lick_end = np.where((lick[:-1] == 1) & (lick[1:] == 0))[0] + 1
    # observe steps, and observe-with-lick-seen
    obs_idx = np.where(obs == 1)[0]
    if seen is not None:
        obs_seen_lick_idx = np.where((obs == 1) & (seen == 1))[0]
    else:
        obs_seen_lick_idx = np.array([], dtype=int)
    rew_idx = np.where(reward == 1)[0]
    return {
        "lick_end": lick_end,
        "obs": obs_idx,
        "obs_seen_lick": obs_seen_lick_idx,
        "reward": rew_idx,
    }

def peri_event_average(signal, event_idx, pre_steps=10, post_steps=20):
    """Mean peri-event trace for one episode (can average later across episodes)."""
    signal = np.asarray(signal, dtype=np.float32)
    out = []
    for e in event_idx:
        a = e - pre_steps
        b = e + post_steps + 1
        if a < 0 or b > len(signal):
            continue
        out.append(signal[a:b])
    if len(out) == 0:
        return None
    return np.mean(np.stack(out, axis=0), axis=0)

def simulate_da_models_for_episode(tr, agent, gamma=0.99, ep_frac=0.0):
    """
    Build several candidate DA impulse trains from one episode trajectory.
    Models (impulse-level):
      1) reward_RPE: δ_t = r_t + γ V(s_{t+1}) - V(s_t)
      2) social_cue_RPE: on OBS steps, ΔV due to social features being revealed (V_with - V_without)
      3) social_value: V_with - V_without (continuous but only "available" if OBS; here we gate by OBS)
      4) action_salience: spikes at OBS onset and EAT onset (vigor/initiations)
      5) info_gain_surprise: spikes when OBS sees lick, scaled by surprise under a timing prior
    Notes:
      - These are *hypotheses*. Later you can compare which best matches DA photometry.
      - We keep the models “cheap” and interpretable.
    """
    # arrays
    r = np.array(tr["reward"], dtype=np.float32) # step reward (includes costs + success)
    obs = np.array(tr["observe"], dtype=int)
    eat = np.array(tr["eat"], dtype=int)
    seen_lick = np.array(tr["seen_lick"], dtype=int)
    phase = np.array(tr["phase"], dtype=np.float32) # from state (0..1), smaller ≈ closer to expected lick time
    # states
    S = np.array(tr["state"], dtype=np.float32)
    S2 = np.array(tr["next_state"], dtype=np.float32)
    T = len(r)
    # helper to compute V(s)=max_a Q(s,a) quickly in batch
    def V_batch(states):
        st = torch.tensor(states, dtype=torch.float32, device=agent.device)
        with torch.no_grad():
            qv = agent.q(st)
            v = torch.max(qv, dim=1).values
        return v.detach().cpu().numpy().astype(np.float32)
    V_s = V_batch(S)
    V_s2 = V_batch(S2)
    done = np.array(tr["done"], dtype=np.float32)
    # ---- Model 1: reward RPE ----
    rpe = r + gamma * V_s2 * (1.0 - done) - V_s
    # ---- Model 2: social cue RPE (value jump on OBS because social features appear) ----
    # Use next_state as the "post-observation" state (it contains lick_sig & win_rem when OBS),
    # and a "prior" version with social features zeroed out.
    S2_prior = S2.copy()
    S2_prior[:, 2] = 0.0 # lick_sig
    S2_prior[:, 3] = 0.0 # win_rem
    V_post = V_s2
    V_prior = V_batch(S2_prior)
    cue_deltaV = (V_post - V_prior) * obs.astype(np.float32)
    # ---- Model 3: social value (gated by OBS availability) ----
    # Could also be continuous if you assume the animal has continuous social channel.
    social_value = (V_post - V_prior) * obs.astype(np.float32)
    # ---- Model 4: action salience (OBS/EAT initiation) ----
    # Spikes at action onset; you can tune weights later.
    sal = np.zeros(T, dtype=np.float32)
    sal += 1.0 * (obs == 1).astype(np.float32)
    sal += 0.6 * (eat == 1).astype(np.float32)
    # ---- Model 5: information-gain / surprise on observing lick ----
    # prior probability of lick given phase (smaller phase => closer to predicted lick)
    # We also shrink sigma with learning (ep_frac), imitating better internal timing prediction.
    sigma = 0.25 * (1.0 - ep_frac) + 0.10 * ep_frac # early broad, late sharper
    p = np.exp(-(phase / max(1e-6, sigma)) ** 2) # 0..1
    surprise = -np.log(p + 1e-6)
    info_gain = surprise * ((obs == 1) & (seen_lick == 1)).astype(np.float32)
    # Scale impulses to comparable magnitude
    # (DA amplitude is arbitrary in simulation; later you fit scaling to real data.)
    impulses = {
        "reward_RPE": 0.8 * rpe,
        "social_cue_RPE": 1.2 * cue_deltaV,
        "social_value": 0.6 * social_value,
        "action_salience": 0.3 * sal,
        "info_gain": 0.25 * info_gain,
    }
    return impulses

# ============================================================
# Training + logging (stores state transitions for DA sims)
# ============================================================
def train_social_with_latents(env, agent, episodes=1200, log_every=100, updates_per_step=4, early_stop_threshold=0.8, early_stop_window=100):
    """
    Train agent and record per-step data needed for DA simulations.
    Returns:
      logs: per-episode summary metrics
      traces: list of dict traces, each trace stores:
        state, next_state, reward, done, events, plus a few extras (phase)
    """
    logs = []
    traces = []
    success_history = deque(maxlen=early_stop_window)
    for ep in range(1, episodes + 1):
        s = env.reset()
        done = False
        ep_reward = 0.0
        succ_bouts = 0
        n_bouts = max(1, env.n_bouts)
        obs_steps = 0
        obs_seen = 0
        eat_in_win = 0
        tr = defaultdict(list)
        while not done:
            a = agent.act(s)
            s2, r, done, info = env.step(a)
            ep_reward += r
            if info["eat_valid"] == 1:
                succ_bouts += 1
            if info["observe"] == 1:
                obs_steps += 1
                if info["seen_lick"] == 1:
                    obs_seen += 1
            if info["eat"] == 1 and info["window_open"] == 1:
                eat_in_win += 1
            # store transition for DA simulation
            tr["state"].append(s.copy())
            tr["next_state"].append(s2.copy())
            tr["reward"].append(float(r))
            tr["done"].append(float(done))
            # store events
            tr["teacher_lick"].append(int(info["teacher_lick"]))
            tr["window_open"].append(int(info["window_open"]))
            tr["observe"].append(int(info["observe"]))
            tr["eat"].append(int(info["eat"]))
            tr["eat_valid"].append(int(info["eat_valid"]))
            tr["seen_lick"].append(int(info["seen_lick"]))
            tr["win_rem_est"].append(float(info["win_rem_est"]))
            # store phase (from state feature)
            tr["phase"].append(float(s[4]))
            # learning
            agent.push(s, a, r, s2, done)
            agent.update(updates_per_step=updates_per_step)
            s = s2
        # finalize arrays
        for k in list(tr.keys()):
            tr[k] = np.array(tr[k])
        traces.append(tr)
        logs.append({
            "ep": ep,
            "bout_success_rate": float(succ_bouts / n_bouts),
            "mean_reward": float(ep_reward),
            "obs_rate": float(obs_steps / max(1, env.max_steps)),
            "obs_seen_lick_rate": float(obs_seen / max(1, env.max_steps)),
            "eat_in_window_rate": float(eat_in_win / max(1, env.max_steps)),
            "eps": float(agent.eps),
        })
        success_history.append(logs[-1]["bout_success_rate"])
        if ep % log_every == 0:
            recent = logs[-log_every:]
            print(
                f"[SOC ep {ep:4d}] "
                f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f} "
                f"obs={np.mean([x['obs_rate'] for x in recent]):.3f} "
                f"obsSeen={np.mean([x['obs_seen_lick_rate'] for x in recent]):.3f} "
                f"eatWin={np.mean([x['eat_in_window_rate'] for x in recent]):.3f} "
                f"R={np.mean([x['mean_reward'] for x in recent]):.3f} "
                f"eps={agent.eps:.3f}"
            )
        # Early stopping check
        if len(success_history) == early_stop_window and np.mean(success_history) > early_stop_threshold:
            print(f"Early stopping at episode {ep}: Avg success rate {np.mean(success_history):.3f} > {early_stop_threshold}")
            break
    return logs, traces

# ============================================================
# Plotting helpers
# ============================================================
def ema(x, alpha=0.02):
    x = np.asarray(x, dtype=np.float32)
    y = np.zeros_like(x)
    m = 0.0
    for i in range(len(x)):
        m = (1 - alpha) * m + alpha * x[i]
        y[i] = m
    return y

def plot_learning_curves(logs, save_dir='plots'):
    os.makedirs(save_dir, exist_ok=True)
    ep = np.array([x["ep"] for x in logs])
    succ = ema([x["bout_success_rate"] for x in logs])
    obs = ema([x["obs_rate"] for x in logs])
    obsSeen = ema([x["obs_seen_lick_rate"] for x in logs])
    eatWin = ema([x["eat_in_window_rate"] for x in logs])
    fig = plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(ep, succ)
    plt.title("EMA bout success rate")
    plt.xlabel("episode"); plt.ylabel("rate")
    plt.subplot(1, 3, 2)
    plt.plot(ep, obs, label="observe")
    plt.plot(ep, obsSeen, label="obs sees lick")
    plt.title("EMA observation metrics")
    plt.xlabel("episode"); plt.legend()
    plt.subplot(1, 3, 3)
    plt.plot(ep, eatWin)
    plt.title("EMA eat-in-window rate")
    plt.xlabel("episode"); plt.ylabel("rate")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'learning_curves.png'))
    plt.close()

def plot_da_peri_event(da_by_ep, traces, dt_s, model_name, event_key="reward", pre_s=5, post_s=8, save_dir='plots'):
    os.makedirs(save_dir, exist_ok=True)
    pre = int(round(pre_s / dt_s))
    post = int(round(post_s / dt_s))
    x = (np.arange(pre + post + 1) - pre) * dt_s
    n = len(traces)
    early_idx = range(0, max(1, n // 5)) # first 20%
    late_idx = range(max(0, n - n // 5), n) # last 20%
    def mean_peri(idxs):
        mats = []
        for i in idxs:
            ev = make_event_indices(traces[i])[event_key]
            m = peri_event_average(da_by_ep[i], ev, pre_steps=pre, post_steps=post)
            if m is not None:
                mats.append(m)
        if len(mats) == 0:
            return None
        return np.mean(np.stack(mats, axis=0), axis=0)
    e = mean_peri(early_idx)
    l = mean_peri(late_idx)
    fig = plt.figure(figsize=(6, 4))
    if e is not None:
        plt.plot(x, e, label="early (first 20%)")
    if l is not None:
        plt.plot(x, l, label="late (last 20%)")
    plt.axvline(0, linestyle="--")
    plt.title(f"{model_name}: peri-{event_key} DA")
    plt.xlabel("time (s)")
    plt.ylabel("simulated photometry")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{model_name}_peri_{event_key}.png'))
#     plt.close()

def simulate_and_plot_da(traces, agent, dt_s, gamma=0.99, save_dir='plots'):
    """
    Simulate DA for each episode trace under multiple models, filter to photometry,
    and plot peri-event averages around lick_end and reward.
    """
    # 1) build impulses per episode
    impulses_by_model = defaultdict(list)
    n = len(traces)
    for i, tr in enumerate(traces):
        ep_frac = i / max(1, n - 1)
        imp = simulate_da_models_for_episode(tr, agent, gamma=gamma, ep_frac=ep_frac)
        for k, v in imp.items():
            impulses_by_model[k].append(v)
    # 2) filter to photometry-like signals
    phot_by_model = {}
    for k, imps in impulses_by_model.items():
        phot_by_model[k] = [calcium_filter(x, dt_s=dt_s, tau_rise_s=0.2, tau_decay_s=1.2) for x in imps]
    # 3) plot peri-event for each model
    for k, da_eps in phot_by_model.items():
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="lick_end", pre_s=6, post_s=10, save_dir=save_dir)
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="reward", pre_s=6, post_s=10, save_dir=save_dir)
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="obs_seen_lick", pre_s=6, post_s=10, save_dir=save_dir)
    return phot_by_model

# ============================================================
# Run experiment
# ============================================================
def run_experiment(args):
    set_seed(args.seed)
    env = SocialLickEnv1D(
        grid_size=15,
        dt_s=0.5,
        max_time_s=120.0,
        teacher_period_s=30.0,
        teacher_jitter_s=6.0,
        lick_mean_s=1.0,
        lick_dist="gamma", # distribution, mean ~1s
        # lick_dist="lognormal", lick_cv=0.8,
        window_s=3.0,
        eat_cooldown_s=2.0,
        observe_cost=0.002,
        eat_cost=0.005,
        attend_bonus=0.004,
        p_detect_per_obs=0.70,
        win_rem_noise_steps=1,
    )
    agent = DQNAgent(state_dim=6, action_dim=5)
    logs, traces = train_social_with_latents(
        env, agent,
        episodes=args.episodes,
        log_every=max(100, args.episodes // 12),
        updates_per_step=4,
    )
    plot_learning_curves(logs)
    # Simulate DA under multiple models and plot early vs late peri-event signals
    phot_by_model = simulate_and_plot_da(traces, agent, dt_s=env.dt_s, gamma=agent.gamma)
    return logs, traces, phot_by_model

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Social Lick Environment DQN Training")
    parser.add_argument('--episodes', type=int, default=1200, help='Number of episodes')
    parser.add_argument('--seed', type=int, default=42, help='Random seed')
    args, unknown = parser.parse_known_args()
    run_experiment(args)

## 2D Final version

In [ ]:
# import numpy as np
# import random
# from collections import deque, defaultdict
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import seaborn as sns
# import matplotlib.pyplot as plt
# # -------------------------
# # Seed
# # -------------------------
# def set_seed(seed: int):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False
# def clamp_int(x: int, lo: int, hi: int) -> int:
#     return max(lo, min(hi, x))
# # -------------------------
# # Action space (7 actions for 2D)
# # 0 up, 1 down, 2 left, 3 right, 4 stay, 5 OBSERVE, 6 EAT
# # OBSERVE consumes time (no movement/eat)
# # -------------------------
# A_UP = 0
# A_DOWN = 1
# A_LEFT = 2
# A_RIGHT = 3
# A_STAY = 4
# A_OBS = 5
# A_EAT = 6
# # ============================================================
# # Environment (2D extension)
# # ============================================================
# class SocialLickEnv2D:
#     """
#     Extended to 2D grid for naturalistic movement.
#     dt_s = seconds per step (default 0.5s) for speed.
#     Teacher lick bout duration is sampled (Gamma or LogNormal) with mean ~ 1s, then discretized.
#     Reward window: 3s after the LAST lick step of the bout.
#       window_steps = round(3.0 / dt_s)
#     Learner must observe >= 1 step DURING licking to "register" that bout.
#     Social cues are only revealed on OBSERVE steps.
#     Success requires BOTH:
#       - social registration of the bout (detected_bout_id == last_lick_bout_id)
#       - at patch
#       - EAT within window
#     Episode ends at fixed horizon (NOT at first reward) so windows exist.
#     """
#     def __init__(
#         self,
#         grid_size=(9, 9),
#         dt_s=0.5,
#         # --- speed/learning signal ---
#         max_time_s=300.0, # longer to accommodate longer periods
#         teacher_period_s=60.0, # mean interval 60s
#         teacher_jitter_s=20.0, # jitter ±20s
#         # --- lick bout distribution ---
#         lick_mean_s=1.0,
#         lick_dist="gamma", # "gamma" or "lognormal"
#         lick_cv=0.8, # lognormal only
#         # --- reward window ---
#         window_s=3.0,
#         eat_cooldown_s=2.0,
#         # --- costs ---
#         observe_cost=0.002,
#         eat_cost=0.005,
#         # --- shaping (small) ---
#         attend_bonus=0.003,
#         # --- imperfect perception (more realistic) ---
#         base_p_detect=0.85, # base when dist=0
#         detect_sigma=3.0, # for exp decay with distance
#         win_rem_noise_steps=1,
#     ):
#         self.grid_width, self.grid_height = grid_size if isinstance(grid_size, tuple) else (grid_size, grid_size)
#         self.dt_s = float(dt_s)
#         self.max_steps = int(round(max_time_s / self.dt_s))
#         self.teacher_period_steps = int(round(teacher_period_s / self.dt_s))
#         self.teacher_jitter_steps = int(round(teacher_jitter_s / self.dt_s))
#         self.lick_mean_s = float(lick_mean_s)
#         self.lick_dist = str(lick_dist)
#         self.lick_cv = float(lick_cv)
#         self.window_steps = int(round(window_s / self.dt_s))
#         self.eat_cooldown_steps = int(round(eat_cooldown_s / self.dt_s))
#         self.observe_cost = float(observe_cost)
#         self.eat_cost = float(eat_cost)
#         self.attend_bonus = float(attend_bonus)
#         self.base_p_detect = float(base_p_detect)
#         self.detect_sigma = float(detect_sigma)
#         self.win_rem_noise_steps = int(win_rem_noise_steps)
#         assert self.teacher_period_steps > self.window_steps + 1
#         assert self.max_steps > self.teacher_period_steps + 5
#         self.reset()
#     def _sample_lick_duration_steps(self):
#         if self.lick_dist == "gamma":
#             shape = 2.0
#             scale = self.lick_mean_s / shape
#             dur_s = float(np.random.gamma(shape, scale))
#         elif self.lick_dist == "lognormal":
#             cv = max(1e-6, self.lick_cv)
#             sigma2 = np.log(1.0 + cv**2)
#             sigma = np.sqrt(sigma2)
#             mu = np.log(self.lick_mean_s) - 0.5 * sigma2
#             dur_s = float(np.random.lognormal(mean=mu, sigma=sigma))
#         else:
#             raise ValueError(f"Unknown lick_dist={self.lick_dist}")
#         steps = int(np.round(dur_s / self.dt_s))
#         steps = max(1, steps)
#         return steps, dur_s
#     def reset(self):
#         self.t = 0
#         # positions in 2D
#         self.teacher_pos = (int(np.random.randint(0, self.grid_width)), int(np.random.randint(0, self.grid_height)))
#         while True:
#             food_pos = (int(np.random.randint(0, self.grid_width)), int(np.random.randint(0, self.grid_height)))
#             if food_pos != self.teacher_pos:
#                 self.learner_food_pos = food_pos
#                 break
#         self.learner_pos = (int(np.random.randint(0, self.grid_width)), int(np.random.randint(0, self.grid_height)))
#         # cooldown
#         self.eat_cd = 0
#         # lick timing
#         self.last_lick_step = None
#         # social registration
#         self.detected_bout_id = None
#         self.rewarded_bout_ids = set()
#         # teacher lick timeline
#         self.bout_id_at_step = -np.ones(self.max_steps + 1, dtype=np.int32)
#         self.bout_start_steps = []
#         self.bout_end_steps = []
#         self.bout_dur_s_list = []
#         bout_id = 0
#         k = 1
#         while True:
#             nominal = k * self.teacher_period_steps
#             if nominal >= self.max_steps:
#                 break
#             jitter = int(np.random.randint(-self.teacher_jitter_steps, self.teacher_jitter_steps + 1))
#             start = clamp_int(nominal + jitter, 1, self.max_steps - 2)
#             dur_steps, dur_s = self._sample_lick_duration_steps()
#             end = clamp_int(start + dur_steps - 1, start, self.max_steps - 1)
#             self.bout_id_at_step[start:end + 1] = bout_id
#             self.bout_start_steps.append(start)
#             self.bout_end_steps.append(end)
#             self.bout_dur_s_list.append(dur_s)
#             bout_id += 1
#             k += 1
#         self.n_bouts = int(bout_id)
#         obs = self._get_obs(lick_sig=0.0, win_rem=0.0)
#         return obs
#     def _teacher_lick_now(self, t: int):
#         bid = int(self.bout_id_at_step[t]) if (0 <= t <= self.max_steps) else -1
#         return (1, bid) if bid >= 0 else (0, -1)
#     def _window_open(self, t: int) -> int:
#         if self.last_lick_step is None:
#             return 0
#         dt = t - self.last_lick_step
#         return 1 if (0 <= dt < self.window_steps) else 0
#     def _window_remaining(self, t: int) -> int:
#         if self.last_lick_step is None:
#             return 0
#         dt = t - self.last_lick_step
#         rem = self.window_steps - dt
#         return int(max(0, rem))
#     def _phase_to_next_nominal(self, t: int) -> int:
#         mod = t % self.teacher_period_steps
#         return self.teacher_period_steps - mod
#     def _get_obs(self, lick_sig: float, win_rem: float):
#         """
#         State features (dim=10 for 2D):
#           [learner_pos_x, learner_pos_y, food_pos_x, food_pos_y, teacher_pos_x, teacher_pos_y, lick_sig, win_rem, phase_to_next, eat_cd]
#         lick_sig/win_rem are ONLY non-zero when action==OBSERVE.
#         """
#         lp_x = float(self.learner_pos[0]) / float(self.grid_width - 1)
#         lp_y = float(self.learner_pos[1]) / float(self.grid_height - 1)
#         lf_x = float(self.learner_food_pos[0]) / float(self.grid_width - 1)
#         lf_y = float(self.learner_food_pos[1]) / float(self.grid_height - 1)
#         lt_x = float(self.teacher_pos[0]) / float(self.grid_width - 1)
#         lt_y = float(self.teacher_pos[1]) / float(self.grid_height - 1)
#         phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period_steps)
#         cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown_steps))
#         return np.array([lp_x, lp_y, lf_x, lf_y, lt_x, lt_y, float(lick_sig), float(win_rem), phase, cdn], dtype=np.float32)
#     def step(self, action: int):
#         self.t += 1
#         teacher_lick, bout_id = self._teacher_lick_now(self.t)
#         if teacher_lick == 1:
#             self.last_lick_step = self.t
#         window_open = self._window_open(self.t)
#         win_rem_true = self._window_remaining(self.t)
#         # identify which bout produced the most recent lick
#         if self.last_lick_step is None:
#             last_lick_bout_id = -1
#             dt_from_last_lick = 10**9
#         else:
#             last_lick_bout_id = int(self.bout_id_at_step[self.last_lick_step])
#             dt_from_last_lick = int(self.t - self.last_lick_step)
#         if self.eat_cd > 0:
#             self.eat_cd -= 1
#         lick_sig = 0.0
#         win_rem = 0.0
#         did_observe = 0
#         did_eat = 0
#         seen_lick = 0
#         if action == A_UP:
#             self.learner_pos = (self.learner_pos[0], self.learner_pos[1] + 1)
#         elif action == A_DOWN:
#             self.learner_pos = (self.learner_pos[0], self.learner_pos[1] - 1)
#         elif action == A_LEFT:
#             self.learner_pos = (self.learner_pos[0] - 1, self.learner_pos[1])
#         elif action == A_RIGHT:
#             self.learner_pos = (self.learner_pos[0] + 1, self.learner_pos[1])
#         elif action == A_OBS:
#             did_observe = 1
#             dist = abs(self.learner_pos[0] - self.teacher_pos[0]) + abs(self.learner_pos[1] - self.teacher_pos[1])
#             p_detect = self.base_p_detect * np.exp(-dist / self.detect_sigma)
#             p_detect = max(0.1, min(1.0, p_detect))  # clamp
#             if teacher_lick == 1 and bout_id >= 0 and (np.random.rand() < p_detect):
#                 seen_lick = 1
#                 self.detected_bout_id = int(bout_id)
#             lick_sig = float(seen_lick)
#             if win_rem_true > 0:
#                 noise = int(np.random.randint(-self.win_rem_noise_steps, self.win_rem_noise_steps + 1))
#                 est = clamp_int(win_rem_true + noise, 0, self.window_steps)
#                 win_rem = float(est) / float(max(1, self.window_steps))
#             else:
#                 win_rem = 0.0
#         elif action == A_EAT:
#             did_eat = 1
#         self.learner_pos = (
#             clamp_int(self.learner_pos[0], 0, self.grid_width - 1),
#             clamp_int(self.learner_pos[1], 0, self.grid_height - 1)
#         )
#         reward = 0.0
#         reward -= self.observe_cost * float(did_observe)
#         reward -= self.eat_cost * float(did_eat)
#         # small bonus: “trying to attend” during lick even if not detected
#         if did_observe == 1 and teacher_lick == 1 and bout_id >= 0 and seen_lick == 0:
#             reward += self.attend_bonus
#         eat_allowed = (did_eat == 1 and self.eat_cd == 0)
#         if eat_allowed:
#             self.eat_cd = self.eat_cooldown_steps
#         eat_valid = 0
#         if (
#             eat_allowed
#             and (self.learner_pos == self.learner_food_pos)
#             and (window_open == 1)
#             and (last_lick_bout_id >= 0)
#         ):
#             det = -1 if self.detected_bout_id is None else int(self.detected_bout_id)
#             if (det == last_lick_bout_id) and (last_lick_bout_id not in self.rewarded_bout_ids):
#                 eat_valid = 1
#                 self.rewarded_bout_ids.add(last_lick_bout_id)
#                 reward += 1.0
#         done = bool(self.t >= self.max_steps)
#         info = {
#             "t": self.t,
#             "teacher_lick": int(teacher_lick),
#             "bout_id": int(bout_id),
#             "window_open": int(window_open),
#             "win_rem_true": int(win_rem_true),
#             "last_lick_bout_id": int(last_lick_bout_id),
#             "dt_from_last_lick": int(dt_from_last_lick),
#             "action": int(action),
#             "observe": int(did_observe),
#             "seen_lick": int(seen_lick),
#             "eat": int(did_eat),
#             "eat_allowed": int(eat_allowed),
#             "eat_valid": int(eat_valid),
#             "learner_pos": self.learner_pos,
#             "food_pos": self.learner_food_pos,
#             "teacher_pos": self.teacher_pos,
#             "detected_bout_id": -1 if self.detected_bout_id is None else int(self.detected_bout_id),
#             "rewarded_bouts": len(self.rewarded_bout_ids),
#             "n_bouts": int(self.n_bouts),
#             "dt_s": float(self.dt_s),
#             "window_steps": int(self.window_steps),
#             "lick_dur_mean_s": float(np.mean(self.bout_dur_s_list) if len(self.bout_dur_s_list) else 0.0),
#         }
#         obs = self._get_obs(lick_sig=lick_sig, win_rem=win_rem)
#         return obs, reward, done, info
# # ============================================================
# # DQN (Dueling + Double + Huber + soft target) - optimized params for speed
# # ============================================================
# class DuelingQNet(nn.Module):
#     def __init__(self, state_dim, action_dim, hidden=128):
#         super().__init__()
#         self.backbone = nn.Sequential(
#             nn.Linear(state_dim, hidden),
#             nn.ReLU(),
#             nn.Linear(hidden, hidden),
#             nn.ReLU(),
#         )
#         self.V = nn.Linear(hidden, 1)
#         self.A = nn.Linear(hidden, action_dim)
#     def forward(self, x):
#         h = self.backbone(x)
#         v = self.V(h)
#         a = self.A(h)
#         return v + (a - a.mean(dim=1, keepdim=True))
# class FastReplayBuffer:
#     def __init__(self, capacity, state_dim):
#         self.capacity = int(capacity)
#         self.state = np.zeros((capacity, state_dim), dtype=np.float32)
#         self.action = np.zeros(capacity, dtype=np.int32)
#         self.reward = np.zeros(capacity, dtype=np.float32)
#         self.next_state = np.zeros((capacity, state_dim), dtype=np.float32)
#         self.done = np.zeros(capacity, dtype=np.float32)
#         self.idx = 0
#         self.size = 0
#     def push(self, s, a, r, s2, done):
#         self.state[self.idx] = s
#         self.action[self.idx] = a
#         self.reward[self.idx] = r
#         self.next_state[self.idx] = s2
#         self.done[self.idx] = done
#         self.idx = (self.idx + 1) % self.capacity
#         self.size = min(self.size + 1, self.capacity)
#     def sample(self, batch_size):
#         indices = np.random.randint(0, self.size, size=batch_size)
#         return (
#             self.state[indices],
#             self.action[indices],
#             self.reward[indices],
#             self.next_state[indices],
#             self.done[indices],
#         )
#     def __len__(self):
#         return self.size
# class DQNAgent:
#     def __init__(
#         self,
#         state_dim,
#         action_dim,
#         lr=1e-3,
#         gamma=0.99,
#         batch_size=256,
#         buffer_size=50000,
#         eps_start=1.0,
#         eps_end=0.05,
#         eps_decay=0.998,
#         tau=0.02,
#         grad_clip=10.0,
#         device=None,
#     ):
#         self.state_dim = int(state_dim)
#         self.action_dim = int(action_dim)
#         self.gamma = float(gamma)
#         self.batch_size = int(batch_size)
#         self.tau = float(tau)
#         self.grad_clip = float(grad_clip)
#         self.eps = float(eps_start)
#         self.eps_end = float(eps_end)
#         self.eps_decay = float(eps_decay)
#         self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
#         self.q = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
#         self.qt = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
#         self.qt.load_state_dict(self.q.state_dict())
#         self.opt = optim.Adam(self.q.parameters(), lr=float(lr))
#         self.loss_fn = nn.SmoothL1Loss()
#         self.rb = FastReplayBuffer(buffer_size, state_dim)
#     def act(self, s):
#         if np.random.rand() < self.eps:
#             return np.random.randint(self.action_dim)
#         return self.act_greedy(s)
#     def act_greedy(self, s):
#         st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
#         with torch.no_grad():
#             qv = self.q(st)
#         return int(torch.argmax(qv, dim=1).item())
#     def entropy(self, s):
#         st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
#         with torch.no_grad():
#             qv = self.q(st)
#         p = torch.softmax(qv / 1.0, dim=1)[0]
#         ent = - (p * torch.log(p + 1e-6)).sum().item()
#         return ent
#     def push(self, s, a, r, s2, done):
#         self.rb.push(s, int(a), float(r), s2, float(done))
#     def update(self, updates_per_step=1):
#         if len(self.rb) < self.batch_size:
#             return None
#         last_loss = None
#         for _ in range(int(updates_per_step)):
#             if len(self.rb) < self.batch_size:
#                 break
#             s, a, r, s2, d = self.rb.sample(self.batch_size)
#             s = torch.tensor(s, dtype=torch.float32, device=self.device)
#             a = torch.tensor(a, dtype=torch.int64, device=self.device).unsqueeze(1)
#             r = torch.tensor(r, dtype=torch.float32, device=self.device)
#             s2 = torch.tensor(s2, dtype=torch.float32, device=self.device)
#             d = torch.tensor(d, dtype=torch.float32, device=self.device)
#             q_sa = self.q(s).gather(1, a).squeeze(1)
#             with torch.no_grad():
#                 a_star = torch.argmax(self.q(s2), dim=1, keepdim=True) # Double DQN
#                 q_next = self.qt(s2).gather(1, a_star).squeeze(1)
#                 target = r + self.gamma * q_next * (1.0 - d)
#             loss = self.loss_fn(q_sa, target)
#             self.opt.zero_grad()
#             loss.backward()
#             nn.utils.clip_grad_norm_(self.q.parameters(), self.grad_clip)
#             self.opt.step()
#             with torch.no_grad():
#                 for p, pt in zip(self.q.parameters(), self.qt.parameters()):
#                     pt.data.mul_(1.0 - self.tau).add_(self.tau * p.data)
#             last_loss = float(loss.item())
#         self.eps = max(self.eps_end, self.eps * self.eps_decay)
#         return last_loss
# # ============================================================
# # Random control (no training)
# # ============================================================
# class RandomPolicy:
#     def __init__(self, p_obs=0.06, p_eat=0.06):
#         self.p_obs = float(p_obs)
#         self.p_eat = float(p_eat)
#     def act(self, s):
#         if np.random.rand() < self.p_obs:
#             return A_OBS
#         if np.random.rand() < self.p_eat:
#             return A_EAT
#         return np.random.choice([A_UP, A_DOWN, A_LEFT, A_RIGHT, A_STAY])
# # ============================================================
# # Helpers
# # ============================================================
# def ema(x, alpha=0.03):
#     x = np.asarray(x, dtype=np.float32)
#     y = np.zeros_like(x)
#     m = 0.0
#     for i in range(len(x)):
#         m = (1 - alpha) * m + alpha * x[i]
#         y[i] = m
#     return y
# def observe_run_stats(obs_arr):
#     obs = np.asarray(obs_arr, dtype=np.int32)
#     runs = []
#     cur = 0
#     for v in obs:
#         if v == 1:
#             cur += 1
#         else:
#             if cur > 0:
#                 runs.append(cur)
#             cur = 0
#     if cur > 0:
#         runs.append(cur)
#     if len(runs) == 0:
#         return 0.0, 0.0
#     return float(np.mean(runs)), float(np.median(runs))
# def bin_mean_sem(data, bin_size=50):
#     means = []
#     sems = []
#     for i in range(0, len(data), bin_size):
#         chunk = data[i:i+bin_size]
#         if len(chunk) == 0:
#             break
#         m = np.mean(chunk)
#         s = np.std(chunk) / np.sqrt(len(chunk))
#         means.append(m)
#         sems.append(s)
#     return np.array(means), np.array(sems)
# # ============================================================
# # Training with full behavior breakdown - added latency metric
# # ============================================================
# def train_social_with_traces(
#     env,
#     agent,
#     episodes=1500,  # increased to 1500
#     log_every=100,
#     updates_per_step=8, # increased for faster convergence
#     trace_stride=4, # store 1/4 episodes for plots & DA sim (saves memory, still enough)
# ):
#     logs = []
#     traces = []
#     for ep in range(1, episodes + 1):
#         s = env.reset()
#         done = False
#         steps = 0
#         ep_reward = 0.0
#         ep_entropy = 0.0
#         n_entropy = 0
#         # ===== step-context denominators =====
#         n_active = 0 # teacher_lick==1
#         n_window = 0 # window_open==1 & teacher_lick==0
#         n_passive = 0 # else
#         # ===== eat numerators by context =====
#         eat_active = 0
#         eat_window = 0
#         eat_passive = 0
#         # ===== window split by detected/not =====
#         n_win_det = 0
#         n_win_nodet = 0
#         eat_win_det = 0
#         eat_win_nodet = 0
#         # ===== observation alignment =====
#         obs_steps = 0
#         obs_at_lick = 0
#         obs_seen = 0
#         # ===== eat success =====
#         succ_bouts = 0
#         n_bouts = max(1, env.n_bouts)
#         # ===== EAT-run start timing =====
#         prev_eat = 0
#         eat_run_start_active = 0 # starts during lick
#         eat_run_start_window = 0 # starts after lick (in window)
#         eat_run_start_passive = 0
#         eat_run_starts = 0
#         # ===== additional metrics =====
#         latencies = []  # dt_from_last_lick for successful eats
#         store_trace = ((ep % trace_stride) == 0)
#         tr = defaultdict(list)
#         # for bout-level conditional probabilities
#         bout_detect = defaultdict(int) # bout_id -> detected?
#         bout_reward = defaultdict(int) # bout_id -> rewarded?
#         bout_eat_attempt = defaultdict(int)  # bout_id -> attempted eat in window
#         while not done:
#             a = agent.act(s)
#             s2, r, done, info = env.step(a)
#             steps += 1
#             ep_reward += r
#             if a != agent.act_greedy(s):  # only compute entropy for greedy actions
#                 ep_entropy += agent.entropy(s)
#                 n_entropy += 1
#             teacher_lick = info["teacher_lick"]
#             window_open = info["window_open"]
#             last_bid = info["last_lick_bout_id"]
#             det_bid = info["detected_bout_id"]
#             # ---- step context classification ----
#             if teacher_lick == 1:
#                 n_active += 1
#             elif window_open == 1:
#                 n_window += 1
#             else:
#                 n_passive += 1
#             # ---- observation stats ----
#             if info["observe"] == 1:
#                 obs_steps += 1
#                 if teacher_lick == 1:
#                     obs_at_lick += 1
#                 if info["seen_lick"] == 1:
#                     obs_seen += 1
#                     # seen_lick only happens during lick; bout_id is valid
#                     if info["bout_id"] >= 0:
#                         bout_detect[int(info["bout_id"])] = 1
#             # ---- EAT by context ----
#             did_eat = info["eat"]
#             if did_eat == 1:
#                 if teacher_lick == 1:
#                     eat_active += 1
#                 elif window_open == 1:
#                     eat_window += 1
#                 else:
#                     eat_passive += 1
#             # ---- window detected vs not-detected ----
#             if (teacher_lick == 0) and (window_open == 1) and (last_bid >= 0):
#                 if det_bid == last_bid:
#                     n_win_det += 1
#                     if did_eat == 1:
#                         eat_win_det += 1
#                         bout_eat_attempt[last_bid] = 1
#                 else:
#                     n_win_nodet += 1
#                     if did_eat == 1:
#                         eat_win_nodet += 1
#                         bout_eat_attempt[last_bid] = 1
#             # ---- reward / bout success ----
#             if info["eat_valid"] == 1:
#                 succ_bouts += 1
#                 latencies.append(info["dt_from_last_lick"])
#                 if last_bid >= 0:
#                     bout_reward[int(last_bid)] = 1
#             # ---- EAT-run start timing (start of consecutive EAT steps) ----
#             if (did_eat == 1) and (prev_eat == 0):
#                 eat_run_starts += 1
#                 if teacher_lick == 1:
#                     eat_run_start_active += 1 # starts before lick-end (during lick)
#                 elif window_open == 1:
#                     eat_run_start_window += 1 # starts after lick-end (window)
#                 else:
#                     eat_run_start_passive += 1
#             prev_eat = did_eat
#             # ---- trace storage ----
#             if store_trace:
#                 tr["t"].append(info["t"])
#                 tr["learner_pos_x"].append(info["learner_pos"][0])
#                 tr["learner_pos_y"].append(info["learner_pos"][1])
#                 tr["food_pos_x"].append(info["food_pos"][0])
#                 tr["food_pos_y"].append(info["food_pos"][1])
#                 tr["teacher_lick"].append(info["teacher_lick"])
#                 tr["window_open"].append(info["window_open"])
#                 tr["observe"].append(info["observe"])
#                 tr["seen_lick"].append(info["seen_lick"])
#                 tr["eat"].append(info["eat"])
#                 tr["eat_valid"].append(info["eat_valid"])
#             agent.push(s, a, r, s2, done)
#             agent.update(updates_per_step=updates_per_step)
#             s = s2
#         if store_trace:
#             for k in list(tr.keys()):
#                 tr[k] = np.array(tr[k])
#             traces.append(tr)
#             obs_mean_run, obs_med_run = observe_run_stats(tr["observe"])
#         else:
#             obs_mean_run, obs_med_run = 0.0, 0.0
#         avg_entropy = ep_entropy / max(1, n_entropy)
#         # ===== bout-level conditionals =====
#         # P(reward | detected) and P(reward | not detected)
#         det_list = []
#         rew_list = []
#         attempt_list = []
#         for bid in range(env.n_bouts):
#             det = bout_detect.get(bid, 0)
#             rew = bout_reward.get(bid, 0)
#             attempt = bout_eat_attempt.get(bid, 0)
#             det_list.append(det)
#             rew_list.append(rew)
#             attempt_list.append(attempt)
#         det_list = np.array(det_list, dtype=np.int32)
#         rew_list = np.array(rew_list, dtype=np.int32)
#         attempt_list = np.array(attempt_list, dtype=np.int32)
#         n_det = int(det_list.sum())
#         n_nodet = int((1 - det_list).sum())
#         p_rew_given_det = float(rew_list[det_list == 1].mean()) if n_det > 0 else 0.0
#         p_rew_given_nodet = float(rew_list[det_list == 0].mean()) if n_nodet > 0 else 0.0
#         # additional: P(attempt | det), P(attempt | nodet) for eat in window
#         p_attempt_given_det = float(attempt_list[det_list == 1].mean()) if n_det > 0 else 0.0
#         p_attempt_given_nodet = float(attempt_list[det_list == 0].mean()) if n_nodet > 0 else 0.0
#         logs.append({
#             "ep": ep,
#             "bout_success_rate": float(succ_bouts / n_bouts),
#             "mean_reward": float(ep_reward),
#             # observation
#             "observe_rate": float(obs_steps / max(1, steps)),
#             "obs_at_lick_rate": float(obs_at_lick / max(1, steps)),
#             "obs_seen_lick_rate": float(obs_seen / max(1, steps)),
#             # EAT probabilities by context (per-step conditional)
#             "p_eat_active": float(eat_active / max(1, n_active)),
#             "p_eat_window": float(eat_window / max(1, n_window)),
#             "p_eat_passive": float(eat_passive / max(1, n_passive)),
#             # window split (this is your “active social learning” signature)
#             "p_eat_win_detected": float(eat_win_det / max(1, n_win_det)),
#             "p_eat_win_notdet": float(eat_win_nodet / max(1, n_win_nodet)),
#             # bout-level conditional rewards
#             "p_rew_given_det": float(p_rew_given_det),
#             "p_rew_given_nodet": float(p_rew_given_nodet),
#             # bout-level eat attempts
#             "p_attempt_given_det": float(p_attempt_given_det),
#             "p_attempt_given_nodet": float(p_attempt_given_nodet),
#             # EAT-run start timing
#             "frac_eat_run_start_active": float(eat_run_start_active / max(1, eat_run_starts)),
#             "frac_eat_run_start_window": float(eat_run_start_window / max(1, eat_run_starts)),
#             "frac_eat_run_start_passive": float(eat_run_start_passive / max(1, eat_run_starts)),
#             "obs_run_mean_steps": float(obs_mean_run),
#             "obs_run_median_steps": float(obs_med_run),
#             # new metrics
#             "mean_eat_latency_steps": float(np.mean(latencies)) if latencies else 0.0,
#             "avg_policy_entropy": float(avg_entropy),
#             "eps": float(agent.eps),
#             "lick_dur_mean_s": float(np.mean(env.bout_dur_s_list) if len(env.bout_dur_s_list) else 0.0),
#             "dt_s": float(env.dt_s),
#         })
#         if ep % log_every == 0:
#             recent = logs[-log_every:]
#             print(
#                 f"[SOC ep {ep:4d}] "
#                 f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f} "
#                 f"eatWin(det)={np.mean([x['p_eat_win_detected'] for x in recent]):.3f} "
#                 f"eatWin(!det)={np.mean([x['p_eat_win_notdet'] for x in recent]):.3f} "
#                 f"P(rew|det)={np.mean([x['p_rew_given_det'] for x in recent]):.3f} "
#                 f"P(rew|!det)={np.mean([x['p_rew_given_nodet'] for x in recent]):.3f} "
#                 f"P(attempt|det)={np.mean([x['p_attempt_given_det'] for x in recent]):.3f} "
#                 f"latency={np.mean([x['mean_eat_latency_steps'] for x in recent]):.3f} "
#                 f"entropy={np.mean([x['avg_policy_entropy'] for x in recent]):.3f} "
#                 f"obs={np.mean([x['observe_rate'] for x in recent]):.3f} "
#                 f"eps={agent.eps:.3f}"
#             )
#     return logs, traces
# def run_random_control(env, policy, episodes=1500, log_every=100):
#     logs = []
#     for ep in range(1, episodes + 1):
#         s = env.reset()
#         done = False
#         succ_bouts = 0
#         n_bouts = max(1, env.n_bouts)
#         ep_reward = 0.0
#         while not done:
#             a = policy.act(s)
#             s, r, done, info = env.step(a)
#             ep_reward += r
#             if info["eat_valid"] == 1:
#                 succ_bouts += 1
#         logs.append({
#             "ep": ep,
#             "bout_success_rate": float(succ_bouts / n_bouts),
#             "mean_reward": float(ep_reward),
#         })
#         if ep % log_every == 0:
#             recent = logs[-log_every:]
#             print(
#                 f"[RND ep {ep:4d}] "
#                 f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f} "
#                 f"R={np.mean([x['mean_reward'] for x in recent]):.3f}"
#             )
#     return logs
# # ============================================================
# # DA Simulation - batched
# # ============================================================
# def simulate_da(agent, env, n_ep=20, hypotheses=["rpe", "unsigned_pe", "policy_learning"]):
#     traces = []
#     for _ in range(n_ep):
#         s = env.reset()
#         done = False
#         tr = defaultdict(list)
#         states = []
#         actions = []
#         rewards = []
#         qs = []
#         ents = []
#         states.append(s.copy())
#         while not done:
#             a = agent.act_greedy(s)
#             actions.append(a)
#             st = torch.tensor(s, dtype=torch.float32, device=agent.device).unsqueeze(0)
#             with torch.no_grad():
#                 qv = agent.q(st)
#                 q_sa = qv[0, a].item()
#                 p = torch.softmax(qv / 1.0, dim=1)[0]
#                 ent = - (p * torch.log(p + 1e-6)).sum().item()
#             qs.append(q_sa)
#             ents.append(ent)
#             s2, r, done, info = env.step(a)
#             rewards.append(r)
#             tr["t"].append(info["t"])
#             tr["learner_pos_x"].append(info["learner_pos"][0])
#             tr["learner_pos_y"].append(info["learner_pos"][1])
#             tr["food_pos_x"].append(info["food_pos"][0])
#             tr["food_pos_y"].append(info["food_pos"][1])
#             tr["teacher_lick"].append(info["teacher_lick"])
#             tr["window_open"].append(info["window_open"])
#             tr["observe"].append(info["observe"])
#             tr["seen_lick"].append(info["seen_lick"])
#             tr["eat"].append(info["eat"])
#             tr["eat_valid"].append(info["eat_valid"])
#             tr["entropy"].append(ent)
#             s = s2
#             states.append(s.copy())
#         states = np.array(states)
#         rewards = np.array(rewards)
#         actions = np.array(actions)
#         qs = np.array(qs)
#         ents = np.array(ents)
#         st2 = torch.tensor(states[1:], dtype=torch.float32, device=agent.device)
#         with torch.no_grad():
#             max_q_next = agent.q(st2).max(1)[0].cpu().numpy()
#         rpe = rewards + agent.gamma * max_q_next - qs
#         for hyp in hypotheses:
#             if hyp == "rpe":
#                 tr["DA_" + hyp] = np.concatenate([rpe, [0.0]])
#             elif hyp == "unsigned_pe":
#                 tr["DA_" + hyp] = np.concatenate([np.abs(rpe), [0.0]])
#             elif hyp == "policy_learning":
#                 tr["DA_" + hyp] = np.concatenate([ents, [0.0]])  # using entropy as proxy for policy uncertainty/learning
#         tr["entropy"] = np.concatenate([ents, [0.0]])
#         for k in list(tr.keys()):
#             tr[k] = np.array(tr[k])
#         traces.append(tr)
#     return traces
# # ============================================================
# # Visualization - use seaborn for better plots, added bin_sem for error bars
# # ============================================================
# def plot_learning(logs_social, logs_random=None, bin_size=50):
#     def get(logs, key):
#         return np.array([x[key] for x in logs], dtype=np.float32)
#     fig, axs = plt.subplots(2, 2, figsize=(13, 10))
#     # Bout success
#     ax = axs[0, 0]
#     data_soc = get(logs_social, "bout_success_rate")
#     means, sems = bin_mean_sem(data_soc, bin_size)
#     x = np.arange(len(means)) * bin_size + bin_size / 2
#     ax.plot(x, means, label="social")
#     ax.fill_between(x, means - sems, means + sems, alpha=0.2)
#     if logs_random is not None:
#         data_rnd = get(logs_random, "bout_success_rate")
#         means_r, sems_r = bin_mean_sem(data_rnd, bin_size)
#         ax.plot(x, means_r, label="random control")
#         ax.fill_between(x, means_r - sems_r, means_r + sems_r, alpha=0.2)
#     ax.set_title("Binned bout success rate (with SEM)")
#     ax.set_xlabel("episode")
#     ax.set_ylabel("rate")
#     ax.legend()
#     # Observation alignment
#     ax = axs[0, 1]
#     for key, label in [("observe_rate", "observe_rate"), ("obs_at_lick_rate", "obs@lick"), ("obs_seen_lick_rate", "obs detects lick")]:
#         data = get(logs_social, key)
#         means, sems = bin_mean_sem(data, bin_size)
#         ax.plot(x, means, label=label)
#         ax.fill_between(x, means - sems, means + sems, alpha=0.2)
#     ax.set_title("Binned observation alignment (with SEM)")
#     ax.set_xlabel("episode")
#     ax.set_ylabel("rate")
#     ax.legend()
#     # Eat prob by context
#     ax = axs[1, 0]
#     for key, label in [("p_eat_active", "P(EAT | active lick)"), ("p_eat_window", "P(EAT | window)"), ("p_eat_passive", "P(EAT | passive)")]:
#         data = get(logs_social, key)
#         means, sems = bin_mean_sem(data, bin_size)
#         ax.plot(x, means, label=label)
#         ax.fill_between(x, means - sems, means + sems, alpha=0.2)
#     ax.set_title("Eat probability by state (per-step conditional)")
#     ax.set_xlabel("episode")
#     ax.set_ylabel("prob")
#     ax.legend()
#     # Social learning signature
#     ax = axs[1, 1]
#     for key, label in [("p_eat_win_detected", "P(EAT | window & detected)"), ("p_eat_win_notdet", "P(EAT | window & NOT detected)"),
#                        ("p_rew_given_det", "P(reward | detected bout)"), ("p_rew_given_nodet", "P(reward | NOT detected bout)"),
#                        ("p_attempt_given_det", "P(attempt | detected bout)"), ("p_attempt_given_nodet", "P(attempt | NOT detected bout)")]:
#         data = get(logs_social, key)
#         means, sems = bin_mean_sem(data, bin_size)
#         ax.plot(x, means, label=label)
#         ax.fill_between(x, means - sems, means + sems, alpha=0.2)
#     ax.set_title("The social-learning signature (should separate)")
#     ax.set_xlabel("episode")
#     ax.set_ylabel("prob")
#     ax.legend()
#     plt.tight_layout()
#     plt.show()
# def plot_eat_run_starts(logs_social, bin_size=50):
#     def get(key):
#         return np.array([x[key] for x in logs_social], dtype=np.float32)
#     fig, ax = plt.subplots(figsize=(10, 4))
#     for key, label in [("frac_eat_run_start_active", "EAT-run starts during lick (before lick end)"),
#                        ("frac_eat_run_start_window", "EAT-run starts in window (after lick end)"),
#                        ("frac_eat_run_start_passive", "EAT-run starts passive")]:
#         data = get(key)
#         means, sems = bin_mean_sem(data, bin_size)
#         x = np.arange(len(means)) * bin_size + bin_size / 2
#         ax.plot(x, means, label=label)
#         ax.fill_between(x, means - sems, means + sems, alpha=0.2)
#     ax.set_title("When does the learner *start* an eating attempt sequence? (with SEM)")
#     ax.set_xlabel("episode")
#     ax.set_ylabel("fraction")
#     ax.legend()
#     plt.tight_layout()
#     plt.show()
# def peri_lick_alignment(traces, max_dt_steps=20):
#     dt_range = np.arange(-max_dt_steps, max_dt_steps + 1)
#     obs_counts = np.zeros_like(dt_range, dtype=np.float32)
#     eat_counts = np.zeros_like(dt_range, dtype=np.float32)
#     patch_counts = np.zeros_like(dt_range, dtype=np.float32)
#     da_sum = np.zeros_like(dt_range, dtype=np.float32)
#     ent_sum = np.zeros_like(dt_range, dtype=np.float32)
#     denom = np.zeros_like(dt_range, dtype=np.float32)
#     for tr in traces:
#         t = np.array(tr["t"], dtype=int)
#         lick = np.array(tr["teacher_lick"], dtype=int)
#         obs = np.array(tr["observe"], dtype=int)
#         eat = np.array(tr["eat"], dtype=int)
#         lp_x = np.array(tr["learner_pos_x"], dtype=int)
#         lp_y = np.array(tr["learner_pos_y"], dtype=int)
#         fp_x = np.array(tr["food_pos_x"], dtype=int)
#         fp_y = np.array(tr["food_pos_y"], dtype=int)
#         da = np.array(tr.get("DA_rpe", np.zeros_like(t)), dtype=np.float32)
#         ent = np.array(tr.get("entropy", np.zeros_like(t)), dtype=np.float32)
#         lick_ts = t[lick == 1]
#         for L in lick_ts:
#             for i, dt in enumerate(dt_range):
#                 tt = L + dt
#                 idx = np.where(t == tt)[0]
#                 if len(idx) == 0:
#                     continue
#                 j = idx[0]
#                 denom[i] += 1.0
#                 obs_counts[i] += float(obs[j] == 1)
#                 eat_counts[i] += float(eat[j] == 1)
#                 patch_counts[i] += float((lp_x[j] == fp_x[j]) and (lp_y[j] == fp_y[j]))
#                 da_sum[i] += da[j]
#                 ent_sum[i] += ent[j]
#     denom = np.maximum(denom, 1.0)
#     return dt_range, obs_counts / denom, eat_counts / denom, patch_counts / denom, da_sum / denom, ent_sum / denom
# def plot_peri_lick(traces, dt_s, max_dt_steps=20):
#     dt, p_obs, p_eat, p_patch, mean_da, mean_ent = peri_lick_alignment(traces, max_dt_steps=max_dt_steps)
#     dt_sec = dt * float(dt_s)
#     fig, axs = plt.subplots(1, 5, figsize=(20, 4))
#     axs[0].plot(dt_sec, p_obs)
#     axs[0].axvline(0, linestyle="--")
#     axs[0].set_title("P(OBSERVE) aligned to lick")
#     axs[0].set_xlabel("dt (s)")
#     axs[0].set_ylabel("prob")
#     axs[1].plot(dt_sec, p_eat)
#     axs[1].axvline(0, linestyle="--")
#     axs[1].set_title("P(EAT) aligned to lick")
#     axs[1].set_xlabel("dt (s)")
#     axs[1].set_ylabel("prob")
#     axs[2].plot(dt_sec, p_patch)
#     axs[2].axvline(0, linestyle="--")
#     axs[2].set_title("P(at patch) aligned to lick")
#     axs[2].set_xlabel("dt (s)")
#     axs[2].set_ylabel("prob")
#     axs[3].plot(dt_sec, mean_da)
#     axs[3].axvline(0, linestyle="--")
#     axs[3].set_title("Mean DA (RPE) aligned to lick")
#     axs[3].set_xlabel("dt (s)")
#     axs[3].set_ylabel("DA signal")
#     axs[4].plot(dt_sec, mean_ent)
#     axs[4].axvline(0, linestyle="--")
#     axs[4].set_title("Mean policy entropy aligned to lick")
#     axs[4].set_xlabel("dt (s)")
#     axs[4].set_ylabel("entropy")
#     plt.tight_layout()
#     plt.show()
# def plot_raster(tr, dt_s, title="Episode raster"):
#     t = np.array(tr["t"], dtype=int)
#     tt = t * float(dt_s)
#     lp_x = np.array(tr["learner_pos_x"], dtype=float)
#     lp_y = np.array(tr["learner_pos_y"], dtype=float)
#     fp_x = np.array(tr["food_pos_x"], dtype=float)
#     fp_y = np.array(tr["food_pos_y"], dtype=float)
#     lick = np.array(tr["teacher_lick"], dtype=int)
#     win = np.array(tr["window_open"], dtype=int)
#     obs = np.array(tr["observe"], dtype=int)
#     eat = np.array(tr["eat"], dtype=int)
#     succ = np.array(tr["eat_valid"], dtype=int)
#     fig, ax = plt.subplots(figsize=(11, 4))
#     idx = np.where(win == 1)[0]
#     if len(idx) > 0:
#         start = idx[0]
#         for i in range(1, len(idx)):
#             if idx[i] != idx[i - 1] + 1:
#                 ax.axvspan(tt[start], tt[idx[i - 1]] + dt_s, alpha=0.12)
#                 start = idx[i]
#         ax.axvspan(tt[start], tt[idx[-1]] + dt_s, alpha=0.12)
#     # Plot x and y separately or trajectory
#     ax.plot(tt, lp_x, label="learner_pos_x")
#     ax.plot(tt, lp_y, label="learner_pos_y")
#     ax.plot(tt, np.full_like(tt, fp_x[0]), '--', label="food_x")
#     ax.plot(tt, np.full_like(tt, fp_y[0]), '--', label="food_y")
#     lidx = np.where(lick == 1)[0]
#     if len(lidx) > 0:
#         ax.scatter(tt[lidx], lp_x[lidx], marker="x", label="teacher lick")
#     oidx = np.where(obs == 1)[0]
#     if len(oidx) > 0:
#         ax.scatter(tt[oidx], lp_x[oidx], marker="o", label="OBSERVE")
#     eidx = np.where(eat == 1)[0]
#     if len(eidx) > 0:
#         ax.scatter(tt[eidx], lp_x[eidx], marker="+", label="EAT")
#     sidx = np.where(succ == 1)[0]
#     if len(sidx) > 0:
#         ax.scatter(tt[sidx], lp_x[sidx], marker="*", s=150, label="SUCCESS")
#     ax.set_title(title)
#     ax.set_xlabel("time (s)")
#     ax.set_ylabel("position")
#     ax.legend()
#     plt.tight_layout()
#     plt.show()
# # ============================================================
# # Run experiment
# # ============================================================
# def run_experiment(seed=42, episodes=1500):
#     set_seed(seed)
#     env = SocialLickEnv2D(
#         grid_size=(9, 9),
#         dt_s=0.5,
#         # SPEED / learning signal
#         max_time_s=300.0,
#         teacher_period_s=60.0,
#         teacher_jitter_s=20.0,
#         lick_mean_s=1.0,
#         lick_dist="gamma", # distributed durations
#         window_s=3.0,
#         eat_cooldown_s=2.0,
#         observe_cost=0.002,
#         eat_cost=0.005,
#         attend_bonus=0.003,
#         base_p_detect=0.85,
#         detect_sigma=3.0,
#         win_rem_noise_steps=1,
#     )
#     agent = DQNAgent(state_dim=10, action_dim=7, lr=1e-3, batch_size=256, buffer_size=50000, eps_decay=0.998)
#     logs_social, traces = train_social_with_traces(
#         env, agent,
#         episodes=episodes,
#         log_every=max(100, episodes // 12),
#         updates_per_step=8,
#         trace_stride=4,
#     )
#     rnd = RandomPolicy(p_obs=0.06, p_eat=0.06)
#     logs_random = run_random_control(
#         env, rnd,
#         episodes=episodes,
#         log_every=max(100, episodes // 12),
#     )
#     plot_learning(logs_social, logs_random)
#     plot_eat_run_starts(logs_social)
#     if len(traces) > 0:
#         plot_peri_lick(traces[-10:], dt_s=env.dt_s, max_dt_steps=20)
#         for i in range(min(3, len(traces))):
#             plot_raster(traces[-(i + 1)], dt_s=env.dt_s, title=f"Recent episode raster #{i+1}")
#     # DA simulation
#     test_traces = simulate_da(agent, env, n_ep=20, hypotheses=["rpe", "unsigned_pe", "policy_learning"])
#     plot_peri_lick(test_traces, dt_s=env.dt_s, max_dt_steps=20)
#     return logs_social, logs_random, traces
# if __name__ == "__main__":
#     run_experiment(seed=42, episodes=1500)

# Multiple models

In [ ]:
from typing import Optional
import numpy as np
import random
from collections import deque, defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import argparse
import os
import math

# -------------------------
# Seed / utils
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def clamp_int(x: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, x))

# -------------------------
# Actions
# -------------------------
A_LEFT = 0
A_STAY = 1
A_RIGHT = 2
A_OBS = 3  # consumes time; reveals social timing info stochastically
A_EAT = 4

# ============================================================
# Environment (same logic as your current code)
# ============================================================
class SocialLickEnv1D:
    """
    Fast but detailed (paper-aligned):
      dt_s = 0.5s/step
      teacher cue bursts ("lick bouts") with stochastic duration (mean ~ 1s)
      reward window: 3s after last cue step
    Observation gating:
      - social signal (cue + noisy window remaining) only appears when action==OBS
      - detection during cue burst is probabilistic per OBS step
    """
    def __init__(
        self,
        grid_size=15,
        dt_s=0.5,
        max_time_s=120.0,
        teacher_period_s=30.0,
        teacher_jitter_s=6.0,
        lick_mean_s=1.0,
        lick_dist="gamma",  # "gamma" or "lognormal"
        lick_cv=0.8,        # for lognormal
        window_s=3.0,
        eat_cooldown_s=2.0,
        observe_cost=0.002,
        eat_cost=0.005,
        attend_bonus=0.004,
        p_detect_per_obs=0.70,
        win_rem_noise_steps=1,
    ):
        self.grid_size = int(grid_size)
        self.dt_s = float(dt_s)
        self.max_steps = int(round(max_time_s / self.dt_s))
        self.teacher_period_steps = int(round(teacher_period_s / self.dt_s))
        self.teacher_jitter_steps = int(round(teacher_jitter_s / self.dt_s))
        self.lick_mean_s = float(lick_mean_s)
        self.lick_dist = str(lick_dist)
        self.lick_cv = float(lick_cv)
        self.window_steps = int(round(window_s / self.dt_s))
        self.eat_cooldown_steps = int(round(eat_cooldown_s / self.dt_s))
        self.observe_cost = float(observe_cost)
        self.eat_cost = float(eat_cost)
        self.attend_bonus = float(attend_bonus)
        self.p_detect_per_obs = float(p_detect_per_obs)
        self.win_rem_noise_steps = int(win_rem_noise_steps)

        assert self.teacher_period_steps > self.window_steps + 1
        assert self.max_steps > self.teacher_period_steps + 5
        self.reset()

    def _sample_lick_duration_steps(self):
        if self.lick_dist == "gamma":
            shape = 2.0
            scale = self.lick_mean_s / shape
            dur_s = float(np.random.gamma(shape, scale))
        elif self.lick_dist == "lognormal":
            cv = max(1e-6, self.lick_cv)
            sigma2 = np.log(1.0 + cv**2)
            sigma = np.sqrt(sigma2)
            mu = np.log(self.lick_mean_s) - 0.5 * sigma2
            dur_s = float(np.random.lognormal(mean=mu, sigma=sigma))
        else:
            raise ValueError(f"Unknown lick_dist={self.lick_dist}")
        steps = int(np.round(dur_s / self.dt_s))
        return max(1, steps), dur_s

    def reset(self):
        self.t = 0
        self.teacher_pos = int(np.random.randint(0, self.grid_size))
        candidates = [i for i in range(self.grid_size) if i != self.teacher_pos]
        self.learner_food_pos = int(np.random.choice(candidates))
        self.learner_pos = int(np.random.randint(0, self.grid_size))

        self.eat_cd = 0
        self.last_lick_step = None
        self.detected_bout_id = None
        self.rewarded_bout_ids = set()

        # build cue-burst timeline
        self.bout_id_at_step = -np.ones(self.max_steps + 1, dtype=np.int32)
        self.bout_start_steps = []
        self.bout_end_steps = []
        self.bout_dur_s_list = []
        bout_id = 0
        k = 1
        while True:
            nominal = k * self.teacher_period_steps
            if nominal >= self.max_steps:
                break
            jitter = int(np.random.randint(-self.teacher_jitter_steps, self.teacher_jitter_steps + 1))
            start = clamp_int(nominal + jitter, 1, self.max_steps - 2)
            dur_steps, dur_s = self._sample_lick_duration_steps()
            end = clamp_int(start + dur_steps - 1, start, self.max_steps - 1)
            self.bout_id_at_step[start:end + 1] = bout_id
            self.bout_start_steps.append(start)
            self.bout_end_steps.append(end)
            self.bout_dur_s_list.append(dur_s)
            bout_id += 1
            k += 1
        self.n_bouts = int(bout_id)
        return self._get_obs(lick_sig=0.0, win_rem=0.0)

    def _teacher_lick_now(self, t: int):
        bid = int(self.bout_id_at_step[t]) if (0 <= t <= self.max_steps) else -1
        return (1, bid) if bid >= 0 else (0, -1)

    def _window_open(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        return 1 if (0 <= dt < self.window_steps) else 0

    def _window_remaining(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        rem = self.window_steps - dt
        return int(max(0, rem))

    def _phase_to_next_nominal(self, t: int) -> int:
        mod = t % self.teacher_period_steps
        return self.teacher_period_steps - mod

    def _get_obs(self, lick_sig: float, win_rem: float):
        lp = float(self.learner_pos) / float(self.grid_size - 1)
        lf = float(self.learner_food_pos) / float(self.grid_size - 1)
        phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period_steps)
        cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown_steps))
        # [pos, food_pos, gated cue, gated window remaining, phase clock, cooldown]
        return np.array([lp, lf, float(lick_sig), float(win_rem), phase, cdn], dtype=np.float32)

    def step(self, action: int):
        self.t += 1
        teacher_lick, bout_id = self._teacher_lick_now(self.t)
        if teacher_lick == 1:
            self.last_lick_step = self.t

        window_open = self._window_open(self.t)
        win_rem_true = self._window_remaining(self.t)

        if self.eat_cd > 0:
            self.eat_cd -= 1

        lick_sig = 0.0
        win_rem = 0.0
        did_observe = 0
        did_eat = 0
        seen_lick = 0

        # movement / observe / eat
        if action == A_LEFT:
            self.learner_pos -= 1
        elif action == A_RIGHT:
            self.learner_pos += 1
        elif action == A_OBS:
            did_observe = 1
            if teacher_lick == 1 and bout_id >= 0 and (np.random.rand() < self.p_detect_per_obs):
                seen_lick = 1
                self.detected_bout_id = int(bout_id)
            lick_sig = float(seen_lick)
            if win_rem_true > 0:
                noise = int(np.random.randint(-self.win_rem_noise_steps, self.win_rem_noise_steps + 1))
                est = clamp_int(win_rem_true + noise, 0, self.window_steps)
                win_rem = float(est) / float(max(1, self.window_steps))
        elif action == A_EAT:
            did_eat = 1
        # A_STAY does nothing

        self.learner_pos = clamp_int(self.learner_pos, 0, self.grid_size - 1)

        reward = 0.0
        reward -= self.observe_cost * float(did_observe)
        reward -= self.eat_cost * float(did_eat)

        # shaping: reward OBS during cue only if bout not yet detected
        if did_observe == 1 and teacher_lick == 1 and bout_id >= 0:
            if self.detected_bout_id != int(bout_id):
                reward += self.attend_bonus

        # eat allowed?
        eat_allowed = (did_eat == 1 and self.eat_cd == 0)
        if eat_allowed:
            self.eat_cd = self.eat_cooldown_steps

        # success logic (1 reward per bout)
        eat_valid = 0
        if eat_allowed and (self.learner_pos == self.learner_food_pos) and (window_open == 1) and (self.last_lick_step is not None):
            last_bid = int(self.bout_id_at_step[self.last_lick_step]) if self.last_lick_step <= self.max_steps else -1
            if (last_bid >= 0) and (self.detected_bout_id == last_bid) and (last_bid not in self.rewarded_bout_ids):
                eat_valid = 1
                self.rewarded_bout_ids.add(last_bid)
                reward += 1.0

        done = bool(self.t >= self.max_steps)
        info = {
            "t": self.t,
            "teacher_lick": int(teacher_lick),
            "bout_id": int(bout_id),
            "window_open": int(window_open),
            "win_rem_true": int(win_rem_true),
            "action": int(action),
            "observe": int(did_observe),
            "eat": int(did_eat),
            "eat_allowed": int(eat_allowed),
            "eat_valid": int(eat_valid),
            "learner_pos": int(self.learner_pos),
            "food_pos": int(self.learner_food_pos),
            "seen_lick": int(seen_lick),
            "win_rem_est": float(win_rem),
            "detected_bout_id": -1 if self.detected_bout_id is None else int(self.detected_bout_id),
            "rewarded_bouts": len(self.rewarded_bout_ids),
            "n_bouts": int(self.n_bouts),
            "dt_s": float(self.dt_s),
            "window_steps": int(self.window_steps),
        }
        obs = self._get_obs(lick_sig=lick_sig, win_rem=win_rem)
        return obs, reward, done, info


# ============================================================
# Plot saving (PDF by default)
# ============================================================
def _save_fig(fig, out_path_no_ext: str, ext: str, pdf_pages: Optional[PdfPages] = None):
    os.makedirs(os.path.dirname(out_path_no_ext), exist_ok=True)
    if ext.lower() == "pdf" and pdf_pages is not None:
        pdf_pages.savefig(fig, bbox_inches="tight")
    else:
        fig.savefig(f"{out_path_no_ext}.{ext}", bbox_inches="tight")

def ema(x, alpha=0.02):
    x = np.asarray(x, dtype=np.float32)
    y = np.zeros_like(x)
    m = 0.0
    for i in range(len(x)):
        m = (1 - alpha) * m + alpha * x[i]
        y[i] = m
    return y

def plot_learning_curves(logs, save_dir="plots", ext="pdf", pdf_pages=None):
    ep = np.array([x["ep"] for x in logs])
    succ = ema([x["bout_success_rate"] for x in logs])
    obs = ema([x["obs_rate"] for x in logs])
    obsSeen = ema([x["obs_seen_lick_rate"] for x in logs])
    eatWin = ema([x["eat_in_window_rate"] for x in logs])

    fig = plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(ep, succ)
    plt.title("EMA bout success rate")
    plt.xlabel("episode"); plt.ylabel("rate")

    plt.subplot(1, 3, 2)
    plt.plot(ep, obs, label="observe")
    plt.plot(ep, obsSeen, label="obs sees cue")
    plt.title("EMA observation metrics")
    plt.xlabel("episode"); plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(ep, eatWin)
    plt.title("EMA eat-in-window rate")
    plt.xlabel("episode"); plt.ylabel("rate")

    plt.tight_layout()
    _save_fig(fig, os.path.join(save_dir, "learning_curves"), ext=ext, pdf_pages=pdf_pages)
#     plt.close(fig)

# ============================================================
# Latent / “DA-like” simulation utilities (algorithm-agnostic)
# ============================================================
def calcium_filter(impulses, dt_s, tau_rise_s=0.2, tau_decay_s=1.2):
    impulses = np.asarray(impulses, dtype=np.float32)
    n = len(impulses)
    L = int(np.ceil(8 * tau_decay_s / dt_s))
    t = np.arange(L, dtype=np.float32) * dt_s
    k = (1 - np.exp(-t / max(1e-6, tau_rise_s))) * np.exp(-t / max(1e-6, tau_decay_s))
    k = k / (k.sum() + 1e-9)
    y = np.convolve(impulses, k, mode="full")[:n]
    return y.astype(np.float32)

def make_event_indices(tr):
    lick = np.array(tr["teacher_lick"], dtype=int)
    obs = np.array(tr["observe"], dtype=int)
    seen = np.array(tr["seen_lick"], dtype=int) if "seen_lick" in tr else None
    reward = np.array(tr["eat_valid"], dtype=int)

    lick_end = np.where((lick[:-1] == 1) & (lick[1:] == 0))[0] + 1
    obs_idx = np.where(obs == 1)[0]
    if seen is not None:
        obs_seen_lick_idx = np.where((obs == 1) & (seen == 1))[0]
    else:
        obs_seen_lick_idx = np.array([], dtype=int)
    rew_idx = np.where(reward == 1)[0]

    return {
        "lick_end": lick_end,
        "obs": obs_idx,
        "obs_seen_lick": obs_seen_lick_idx,
        "reward": rew_idx,
    }

def peri_event_average(signal, event_idx, pre_steps=10, post_steps=20):
    signal = np.asarray(signal, dtype=np.float32)
    out = []
    for e in event_idx:
        a = e - pre_steps
        b = e + post_steps + 1
        if a < 0 or b > len(signal):
            continue
        out.append(signal[a:b])
    if len(out) == 0:
        return None
    return np.mean(np.stack(out, axis=0), axis=0)

def gae_advantages(rewards, values, dones, gamma=0.99, lam=0.95):
    """GAE(λ) computed from r_t, V_t, done_t. values length T+1 (bootstrapped)."""
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        nonterminal = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * values[t+1] * nonterminal - values[t]
        gae = delta + gamma * lam * nonterminal * gae
        adv[t] = gae
    return adv

def simulate_latent_models_for_episode(tr, agent, gamma=0.99, ep_frac=0.0, gae_lam=0.95):
    """
    Candidate latent signal hypotheses (paper-aligned, but algorithm-agnostic):
      - reward_RPE (TD error)
      - gae_adv (advantage-like)
      - social_cue_RPE / social_value (ΔV on Observe due to revealing gated cue features)
      - action_salience
      - info_gain (surprise-scaled Observe-detect)
      - optional: policy_entropy (if agent exposes policy entropy)
    """
    r = np.array(tr["reward"], dtype=np.float32)
    obs = np.array(tr["observe"], dtype=int)
    eat = np.array(tr["eat"], dtype=int)
    seen_lick = np.array(tr["seen_lick"], dtype=int)
    phase = np.array(tr["phase"], dtype=np.float32)
    S = np.array(tr["state"], dtype=np.float32)
    S2 = np.array(tr["next_state"], dtype=np.float32)
    done = np.array(tr["done"], dtype=np.float32)

    V_s = agent.value_batch(S)
    V_s2 = agent.value_batch(S2)

    # TD error (reward_RPE)
    rpe = r + gamma * V_s2 * (1.0 - done) - V_s

    # GAE advantage-like signal
    V_boot = np.concatenate([V_s, V_s2[-1:]], axis=0)  # crude: last bootstrap
    adv = gae_advantages(r, V_boot[:len(r)+1], done, gamma=gamma, lam=gae_lam)

    # Observation-driven ΔV: V_post - V_prior (social features zeroed)
    S2_prior = S2.copy()
    S2_prior[:, 2] = 0.0  # lick_sig
    S2_prior[:, 3] = 0.0  # win_rem
    V_post = V_s2
    V_prior = agent.value_batch(S2_prior)
    deltaV = (V_post - V_prior) * obs.astype(np.float32)

    # Action salience
    sal = np.zeros(len(r), dtype=np.float32)
    sal += 1.0 * (obs == 1).astype(np.float32)
    sal += 0.6 * (eat == 1).astype(np.float32)

    # Info gain / surprise when OBS successfully detects cue
    sigma = 0.25 * (1.0 - ep_frac) + 0.10 * ep_frac
    p = np.exp(-(phase / max(1e-6, sigma)) ** 2)  # 0..1
    surprise = -np.log(p + 1e-6)
    info_gain = surprise * ((obs == 1) & (seen_lick == 1)).astype(np.float32)

    impulses = {
        "reward_RPE": 0.8 * rpe,
        "gae_adv": 0.7 * adv,
        "social_cue_RPE": 1.2 * deltaV,
        "social_value": 0.6 * deltaV,
        "action_salience": 0.3 * sal,
        "info_gain": 0.25 * info_gain,
    }

    # Optional policy entropy latent (actor-critic learners)
    if hasattr(agent, "policy_entropy_batch"):
        ent = agent.policy_entropy_batch(S)  # shape [T]
        if ent is not None:
            impulses["policy_entropy"] = 0.15 * ent.astype(np.float32)

    return impulses

def plot_da_peri_event(da_by_ep, traces, dt_s, model_name, event_key="reward",
                       pre_s=6, post_s=10, save_dir="plots", ext="pdf", pdf_pages=None):
    pre = int(round(pre_s / dt_s))
    post = int(round(post_s / dt_s))
    x = (np.arange(pre + post + 1) - pre) * dt_s

    n = len(traces)
    early_idx = range(0, max(1, n // 5))
    late_idx = range(max(0, n - n // 5), n)

    def mean_peri(idxs):
        mats = []
        for i in idxs:
            ev = make_event_indices(traces[i])[event_key]
            m = peri_event_average(da_by_ep[i], ev, pre_steps=pre, post_steps=post)
            if m is not None:
                mats.append(m)
        if len(mats) == 0:
            return None
        return np.mean(np.stack(mats, axis=0), axis=0)

    e = mean_peri(early_idx)
    l = mean_peri(late_idx)

    fig = plt.figure(figsize=(6, 4))
    if e is not None:
        plt.plot(x, e, label="early (first 20%)")
    if l is not None:
        plt.plot(x, l, label="late (last 20%)")
    plt.axvline(0, linestyle="--")
    plt.title(f"{model_name}: peri-{event_key}")
    plt.xlabel("time (s)")
    plt.ylabel("simulated signal")
    plt.legend()
    plt.tight_layout()

    out = os.path.join(save_dir, f"{model_name}_peri_{event_key}")
    _save_fig(fig, out, ext=ext, pdf_pages=pdf_pages)
#     plt.close(fig)

def simulate_and_plot_latents(traces, agent, dt_s, gamma=0.99,
                              save_dir="plots", ext="pdf", pdf_pages=None):
    impulses_by_model = defaultdict(list)
    n = len(traces)
    for i, tr in enumerate(traces):
        ep_frac = i / max(1, n - 1)
        imp = simulate_latent_models_for_episode(tr, agent, gamma=gamma, ep_frac=ep_frac)
        for k, v in imp.items():
            impulses_by_model[k].append(v)

    phot_by_model = {}
    for k, imps in impulses_by_model.items():
        phot_by_model[k] = [calcium_filter(x, dt_s=dt_s, tau_rise_s=0.2, tau_decay_s=1.2) for x in imps]

    for k, da_eps in phot_by_model.items():
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="lick_end",
                           pre_s=6, post_s=10, save_dir=save_dir, ext=ext, pdf_pages=pdf_pages)
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="reward",
                           pre_s=6, post_s=10, save_dir=save_dir, ext=ext, pdf_pages=pdf_pages)
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="obs_seen_lick",
                           pre_s=6, post_s=10, save_dir=save_dir, ext=ext, pdf_pages=pdf_pages)

    return phot_by_model


# ============================================================
# Agents (multiple learner models)
# ============================================================
class BaseAgent:
    def act(self, s: np.ndarray):
        """Return (action:int, extra:dict)."""
        raise NotImplementedError

    def record_transition(self, s, a, r, s2, done, extra):
        pass

    def step_update(self):
        return None

    def episode_update(self):
        return None

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        raise NotImplementedError


# -------------------------
# DQN (dueling + double + huber + soft target)
# -------------------------
class DuelingQNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.V = nn.Linear(hidden, 1)
        self.A = nn.Linear(hidden, action_dim)

    def forward(self, x):
        h = self.backbone(x)
        v = self.V(h)
        a = self.A(h)
        return v + (a - a.mean(dim=1, keepdim=True))

class TorchReplayBuffer:
    def __init__(self, capacity, state_dim, device):
        self.capacity = int(capacity)
        self.device = device
        self.state = torch.zeros((capacity, state_dim), dtype=torch.float32, device=device)
        self.action = torch.zeros(capacity, dtype=torch.int64, device=device)
        self.reward = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.next_state = torch.zeros((capacity, state_dim), dtype=torch.float32, device=device)
        self.done = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.idx = 0
        self.size = 0

    def push(self, s, a, r, s2, done):
        self.state[self.idx] = torch.tensor(s, dtype=torch.float32, device=self.device)
        self.action[self.idx] = int(a)
        self.reward[self.idx] = float(r)
        self.next_state[self.idx] = torch.tensor(s2, dtype=torch.float32, device=self.device)
        self.done[self.idx] = float(done)
        self.idx = (self.idx + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        indices = torch.randint(0, self.size, (batch_size,), device=self.device)
        return (
            self.state[indices],
            self.action[indices].unsqueeze(1),
            self.reward[indices],
            self.next_state[indices],
            self.done[indices],
        )

    def __len__(self):
        return self.size

class DQNAgent(BaseAgent):
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        batch_size=256,
        buffer_size=50000,
        eps_start=1.0,
        eps_end=0.05,
        eps_decay=0.9995,
        tau=0.02,
        grad_clip=10.0,
        device=None,
        use_compile=False,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.batch_size = int(batch_size)
        self.tau = float(tau)
        self.grad_clip = float(grad_clip)

        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.q = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt.load_state_dict(self.q.state_dict())

        if use_compile and hasattr(torch, "compile"):
            try:
                self.q = torch.compile(self.q)
            except Exception:
                pass

        self.opt = optim.Adam(self.q.parameters(), lr=float(lr))
        self.loss_fn = nn.SmoothL1Loss()
        self.rb = TorchReplayBuffer(buffer_size, state_dim, self.device)
        self.scaler = GradScaler(enabled=self.device.type == "cuda")

    def act(self, s):
        if np.random.rand() < self.eps:
            return int(np.random.randint(self.action_dim)), {}
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            qv = self.q(st)
        return int(torch.argmax(qv, dim=1).item()), {}

    def record_transition(self, s, a, r, s2, done, extra):
        self.rb.push(s, a, r, s2, done)

    def step_update(self, updates_per_step=1):
        if len(self.rb) < self.batch_size:
            self.eps = max(self.eps_end, self.eps * self.eps_decay)
            return None

        last_loss = None
        for _ in range(int(updates_per_step)):
            s, a, r, s2, d = self.rb.sample(self.batch_size)
            with autocast(enabled=self.device.type == "cuda"):
                q_sa = self.q(s).gather(1, a).squeeze(1)
                with torch.no_grad():
                    a_star = torch.argmax(self.q(s2), dim=1, keepdim=True)
                    q_next = self.qt(s2).gather(1, a_star).squeeze(1)
                    target = r + self.gamma * q_next * (1.0 - d)
                loss = self.loss_fn(q_sa, target)

            self.opt.zero_grad()
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.opt)
            nn.utils.clip_grad_norm_(self.q.parameters(), self.grad_clip)
            self.scaler.step(self.opt)
            self.scaler.update()

            with torch.no_grad():
                for p, pt in zip(self.q.parameters(), self.qt.parameters()):
                    pt.data.mul_(1.0 - self.tau).add_(self.tau * p.data)

            last_loss = float(loss.item())

        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        return last_loss

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            qv = self.q(st)
            v = torch.max(qv, dim=1).values
        return v.detach().cpu().numpy().astype(np.float32)

# -------------------------
# PPO (actor-critic)
# -------------------------
class ActorCriticNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.pi = nn.Linear(hidden, action_dim)
        self.v = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.body(x)
        logits = self.pi(h)
        value = self.v(h).squeeze(-1)
        return logits, value

class PPOAgent(BaseAgent):
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_ratio=0.2,
        vf_coef=0.5,
        ent_coef=0.01,
        train_epochs=4,
        minibatch_size=256,
        max_grad_norm=1.0,
        device=None,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.gae_lambda = float(gae_lambda)
        self.clip_ratio = float(clip_ratio)
        self.vf_coef = float(vf_coef)
        self.ent_coef = float(ent_coef)
        self.train_epochs = int(train_epochs)
        self.minibatch_size = int(minibatch_size)
        self.max_grad_norm = float(max_grad_norm)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.net = ActorCriticNet(self.state_dim, self.action_dim).to(self.device)
        self.opt = optim.Adam(self.net.parameters(), lr=float(lr))

        self.roll = []  # stores dicts per step

    def act(self, s):
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            logits, v = self.net(st)
            dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample()
            logp = dist.log_prob(a)
            ent = dist.entropy()
        return int(a.item()), {"logp": float(logp.item()), "v": float(v.item()), "ent": float(ent.item())}

    def record_transition(self, s, a, r, s2, done, extra):
        self.roll.append({
            "s": np.array(s, dtype=np.float32),
            "a": int(a),
            "r": float(r),
            "done": float(done),
            "logp": float(extra.get("logp", 0.0)),
            "v": float(extra.get("v", 0.0)),
        })

    def episode_update(self):
        if len(self.roll) < 8:
            self.roll.clear()
            return None

        S = np.stack([x["s"] for x in self.roll], axis=0).astype(np.float32)
        A = np.array([x["a"] for x in self.roll], dtype=np.int64)
        R = np.array([x["r"] for x in self.roll], dtype=np.float32)
        D = np.array([x["done"] for x in self.roll], dtype=np.float32)
        LOGP_OLD = np.array([x["logp"] for x in self.roll], dtype=np.float32)
        V = np.array([x["v"] for x in self.roll], dtype=np.float32)

        # bootstrap last value with current critic (on last state)
        with torch.no_grad():
            st_last = torch.tensor(S[-1], dtype=torch.float32, device=self.device).unsqueeze(0)
            _, v_last = self.net(st_last)
            v_last = float(v_last.item())
        V_ext = np.concatenate([V, np.array([v_last], dtype=np.float32)], axis=0)

        ADV = gae_advantages(R, V_ext, D, gamma=self.gamma, lam=self.gae_lambda)
        RET = ADV + V  # return target

        # normalize advantages
        ADV = (ADV - ADV.mean()) / (ADV.std() + 1e-8)

        # torch tensors
        ts = torch.tensor(S, dtype=torch.float32, device=self.device)
        ta = torch.tensor(A, dtype=torch.int64, device=self.device)
        told = torch.tensor(LOGP_OLD, dtype=torch.float32, device=self.device)
        tadv = torch.tensor(ADV, dtype=torch.float32, device=self.device)
        tret = torch.tensor(RET, dtype=torch.float32, device=self.device)

        n = len(S)
        idx = np.arange(n)
        info = {}

        for _ in range(self.train_epochs):
            np.random.shuffle(idx)
            for start in range(0, n, self.minibatch_size):
                mb = idx[start:start+self.minibatch_size]
                logits, vpred = self.net(ts[mb])
                dist = torch.distributions.Categorical(logits=logits)

                logp = dist.log_prob(ta[mb])
                ratio = torch.exp(logp - told[mb])

                clip = torch.clamp(ratio, 1.0 - self.clip_ratio, 1.0 + self.clip_ratio)
                pg_loss = -(torch.min(ratio * tadv[mb], clip * tadv[mb])).mean()

                v_loss = ((vpred - tret[mb]) ** 2).mean()
                ent = dist.entropy().mean()

                loss = pg_loss + self.vf_coef * v_loss - self.ent_coef * ent

                self.opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), self.max_grad_norm)
                self.opt.step()

                info = {
                    "pg_loss": float(pg_loss.item()),
                    "v_loss": float(v_loss.item()),
                    "ent": float(ent.item()),
                    "loss": float(loss.item()),
                }

        self.roll.clear()
        return info

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            _, v = self.net(st)
        return v.detach().cpu().numpy().astype(np.float32)

    def policy_entropy_batch(self, states: np.ndarray):
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            logits, _ = self.net(st)
            dist = torch.distributions.Categorical(logits=logits)
            ent = dist.entropy()
        return ent.detach().cpu().numpy().astype(np.float32)

# -------------------------
# Tabular Q-learning (light baseline)
# -------------------------
class TabularQAgent(BaseAgent):
    def __init__(self, action_dim=5, alpha=0.25, gamma=0.99, eps_start=1.0, eps_end=0.05, eps_decay=0.9995,
                 phase_bins=10, win_bins=7, cd_bins=6, device=None):
        self.action_dim = int(action_dim)
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.phase_bins = int(phase_bins)
        self.win_bins = int(win_bins)
        self.cd_bins = int(cd_bins)

        self.Q = defaultdict(lambda: np.zeros(self.action_dim, dtype=np.float32))
        self._last = None

    def _disc(self, s: np.ndarray):
        # s = [lp, lf, lick_sig, win_rem, phase, cdn]
        lp, lf, lick, win, phase, cd = s.tolist()
        # positions already normalized, discretize to 0..14-ish by rounding*14
        p = int(np.round(lp * 14))
        f = int(np.round(lf * 14))
        lickb = int(lick > 0.5)
        winb = int(np.round(win * (self.win_bins - 1)))
        winb = clamp_int(winb, 0, self.win_bins - 1)
        phb = int(np.floor(phase * self.phase_bins))
        phb = clamp_int(phb, 0, self.phase_bins - 1)
        cdb = int(np.round(cd * (self.cd_bins - 1)))
        cdb = clamp_int(cdb, 0, self.cd_bins - 1)
        return (p, f, lickb, winb, phb, cdb)

    def act(self, s):
        key = self._disc(s)
        if np.random.rand() < self.eps:
            a = int(np.random.randint(self.action_dim))
        else:
            a = int(np.argmax(self.Q[key]))
        return a, {"key": key}

    def record_transition(self, s, a, r, s2, done, extra):
        k = extra["key"]
        k2 = self._disc(s2)
        q = self.Q[k]
        target = float(r) + self.gamma * (0.0 if done else float(np.max(self.Q[k2])))
        q[a] = (1 - self.alpha) * q[a] + self.alpha * target

        self.eps = max(self.eps_end, self.eps * self.eps_decay)

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        out = np.zeros(len(states), dtype=np.float32)
        for i, s in enumerate(states):
            out[i] = float(np.max(self.Q[self._disc(s)]))
        return out


# ============================================================
# Training + logging (works for all agents above)
# ============================================================
def train_social_with_latents(env, agent: BaseAgent, episodes=1200, log_every=100,
                             updates_per_step=4, early_stop_threshold=0.8, early_stop_window=100):
    logs = []
    traces = []
    success_history = deque(maxlen=early_stop_window)

    for ep in range(1, episodes + 1):
        s = env.reset()
        done = False

        ep_reward = 0.0
        succ_bouts = 0
        n_bouts = max(1, env.n_bouts)
        obs_steps = 0
        obs_seen = 0
        eat_in_win = 0

        tr = defaultdict(list)

        while not done:
            a, extra = agent.act(s)
            s2, r, done, info = env.step(a)
            ep_reward += r

            if info["eat_valid"] == 1:
                succ_bouts += 1
            if info["observe"] == 1:
                obs_steps += 1
                if info["seen_lick"] == 1:
                    obs_seen += 1
            if info["eat"] == 1 and info["window_open"] == 1:
                eat_in_win += 1

            # trace for latent sims
            tr["state"].append(s.copy())
            tr["next_state"].append(s2.copy())
            tr["reward"].append(float(r))
            tr["done"].append(float(done))

            tr["teacher_lick"].append(int(info["teacher_lick"]))
            tr["window_open"].append(int(info["window_open"]))
            tr["observe"].append(int(info["observe"]))
            tr["eat"].append(int(info["eat"]))
            tr["eat_valid"].append(int(info["eat_valid"]))
            tr["seen_lick"].append(int(info["seen_lick"]))
            tr["win_rem_est"].append(float(info["win_rem_est"]))
            tr["phase"].append(float(s[4]))

            # learner updates
            agent.record_transition(s, a, r, s2, done, extra)

            if hasattr(agent, "step_update"):
                # DQN uses step_update; PPO ignores it
                if isinstance(agent, DQNAgent):
                    agent.step_update(updates_per_step=updates_per_step)

            s = s2

        if hasattr(agent, "episode_update"):
            agent.episode_update()

        for k in list(tr.keys()):
            tr[k] = np.array(tr[k])
        traces.append(tr)

        logs.append({
            "ep": ep,
            "bout_success_rate": float(succ_bouts / n_bouts),
            "mean_reward": float(ep_reward),
            "obs_rate": float(obs_steps / max(1, env.max_steps)),
            "obs_seen_lick_rate": float(obs_seen / max(1, env.max_steps)),
            "eat_in_window_rate": float(eat_in_win / max(1, env.max_steps)),
            "eps": float(getattr(agent, "eps", np.nan)),
        })

        success_history.append(logs[-1]["bout_success_rate"])

        if ep % log_every == 0:
            recent = logs[-log_every:]
            print(
                f"[ep {ep:4d}] "
                f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f} "
                f"obs={np.mean([x['obs_rate'] for x in recent]):.3f} "
                f"obsSeen={np.mean([x['obs_seen_lick_rate'] for x in recent]):.3f} "
                f"eatWin={np.mean([x['eat_in_window_rate'] for x in recent]):.3f} "
                f"R={np.mean([x['mean_reward'] for x in recent]):.3f} "
                f"eps={logs[-1]['eps']:.3f}"
            )

        if len(success_history) == early_stop_window and np.mean(success_history) > early_stop_threshold:
            print(f"Early stopping at episode {ep}: Avg success rate {np.mean(success_history):.3f} > {early_stop_threshold}")
            break

    return logs, traces


# ============================================================
# Run experiment
# ============================================================
def run_experiment(args):
    set_seed(args.seed)

    env = SocialLickEnv1D(
        grid_size=args.grid_size,
        dt_s=args.dt_s,
        max_time_s=args.max_time_s,
        teacher_period_s=args.teacher_period_s,
        teacher_jitter_s=args.teacher_jitter_s,
        lick_mean_s=args.lick_mean_s,
        lick_dist=args.lick_dist,
        lick_cv=args.lick_cv,
        window_s=args.window_s,
        eat_cooldown_s=args.eat_cooldown_s,
        observe_cost=args.observe_cost,
        eat_cost=args.eat_cost,
        attend_bonus=args.attend_bonus,
        p_detect_per_obs=args.p_detect_per_obs,
        win_rem_noise_steps=args.win_rem_noise_steps,
    )

    state_dim = 6
    action_dim = 5

    if args.algo == "dqn":
        agent = DQNAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            lr=args.lr,
            gamma=args.gamma,
            batch_size=args.batch_size,
            buffer_size=args.buffer_size,
            eps_start=args.eps_start,
            eps_end=args.eps_end,
            eps_decay=args.eps_decay,
            tau=args.tau,
            grad_clip=args.grad_clip,
            use_compile=args.use_compile,
        )
    elif args.algo == "ppo":
        agent = PPOAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            lr=args.lr,
            gamma=args.gamma,
            gae_lambda=args.gae_lambda,
            clip_ratio=args.clip_ratio,
            vf_coef=args.vf_coef,
            ent_coef=args.ent_coef,
            train_epochs=args.ppo_epochs,
            minibatch_size=args.minibatch_size,
            max_grad_norm=args.max_grad_norm,
        )
    elif args.algo == "tabular":
        agent = TabularQAgent(
            action_dim=action_dim,
            alpha=args.tab_alpha,
            gamma=args.gamma,
            eps_start=args.eps_start,
            eps_end=args.eps_end,
            eps_decay=args.eps_decay,
        )
    else:
        raise ValueError(f"Unknown algo: {args.algo}")

    save_dir = args.save_dir
    os.makedirs(save_dir, exist_ok=True)

    pdf_pages = None
    if args.plot_ext.lower() == "pdf" and args.multipage_pdf:
        pdf_pages = PdfPages(os.path.join(save_dir, f"all_figures_{args.algo}.pdf"))

    logs, traces = train_social_with_latents(
        env, agent,
        episodes=args.episodes,
        log_every=max(50, args.episodes // 12),
        updates_per_step=args.updates_per_step,
        early_stop_threshold=args.early_stop_threshold,
        early_stop_window=args.early_stop_window,
    )

    if not args.no_plots:
        plot_learning_curves(logs, save_dir=save_dir, ext=args.plot_ext, pdf_pages=pdf_pages)
        simulate_and_plot_latents(traces, agent, dt_s=env.dt_s, gamma=args.gamma,
                                  save_dir=save_dir, ext=args.plot_ext, pdf_pages=pdf_pages)

    if pdf_pages is not None:
        pdf_pages.close()

    return logs, traces


def build_argparser():
    p = argparse.ArgumentParser(description="Active observational learning (multi-algo) + latent signal sims")

#     # algo
#     p.add_argument("--algo", type=str, default="dqn", choices=["dqn", "ppo", "tabular"])
    # algo
    p.add_argument("--algo", type=str, default="dqn", choices=["dqn", "ppo", "tabular"])

    # run
    p.add_argument("--episodes", type=int, default=1200)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--save_dir", type=str, default="plots")
    p.add_argument("--no_plots", action="store_true")
    p.add_argument("--plot_ext", type=str, default="pdf", choices=["pdf", "png"])
    p.add_argument("--multipage_pdf", action="store_true", help="If set + plot_ext=pdf, also writes one multi-page PDF.")
    p.add_argument("--updates_per_step", type=int, default=4)

    # environment
    p.add_argument("--grid_size", type=int, default=15)
    p.add_argument("--dt_s", type=float, default=0.5)
    p.add_argument("--max_time_s", type=float, default=120.0)
    p.add_argument("--teacher_period_s", type=float, default=30.0)
    p.add_argument("--teacher_jitter_s", type=float, default=6.0)
    p.add_argument("--lick_mean_s", type=float, default=1.0)
    p.add_argument("--lick_dist", type=str, default="gamma", choices=["gamma", "lognormal"])
    p.add_argument("--lick_cv", type=float, default=0.8)
    p.add_argument("--window_s", type=float, default=3.0)
    p.add_argument("--eat_cooldown_s", type=float, default=2.0)
    p.add_argument("--observe_cost", type=float, default=0.002)
    p.add_argument("--eat_cost", type=float, default=0.005)
    p.add_argument("--attend_bonus", type=float, default=0.004)
    p.add_argument("--p_detect_per_obs", type=float, default=0.70)
    p.add_argument("--win_rem_noise_steps", type=int, default=1)

    # common learning
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--eps_start", type=float, default=1.0)
    p.add_argument("--eps_end", type=float, default=0.05)
    p.add_argument("--eps_decay", type=float, default=0.9995)

    # DQN
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--buffer_size", type=int, default=50000)
    p.add_argument("--tau", type=float, default=0.02)
    p.add_argument("--grad_clip", type=float, default=10.0)
    p.add_argument("--use_compile", action="store_true")

    # PPO
    p.add_argument("--gae_lambda", type=float, default=0.95)
    p.add_argument("--clip_ratio", type=float, default=0.2)
    p.add_argument("--vf_coef", type=float, default=0.5)
    p.add_argument("--ent_coef", type=float, default=0.01)
    p.add_argument("--ppo_epochs", type=int, default=4)
    p.add_argument("--minibatch_size", type=int, default=256)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    # Tabular Q
    p.add_argument("--tab_alpha", type=float, default=0.25)

    # early stopping
    p.add_argument("--early_stop_threshold", type=float, default=0.8)
    p.add_argument("--early_stop_window", type=int, default=100)

    return p


if __name__ == "__main__":
    parser = build_argparser()
    args, _ = parser.parse_known_args()
    run_experiment(args)

In [ ]:
if __name__ == "__main__":
    parser = build_argparser()
    args, _ = parser.parse_known_args()
    run_experiment(args)

In [ ]:
# # DQN (your original baseline), plots as PDF
# python social_rl_multi_algo.py --algo dqn --plot_ext pdf --save_dir plots_dqn

# # PPO actor-critic (drop-in learner model)
# python social_rl_multi_algo.py --algo ppo --plot_ext pdf --save_dir plots_ppo

# # Tabular Q (light baseline / sanity check)
# python social_rl_multi_algo.py --algo tabular --plot_ext pdf --save_dir plots_tab
# # All
# python social_rl_multi_algo.py --algo dqn --plot_ext pdf --multipage_pdf --save_dir plots_dqn


## unfamilar

In [ ]:
from typing import Optional, Dict, Any, List, Tuple
import numpy as np
import random
from collections import deque, defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import argparse
import os
import math

# -------------------------
# Seed / utils
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def clamp_int(x: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, x))

def clamp_float(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, float(x)))

# -------------------------
# Actions
# -------------------------
A_LEFT = 0
A_STAY = 1
A_RIGHT = 2
A_OBS = 3  # consumes time; reveals social timing info stochastically
A_EAT = 4

# ============================================================
# Environment (same logic as your current code + new condition knobs)
# ============================================================
class SocialLickEnv1D:
    """
    Fast but detailed (paper-aligned):
      dt_s = 0.5s/step
      teacher cue bursts ("lick bouts") with stochastic duration (mean ~ 1s)
      reward window: 3s after last cue step

    Observation gating:
      - social signal (cue + noisy window remaining) only appears when action==OBS
      - detection during cue burst is probabilistic per OBS step

    NEW (optional) knobs for condition comparisons:
      - social_visibility: 0.0 => "social blind" (OBS yields no social info)
      - detect_memory_steps: if set, detected_bout_id expires after N steps
      - familiarity_enabled: if True, a latent familiarity variable accumulates across episodes and
        increases effective detect prob and reduces window-remaining noise
    """
    def __init__(
        self,
        grid_size=15,
        dt_s=0.5,
        max_time_s=120.0,
        teacher_period_s=30.0,
        teacher_jitter_s=6.0,
        lick_mean_s=1.0,
        lick_dist="gamma",  # "gamma" or "lognormal"
        lick_cv=0.8,        # for lognormal
        window_s=3.0,
        eat_cooldown_s=2.0,
        observe_cost=0.002,
        eat_cost=0.005,
        attend_bonus=0.004,
        p_detect_per_obs=0.70,
        win_rem_noise_steps=1,

        # perception / cognition constraints
        social_visibility: float = 1.0,             # 1.0 normal, 0.0 blind
        detect_memory_steps: Optional[int] = None,  # None => perfect retention
        # familiarity dynamics
        familiarity_enabled: bool = False,
        familiarity_init: float = 0.0,              # persistent across episodes
        fam_decay_ep: float = 0.00,                 # applied at reset (episode boundary)
        fam_gain_obs: float = 0.00,                 # gain per OBS during teacher lick
        fam_gain_detect: float = 0.00,              # extra gain when OBS successfully detects lick
        p_detect_fam_boost: float = 0.00,           # added detect prob at fam=1
        noise_fam_reduction: float = 0.00,          # fraction reduction of noise_steps at fam=1
    ):
        self.grid_size = int(grid_size)
        self.dt_s = float(dt_s)
        self.max_steps = int(round(max_time_s / self.dt_s))
        self.teacher_period_steps = int(round(teacher_period_s / self.dt_s))
        self.teacher_jitter_steps = int(round(teacher_jitter_s / self.dt_s))
        self.lick_mean_s = float(lick_mean_s)
        self.lick_dist = str(lick_dist)
        self.lick_cv = float(lick_cv)
        self.window_steps = int(round(window_s / self.dt_s))
        self.eat_cooldown_steps = int(round(eat_cooldown_s / self.dt_s))
        self.observe_cost = float(observe_cost)
        self.eat_cost = float(eat_cost)
        self.attend_bonus = float(attend_bonus)
        self.p_detect_per_obs = float(p_detect_per_obs)
        self.win_rem_noise_steps = int(win_rem_noise_steps)

        self.social_visibility = clamp_float(social_visibility, 0.0, 1.0)
        self.detect_memory_steps = None if detect_memory_steps is None else int(detect_memory_steps)

        self.familiarity_enabled = bool(familiarity_enabled)
        self.familiarity = clamp_float(familiarity_init, 0.0, 1.0)
        self.fam_decay_ep = clamp_float(fam_decay_ep, 0.0, 1.0)
        self.fam_gain_obs = float(fam_gain_obs)
        self.fam_gain_detect = float(fam_gain_detect)
        self.p_detect_fam_boost = float(p_detect_fam_boost)
        self.noise_fam_reduction = clamp_float(noise_fam_reduction, 0.0, 1.0)

        assert self.teacher_period_steps > self.window_steps + 1
        assert self.max_steps > self.teacher_period_steps + 5
        self.reset()

    def _sample_lick_duration_steps(self):
        if self.lick_dist == "gamma":
            shape = 2.0
            scale = self.lick_mean_s / shape
            dur_s = float(np.random.gamma(shape, scale))
        elif self.lick_dist == "lognormal":
            cv = max(1e-6, self.lick_cv)
            sigma2 = np.log(1.0 + cv**2)
            sigma = np.sqrt(sigma2)
            mu = np.log(self.lick_mean_s) - 0.5 * sigma2
            dur_s = float(np.random.lognormal(mean=mu, sigma=sigma))
        else:
            raise ValueError(f"Unknown lick_dist={self.lick_dist}")
        steps = int(np.round(dur_s / self.dt_s))
        return max(1, steps), dur_s

    def _effective_detect_prob(self) -> float:
        p = self.p_detect_per_obs
        if self.familiarity_enabled:
            p += self.p_detect_fam_boost * self.familiarity
        return clamp_float(p, 0.0, 1.0)

    def _effective_noise_steps(self) -> int:
        n = int(self.win_rem_noise_steps)
        if self.familiarity_enabled and n > 0:
            n = int(round(n * (1.0 - self.noise_fam_reduction * self.familiarity)))
        return max(0, n)

    def reset(self):
        # Episode boundary familiarity decay (persistent across episodes)
        if self.familiarity_enabled and self.fam_decay_ep > 0.0:
            self.familiarity = clamp_float(self.familiarity * (1.0 - self.fam_decay_ep), 0.0, 1.0)

        self.t = 0
        self.teacher_pos = int(np.random.randint(0, self.grid_size))
        candidates = [i for i in range(self.grid_size) if i != self.teacher_pos]
        self.learner_food_pos = int(np.random.choice(candidates))
        self.learner_pos = int(np.random.randint(0, self.grid_size))

        self.eat_cd = 0
        self.last_lick_step = None
        self.detected_bout_id = None
        self._detect_ttl = None
        self.rewarded_bout_ids = set()

        # build cue-burst timeline
        self.bout_id_at_step = -np.ones(self.max_steps + 1, dtype=np.int32)
        self.bout_start_steps = []
        self.bout_end_steps = []
        self.bout_dur_s_list = []
        bout_id = 0
        k = 1
        while True:
            nominal = k * self.teacher_period_steps
            if nominal >= self.max_steps:
                break
            jitter = int(np.random.randint(-self.teacher_jitter_steps, self.teacher_jitter_steps + 1))
            start = clamp_int(nominal + jitter, 1, self.max_steps - 2)
            dur_steps, dur_s = self._sample_lick_duration_steps()
            end = clamp_int(start + dur_steps - 1, start, self.max_steps - 1)
            self.bout_id_at_step[start:end + 1] = bout_id
            self.bout_start_steps.append(start)
            self.bout_end_steps.append(end)
            self.bout_dur_s_list.append(dur_s)
            bout_id += 1
            k += 1
        self.n_bouts = int(bout_id)
        return self._get_obs(lick_sig=0.0, win_rem=0.0)

    def _teacher_lick_now(self, t: int):
        bid = int(self.bout_id_at_step[t]) if (0 <= t <= self.max_steps) else -1
        return (1, bid) if bid >= 0 else (0, -1)

    def _window_open(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        return 1 if (0 <= dt < self.window_steps) else 0

    def _window_remaining(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        rem = self.window_steps - dt
        return int(max(0, rem))

    def _phase_to_next_nominal(self, t: int) -> int:
        mod = t % self.teacher_period_steps
        return self.teacher_period_steps - mod

    def _get_obs(self, lick_sig: float, win_rem: float):
        lp = float(self.learner_pos) / float(self.grid_size - 1)
        lf = float(self.learner_food_pos) / float(self.grid_size - 1)
        phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period_steps)
        cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown_steps))
        # [pos, food_pos, gated cue, gated window remaining, phase clock, cooldown]
        return np.array([lp, lf, float(lick_sig), float(win_rem), phase, cdn], dtype=np.float32)

    def step(self, action: int):
        self.t += 1
        teacher_lick, bout_id = self._teacher_lick_now(self.t)
        if teacher_lick == 1:
            self.last_lick_step = self.t

        window_open = self._window_open(self.t)
        win_rem_true = self._window_remaining(self.t)

        if self.eat_cd > 0:
            self.eat_cd -= 1

        # optional memory decay for detected bout id
        if self.detect_memory_steps is not None and self._detect_ttl is not None:
            self._detect_ttl -= 1
            if self._detect_ttl <= 0:
                self._detect_ttl = None
                self.detected_bout_id = None

        lick_sig = 0.0
        win_rem = 0.0
        did_observe = 0
        did_eat = 0
        seen_lick = 0

        # movement / observe / eat
        if action == A_LEFT:
            self.learner_pos -= 1
        elif action == A_RIGHT:
            self.learner_pos += 1
        elif action == A_OBS:
            did_observe = 1

            # social blind => OBS reveals nothing
            if self.social_visibility > 0.0:
                p_det = self._effective_detect_prob()
                if teacher_lick == 1 and bout_id >= 0 and (np.random.rand() < p_det):
                    seen_lick = 1
                    self.detected_bout_id = int(bout_id)
                    if self.detect_memory_steps is not None:
                        self._detect_ttl = int(self.detect_memory_steps)

                lick_sig = float(seen_lick)

                if win_rem_true > 0:
                    noise_steps = self._effective_noise_steps()
                    noise = int(np.random.randint(-noise_steps, noise_steps + 1)) if noise_steps > 0 else 0
                    est = clamp_int(win_rem_true + noise, 0, self.window_steps)
                    win_rem = float(est) / float(max(1, self.window_steps))

                # familiarity update: "later can get more information"
                if self.familiarity_enabled and teacher_lick == 1 and bout_id >= 0:
                    self.familiarity = clamp_float(self.familiarity + self.fam_gain_obs, 0.0, 1.0)
                    if seen_lick == 1:
                        self.familiarity = clamp_float(self.familiarity + self.fam_gain_detect, 0.0, 1.0)
        elif action == A_EAT:
            did_eat = 1
        # A_STAY does nothing

        self.learner_pos = clamp_int(self.learner_pos, 0, self.grid_size - 1)

        reward = 0.0
        reward -= self.observe_cost * float(did_observe)
        reward -= self.eat_cost * float(did_eat)

        # shaping: reward OBS during cue only if bout not yet detected
        if did_observe == 1 and teacher_lick == 1 and bout_id >= 0:
            if self.detected_bout_id != int(bout_id):
                reward += self.attend_bonus

        # eat allowed?
        eat_allowed = (did_eat == 1 and self.eat_cd == 0)
        if eat_allowed:
            self.eat_cd = self.eat_cooldown_steps

        # success logic (1 reward per bout)
        eat_valid = 0
        if eat_allowed and (self.learner_pos == self.learner_food_pos) and (window_open == 1) and (self.last_lick_step is not None):
            last_bid = int(self.bout_id_at_step[self.last_lick_step]) if self.last_lick_step <= self.max_steps else -1
            if (last_bid >= 0) and (self.detected_bout_id == last_bid) and (last_bid not in self.rewarded_bout_ids):
                eat_valid = 1
                self.rewarded_bout_ids.add(last_bid)
                reward += 1.0

        done = bool(self.t >= self.max_steps)
        info = {
            "t": self.t,
            "teacher_lick": int(teacher_lick),
            "bout_id": int(bout_id),
            "window_open": int(window_open),
            "win_rem_true": int(win_rem_true),
            "action": int(action),
            "observe": int(did_observe),
            "eat": int(did_eat),
            "eat_allowed": int(eat_allowed),
            "eat_valid": int(eat_valid),
            "learner_pos": int(self.learner_pos),
            "food_pos": int(self.learner_food_pos),
            "seen_lick": int(seen_lick),
            "win_rem_est": float(win_rem),
            "detected_bout_id": -1 if self.detected_bout_id is None else int(self.detected_bout_id),
            "rewarded_bouts": len(self.rewarded_bout_ids),
            "n_bouts": int(self.n_bouts),
            "dt_s": float(self.dt_s),
            "window_steps": int(self.window_steps),
            "social_visibility": float(self.social_visibility),
            "familiarity": float(self.familiarity),
            "p_detect_eff": float(self._effective_detect_prob()) if (self.social_visibility > 0.0) else 0.0,
        }
        obs = self._get_obs(lick_sig=lick_sig, win_rem=win_rem)
        return obs, reward, done, info


# ============================================================
# Plot saving (PDF by default)
# ============================================================
def _save_fig(fig, out_path_no_ext: str, ext: str, pdf_pages: Optional[PdfPages] = None):
    os.makedirs(os.path.dirname(out_path_no_ext), exist_ok=True)
    if ext.lower() == "pdf" and pdf_pages is not None:
        pdf_pages.savefig(fig, bbox_inches="tight")
    else:
        fig.savefig(f"{out_path_no_ext}.{ext}", bbox_inches="tight")

def ema(x, alpha=0.02):
    x = np.asarray(x, dtype=np.float32)
    y = np.zeros_like(x)
    m = 0.0
    for i in range(len(x)):
        m = (1 - alpha) * m + alpha * x[i]
        y[i] = m
    return y

def plot_learning_curves(logs, save_dir="plots", ext="pdf", pdf_pages=None, title_suffix=""):
    ep = np.array([x["ep"] for x in logs])
    succ = ema([x["bout_success_rate"] for x in logs])
    obs = ema([x["obs_rate"] for x in logs])
    obsSeen = ema([x["obs_seen_lick_rate"] for x in logs])
    eatWin = ema([x["eat_in_window_rate"] for x in logs])
    fam = ema([x.get("familiarity", 0.0) for x in logs])

    fig = plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(ep, succ)
    plt.title(f"EMA bout success rate{title_suffix}")
    plt.xlabel("episode"); plt.ylabel("rate")

    plt.subplot(1, 3, 2)
    plt.plot(ep, obs, label="observe")
    plt.plot(ep, obsSeen, label="obs sees cue")
    plt.title("EMA observation metrics")
    plt.xlabel("episode"); plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(ep, eatWin, label="eat in window")
    if np.any(fam > 0):
        plt.plot(ep, fam, label="familiarity")
    plt.title("EMA eating + familiarity")
    plt.xlabel("episode"); plt.legend()

    plt.tight_layout()
    _save_fig(fig, os.path.join(save_dir, "learning_curves"), ext=ext, pdf_pages=pdf_pages)
#     plt.close(fig)

def plot_comparison_curves(results: Dict[str, List[Dict[str, float]]],
                           metric: str,
                           save_dir="plots",
                           ext="pdf",
                           pdf_pages=None,
                           alpha_ema: float = 0.02,
                           title: Optional[str] = None):
    """
    results: {label -> logs}
    metric: key in each logs[i]
    """
    fig = plt.figure(figsize=(7.5, 4.5))
    for label, logs in results.items():
        ep = np.array([x["ep"] for x in logs])
        y = ema([x.get(metric, 0.0) for x in logs], alpha=alpha_ema)
        plt.plot(ep, y, label=label)
    plt.xlabel("episode")
    plt.ylabel(metric)
    plt.title(title or f"EMA {metric} (comparison)")
    plt.legend(fontsize=8)
    plt.tight_layout()
    _save_fig(fig, os.path.join(save_dir, f"compare_{metric}"), ext=ext, pdf_pages=pdf_pages)
#     plt.close(fig)

# ============================================================
# Latent / “DA-like” simulation utilities (algorithm-agnostic)
# ============================================================
def calcium_filter(impulses, dt_s, tau_rise_s=0.2, tau_decay_s=1.2):
    impulses = np.asarray(impulses, dtype=np.float32)
    n = len(impulses)
    L = int(np.ceil(8 * tau_decay_s / dt_s))
    t = np.arange(L, dtype=np.float32) * dt_s
    k = (1 - np.exp(-t / max(1e-6, tau_rise_s))) * np.exp(-t / max(1e-6, tau_decay_s))
    k = k / (k.sum() + 1e-9)
    y = np.convolve(impulses, k, mode="full")[:n]
    return y.astype(np.float32)

def make_event_indices(tr):
    lick = np.array(tr["teacher_lick"], dtype=int)
    obs = np.array(tr["observe"], dtype=int)
    seen = np.array(tr["seen_lick"], dtype=int) if "seen_lick" in tr else None
    reward = np.array(tr["eat_valid"], dtype=int)

    lick_end = np.where((lick[:-1] == 1) & (lick[1:] == 0))[0] + 1
    obs_idx = np.where(obs == 1)[0]
    if seen is not None:
        obs_seen_lick_idx = np.where((obs == 1) & (seen == 1))[0]
    else:
        obs_seen_lick_idx = np.array([], dtype=int)
    rew_idx = np.where(reward == 1)[0]

    return {
        "lick_end": lick_end,
        "obs": obs_idx,
        "obs_seen_lick": obs_seen_lick_idx,
        "reward": rew_idx,
    }

def peri_event_average(signal, event_idx, pre_steps=10, post_steps=20):
    signal = np.asarray(signal, dtype=np.float32)
    out = []
    for e in event_idx:
        a = e - pre_steps
        b = e + post_steps + 1
        if a < 0 or b > len(signal):
            continue
        out.append(signal[a:b])
    if len(out) == 0:
        return None
    return np.mean(np.stack(out, axis=0), axis=0)

def gae_advantages(rewards, values, dones, gamma=0.99, lam=0.95):
    """GAE(λ) computed from r_t, V_t, done_t. values length T+1 (bootstrapped)."""
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        nonterminal = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * values[t+1] * nonterminal - values[t]
        gae = delta + gamma * lam * nonterminal * gae
        adv[t] = gae
    return adv

def simulate_latent_models_for_episode(tr, agent, gamma=0.99, ep_frac=0.0, gae_lam=0.95):
    """
    Candidate latent signal hypotheses (paper-aligned, but algorithm-agnostic):
      - reward_RPE (TD error)
      - gae_adv (advantage-like)
      - social_cue_RPE / social_value (ΔV on Observe due to revealing gated cue features)
      - action_salience
      - info_gain (surprise-scaled Observe-detect)
      - optional: policy_entropy (if agent exposes policy entropy)
    """
    r = np.array(tr["reward"], dtype=np.float32)
    obs = np.array(tr["observe"], dtype=int)
    eat = np.array(tr["eat"], dtype=int)
    seen_lick = np.array(tr["seen_lick"], dtype=int)
    phase = np.array(tr["phase"], dtype=np.float32)
    S = np.array(tr["state"], dtype=np.float32)
    S2 = np.array(tr["next_state"], dtype=np.float32)
    done = np.array(tr["done"], dtype=np.float32)

    V_s = agent.value_batch(S)
    V_s2 = agent.value_batch(S2)

    # TD error (reward_RPE)
    rpe = r + gamma * V_s2 * (1.0 - done) - V_s

    # GAE advantage-like signal
    V_boot = np.concatenate([V_s, V_s2[-1:]], axis=0)  # crude: last bootstrap
    adv = gae_advantages(r, V_boot[:len(r)+1], done, gamma=gamma, lam=gae_lam)

    # Observation-driven ΔV: V_post - V_prior (social features zeroed)
    S2_prior = S2.copy()
    S2_prior[:, 2] = 0.0  # lick_sig
    S2_prior[:, 3] = 0.0  # win_rem
    V_post = V_s2
    V_prior = agent.value_batch(S2_prior)
    deltaV = (V_post - V_prior) * obs.astype(np.float32)

    # Action salience
    sal = np.zeros(len(r), dtype=np.float32)
    sal += 1.0 * (obs == 1).astype(np.float32)
    sal += 0.6 * (eat == 1).astype(np.float32)

    # Info gain / surprise when OBS successfully detects cue
    sigma = 0.25 * (1.0 - ep_frac) + 0.10 * ep_frac
    p = np.exp(-(phase / max(1e-6, sigma)) ** 2)  # 0..1
    surprise = -np.log(p + 1e-6)
    info_gain = surprise * ((obs == 1) & (seen_lick == 1)).astype(np.float32)

    impulses = {
        "reward_RPE": 0.8 * rpe,
        "gae_adv": 0.7 * adv,
        "social_cue_RPE": 1.2 * deltaV,
        "social_value": 0.6 * deltaV,
        "action_salience": 0.3 * sal,
        "info_gain": 0.25 * info_gain,
    }

    # Optional policy entropy latent (actor-critic learners)
    if hasattr(agent, "policy_entropy_batch"):
        ent = agent.policy_entropy_batch(S)  # shape [T]
        if ent is not None:
            impulses["policy_entropy"] = 0.15 * ent.astype(np.float32)

    return impulses

def plot_da_peri_event(da_by_ep, traces, dt_s, model_name, event_key="reward",
                       pre_s=6, post_s=10, save_dir="plots", ext="pdf", pdf_pages=None):
    pre = int(round(pre_s / dt_s))
    post = int(round(post_s / dt_s))
    x = (np.arange(pre + post + 1) - pre) * dt_s

    n = len(traces)
    early_idx = range(0, max(1, n // 5))
    late_idx = range(max(0, n - n // 5), n)

    def mean_peri(idxs):
        mats = []
        for i in idxs:
            ev = make_event_indices(traces[i])[event_key]
            m = peri_event_average(da_by_ep[i], ev, pre_steps=pre, post_steps=post)
            if m is not None:
                mats.append(m)
        if len(mats) == 0:
            return None
        return np.mean(np.stack(mats, axis=0), axis=0)

    e = mean_peri(early_idx)
    l = mean_peri(late_idx)

    fig = plt.figure(figsize=(6, 4))
    if e is not None:
        plt.plot(x, e, label="early (first 20%)")
    if l is not None:
        plt.plot(x, l, label="late (last 20%)")
    plt.axvline(0, linestyle="--")
    plt.title(f"{model_name}: peri-{event_key}")
    plt.xlabel("time (s)")
    plt.ylabel("simulated signal")
    plt.legend()
    plt.tight_layout()

    out = os.path.join(save_dir, f"{model_name}_peri_{event_key}")
    _save_fig(fig, out, ext=ext, pdf_pages=pdf_pages)
#     plt.close(fig)

def simulate_and_plot_latents(traces, agent, dt_s, gamma=0.99,
                              save_dir="plots", ext="pdf", pdf_pages=None):
    impulses_by_model = defaultdict(list)
    n = len(traces)
    for i, tr in enumerate(traces):
        ep_frac = i / max(1, n - 1)
        imp = simulate_latent_models_for_episode(tr, agent, gamma=gamma, ep_frac=ep_frac)
        for k, v in imp.items():
            impulses_by_model[k].append(v)

    phot_by_model = {}
    for k, imps in impulses_by_model.items():
        phot_by_model[k] = [calcium_filter(x, dt_s=dt_s, tau_rise_s=0.2, tau_decay_s=1.2) for x in imps]

    for k, da_eps in phot_by_model.items():
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="lick_end",
                           pre_s=6, post_s=10, save_dir=save_dir, ext=ext, pdf_pages=pdf_pages)
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="reward",
                           pre_s=6, post_s=10, save_dir=save_dir, ext=ext, pdf_pages=pdf_pages)
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="obs_seen_lick",
                           pre_s=6, post_s=10, save_dir=save_dir, ext=ext, pdf_pages=pdf_pages)

    return phot_by_model

def summarize_reward_rpe_peri(traces, phot_by_model, dt_s, pre_s=6, post_s=10, which="late"):
    """Return peri-event mean for reward_RPE around reward events, averaged over early/late episodes."""
    if phot_by_model is None or "reward_RPE" not in phot_by_model:
        return None
    da_eps = phot_by_model["reward_RPE"]
    pre = int(round(pre_s / dt_s))
    post = int(round(post_s / dt_s))
    n = len(traces)
    if n == 0:
        return None
    if which == "early":
        idxs = range(0, max(1, n // 5))
    else:
        idxs = range(max(0, n - n // 5), n)
    mats = []
    for i in idxs:
        ev = make_event_indices(traces[i])["reward"]
        m = peri_event_average(da_eps[i], ev, pre_steps=pre, post_steps=post)
        if m is not None:
            mats.append(m)
    if len(mats) == 0:
        return None
    return np.mean(np.stack(mats, axis=0), axis=0)

def plot_reward_rpe_peri_comparison(peri_by_label: Dict[str, np.ndarray], dt_s, pre_s=6, post_s=10,
                                   save_dir="plots", ext="pdf", pdf_pages=None, title="Reward_RPE peri-reward (late)"):
    pre = int(round(pre_s / dt_s))
    post = int(round(post_s / dt_s))
    x = (np.arange(pre + post + 1) - pre) * dt_s

    fig = plt.figure(figsize=(7.5, 4.5))
    for label, y in peri_by_label.items():
        if y is None:
            continue
        plt.plot(x, y, label=label)
    plt.axvline(0, linestyle="--")
    plt.xlabel("time (s)")
    plt.ylabel("simulated signal")
    plt.title(title)
    plt.legend(fontsize=8)
    plt.tight_layout()
    _save_fig(fig, os.path.join(save_dir, "compare_reward_RPE_peri_reward_late"), ext=ext, pdf_pages=pdf_pages)

# ============================================================
# Agents (multiple learner models)
# ============================================================
class BaseAgent:
    def act(self, s: np.ndarray):
        """Return (action:int, extra:dict)."""
        raise NotImplementedError

    def record_transition(self, s, a, r, s2, done, extra):
        pass

    def step_update(self):
        return None

    def episode_update(self):
        return None

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        raise NotImplementedError

# -------------------------
# DQN (dueling + double + huber + soft target)
# -------------------------
class DuelingQNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.V = nn.Linear(hidden, 1)
        self.A = nn.Linear(hidden, action_dim)

    def forward(self, x):
        h = self.backbone(x)
        v = self.V(h)
        a = self.A(h)
        return v + (a - a.mean(dim=1, keepdim=True))

class TorchReplayBuffer:
    def __init__(self, capacity, state_dim, device):
        self.capacity = int(capacity)
        self.device = device
        self.state = torch.zeros((capacity, state_dim), dtype=torch.float32, device=device)
        self.action = torch.zeros(capacity, dtype=torch.int64, device=device)
        self.reward = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.next_state = torch.zeros((capacity, state_dim), dtype=torch.float32, device=device)
        self.done = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.idx = 0
        self.size = 0

    def push(self, s, a, r, s2, done):
        self.state[self.idx] = torch.tensor(s, dtype=torch.float32, device=self.device)
        self.action[self.idx] = int(a)
        self.reward[self.idx] = float(r)
        self.next_state[self.idx] = torch.tensor(s2, dtype=torch.float32, device=self.device)
        self.done[self.idx] = float(done)
        self.idx = (self.idx + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        indices = torch.randint(0, self.size, (batch_size,), device=self.device)
        return (
            self.state[indices],
            self.action[indices].unsqueeze(1),
            self.reward[indices],
            self.next_state[indices],
            self.done[indices],
        )

    def __len__(self):
        return self.size

class DQNAgent(BaseAgent):
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        batch_size=256,
        buffer_size=50000,
        eps_start=1.0,
        eps_end=0.05,
        eps_decay=0.9995,
        tau=0.02,
        grad_clip=10.0,
        device=None,
        use_compile=False,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.batch_size = int(batch_size)
        self.tau = float(tau)
        self.grad_clip = float(grad_clip)

        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.q = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt.load_state_dict(self.q.state_dict())

        if use_compile and hasattr(torch, "compile"):
            try:
                self.q = torch.compile(self.q)
            except Exception:
                pass

        self.opt = optim.Adam(self.q.parameters(), lr=float(lr))
        self.loss_fn = nn.SmoothL1Loss()
        self.rb = TorchReplayBuffer(buffer_size, state_dim, self.device)
        self.scaler = GradScaler(enabled=self.device.type == "cuda")

    def act(self, s):
        if np.random.rand() < self.eps:
            return int(np.random.randint(self.action_dim)), {}
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            qv = self.q(st)
        return int(torch.argmax(qv, dim=1).item()), {}

    def record_transition(self, s, a, r, s2, done, extra):
        self.rb.push(s, a, r, s2, done)

    def step_update(self, updates_per_step=1):
        if len(self.rb) < self.batch_size:
            self.eps = max(self.eps_end, self.eps * self.eps_decay)
            return None

        last_loss = None
        for _ in range(int(updates_per_step)):
            s, a, r, s2, d = self.rb.sample(self.batch_size)
            with autocast(enabled=self.device.type == "cuda"):
                q_sa = self.q(s).gather(1, a).squeeze(1)
                with torch.no_grad():
                    a_star = torch.argmax(self.q(s2), dim=1, keepdim=True)
                    q_next = self.qt(s2).gather(1, a_star).squeeze(1)
                    target = r + self.gamma * q_next * (1.0 - d)
                loss = self.loss_fn(q_sa, target)

            self.opt.zero_grad()
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.opt)
            nn.utils.clip_grad_norm_(self.q.parameters(), self.grad_clip)
            self.scaler.step(self.opt)
            self.scaler.update()

            with torch.no_grad():
                for p, pt in zip(self.q.parameters(), self.qt.parameters()):
                    pt.data.mul_(1.0 - self.tau).add_(self.tau * p.data)

            last_loss = float(loss.item())

        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        return last_loss

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            qv = self.q(st)
            v = torch.max(qv, dim=1).values
        return v.detach().cpu().numpy().astype(np.float32)

# -------------------------
# PPO (actor-critic)
# -------------------------
class ActorCriticNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.pi = nn.Linear(hidden, action_dim)
        self.v = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.body(x)
        logits = self.pi(h)
        value = self.v(h).squeeze(-1)
        return logits, value

class PPOAgent(BaseAgent):
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_ratio=0.2,
        vf_coef=0.5,
        ent_coef=0.01,
        train_epochs=4,
        minibatch_size=256,
        max_grad_norm=1.0,
        device=None,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.gae_lambda = float(gae_lambda)
        self.clip_ratio = float(clip_ratio)
        self.vf_coef = float(vf_coef)
        self.ent_coef = float(ent_coef)
        self.train_epochs = int(train_epochs)
        self.minibatch_size = int(minibatch_size)
        self.max_grad_norm = float(max_grad_norm)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.net = ActorCriticNet(self.state_dim, self.action_dim).to(self.device)
        self.opt = optim.Adam(self.net.parameters(), lr=float(lr))

        self.roll = []  # stores dicts per step

    def act(self, s):
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            logits, v = self.net(st)
            dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample()
            logp = dist.log_prob(a)
            ent = dist.entropy()
        return int(a.item()), {"logp": float(logp.item()), "v": float(v.item()), "ent": float(ent.item())}

    def record_transition(self, s, a, r, s2, done, extra):
        self.roll.append({
            "s": np.array(s, dtype=np.float32),
            "a": int(a),
            "r": float(r),
            "done": float(done),
            "logp": float(extra.get("logp", 0.0)),
            "v": float(extra.get("v", 0.0)),
        })

    def episode_update(self):
        if len(self.roll) < 8:
            self.roll.clear()
            return None

        S = np.stack([x["s"] for x in self.roll], axis=0).astype(np.float32)
        A = np.array([x["a"] for x in self.roll], dtype=np.int64)
        R = np.array([x["r"] for x in self.roll], dtype=np.float32)
        D = np.array([x["done"] for x in self.roll], dtype=np.float32)
        LOGP_OLD = np.array([x["logp"] for x in self.roll], dtype=np.float32)
        V = np.array([x["v"] for x in self.roll], dtype=np.float32)

        # bootstrap last value with current critic (on last state)
        with torch.no_grad():
            st_last = torch.tensor(S[-1], dtype=torch.float32, device=self.device).unsqueeze(0)
            _, v_last = self.net(st_last)
            v_last = float(v_last.item())
        V_ext = np.concatenate([V, np.array([v_last], dtype=np.float32)], axis=0)

        ADV = gae_advantages(R, V_ext, D, gamma=self.gamma, lam=self.gae_lambda)
        RET = ADV + V  # return target

        # normalize advantages
        ADV = (ADV - ADV.mean()) / (ADV.std() + 1e-8)

        # torch tensors
        ts = torch.tensor(S, dtype=torch.float32, device=self.device)
        ta = torch.tensor(A, dtype=torch.int64, device=self.device)
        told = torch.tensor(LOGP_OLD, dtype=torch.float32, device=self.device)
        tadv = torch.tensor(ADV, dtype=torch.float32, device=self.device)
        tret = torch.tensor(RET, dtype=torch.float32, device=self.device)

        n = len(S)
        idx = np.arange(n)
        info = {}

        for _ in range(self.train_epochs):
            np.random.shuffle(idx)
            for start in range(0, n, self.minibatch_size):
                mb = idx[start:start+self.minibatch_size]
                logits, vpred = self.net(ts[mb])
                dist = torch.distributions.Categorical(logits=logits)

                logp = dist.log_prob(ta[mb])
                ratio = torch.exp(logp - told[mb])

                clip = torch.clamp(ratio, 1.0 - self.clip_ratio, 1.0 + self.clip_ratio)
                pg_loss = -(torch.min(ratio * tadv[mb], clip * tadv[mb])).mean()

                v_loss = ((vpred - tret[mb]) ** 2).mean()
                ent = dist.entropy().mean()

                loss = pg_loss + self.vf_coef * v_loss - self.ent_coef * ent

                self.opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), self.max_grad_norm)
                self.opt.step()

                info = {
                    "pg_loss": float(pg_loss.item()),
                    "v_loss": float(v_loss.item()),
                    "ent": float(ent.item()),
                    "loss": float(loss.item()),
                }

        self.roll.clear()
        return info

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            _, v = self.net(st)
        return v.detach().cpu().numpy().astype(np.float32)

    def policy_entropy_batch(self, states: np.ndarray):
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            logits, _ = self.net(st)
            dist = torch.distributions.Categorical(logits=logits)
            ent = dist.entropy()
        return ent.detach().cpu().numpy().astype(np.float32)

# -------------------------
# Tabular Q-learning (light baseline)
# -------------------------
class TabularQAgent(BaseAgent):
    def __init__(self, action_dim=5, alpha=0.25, gamma=0.99, eps_start=1.0, eps_end=0.05, eps_decay=0.9995,
                 phase_bins=10, win_bins=7, cd_bins=6):
        self.action_dim = int(action_dim)
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.phase_bins = int(phase_bins)
        self.win_bins = int(win_bins)
        self.cd_bins = int(cd_bins)

        self.Q = defaultdict(lambda: np.zeros(self.action_dim, dtype=np.float32))

    def _disc(self, s: np.ndarray):
        # s = [lp, lf, lick_sig, win_rem, phase, cdn]
        lp, lf, lick, win, phase, cd = s.tolist()
        p = int(np.round(lp * 14))
        f = int(np.round(lf * 14))
        lickb = int(lick > 0.5)
        winb = int(np.round(win * (self.win_bins - 1)))
        winb = clamp_int(winb, 0, self.win_bins - 1)
        phb = int(np.floor(phase * self.phase_bins))
        phb = clamp_int(phb, 0, self.phase_bins - 1)
        cdb = int(np.round(cd * (self.cd_bins - 1)))
        cdb = clamp_int(cdb, 0, self.cd_bins - 1)
        return (p, f, lickb, winb, phb, cdb)

    def act(self, s):
        key = self._disc(s)
        if np.random.rand() < self.eps:
            a = int(np.random.randint(self.action_dim))
        else:
            a = int(np.argmax(self.Q[key]))
        return a, {"key": key}

    def record_transition(self, s, a, r, s2, done, extra):
        k = extra["key"]
        k2 = self._disc(s2)
        q = self.Q[k]
        target = float(r) + self.gamma * (0.0 if done else float(np.max(self.Q[k2])))
        q[a] = (1 - self.alpha) * q[a] + self.alpha * target
        self.eps = max(self.eps_end, self.eps * self.eps_decay)

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        out = np.zeros(len(states), dtype=np.float32)
        for i, s in enumerate(states):
            out[i] = float(np.max(self.Q[self._disc(s)]))
        return out


# ============================================================
# Training + logging (works for all agents above)
# ============================================================
def train_social_with_latents(env, agent: BaseAgent, episodes=1200, log_every=100,
                             updates_per_step=4, early_stop_threshold=0.8, early_stop_window=100):
    logs = []
    traces = []
    success_history = deque(maxlen=early_stop_window)

    for ep in range(1, episodes + 1):
        s = env.reset()
        done = False

        ep_reward = 0.0
        succ_bouts = 0
        n_bouts = max(1, env.n_bouts)
        obs_steps = 0
        obs_seen = 0
        eat_in_win = 0

        tr = defaultdict(list)

        while not done:
            a, extra = agent.act(s)
            s2, r, done, info = env.step(a)
            ep_reward += r

            if info["eat_valid"] == 1:
                succ_bouts += 1
            if info["observe"] == 1:
                obs_steps += 1
                if info["seen_lick"] == 1:
                    obs_seen += 1
            if info["eat"] == 1 and info["window_open"] == 1:
                eat_in_win += 1

            # trace for latent sims
            tr["state"].append(s.copy())
            tr["next_state"].append(s2.copy())
            tr["reward"].append(float(r))
            tr["done"].append(float(done))

            tr["teacher_lick"].append(int(info["teacher_lick"]))
            tr["window_open"].append(int(info["window_open"]))
            tr["observe"].append(int(info["observe"]))
            tr["eat"].append(int(info["eat"]))
            tr["eat_valid"].append(int(info["eat_valid"]))
            tr["seen_lick"].append(int(info["seen_lick"]))
            tr["win_rem_est"].append(float(info["win_rem_est"]))
            tr["phase"].append(float(s[4]))

            # learner updates
            agent.record_transition(s, a, r, s2, done, extra)

            if isinstance(agent, DQNAgent):
                agent.step_update(updates_per_step=updates_per_step)

            s = s2

        if hasattr(agent, "episode_update"):
            agent.episode_update()

        for k in list(tr.keys()):
            tr[k] = np.array(tr[k])
        traces.append(tr)

        logs.append({
            "ep": ep,
            "bout_success_rate": float(succ_bouts / n_bouts),
            "mean_reward": float(ep_reward),
            "obs_rate": float(obs_steps / max(1, env.max_steps)),
            "obs_seen_lick_rate": float(obs_seen / max(1, env.max_steps)),
            "eat_in_window_rate": float(eat_in_win / max(1, env.max_steps)),
            "eps": float(getattr(agent, "eps", np.nan)),
            "familiarity": float(getattr(env, "familiarity", 0.0)),
            "p_detect_eff": float(getattr(env, "_effective_detect_prob")()) if getattr(env, "social_visibility", 1.0) > 0.0 else 0.0,
        })

        success_history.append(logs[-1]["bout_success_rate"])

        if ep % log_every == 0:
            recent = logs[-log_every:]
            print(
                f"[ep {ep:4d}] "
                f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f} "
                f"obs={np.mean([x['obs_rate'] for x in recent]):.3f} "
                f"obsSeen={np.mean([x['obs_seen_lick_rate'] for x in recent]):.3f} "
                f"eatWin={np.mean([x['eat_in_window_rate'] for x in recent]):.3f} "
                f"R={np.mean([x['mean_reward'] for x in recent]):.3f} "
                f"eps={logs[-1]['eps']:.3f} "
                f"fam={logs[-1]['familiarity']:.2f}"
            )

        if len(success_history) == early_stop_window and np.mean(success_history) > early_stop_threshold:
            print(f"Early stopping at episode {ep}: Avg success rate {np.mean(success_history):.3f} > {early_stop_threshold}")
            break

    return logs, traces


# ============================================================
# Condition builder (keeps your old details; only overrides when you ask)
# ============================================================
def build_env_for_condition(args, teacher_pal: str, learner_profile: str, familiarity_mode: str):
    # teacher palatability -> bout duration mean
    lick_mean = args.lick_mean_s
    if teacher_pal == "high":
        lick_mean = args.pal_high_lick_mean_s
    elif teacher_pal == "low":
        lick_mean = args.pal_low_lick_mean_s

    # defaults = your old behavior
    env_kwargs = dict(
        grid_size=args.grid_size,
        dt_s=args.dt_s,
        max_time_s=args.max_time_s,
        teacher_period_s=args.teacher_period_s,
        teacher_jitter_s=args.teacher_jitter_s,
        lick_mean_s=lick_mean,
        lick_dist=args.lick_dist,
        lick_cv=args.lick_cv,
        window_s=args.window_s,
        eat_cooldown_s=args.eat_cooldown_s,
        observe_cost=args.observe_cost,
        eat_cost=args.eat_cost,
        attend_bonus=args.attend_bonus,
        p_detect_per_obs=args.p_detect_per_obs,
        win_rem_noise_steps=args.win_rem_noise_steps,

        social_visibility=1.0,
        detect_memory_steps=None,
        familiarity_enabled=(familiarity_mode == "on"),
        familiarity_init=args.fam_init,
        fam_decay_ep=args.fam_decay_ep,
        fam_gain_obs=args.fam_gain_obs,
        fam_gain_detect=args.fam_gain_detect,
        p_detect_fam_boost=args.p_detect_fam_boost,
        noise_fam_reduction=args.noise_fam_reduction,
    )

    # learner profile modifies "access to others" and tracking fidelity
    if learner_profile == "normal":
        pass
    elif learner_profile == "blind":
        env_kwargs["social_visibility"] = 0.0
        # familiarity doesn't matter if blind; keep but harmless
    elif learner_profile == "autistic":
        # "hardly track others": lower detect prob, noisier window estimate, limited memory
        env_kwargs["p_detect_per_obs"] = args.aut_p_detect_per_obs
        env_kwargs["win_rem_noise_steps"] = args.aut_win_rem_noise_steps
        env_kwargs["detect_memory_steps"] = args.aut_detect_memory_steps
        # optionally make observation more costly (attention cost)
        env_kwargs["observe_cost"] = args.aut_observe_cost

        # typically slower familiarity accumulation
        if familiarity_mode == "on":
            env_kwargs["fam_gain_obs"] = args.aut_fam_gain_obs
            env_kwargs["fam_gain_detect"] = args.aut_fam_gain_detect
            env_kwargs["p_detect_fam_boost"] = args.aut_p_detect_fam_boost
            env_kwargs["noise_fam_reduction"] = args.aut_noise_fam_reduction
    else:
        raise ValueError(f"Unknown learner_profile={learner_profile}")

    return SocialLickEnv1D(**env_kwargs)

def build_agent(args, state_dim=6, action_dim=5):
    if args.algo == "dqn":
        return DQNAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            lr=args.lr,
            gamma=args.gamma,
            batch_size=args.batch_size,
            buffer_size=args.buffer_size,
            eps_start=args.eps_start,
            eps_end=args.eps_end,
            eps_decay=args.eps_decay,
            tau=args.tau,
            grad_clip=args.grad_clip,
            use_compile=args.use_compile,
        )
    elif args.algo == "ppo":
        return PPOAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            lr=args.lr,
            gamma=args.gamma,
            gae_lambda=args.gae_lambda,
            clip_ratio=args.clip_ratio,
            vf_coef=args.vf_coef,
            ent_coef=args.ent_coef,
            train_epochs=args.ppo_epochs,
            minibatch_size=args.minibatch_size,
            max_grad_norm=args.max_grad_norm,
        )
    elif args.algo == "tabular":
        return TabularQAgent(
            action_dim=action_dim,
            alpha=args.tab_alpha,
            gamma=args.gamma,
            eps_start=args.eps_start,
            eps_end=args.eps_end,
            eps_decay=args.eps_decay,
        )
    else:
        raise ValueError(f"Unknown algo: {args.algo}")

# ============================================================
# Run experiment (single condition OR comparison suite)
# ============================================================
def run_single_condition(args, teacher_pal: str, learner_profile: str, familiarity_mode: str, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)

    env = build_env_for_condition(args, teacher_pal=teacher_pal, learner_profile=learner_profile, familiarity_mode=familiarity_mode)
    agent = build_agent(args, state_dim=6, action_dim=5)

    pdf_pages = None
    if args.plot_ext.lower() == "pdf" and args.multipage_pdf:
        pdf_pages = PdfPages(os.path.join(out_dir, f"all_figures_{args.algo}.pdf"))

    logs, traces = train_social_with_latents(
        env, agent,
        episodes=args.episodes,
        log_every=max(50, args.episodes // 12),
        updates_per_step=args.updates_per_step,
        early_stop_threshold=args.early_stop_threshold,
        early_stop_window=args.early_stop_window,
    )

    phot_by_model = None
    if not args.no_plots:
        title_suffix = f" ({teacher_pal}, {learner_profile}, fam={familiarity_mode})"
        plot_learning_curves(logs, save_dir=out_dir, ext=args.plot_ext, pdf_pages=pdf_pages, title_suffix=title_suffix)
        phot_by_model = simulate_and_plot_latents(traces, agent, dt_s=env.dt_s, gamma=args.gamma,
                                                  save_dir=out_dir, ext=args.plot_ext, pdf_pages=pdf_pages)

    if pdf_pages is not None:
        pdf_pages.close()

    return env, agent, logs, traces, phot_by_model

def run_comparison_suite(args):
    """
    Compares:
      - learner profiles: normal vs social-blind vs autistic
      - teacher palatability: high vs low (shorter consumption bouts)
      - familiarity: off vs on (learning-to-track others)
    Keeps your old environment/task details; just runs multiple conditions and plots overlays.
    """
    base_dir = args.save_dir
    os.makedirs(base_dir, exist_ok=True)

    # A single multi-page PDF collecting the *comparison* figures (optional)
    cmp_pdf = None
    if args.plot_ext.lower() == "pdf" and args.multipage_pdf:
        cmp_pdf = PdfPages(os.path.join(base_dir, f"comparison_suite_{args.algo}.pdf"))

    teacher_pals = ["high", "low"] if args.compare_teacher_pal else [args.teacher_pal]
    profiles = ["normal", "blind", "autistic"] if args.compare_profiles else [args.learner_profile]
    fam_modes = ["off", "on"] if args.compare_familiarity else [args.familiarity]

    results_logs = {}
    late_peri = {}

    # Use deterministic but different seeds per condition unless shared_seed is requested
    cond_list = []
    for tp in teacher_pals:
        for pr in profiles:
            for fm in fam_modes:
                cond_list.append((tp, pr, fm))

    for i, (tp, pr, fm) in enumerate(cond_list):
        if args.shared_seed_across_conditions:
            set_seed(args.seed)
        else:
            set_seed(args.seed + 1000 * i)

        label = f"T={tp}|L={pr}|F={fm}"
        out_dir = os.path.join(base_dir, label.replace("|", "__").replace("=", "_"))
        print("\n" + "=" * 80)
        print(f"Running condition: {label} -> {out_dir}")
        env, agent, logs, traces, phot_by_model = run_single_condition(args, tp, pr, fm, out_dir=out_dir)
        results_logs[label] = logs

        if phot_by_model is not None and not args.no_plots:
            lp = summarize_reward_rpe_peri(traces, phot_by_model, dt_s=env.dt_s, which="late")
            late_peri[label] = lp

    # overlay comparison plots (EMA curves)
    if not args.no_plots:
        plot_comparison_curves(results_logs, metric="bout_success_rate", save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                               title="EMA bout success rate (conditions)")
        plot_comparison_curves(results_logs, metric="mean_reward", save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                               title="EMA episode mean reward (conditions)")
        plot_comparison_curves(results_logs, metric="obs_rate", save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                               title="EMA observe rate (conditions)")
        plot_comparison_curves(results_logs, metric="obs_seen_lick_rate", save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                               title="EMA OBS sees cue rate (conditions)")

        if any(v is not None for v in late_peri.values()):
            plot_reward_rpe_peri_comparison(late_peri, dt_s=args.dt_s, pre_s=6, post_s=10,
                                            save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf)

    if cmp_pdf is not None:
        cmp_pdf.close()

    # Print a compact summary table to terminal
    def early_late_mean(logs, key):
        y = np.array([x.get(key, 0.0) for x in logs], dtype=np.float32)
        if len(y) == 0:
            return (np.nan, np.nan)
        k = max(1, len(y) // 5)
        return (float(np.mean(y[:k])), float(np.mean(y[-k:])))

    print("\n" + "=" * 80)
    print("SUMMARY (mean over first/last 20% episodes)")
    for label, logs in results_logs.items():
        es, ls = early_late_mean(logs, "bout_success_rate")
        er, lr = early_late_mean(logs, "mean_reward")
        print(f"{label:28s} | succ early/late: {es:6.3f}/{ls:6.3f} | R early/late: {er:7.3f}/{lr:7.3f}")

    return results_logs

def run_experiment(args):
    set_seed(args.seed)

    if args.compare_suite:
        return run_comparison_suite(args)

    # single condition (backwards compatible)
    env = build_env_for_condition(args, teacher_pal=args.teacher_pal, learner_profile=args.learner_profile, familiarity_mode=args.familiarity)
    agent = build_agent(args, state_dim=6, action_dim=5)

    save_dir = args.save_dir
    os.makedirs(save_dir, exist_ok=True)

    pdf_pages = None
    if args.plot_ext.lower() == "pdf" and args.multipage_pdf:
        pdf_pages = PdfPages(os.path.join(save_dir, f"all_figures_{args.algo}.pdf"))

    logs, traces = train_social_with_latents(
        env, agent,
        episodes=args.episodes,
        log_every=max(50, args.episodes // 12),
        updates_per_step=args.updates_per_step,
        early_stop_threshold=args.early_stop_threshold,
        early_stop_window=args.early_stop_window,
    )

    if not args.no_plots:
        plot_learning_curves(logs, save_dir=save_dir, ext=args.plot_ext, pdf_pages=pdf_pages,
                             title_suffix=f" (T={args.teacher_pal}, L={args.learner_profile}, fam={args.familiarity})")
        simulate_and_plot_latents(traces, agent, dt_s=env.dt_s, gamma=args.gamma,
                                  save_dir=save_dir, ext=args.plot_ext, pdf_pages=pdf_pages)

    if pdf_pages is not None:
        pdf_pages.close()

    return logs, traces


def build_argparser():
    p = argparse.ArgumentParser(description="Active observational learning (multi-algo) + latent signal sims + condition suite")

    # algo
    p.add_argument("--algo", type=str, default="dqn", choices=["dqn", "ppo", "tabular"])

    # run
    p.add_argument("--episodes", type=int, default=1200)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--save_dir", type=str, default="plots")
    p.add_argument("--no_plots", action="store_true")
    p.add_argument("--plot_ext", type=str, default="pdf", choices=["pdf", "png"])
    p.add_argument("--multipage_pdf", action="store_true", help="If set + plot_ext=pdf, also writes one multi-page PDF.")
    p.add_argument("--updates_per_step", type=int, default=4)

    # environment (your original knobs)
    p.add_argument("--grid_size", type=int, default=15)
    p.add_argument("--dt_s", type=float, default=0.5)
    p.add_argument("--max_time_s", type=float, default=120.0)
    p.add_argument("--teacher_period_s", type=float, default=30.0)
    p.add_argument("--teacher_jitter_s", type=float, default=6.0)
    p.add_argument("--lick_mean_s", type=float, default=1.0)
    p.add_argument("--lick_dist", type=str, default="gamma", choices=["gamma", "lognormal"])
    p.add_argument("--lick_cv", type=float, default=0.8)
    p.add_argument("--window_s", type=float, default=3.0)
    p.add_argument("--eat_cooldown_s", type=float, default=2.0)
    p.add_argument("--observe_cost", type=float, default=0.002)
    p.add_argument("--eat_cost", type=float, default=0.005)
    p.add_argument("--attend_bonus", type=float, default=0.004)
    p.add_argument("--p_detect_per_obs", type=float, default=0.70)
    p.add_argument("--win_rem_noise_steps", type=int, default=1)

    # common learning
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--eps_start", type=float, default=1.0)
    p.add_argument("--eps_end", type=float, default=0.05)
    p.add_argument("--eps_decay", type=float, default=0.9995)

    # DQN
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--buffer_size", type=int, default=50000)
    p.add_argument("--tau", type=float, default=0.02)
    p.add_argument("--grad_clip", type=float, default=10.0)
    p.add_argument("--use_compile", action="store_true")

    # PPO
    p.add_argument("--gae_lambda", type=float, default=0.95)
    p.add_argument("--clip_ratio", type=float, default=0.2)
    p.add_argument("--vf_coef", type=float, default=0.5)
    p.add_argument("--ent_coef", type=float, default=0.01)
    p.add_argument("--ppo_epochs", type=int, default=4)
    p.add_argument("--minibatch_size", type=int, default=256)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    # Tabular Q
    p.add_argument("--tab_alpha", type=float, default=0.25)

    # early stopping
    p.add_argument("--early_stop_threshold", type=float, default=0.8)
    p.add_argument("--early_stop_window", type=int, default=100)

    # ============================================================
    # NEW: condition controls (do not change old behavior unless used)
    # ============================================================
    p.add_argument("--teacher_pal", type=str, default="default", choices=["default", "high", "low"],
                   help="Teacher palatability condition. 'high'/'low' only changes lick_mean_s via pal_* args.")
    p.add_argument("--pal_high_lick_mean_s", type=float, default=1.4,
                   help="Mean bout duration (s) for high palatability teacher.")
    p.add_argument("--pal_low_lick_mean_s", type=float, default=0.6,
                   help="Mean bout duration (s) for low palatability teacher (shorter consumption).")

    p.add_argument("--learner_profile", type=str, default="normal", choices=["normal", "blind", "autistic"],
                   help="Learner perception/tracking profile.")
    p.add_argument("--familiarity", type=str, default="off", choices=["off", "on"],
                   help="If on, familiarity accumulates across episodes and improves social inference.")

    # familiarity baseline (used for normal profile when fam=on)
    p.add_argument("--fam_init", type=float, default=0.0)
    p.add_argument("--fam_decay_ep", type=float, default=0.00)
    p.add_argument("--fam_gain_obs", type=float, default=0.01)
    p.add_argument("--fam_gain_detect", type=float, default=0.04)
    p.add_argument("--p_detect_fam_boost", type=float, default=0.20)
    p.add_argument("--noise_fam_reduction", type=float, default=0.70)

    # autistic profile overrides (still uses your task logic)
    p.add_argument("--aut_p_detect_per_obs", type=float, default=0.25)
    p.add_argument("--aut_win_rem_noise_steps", type=int, default=3)
    p.add_argument("--aut_detect_memory_steps", type=int, default=12, help="How long the detected bout id is retained (steps).")
    p.add_argument("--aut_observe_cost", type=float, default=0.004)

    p.add_argument("--aut_fam_gain_obs", type=float, default=0.006)
    p.add_argument("--aut_fam_gain_detect", type=float, default=0.020)
    p.add_argument("--aut_p_detect_fam_boost", type=float, default=0.12)
    p.add_argument("--aut_noise_fam_reduction", type=float, default=0.45)

    # comparison suite
    p.add_argument("--compare_suite", action="store_true",
                   help="Run comparison suite across profiles/palatability/familiarity and save overlay plots.")
    p.add_argument("--compare_profiles", action="store_true", help="In suite: compare normal vs blind vs autistic.")
    p.add_argument("--compare_teacher_pal", action="store_true", help="In suite: compare teacher palatability high vs low.")
    p.add_argument("--compare_familiarity", action="store_true", help="In suite: compare familiarity off vs on.")
    p.add_argument("--shared_seed_across_conditions", action="store_true",
                   help="If set, each condition uses the same seed (otherwise seed+1000*i).")

    return p


if __name__ == "__main__":
    parser = build_argparser()
    args, _ = parser.parse_known_args()
    run_experiment(args)


## Autistic 

In [ ]:
def build_argparser():
    p = argparse.ArgumentParser(description="Active observational learning (multi-algo) + latent signal sims + condition suite")

    # algo
    p.add_argument("--algo", type=str, default="dqn", choices=["dqn", "ppo", "tabular"])

    # run
    p.add_argument("--episodes", type=int, default=1200)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--save_dir", type=str, default="plots")
    p.add_argument("--no_plots", action="store_true")
    p.add_argument("--plot_ext", type=str, default="pdf", choices=["pdf", "png"])
    p.add_argument("--multipage_pdf", action="store_true", help="If set + plot_ext=pdf, also writes one multi-page PDF.")
    p.add_argument("--updates_per_step", type=int, default=4)

    # environment (your original knobs)
    p.add_argument("--grid_size", type=int, default=15)
    p.add_argument("--dt_s", type=float, default=0.5)
    p.add_argument("--max_time_s", type=float, default=120.0)
    p.add_argument("--teacher_period_s", type=float, default=30.0)
    p.add_argument("--teacher_jitter_s", type=float, default=6.0)
    p.add_argument("--lick_mean_s", type=float, default=1.0)
    p.add_argument("--lick_dist", type=str, default="gamma", choices=["gamma", "lognormal"])
    p.add_argument("--lick_cv", type=float, default=0.8)
    p.add_argument("--window_s", type=float, default=3.0)
    p.add_argument("--eat_cooldown_s", type=float, default=2.0)
    p.add_argument("--observe_cost", type=float, default=0.002)
    p.add_argument("--eat_cost", type=float, default=0.005)
    p.add_argument("--attend_bonus", type=float, default=0.004)
    p.add_argument("--p_detect_per_obs", type=float, default=0.70)
    p.add_argument("--win_rem_noise_steps", type=int, default=1)

    # common learning
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--eps_start", type=float, default=1.0)
    p.add_argument("--eps_end", type=float, default=0.05)
    p.add_argument("--eps_decay", type=float, default=0.9995)

    # DQN
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--buffer_size", type=int, default=50000)
    p.add_argument("--tau", type=float, default=0.02)
    p.add_argument("--grad_clip", type=float, default=10.0)
    p.add_argument("--use_compile", action="store_true")

    # PPO
    p.add_argument("--gae_lambda", type=float, default=0.95)
    p.add_argument("--clip_ratio", type=float, default=0.2)
    p.add_argument("--vf_coef", type=float, default=0.5)
    p.add_argument("--ent_coef", type=float, default=0.01)
    p.add_argument("--ppo_epochs", type=int, default=4)
    p.add_argument("--minibatch_size", type=int, default=256)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    # Tabular Q
    p.add_argument("--tab_alpha", type=float, default=0.25)

    # early stopping
    p.add_argument("--early_stop_threshold", type=float, default=0.8)
    p.add_argument("--early_stop_window", type=int, default=100)

    # ============================================================
    # NEW: condition controls (do not change old behavior unless used)
    # ============================================================
    p.add_argument("--teacher_pal", type=str, default="default", choices=["default", "high", "low"],
                   help="Teacher palatability condition. 'high'/'low' only changes lick_mean_s via pal_* args.")
    p.add_argument("--pal_high_lick_mean_s", type=float, default=1.4,
                   help="Mean bout duration (s) for high palatability teacher.")
    p.add_argument("--pal_low_lick_mean_s", type=float, default=0.6,
                   help="Mean bout duration (s) for low palatability teacher (shorter consumption).")

    p.add_argument("--learner_profile", type=str, default="autistic", choices=["normal", "blind", "autistic"],
                   help="Learner perception/tracking profile.")
    p.add_argument("--familiarity", type=str, default="on", choices=["off", "on"],
                   help="If on, familiarity accumulates across episodes and improves social inference.")

    # familiarity baseline (used for normal profile when fam=on)
    p.add_argument("--fam_init", type=float, default=0.0)
    p.add_argument("--fam_decay_ep", type=float, default=0.00)
    p.add_argument("--fam_gain_obs", type=float, default=0.01)
    p.add_argument("--fam_gain_detect", type=float, default=0.04)
    p.add_argument("--p_detect_fam_boost", type=float, default=0.20)
    p.add_argument("--noise_fam_reduction", type=float, default=0.70)

    # autistic profile overrides (still uses your task logic)
    p.add_argument("--aut_p_detect_per_obs", type=float, default=0.25)
    p.add_argument("--aut_win_rem_noise_steps", type=int, default=3)
    p.add_argument("--aut_detect_memory_steps", type=int, default=12, help="How long the detected bout id is retained (steps).")
    p.add_argument("--aut_observe_cost", type=float, default=0.004)

    p.add_argument("--aut_fam_gain_obs", type=float, default=0.006)
    p.add_argument("--aut_fam_gain_detect", type=float, default=0.020)
    p.add_argument("--aut_p_detect_fam_boost", type=float, default=0.12)
    p.add_argument("--aut_noise_fam_reduction", type=float, default=0.45)

    # comparison suite
    p.add_argument("--compare_suite", action="store_true",
                   help="Run comparison suite across profiles/palatability/familiarity and save overlay plots.")
    p.add_argument("--compare_profiles", action="store_true", help="In suite: compare normal vs blind vs autistic.")
    p.add_argument("--compare_teacher_pal", action="store_true", help="In suite: compare teacher palatability high vs low.")
    p.add_argument("--compare_familiarity", action="store_true", help="In suite: compare familiarity off vs on.")
    p.add_argument("--shared_seed_across_conditions", action="store_true",
                   help="If set, each condition uses the same seed (otherwise seed+1000*i).")

    return p


if __name__ == "__main__":
    parser = build_argparser()
    args, _ = parser.parse_known_args()
    run_experiment(args)


## Blind

In [ ]:
def build_argparser():
    p = argparse.ArgumentParser(description="Active observational learning (multi-algo) + latent signal sims + condition suite")

    # algo
    p.add_argument("--algo", type=str, default="dqn", choices=["dqn", "ppo", "tabular"])

    # run
    p.add_argument("--episodes", type=int, default=1200)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--save_dir", type=str, default="plots")
    p.add_argument("--no_plots", action="store_true")
    p.add_argument("--plot_ext", type=str, default="pdf", choices=["pdf", "png"])
    p.add_argument("--multipage_pdf", action="store_true", help="If set + plot_ext=pdf, also writes one multi-page PDF.")
    p.add_argument("--updates_per_step", type=int, default=4)

    # environment (your original knobs)
    p.add_argument("--grid_size", type=int, default=15)
    p.add_argument("--dt_s", type=float, default=0.5)
    p.add_argument("--max_time_s", type=float, default=120.0)
    p.add_argument("--teacher_period_s", type=float, default=30.0)
    p.add_argument("--teacher_jitter_s", type=float, default=6.0)
    p.add_argument("--lick_mean_s", type=float, default=1.0)
    p.add_argument("--lick_dist", type=str, default="gamma", choices=["gamma", "lognormal"])
    p.add_argument("--lick_cv", type=float, default=0.8)
    p.add_argument("--window_s", type=float, default=3.0)
    p.add_argument("--eat_cooldown_s", type=float, default=2.0)
    p.add_argument("--observe_cost", type=float, default=0.002)
    p.add_argument("--eat_cost", type=float, default=0.005)
    p.add_argument("--attend_bonus", type=float, default=0.004)
    p.add_argument("--p_detect_per_obs", type=float, default=0.70)
    p.add_argument("--win_rem_noise_steps", type=int, default=1)

    # common learning
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--eps_start", type=float, default=1.0)
    p.add_argument("--eps_end", type=float, default=0.05)
    p.add_argument("--eps_decay", type=float, default=0.9995)

    # DQN
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--buffer_size", type=int, default=50000)
    p.add_argument("--tau", type=float, default=0.02)
    p.add_argument("--grad_clip", type=float, default=10.0)
    p.add_argument("--use_compile", action="store_true")

    # PPO
    p.add_argument("--gae_lambda", type=float, default=0.95)
    p.add_argument("--clip_ratio", type=float, default=0.2)
    p.add_argument("--vf_coef", type=float, default=0.5)
    p.add_argument("--ent_coef", type=float, default=0.01)
    p.add_argument("--ppo_epochs", type=int, default=4)
    p.add_argument("--minibatch_size", type=int, default=256)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    # Tabular Q
    p.add_argument("--tab_alpha", type=float, default=0.25)

    # early stopping
    p.add_argument("--early_stop_threshold", type=float, default=0.8)
    p.add_argument("--early_stop_window", type=int, default=100)

    # ============================================================
    # NEW: condition controls (do not change old behavior unless used)
    # ============================================================
    p.add_argument("--teacher_pal", type=str, default="default", choices=["default", "high", "low"],
                   help="Teacher palatability condition. 'high'/'low' only changes lick_mean_s via pal_* args.")
    p.add_argument("--pal_high_lick_mean_s", type=float, default=1.4,
                   help="Mean bout duration (s) for high palatability teacher.")
    p.add_argument("--pal_low_lick_mean_s", type=float, default=0.6,
                   help="Mean bout duration (s) for low palatability teacher (shorter consumption).")

    p.add_argument("--learner_profile", type=str, default="blind", choices=["normal", "blind", "autistic"],
                   help="Learner perception/tracking profile.")
    p.add_argument("--familiarity", type=str, default="on", choices=["off", "on"],
                   help="If on, familiarity accumulates across episodes and improves social inference.")

    # familiarity baseline (used for normal profile when fam=on)
    p.add_argument("--fam_init", type=float, default=0.0)
    p.add_argument("--fam_decay_ep", type=float, default=0.00)
    p.add_argument("--fam_gain_obs", type=float, default=0.01)
    p.add_argument("--fam_gain_detect", type=float, default=0.04)
    p.add_argument("--p_detect_fam_boost", type=float, default=0.20)
    p.add_argument("--noise_fam_reduction", type=float, default=0.70)

    # autistic profile overrides (still uses your task logic)
    p.add_argument("--aut_p_detect_per_obs", type=float, default=0.25)
    p.add_argument("--aut_win_rem_noise_steps", type=int, default=3)
    p.add_argument("--aut_detect_memory_steps", type=int, default=12, help="How long the detected bout id is retained (steps).")
    p.add_argument("--aut_observe_cost", type=float, default=0.004)

    p.add_argument("--aut_fam_gain_obs", type=float, default=0.006)
    p.add_argument("--aut_fam_gain_detect", type=float, default=0.020)
    p.add_argument("--aut_p_detect_fam_boost", type=float, default=0.12)
    p.add_argument("--aut_noise_fam_reduction", type=float, default=0.45)

    # comparison suite
    p.add_argument("--compare_suite", action="store_true",
                   help="Run comparison suite across profiles/palatability/familiarity and save overlay plots.")
    p.add_argument("--compare_profiles", action="store_true", help="In suite: compare normal vs blind vs autistic.")
    p.add_argument("--compare_teacher_pal", action="store_true", help="In suite: compare teacher palatability high vs low.")
    p.add_argument("--compare_familiarity", action="store_true", help="In suite: compare familiarity off vs on.")
    p.add_argument("--shared_seed_across_conditions", action="store_true",
                   help="If set, each condition uses the same seed (otherwise seed+1000*i).")

    return p


if __name__ == "__main__":
    parser = build_argparser()
    args, _ = parser.parse_known_args()
    run_experiment(args)


## Familarity

In [ ]:
def build_argparser():
    p = argparse.ArgumentParser(description="Active observational learning (multi-algo) + latent signal sims + condition suite")

    # algo
    p.add_argument("--algo", type=str, default="dqn", choices=["dqn", "ppo", "tabular"])

    # run
    p.add_argument("--episodes", type=int, default=1200)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--save_dir", type=str, default="plots")
    p.add_argument("--no_plots", action="store_true")
    p.add_argument("--plot_ext", type=str, default="pdf", choices=["pdf", "png"])
    p.add_argument("--multipage_pdf", action="store_true", help="If set + plot_ext=pdf, also writes one multi-page PDF.")
    p.add_argument("--updates_per_step", type=int, default=4)

    # environment (your original knobs)
    p.add_argument("--grid_size", type=int, default=15)
    p.add_argument("--dt_s", type=float, default=0.5)
    p.add_argument("--max_time_s", type=float, default=120.0)
    p.add_argument("--teacher_period_s", type=float, default=30.0)
    p.add_argument("--teacher_jitter_s", type=float, default=6.0)
    p.add_argument("--lick_mean_s", type=float, default=1.0)
    p.add_argument("--lick_dist", type=str, default="gamma", choices=["gamma", "lognormal"])
    p.add_argument("--lick_cv", type=float, default=0.8)
    p.add_argument("--window_s", type=float, default=3.0)
    p.add_argument("--eat_cooldown_s", type=float, default=2.0)
    p.add_argument("--observe_cost", type=float, default=0.002)
    p.add_argument("--eat_cost", type=float, default=0.005)
    p.add_argument("--attend_bonus", type=float, default=0.004)
    p.add_argument("--p_detect_per_obs", type=float, default=0.70)
    p.add_argument("--win_rem_noise_steps", type=int, default=1)

    # common learning
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--eps_start", type=float, default=1.0)
    p.add_argument("--eps_end", type=float, default=0.05)
    p.add_argument("--eps_decay", type=float, default=0.9995)

    # DQN
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--buffer_size", type=int, default=50000)
    p.add_argument("--tau", type=float, default=0.02)
    p.add_argument("--grad_clip", type=float, default=10.0)
    p.add_argument("--use_compile", action="store_true")

    # PPO
    p.add_argument("--gae_lambda", type=float, default=0.95)
    p.add_argument("--clip_ratio", type=float, default=0.2)
    p.add_argument("--vf_coef", type=float, default=0.5)
    p.add_argument("--ent_coef", type=float, default=0.01)
    p.add_argument("--ppo_epochs", type=int, default=4)
    p.add_argument("--minibatch_size", type=int, default=256)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    # Tabular Q
    p.add_argument("--tab_alpha", type=float, default=0.25)

    # early stopping
    p.add_argument("--early_stop_threshold", type=float, default=0.8)
    p.add_argument("--early_stop_window", type=int, default=100)

    # ============================================================
    # NEW: condition controls (do not change old behavior unless used)
    # ============================================================
    p.add_argument("--teacher_pal", type=str, default="default", choices=["default", "high", "low"],
                   help="Teacher palatability condition. 'high'/'low' only changes lick_mean_s via pal_* args.")
    p.add_argument("--pal_high_lick_mean_s", type=float, default=1.4,
                   help="Mean bout duration (s) for high palatability teacher.")
    p.add_argument("--pal_low_lick_mean_s", type=float, default=0.6,
                   help="Mean bout duration (s) for low palatability teacher (shorter consumption).")

    p.add_argument("--learner_profile", type=str, default="normal", choices=["normal", "blind", "autistic"],
                   help="Learner perception/tracking profile.")
    p.add_argument("--familiarity", type=str, default="on", choices=["off", "on"],
                   help="If on, familiarity accumulates across episodes and improves social inference.")

    # familiarity baseline (used for normal profile when fam=on)
    p.add_argument("--fam_init", type=float, default=0.0)
    p.add_argument("--fam_decay_ep", type=float, default=0.00)
    p.add_argument("--fam_gain_obs", type=float, default=0.01)
    p.add_argument("--fam_gain_detect", type=float, default=0.04)
    p.add_argument("--p_detect_fam_boost", type=float, default=0.20)
    p.add_argument("--noise_fam_reduction", type=float, default=0.70)

    # autistic profile overrides (still uses your task logic)
    p.add_argument("--aut_p_detect_per_obs", type=float, default=0.25)
    p.add_argument("--aut_win_rem_noise_steps", type=int, default=3)
    p.add_argument("--aut_detect_memory_steps", type=int, default=12, help="How long the detected bout id is retained (steps).")
    p.add_argument("--aut_observe_cost", type=float, default=0.004)

    p.add_argument("--aut_fam_gain_obs", type=float, default=0.006)
    p.add_argument("--aut_fam_gain_detect", type=float, default=0.020)
    p.add_argument("--aut_p_detect_fam_boost", type=float, default=0.12)
    p.add_argument("--aut_noise_fam_reduction", type=float, default=0.45)

    # comparison suite
    p.add_argument("--compare_suite", action="store_true",
                   help="Run comparison suite across profiles/palatability/familiarity and save overlay plots.")
    p.add_argument("--compare_profiles", action="store_true", help="In suite: compare normal vs blind vs autistic.")
    p.add_argument("--compare_teacher_pal", action="store_true", help="In suite: compare teacher palatability high vs low.")
    p.add_argument("--compare_familiarity", action="store_true", help="In suite: compare familiarity off vs on.")
    p.add_argument("--shared_seed_across_conditions", action="store_true",
                   help="If set, each condition uses the same seed (otherwise seed+1000*i).")

    return p


if __name__ == "__main__":
    parser = build_argparser()
    args, _ = parser.parse_known_args()
    run_experiment(args)


## low palatability Familar

In [ ]:
def build_argparser():
    p = argparse.ArgumentParser(description="Active observational learning (multi-algo) + latent signal sims + condition suite")

    # algo
    p.add_argument("--algo", type=str, default="dqn", choices=["dqn", "ppo", "tabular"])

    # run
    p.add_argument("--episodes", type=int, default=1200)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--save_dir", type=str, default="plots")
    p.add_argument("--no_plots", action="store_true")
    p.add_argument("--plot_ext", type=str, default="pdf", choices=["pdf", "png"])
    p.add_argument("--multipage_pdf", action="store_true", help="If set + plot_ext=pdf, also writes one multi-page PDF.")
    p.add_argument("--updates_per_step", type=int, default=4)

    # environment (your original knobs)
    p.add_argument("--grid_size", type=int, default=15)
    p.add_argument("--dt_s", type=float, default=0.5)
    p.add_argument("--max_time_s", type=float, default=120.0)
    p.add_argument("--teacher_period_s", type=float, default=30.0)
    p.add_argument("--teacher_jitter_s", type=float, default=6.0)
    p.add_argument("--lick_mean_s", type=float, default=1.0)
    p.add_argument("--lick_dist", type=str, default="gamma", choices=["gamma", "lognormal"])
    p.add_argument("--lick_cv", type=float, default=0.8)
    p.add_argument("--window_s", type=float, default=3.0)
    p.add_argument("--eat_cooldown_s", type=float, default=2.0)
    p.add_argument("--observe_cost", type=float, default=0.002)
    p.add_argument("--eat_cost", type=float, default=0.005)
    p.add_argument("--attend_bonus", type=float, default=0.004)
    p.add_argument("--p_detect_per_obs", type=float, default=0.70)
    p.add_argument("--win_rem_noise_steps", type=int, default=1)

    # common learning
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--eps_start", type=float, default=1.0)
    p.add_argument("--eps_end", type=float, default=0.05)
    p.add_argument("--eps_decay", type=float, default=0.9995)

    # DQN
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--buffer_size", type=int, default=50000)
    p.add_argument("--tau", type=float, default=0.02)
    p.add_argument("--grad_clip", type=float, default=10.0)
    p.add_argument("--use_compile", action="store_true")

    # PPO
    p.add_argument("--gae_lambda", type=float, default=0.95)
    p.add_argument("--clip_ratio", type=float, default=0.2)
    p.add_argument("--vf_coef", type=float, default=0.5)
    p.add_argument("--ent_coef", type=float, default=0.01)
    p.add_argument("--ppo_epochs", type=int, default=4)
    p.add_argument("--minibatch_size", type=int, default=256)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    # Tabular Q
    p.add_argument("--tab_alpha", type=float, default=0.25)

    # early stopping
    p.add_argument("--early_stop_threshold", type=float, default=0.8)
    p.add_argument("--early_stop_window", type=int, default=100)

    # ============================================================
    # NEW: condition controls (do not change old behavior unless used)
    # ============================================================
    p.add_argument("--teacher_pal", type=str, default="low", choices=["default", "high", "low"],
                   help="Teacher palatability condition. 'high'/'low' only changes lick_mean_s via pal_* args.")
    p.add_argument("--pal_high_lick_mean_s", type=float, default=1.4,
                   help="Mean bout duration (s) for high palatability teacher.")
    p.add_argument("--pal_low_lick_mean_s", type=float, default=0.6,
                   help="Mean bout duration (s) for low palatability teacher (shorter consumption).")

    p.add_argument("--learner_profile", type=str, default="normal", choices=["normal", "blind", "autistic"],
                   help="Learner perception/tracking profile.")
    p.add_argument("--familiarity", type=str, default="on", choices=["off", "on"],
                   help="If on, familiarity accumulates across episodes and improves social inference.")

    # familiarity baseline (used for normal profile when fam=on)
    p.add_argument("--fam_init", type=float, default=0.0)
    p.add_argument("--fam_decay_ep", type=float, default=0.00)
    p.add_argument("--fam_gain_obs", type=float, default=0.01)
    p.add_argument("--fam_gain_detect", type=float, default=0.04)
    p.add_argument("--p_detect_fam_boost", type=float, default=0.20)
    p.add_argument("--noise_fam_reduction", type=float, default=0.70)

    # autistic profile overrides (still uses your task logic)
    p.add_argument("--aut_p_detect_per_obs", type=float, default=0.25)
    p.add_argument("--aut_win_rem_noise_steps", type=int, default=3)
    p.add_argument("--aut_detect_memory_steps", type=int, default=12, help="How long the detected bout id is retained (steps).")
    p.add_argument("--aut_observe_cost", type=float, default=0.004)

    p.add_argument("--aut_fam_gain_obs", type=float, default=0.006)
    p.add_argument("--aut_fam_gain_detect", type=float, default=0.020)
    p.add_argument("--aut_p_detect_fam_boost", type=float, default=0.12)
    p.add_argument("--aut_noise_fam_reduction", type=float, default=0.45)

    # comparison suite
    p.add_argument("--compare_suite", action="store_true",
                   help="Run comparison suite across profiles/palatability/familiarity and save overlay plots.")
    p.add_argument("--compare_profiles", action="store_true", help="In suite: compare normal vs blind vs autistic.")
    p.add_argument("--compare_teacher_pal", action="store_true", help="In suite: compare teacher palatability high vs low.")
    p.add_argument("--compare_familiarity", action="store_true", help="In suite: compare familiarity off vs on.")
    p.add_argument("--shared_seed_across_conditions", action="store_true",
                   help="If set, each condition uses the same seed (otherwise seed+1000*i).")

    return p


if __name__ == "__main__":
    parser = build_argparser()
    args, _ = parser.parse_known_args()
    run_experiment(args)


## High palatablity

In [ ]:
def build_argparser():
    p = argparse.ArgumentParser(description="Active observational learning (multi-algo) + latent signal sims + condition suite")

    # algo
    p.add_argument("--algo", type=str, default="dqn", choices=["dqn", "ppo", "tabular"])

    # run
    p.add_argument("--episodes", type=int, default=1200)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--save_dir", type=str, default="plots")
    p.add_argument("--no_plots", action="store_true")
    p.add_argument("--plot_ext", type=str, default="pdf", choices=["pdf", "png"])
    p.add_argument("--multipage_pdf", action="store_true", help="If set + plot_ext=pdf, also writes one multi-page PDF.")
    p.add_argument("--updates_per_step", type=int, default=4)

    # environment (your original knobs)
    p.add_argument("--grid_size", type=int, default=15)
    p.add_argument("--dt_s", type=float, default=0.5)
    p.add_argument("--max_time_s", type=float, default=120.0)
    p.add_argument("--teacher_period_s", type=float, default=30.0)
    p.add_argument("--teacher_jitter_s", type=float, default=6.0)
    p.add_argument("--lick_mean_s", type=float, default=1.0)
    p.add_argument("--lick_dist", type=str, default="gamma", choices=["gamma", "lognormal"])
    p.add_argument("--lick_cv", type=float, default=0.8)
    p.add_argument("--window_s", type=float, default=3.0)
    p.add_argument("--eat_cooldown_s", type=float, default=2.0)
    p.add_argument("--observe_cost", type=float, default=0.002)
    p.add_argument("--eat_cost", type=float, default=0.005)
    p.add_argument("--attend_bonus", type=float, default=0.004)
    p.add_argument("--p_detect_per_obs", type=float, default=0.70)
    p.add_argument("--win_rem_noise_steps", type=int, default=1)

    # common learning
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--eps_start", type=float, default=1.0)
    p.add_argument("--eps_end", type=float, default=0.05)
    p.add_argument("--eps_decay", type=float, default=0.9995)

    # DQN
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--buffer_size", type=int, default=50000)
    p.add_argument("--tau", type=float, default=0.02)
    p.add_argument("--grad_clip", type=float, default=10.0)
    p.add_argument("--use_compile", action="store_true")

    # PPO
    p.add_argument("--gae_lambda", type=float, default=0.95)
    p.add_argument("--clip_ratio", type=float, default=0.2)
    p.add_argument("--vf_coef", type=float, default=0.5)
    p.add_argument("--ent_coef", type=float, default=0.01)
    p.add_argument("--ppo_epochs", type=int, default=4)
    p.add_argument("--minibatch_size", type=int, default=256)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    # Tabular Q
    p.add_argument("--tab_alpha", type=float, default=0.25)

    # early stopping
    p.add_argument("--early_stop_threshold", type=float, default=0.8)
    p.add_argument("--early_stop_window", type=int, default=100)

    # ============================================================
    # NEW: condition controls (do not change old behavior unless used)
    # ============================================================
    p.add_argument("--teacher_pal", type=str, default="high", choices=["default", "high", "low"],
                   help="Teacher palatability condition. 'high'/'low' only changes lick_mean_s via pal_* args.")
    p.add_argument("--pal_high_lick_mean_s", type=float, default=1.4,
                   help="Mean bout duration (s) for high palatability teacher.")
    p.add_argument("--pal_low_lick_mean_s", type=float, default=0.6,
                   help="Mean bout duration (s) for low palatability teacher (shorter consumption).")

    p.add_argument("--learner_profile", type=str, default="normal", choices=["normal", "blind", "autistic"],
                   help="Learner perception/tracking profile.")
    p.add_argument("--familiarity", type=str, default="on", choices=["off", "on"],
                   help="If on, familiarity accumulates across episodes and improves social inference.")

    # familiarity baseline (used for normal profile when fam=on)
    p.add_argument("--fam_init", type=float, default=0.0)
    p.add_argument("--fam_decay_ep", type=float, default=0.00)
    p.add_argument("--fam_gain_obs", type=float, default=0.01)
    p.add_argument("--fam_gain_detect", type=float, default=0.04)
    p.add_argument("--p_detect_fam_boost", type=float, default=0.20)
    p.add_argument("--noise_fam_reduction", type=float, default=0.70)

    # autistic profile overrides (still uses your task logic)
    p.add_argument("--aut_p_detect_per_obs", type=float, default=0.25)
    p.add_argument("--aut_win_rem_noise_steps", type=int, default=3)
    p.add_argument("--aut_detect_memory_steps", type=int, default=12, help="How long the detected bout id is retained (steps).")
    p.add_argument("--aut_observe_cost", type=float, default=0.004)

    p.add_argument("--aut_fam_gain_obs", type=float, default=0.006)
    p.add_argument("--aut_fam_gain_detect", type=float, default=0.020)
    p.add_argument("--aut_p_detect_fam_boost", type=float, default=0.12)
    p.add_argument("--aut_noise_fam_reduction", type=float, default=0.45)

    # comparison suite
    p.add_argument("--compare_suite", action="store_true",
                   help="Run comparison suite across profiles/palatability/familiarity and save overlay plots.")
    p.add_argument("--compare_profiles", action="store_true", help="In suite: compare normal vs blind vs autistic.")
    p.add_argument("--compare_teacher_pal", action="store_true", help="In suite: compare teacher palatability high vs low.")
    p.add_argument("--compare_familiarity", action="store_true", help="In suite: compare familiarity off vs on.")
    p.add_argument("--shared_seed_across_conditions", action="store_true",
                   help="If set, each condition uses the same seed (otherwise seed+1000*i).")

    return p


if __name__ == "__main__":
    parser = build_argparser()
    args, _ = parser.parse_known_args()
    run_experiment(args)


## Compare two conditions

In [ ]:
from typing import Optional, Dict, Any, List, Tuple
import numpy as np
import random
from collections import deque, defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import argparse
import os
import math

# -------------------------
# Seed / utils
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def clamp_int(x: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, x))

def clamp_float(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, float(x)))

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

# -------------------------
# Actions
# -------------------------
A_LEFT = 0
A_STAY = 1
A_RIGHT = 2
A_OBS = 3  # consumes time; reveals social timing info stochastically
A_EAT = 4

# ============================================================
# Environment (same logic + condition knobs you already added)
# ============================================================
class SocialLickEnv1D:
    """
    Fast but detailed (paper-aligned):
      dt_s = 0.5s/step
      teacher cue bursts ("lick bouts") with stochastic duration (mean ~ 1s)
      reward window: 3s after last cue step

    Observation gating:
      - social signal (cue + noisy window remaining) only appears when action==OBS
      - detection during cue burst is probabilistic per OBS step

    Condition knobs:
      - social_visibility: 0.0 => "social blind" (OBS yields no social info)
      - detect_memory_steps: if set, detected_bout_id expires after N steps
      - familiarity_enabled: if True, a latent familiarity variable accumulates across episodes and
        increases effective detect prob and reduces window-remaining noise
    """
    def __init__(
        self,
        grid_size=15,
        dt_s=0.5,
        max_time_s=120.0,
        teacher_period_s=30.0,
        teacher_jitter_s=6.0,
        lick_mean_s=1.0,
        lick_dist="gamma",  # "gamma" or "lognormal"
        lick_cv=0.8,        # for lognormal
        window_s=3.0,
        eat_cooldown_s=2.0,
        observe_cost=0.002,
        eat_cost=0.005,
        attend_bonus=0.004,
        p_detect_per_obs=0.70,
        win_rem_noise_steps=1,

        # perception / cognition constraints
        social_visibility: float = 1.0,             # 1.0 normal, 0.0 blind
        detect_memory_steps: Optional[int] = None,  # None => perfect retention
        # familiarity dynamics
        familiarity_enabled: bool = False,
        familiarity_init: float = 0.0,              # persistent across episodes
        fam_decay_ep: float = 0.00,                 # applied at reset (episode boundary)
        fam_gain_obs: float = 0.00,                 # gain per OBS during teacher lick
        fam_gain_detect: float = 0.00,              # extra gain when OBS successfully detects lick
        p_detect_fam_boost: float = 0.00,           # added detect prob at fam=1
        noise_fam_reduction: float = 0.00,          # fraction reduction of noise_steps at fam=1
    ):
        self.grid_size = int(grid_size)
        self.dt_s = float(dt_s)
        self.max_steps = int(round(max_time_s / self.dt_s))
        self.teacher_period_steps = int(round(teacher_period_s / self.dt_s))
        self.teacher_jitter_steps = int(round(teacher_jitter_s / self.dt_s))
        self.lick_mean_s = float(lick_mean_s)
        self.lick_dist = str(lick_dist)
        self.lick_cv = float(lick_cv)
        self.window_steps = int(round(window_s / self.dt_s))
        self.eat_cooldown_steps = int(round(eat_cooldown_s / self.dt_s))
        self.observe_cost = float(observe_cost)
        self.eat_cost = float(eat_cost)
        self.attend_bonus = float(attend_bonus)
        self.p_detect_per_obs = float(p_detect_per_obs)
        self.win_rem_noise_steps = int(win_rem_noise_steps)

        self.social_visibility = clamp_float(social_visibility, 0.0, 1.0)
        self.detect_memory_steps = None if detect_memory_steps is None else int(detect_memory_steps)

        self.familiarity_enabled = bool(familiarity_enabled)
        self.familiarity = clamp_float(familiarity_init, 0.0, 1.0)
        self.fam_decay_ep = clamp_float(fam_decay_ep, 0.0, 1.0)
        self.fam_gain_obs = float(fam_gain_obs)
        self.fam_gain_detect = float(fam_gain_detect)
        self.p_detect_fam_boost = float(p_detect_fam_boost)
        self.noise_fam_reduction = clamp_float(noise_fam_reduction, 0.0, 1.0)

        assert self.teacher_period_steps > self.window_steps + 1
        assert self.max_steps > self.teacher_period_steps + 5
        self.reset()

    def _sample_lick_duration_steps(self):
        if self.lick_dist == "gamma":
            shape = 2.0
            scale = self.lick_mean_s / shape
            dur_s = float(np.random.gamma(shape, scale))
        elif self.lick_dist == "lognormal":
            cv = max(1e-6, self.lick_cv)
            sigma2 = np.log(1.0 + cv**2)
            sigma = np.sqrt(sigma2)
            mu = np.log(self.lick_mean_s) - 0.5 * sigma2
            dur_s = float(np.random.lognormal(mean=mu, sigma=sigma))
        else:
            raise ValueError(f"Unknown lick_dist={self.lick_dist}")
        steps = int(np.round(dur_s / self.dt_s))
        return max(1, steps), dur_s

    def _effective_detect_prob(self) -> float:
        p = self.p_detect_per_obs
        if self.familiarity_enabled:
            p += self.p_detect_fam_boost * self.familiarity
        return clamp_float(p, 0.0, 1.0)

    def _effective_noise_steps(self) -> int:
        n = int(self.win_rem_noise_steps)
        if self.familiarity_enabled and n > 0:
            n = int(round(n * (1.0 - self.noise_fam_reduction * self.familiarity)))
        return max(0, n)

    def reset(self):
        # Episode boundary familiarity decay (persistent across episodes)
        if self.familiarity_enabled and self.fam_decay_ep > 0.0:
            self.familiarity = clamp_float(self.familiarity * (1.0 - self.fam_decay_ep), 0.0, 1.0)

        self.t = 0
        self.teacher_pos = int(np.random.randint(0, self.grid_size))
        candidates = [i for i in range(self.grid_size) if i != self.teacher_pos]
        self.learner_food_pos = int(np.random.choice(candidates))
        self.learner_pos = int(np.random.randint(0, self.grid_size))

        self.eat_cd = 0
        self.last_lick_step = None
        self.detected_bout_id = None
        self._detect_ttl = None
        self.rewarded_bout_ids = set()

        # build cue-burst timeline
        self.bout_id_at_step = -np.ones(self.max_steps + 1, dtype=np.int32)
        self.bout_start_steps = []
        self.bout_end_steps = []
        self.bout_dur_s_list = []
        bout_id = 0
        k = 1
        while True:
            nominal = k * self.teacher_period_steps
            if nominal >= self.max_steps:
                break
            jitter = int(np.random.randint(-self.teacher_jitter_steps, self.teacher_jitter_steps + 1))
            start = clamp_int(nominal + jitter, 1, self.max_steps - 2)
            dur_steps, dur_s = self._sample_lick_duration_steps()
            end = clamp_int(start + dur_steps - 1, start, self.max_steps - 1)
            self.bout_id_at_step[start:end + 1] = bout_id
            self.bout_start_steps.append(start)
            self.bout_end_steps.append(end)
            self.bout_dur_s_list.append(dur_s)
            bout_id += 1
            k += 1
        self.n_bouts = int(bout_id)
        return self._get_obs(lick_sig=0.0, win_rem=0.0)

    def _teacher_lick_now(self, t: int):
        bid = int(self.bout_id_at_step[t]) if (0 <= t <= self.max_steps) else -1
        return (1, bid) if bid >= 0 else (0, -1)

    def _window_open(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        return 1 if (0 <= dt < self.window_steps) else 0

    def _window_remaining(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        rem = self.window_steps - dt
        return int(max(0, rem))

    def _phase_to_next_nominal(self, t: int) -> int:
        mod = t % self.teacher_period_steps
        return self.teacher_period_steps - mod

    def _get_obs(self, lick_sig: float, win_rem: float):
        lp = float(self.learner_pos) / float(self.grid_size - 1)
        lf = float(self.learner_food_pos) / float(self.grid_size - 1)
        phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period_steps)
        cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown_steps))
        # [pos, food_pos, gated cue, gated window remaining, phase clock, cooldown]
        return np.array([lp, lf, float(lick_sig), float(win_rem), phase, cdn], dtype=np.float32)

    def step(self, action: int):
        self.t += 1
        teacher_lick, bout_id = self._teacher_lick_now(self.t)
        if teacher_lick == 1:
            self.last_lick_step = self.t

        window_open = self._window_open(self.t)
        win_rem_true = self._window_remaining(self.t)

        if self.eat_cd > 0:
            self.eat_cd -= 1

        # optional memory decay for detected bout id
        if self.detect_memory_steps is not None and self._detect_ttl is not None:
            self._detect_ttl -= 1
            if self._detect_ttl <= 0:
                self._detect_ttl = None
                self.detected_bout_id = None

        lick_sig = 0.0
        win_rem = 0.0
        did_observe = 0
        did_eat = 0
        seen_lick = 0

        # movement / observe / eat
        if action == A_LEFT:
            self.learner_pos -= 1
        elif action == A_RIGHT:
            self.learner_pos += 1
        elif action == A_OBS:
            did_observe = 1

            # social blind => OBS reveals nothing
            if self.social_visibility > 0.0:
                p_det = self._effective_detect_prob()
                if teacher_lick == 1 and bout_id >= 0 and (np.random.rand() < p_det):
                    seen_lick = 1
                    self.detected_bout_id = int(bout_id)
                    if self.detect_memory_steps is not None:
                        self._detect_ttl = int(self.detect_memory_steps)

                lick_sig = float(seen_lick)

                if win_rem_true > 0:
                    noise_steps = self._effective_noise_steps()
                    noise = int(np.random.randint(-noise_steps, noise_steps + 1)) if noise_steps > 0 else 0
                    est = clamp_int(win_rem_true + noise, 0, self.window_steps)
                    win_rem = float(est) / float(max(1, self.window_steps))

                # familiarity update
                if self.familiarity_enabled and teacher_lick == 1 and bout_id >= 0:
                    self.familiarity = clamp_float(self.familiarity + self.fam_gain_obs, 0.0, 1.0)
                    if seen_lick == 1:
                        self.familiarity = clamp_float(self.familiarity + self.fam_gain_detect, 0.0, 1.0)

        elif action == A_EAT:
            did_eat = 1
        # A_STAY does nothing

        self.learner_pos = clamp_int(self.learner_pos, 0, self.grid_size - 1)

        reward = 0.0
        reward -= self.observe_cost * float(did_observe)
        reward -= self.eat_cost * float(did_eat)

        # shaping: reward OBS during cue only if bout not yet detected
        if did_observe == 1 and teacher_lick == 1 and bout_id >= 0:
            if self.detected_bout_id != int(bout_id):
                reward += self.attend_bonus

        # eat allowed?
        eat_allowed = (did_eat == 1 and self.eat_cd == 0)
        if eat_allowed:
            self.eat_cd = self.eat_cooldown_steps

        # success logic (1 reward per bout)
        eat_valid = 0
        if eat_allowed and (self.learner_pos == self.learner_food_pos) and (window_open == 1) and (self.last_lick_step is not None):
            last_bid = int(self.bout_id_at_step[self.last_lick_step]) if self.last_lick_step <= self.max_steps else -1
            if (last_bid >= 0) and (self.detected_bout_id == last_bid) and (last_bid not in self.rewarded_bout_ids):
                eat_valid = 1
                self.rewarded_bout_ids.add(last_bid)
                reward += 1.0

        done = bool(self.t >= self.max_steps)
        info = {
            "t": self.t,
            "teacher_lick": int(teacher_lick),
            "bout_id": int(bout_id),
            "window_open": int(window_open),
            "win_rem_true": int(win_rem_true),
            "action": int(action),
            "observe": int(did_observe),
            "eat": int(did_eat),
            "eat_allowed": int(eat_allowed),
            "eat_valid": int(eat_valid),
            "learner_pos": int(self.learner_pos),
            "food_pos": int(self.learner_food_pos),
            "seen_lick": int(seen_lick),
            "win_rem_est": float(win_rem),
            "detected_bout_id": -1 if self.detected_bout_id is None else int(self.detected_bout_id),
            "rewarded_bouts": len(self.rewarded_bout_ids),
            "n_bouts": int(self.n_bouts),
            "dt_s": float(self.dt_s),
            "window_steps": int(self.window_steps),
            "social_visibility": float(self.social_visibility),
            "familiarity": float(self.familiarity),
            "p_detect_eff": float(self._effective_detect_prob()) if (self.social_visibility > 0.0) else 0.0,
        }
        obs = self._get_obs(lick_sig=lick_sig, win_rem=win_rem)
        return obs, reward, done, info


# ============================================================
# Plot saving
# ============================================================
def _save_fig(fig, out_path_no_ext: str, ext: str, pdf_pages: Optional[PdfPages] = None):
    ensure_dir(os.path.dirname(out_path_no_ext))
    if ext.lower() == "pdf" and pdf_pages is not None:
        pdf_pages.savefig(fig, bbox_inches="tight")
    else:
        fig.savefig(f"{out_path_no_ext}.{ext}", bbox_inches="tight")

def ema_1d(y: np.ndarray, alpha: float = 0.02) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    out = np.zeros_like(y)
    m = 0.0
    for i in range(len(y)):
        if np.isnan(y[i]):
            out[i] = m
            continue
        m = (1 - alpha) * m + alpha * y[i]
        out[i] = m
    return out

def logs_to_array_with_ffill(logs: List[Dict[str, float]], metric: str, target_len: int) -> np.ndarray:
    y = np.full((target_len,), np.nan, dtype=np.float32)
    if len(logs) == 0:
        return y
    vals = [float(d.get(metric, np.nan)) for d in logs]
    L = min(len(vals), target_len)
    y[:L] = np.array(vals[:L], dtype=np.float32)
    # forward-fill for early stop / shorter runs (keeps curves aligned)
    if L < target_len and L > 0 and not np.isnan(y[L-1]):
        y[L:] = y[L-1]
    return y

def mean_sem(arr: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    arr: [N, T] with possible NaNs
    returns mean[T], sem[T]
    """
    mean = np.nanmean(arr, axis=0)
    # sem uses sample std with ddof=1 when possible
    sem = np.zeros_like(mean, dtype=np.float32)
    for t in range(arr.shape[1]):
        col = arr[:, t]
        col = col[~np.isnan(col)]
        if len(col) <= 1:
            sem[t] = 0.0
        else:
            sem[t] = float(np.std(col, ddof=1) / math.sqrt(len(col)))
    return mean.astype(np.float32), sem.astype(np.float32)

def plot_metric_mean_sem(
    ep: np.ndarray,
    mean: np.ndarray,
    sem: np.ndarray,
    label: str,
    alpha_fill: float = 0.20,
):
    plt.plot(ep, mean, label=label)
    lo = mean - sem
    hi = mean + sem
    plt.fill_between(ep, lo, hi, alpha=alpha_fill)

def plot_comparison_curves_mean_sem(
    results_logs_by_seed: Dict[str, List[List[Dict[str, float]]]],
    metric: str,
    episodes: int,
    save_dir: str,
    ext: str = "pdf",
    pdf_pages: Optional[PdfPages] = None,
    alpha_ema: float = 0.02,
    title: Optional[str] = None,
):
    """
    results_logs_by_seed: {label -> list_of_runs, each run is logs(list of dicts)}
    plots mean±SEM across runs (seeds).
    """
    fig = plt.figure(figsize=(8, 4.8))
    ep = np.arange(1, episodes + 1, dtype=np.int32)

    for label, runs in results_logs_by_seed.items():
        arr = np.stack([logs_to_array_with_ffill(run, metric, episodes) for run in runs], axis=0)  # [N,T]
        # smooth per-run then aggregate => SEM reflects across-run variability post-smoothing
        arr_s = np.stack([ema_1d(arr[i], alpha=alpha_ema) for i in range(arr.shape[0])], axis=0)
        m, s = mean_sem(arr_s)
        plot_metric_mean_sem(ep, m, s, label=label)

    plt.xlabel("episode")
    plt.ylabel(metric)
    plt.title(title or f"{metric}: mean ± SEM (across seeds)")
    plt.legend(fontsize=8)
    plt.tight_layout()
    _save_fig(fig, os.path.join(save_dir, f"compare_{metric}_mean_sem"), ext=ext, pdf_pages=pdf_pages)

def plot_single_learning_panel_mean_sem(
    runs: List[List[Dict[str, float]]],
    episodes: int,
    save_dir: str,
    ext: str,
    pdf_pages: Optional[PdfPages],
    title_suffix: str = "",
    alpha_ema: float = 0.02,
):
    """
    Single condition (multiple seeds): 3-panel summary with mean±SEM.
    """
    metrics = [
        ("bout_success_rate", "bout success rate"),
        ("obs_rate", "observe rate"),
        ("obs_seen_lick_rate", "OBS sees cue rate"),
        ("eat_in_window_rate", "eat-in-window rate"),
        ("mean_reward", "episode mean reward"),
        ("familiarity", "familiarity"),
        ("p_detect_eff", "effective p(detect)"),
        ("lr", "learning rate"),
    ]
    ep = np.arange(1, episodes + 1, dtype=np.int32)

    # Build smoothed arrays for selected metrics
    def get_ms(metric):
        arr = np.stack([logs_to_array_with_ffill(run, metric, episodes) for run in runs], axis=0)
        arr_s = np.stack([ema_1d(arr[i], alpha=alpha_ema) for i in range(arr.shape[0])], axis=0)
        return mean_sem(arr_s)

    m_succ, s_succ = get_ms("bout_success_rate")
    m_obs, s_obs = get_ms("obs_rate")
    m_seen, s_seen = get_ms("obs_seen_lick_rate")
    m_eatw, s_eatw = get_ms("eat_in_window_rate")
    m_fam, s_fam = get_ms("familiarity")
    m_lr, s_lr = get_ms("lr")

    fig = plt.figure(figsize=(13, 4))
    plt.subplot(1, 3, 1)
    plot_metric_mean_sem(ep, m_succ, s_succ, label="success")
    plt.title(f"EMA bout success{title_suffix}")
    plt.xlabel("episode"); plt.ylabel("rate")
    plt.legend(fontsize=8)

    plt.subplot(1, 3, 2)
    plot_metric_mean_sem(ep, m_obs, s_obs, label="observe")
    plot_metric_mean_sem(ep, m_seen, s_seen, label="obs sees cue")
    plt.title("EMA observation metrics")
    plt.xlabel("episode"); plt.legend(fontsize=8)

    plt.subplot(1, 3, 3)
    plot_metric_mean_sem(ep, m_eatw, s_eatw, label="eat in window")
    if np.nanmax(m_fam) > 1e-6:
        plot_metric_mean_sem(ep, m_fam, s_fam, label="familiarity")
    if np.nanmax(m_lr) > 0:
        plot_metric_mean_sem(ep, m_lr, s_lr, label="lr")
    plt.title("EMA eating + familiarity + lr")
    plt.xlabel("episode"); plt.legend(fontsize=8)

    plt.tight_layout()
    _save_fig(fig, os.path.join(save_dir, "learning_curves_mean_sem"), ext=ext, pdf_pages=pdf_pages)

# ============================================================
# Significance tests for condition comparisons (no SciPy)
# ============================================================
def cohen_d(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    a = a[~np.isnan(a)]
    b = b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2:
        return float("nan")
    va = np.var(a, ddof=1)
    vb = np.var(b, ddof=1)
    sp = math.sqrt(((len(a)-1)*va + (len(b)-1)*vb) / max(1, (len(a)+len(b)-2)))
    if sp <= 1e-12:
        return 0.0
    return float((np.mean(a) - np.mean(b)) / sp)

def permutation_test_mean_diff(a: np.ndarray, b: np.ndarray, n_perm: int = 5000, seed: int = 0) -> float:
    """
    Two-sided permutation test on |mean(a)-mean(b)|.
    Returns p-value.
    """
    rng = np.random.RandomState(seed)
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    if len(a) == 0 or len(b) == 0:
        return float("nan")
    obs = abs(np.mean(a) - np.mean(b))
    x = np.concatenate([a, b], axis=0)
    na = len(a)
    count = 0
    for _ in range(int(n_perm)):
        rng.shuffle(x)
        d = abs(np.mean(x[:na]) - np.mean(x[na:]))
        if d >= obs - 1e-12:
            count += 1
    return float((count + 1) / (n_perm + 1))

def last_k_window_scores(runs: List[List[Dict[str, float]]], metric: str, episodes: int, last_k: int) -> np.ndarray:
    last_k = int(min(last_k, episodes))
    scores = []
    for logs in runs:
        y = logs_to_array_with_ffill(logs, metric, episodes)
        tail = y[-last_k:]
        scores.append(float(np.nanmean(tail)))
    return np.array(scores, dtype=np.float32)

def write_stats_report(
    out_path: str,
    group_key: str,
    metric: str,
    a_label: str,
    b_label: str,
    a_scores: np.ndarray,
    b_scores: np.ndarray,
    pval: float,
    d: float,
):
    with open(out_path, "a", encoding="utf-8") as f:
        f.write(f"\n[{group_key}] metric={metric}\n")
        f.write(f"  {a_label}: n={len(a_scores)} mean={float(np.mean(a_scores)):.4f} std={float(np.std(a_scores, ddof=1) if len(a_scores)>1 else 0.0):.4f}\n")
        f.write(f"  {b_label}: n={len(b_scores)} mean={float(np.mean(b_scores)):.4f} std={float(np.std(b_scores, ddof=1) if len(b_scores)>1 else 0.0):.4f}\n")
        f.write(f"  perm-test p={pval:.6f}  Cohen_d={d:.4f}\n")

# ============================================================
# Latent / “DA-like” simulation (kept; optional to run)
# ============================================================
def calcium_filter(impulses, dt_s, tau_rise_s=0.2, tau_decay_s=1.2):
    impulses = np.asarray(impulses, dtype=np.float32)
    n = len(impulses)
    L = int(np.ceil(8 * tau_decay_s / dt_s))
    t = np.arange(L, dtype=np.float32) * dt_s
    k = (1 - np.exp(-t / max(1e-6, tau_rise_s))) * np.exp(-t / max(1e-6, tau_decay_s))
    k = k / (k.sum() + 1e-9)
    y = np.convolve(impulses, k, mode="full")[:n]
    return y.astype(np.float32)

def make_event_indices(tr):
    lick = np.array(tr["teacher_lick"], dtype=int)
    obs = np.array(tr["observe"], dtype=int)
    seen = np.array(tr["seen_lick"], dtype=int) if "seen_lick" in tr else None
    reward = np.array(tr["eat_valid"], dtype=int)

    lick_end = np.where((lick[:-1] == 1) & (lick[1:] == 0))[0] + 1
    obs_idx = np.where(obs == 1)[0]
    if seen is not None:
        obs_seen_lick_idx = np.where((obs == 1) & (seen == 1))[0]
    else:
        obs_seen_lick_idx = np.array([], dtype=int)
    rew_idx = np.where(reward == 1)[0]

    return {
        "lick_end": lick_end,
        "obs": obs_idx,
        "obs_seen_lick": obs_seen_lick_idx,
        "reward": rew_idx,
    }

def peri_event_average(signal, event_idx, pre_steps=10, post_steps=20):
    signal = np.asarray(signal, dtype=np.float32)
    out = []
    for e in event_idx:
        a = e - pre_steps
        b = e + post_steps + 1
        if a < 0 or b > len(signal):
            continue
        out.append(signal[a:b])
    if len(out) == 0:
        return None
    return np.mean(np.stack(out, axis=0), axis=0)

def gae_advantages(rewards, values, dones, gamma=0.99, lam=0.95):
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        nonterminal = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * values[t+1] * nonterminal - values[t]
        gae = delta + gamma * lam * nonterminal * gae
        adv[t] = gae
    return adv

def simulate_latent_models_for_episode(tr, agent, gamma=0.99, ep_frac=0.0, gae_lam=0.95):
    r = np.array(tr["reward"], dtype=np.float32)
    obs = np.array(tr["observe"], dtype=int)
    eat = np.array(tr["eat"], dtype=int)
    seen_lick = np.array(tr["seen_lick"], dtype=int)
    phase = np.array(tr["phase"], dtype=np.float32)
    S = np.array(tr["state"], dtype=np.float32)
    S2 = np.array(tr["next_state"], dtype=np.float32)
    done = np.array(tr["done"], dtype=np.float32)

    V_s = agent.value_batch(S)
    V_s2 = agent.value_batch(S2)
    rpe = r + gamma * V_s2 * (1.0 - done) - V_s

    V_boot = np.concatenate([V_s, V_s2[-1:]], axis=0)
    adv = gae_advantages(r, V_boot[:len(r)+1], done, gamma=gamma, lam=gae_lam)

    S2_prior = S2.copy()
    S2_prior[:, 2] = 0.0
    S2_prior[:, 3] = 0.0
    V_post = V_s2
    V_prior = agent.value_batch(S2_prior)
    deltaV = (V_post - V_prior) * obs.astype(np.float32)

    sal = np.zeros(len(r), dtype=np.float32)
    sal += 1.0 * (obs == 1).astype(np.float32)
    sal += 0.6 * (eat == 1).astype(np.float32)

    sigma = 0.25 * (1.0 - ep_frac) + 0.10 * ep_frac
    p = np.exp(-(phase / max(1e-6, sigma)) ** 2)
    surprise = -np.log(p + 1e-6)
    info_gain = surprise * ((obs == 1) & (seen_lick == 1)).astype(np.float32)

    impulses = {
        "reward_RPE": 0.8 * rpe,
        "gae_adv": 0.7 * adv,
        "social_cue_RPE": 1.2 * deltaV,
        "social_value": 0.6 * deltaV,
        "action_salience": 0.3 * sal,
        "info_gain": 0.25 * info_gain,
    }

    if hasattr(agent, "policy_entropy_batch"):
        ent = agent.policy_entropy_batch(S)
        if ent is not None:
            impulses["policy_entropy"] = 0.15 * ent.astype(np.float32)

    return impulses

def simulate_and_plot_latents(traces, agent, dt_s, gamma=0.99,
                              save_dir="plots", ext="pdf", pdf_pages=None):
    impulses_by_model = defaultdict(list)
    n = len(traces)
    for i, tr in enumerate(traces):
        ep_frac = i / max(1, n - 1)
        imp = simulate_latent_models_for_episode(tr, agent, gamma=gamma, ep_frac=ep_frac)
        for k, v in imp.items():
            impulses_by_model[k].append(v)

    phot_by_model = {}
    for k, imps in impulses_by_model.items():
        phot_by_model[k] = [calcium_filter(x, dt_s=dt_s, tau_rise_s=0.2, tau_decay_s=1.2) for x in imps]
    # (kept as before; you can add SEM-peri comparisons similarly if you want)
    return phot_by_model


# ============================================================
# Agents
# ============================================================
class BaseAgent:
    def act(self, s: np.ndarray):
        raise NotImplementedError

    def record_transition(self, s, a, r, s2, done, extra):
        pass

    def step_update(self):
        return None

    def episode_update(self):
        return None

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def get_lr(self) -> float:
        return 0.0

# -------------------------
# DQN (dueling + double + huber + soft target)
# -------------------------
class DuelingQNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.V = nn.Linear(hidden, 1)
        self.A = nn.Linear(hidden, action_dim)

    def forward(self, x):
        h = self.backbone(x)
        v = self.V(h)
        a = self.A(h)
        return v + (a - a.mean(dim=1, keepdim=True))

class TorchReplayBuffer:
    def __init__(self, capacity, state_dim, device):
        self.capacity = int(capacity)
        self.device = device
        self.state = torch.zeros((capacity, state_dim), dtype=torch.float32, device=device)
        self.action = torch.zeros(capacity, dtype=torch.int64, device=device)
        self.reward = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.next_state = torch.zeros((capacity, state_dim), dtype=torch.float32, device=device)
        self.done = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.idx = 0
        self.size = 0

    def push(self, s, a, r, s2, done):
        self.state[self.idx] = torch.tensor(s, dtype=torch.float32, device=self.device)
        self.action[self.idx] = int(a)
        self.reward[self.idx] = float(r)
        self.next_state[self.idx] = torch.tensor(s2, dtype=torch.float32, device=self.device)
        self.done[self.idx] = float(done)
        self.idx = (self.idx + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        indices = torch.randint(0, self.size, (batch_size,), device=self.device)
        return (
            self.state[indices],
            self.action[indices].unsqueeze(1),
            self.reward[indices],
            self.next_state[indices],
            self.done[indices],
        )

    def __len__(self):
        return self.size

class DQNAgent(BaseAgent):
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        batch_size=256,
        buffer_size=50000,
        eps_start=1.0,
        eps_end=0.05,
        eps_decay=0.9995,
        tau=0.02,
        grad_clip=10.0,
        device=None,
        use_compile=False,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.batch_size = int(batch_size)
        self.tau = float(tau)
        self.grad_clip = float(grad_clip)

        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.q = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt.load_state_dict(self.q.state_dict())

        if use_compile and hasattr(torch, "compile"):
            try:
                self.q = torch.compile(self.q)
            except Exception:
                pass

        self.opt = optim.Adam(self.q.parameters(), lr=float(lr))
        self.loss_fn = nn.SmoothL1Loss()
        self.rb = TorchReplayBuffer(buffer_size, state_dim, self.device)
        self.scaler = GradScaler(enabled=self.device.type == "cuda")

    def get_lr(self) -> float:
        return float(self.opt.param_groups[0]["lr"])

    def act(self, s):
        if np.random.rand() < self.eps:
            return int(np.random.randint(self.action_dim)), {}
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            qv = self.q(st)
        return int(torch.argmax(qv, dim=1).item()), {}

    def record_transition(self, s, a, r, s2, done, extra):
        self.rb.push(s, a, r, s2, done)

    def step_update(self, updates_per_step=1):
        if len(self.rb) < self.batch_size:
            self.eps = max(self.eps_end, self.eps * self.eps_decay)
            return None

        last_loss = None
        for _ in range(int(updates_per_step)):
            s, a, r, s2, d = self.rb.sample(self.batch_size)
            with autocast(enabled=self.device.type == "cuda"):
                q_sa = self.q(s).gather(1, a).squeeze(1)
                with torch.no_grad():
                    a_star = torch.argmax(self.q(s2), dim=1, keepdim=True)
                    q_next = self.qt(s2).gather(1, a_star).squeeze(1)
                    target = r + self.gamma * q_next * (1.0 - d)
                loss = self.loss_fn(q_sa, target)

            self.opt.zero_grad()
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.opt)
            nn.utils.clip_grad_norm_(self.q.parameters(), self.grad_clip)
            self.scaler.step(self.opt)
            self.scaler.update()

            with torch.no_grad():
                for p, pt in zip(self.q.parameters(), self.qt.parameters()):
                    pt.data.mul_(1.0 - self.tau).add_(self.tau * p.data)

            last_loss = float(loss.item())

        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        return last_loss

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            qv = self.q(st)
            v = torch.max(qv, dim=1).values
        return v.detach().cpu().numpy().astype(np.float32)

# -------------------------
# PPO (actor-critic)
# -------------------------
class ActorCriticNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.pi = nn.Linear(hidden, action_dim)
        self.v = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.body(x)
        logits = self.pi(h)
        value = self.v(h).squeeze(-1)
        return logits, value

class PPOAgent(BaseAgent):
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_ratio=0.2,
        vf_coef=0.5,
        ent_coef=0.01,
        train_epochs=4,
        minibatch_size=256,
        max_grad_norm=1.0,
        device=None,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.gae_lambda = float(gae_lambda)
        self.clip_ratio = float(clip_ratio)
        self.vf_coef = float(vf_coef)
        self.ent_coef = float(ent_coef)
        self.train_epochs = int(train_epochs)
        self.minibatch_size = int(minibatch_size)
        self.max_grad_norm = float(max_grad_norm)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.net = ActorCriticNet(self.state_dim, self.action_dim).to(self.device)
        self.opt = optim.Adam(self.net.parameters(), lr=float(lr))
        self.roll = []

    def get_lr(self) -> float:
        return float(self.opt.param_groups[0]["lr"])

    def act(self, s):
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            logits, v = self.net(st)
            dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample()
            logp = dist.log_prob(a)
            ent = dist.entropy()
        return int(a.item()), {"logp": float(logp.item()), "v": float(v.item()), "ent": float(ent.item())}

    def record_transition(self, s, a, r, s2, done, extra):
        self.roll.append({
            "s": np.array(s, dtype=np.float32),
            "a": int(a),
            "r": float(r),
            "done": float(done),
            "logp": float(extra.get("logp", 0.0)),
            "v": float(extra.get("v", 0.0)),
        })

    def episode_update(self):
        if len(self.roll) < 8:
            self.roll.clear()
            return None

        S = np.stack([x["s"] for x in self.roll], axis=0).astype(np.float32)
        A = np.array([x["a"] for x in self.roll], dtype=np.int64)
        R = np.array([x["r"] for x in self.roll], dtype=np.float32)
        D = np.array([x["done"] for x in self.roll], dtype=np.float32)
        LOGP_OLD = np.array([x["logp"] for x in self.roll], dtype=np.float32)
        V = np.array([x["v"] for x in self.roll], dtype=np.float32)

        with torch.no_grad():
            st_last = torch.tensor(S[-1], dtype=torch.float32, device=self.device).unsqueeze(0)
            _, v_last = self.net(st_last)
            v_last = float(v_last.item())
        V_ext = np.concatenate([V, np.array([v_last], dtype=np.float32)], axis=0)

        ADV = gae_advantages(R, V_ext, D, gamma=self.gamma, lam=self.gae_lambda)
        RET = ADV + V
        ADV = (ADV - ADV.mean()) / (ADV.std() + 1e-8)

        ts = torch.tensor(S, dtype=torch.float32, device=self.device)
        ta = torch.tensor(A, dtype=torch.int64, device=self.device)
        told = torch.tensor(LOGP_OLD, dtype=torch.float32, device=self.device)
        tadv = torch.tensor(ADV, dtype=torch.float32, device=self.device)
        tret = torch.tensor(RET, dtype=torch.float32, device=self.device)

        n = len(S)
        idx = np.arange(n)
        info = {}

        for _ in range(self.train_epochs):
            np.random.shuffle(idx)
            for start in range(0, n, self.minibatch_size):
                mb = idx[start:start+self.minibatch_size]
                logits, vpred = self.net(ts[mb])
                dist = torch.distributions.Categorical(logits=logits)

                logp = dist.log_prob(ta[mb])
                ratio = torch.exp(logp - told[mb])

                clip = torch.clamp(ratio, 1.0 - self.clip_ratio, 1.0 + self.clip_ratio)
                pg_loss = -(torch.min(ratio * tadv[mb], clip * tadv[mb])).mean()

                v_loss = ((vpred - tret[mb]) ** 2).mean()
                ent = dist.entropy().mean()
                loss = pg_loss + self.vf_coef * v_loss - self.ent_coef * ent

                self.opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), self.max_grad_norm)
                self.opt.step()

                info = {
                    "pg_loss": float(pg_loss.item()),
                    "v_loss": float(v_loss.item()),
                    "ent": float(ent.item()),
                    "loss": float(loss.item()),
                }

        self.roll.clear()
        return info

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            _, v = self.net(st)
        return v.detach().cpu().numpy().astype(np.float32)

    def policy_entropy_batch(self, states: np.ndarray):
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            logits, _ = self.net(st)
            dist = torch.distributions.Categorical(logits=logits)
            ent = dist.entropy()
        return ent.detach().cpu().numpy().astype(np.float32)

# -------------------------
# Tabular Q-learning
# -------------------------
class TabularQAgent(BaseAgent):
    def __init__(self, action_dim=5, alpha=0.25, gamma=0.99, eps_start=1.0, eps_end=0.05, eps_decay=0.9995,
                 phase_bins=10, win_bins=7, cd_bins=6):
        self.action_dim = int(action_dim)
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.phase_bins = int(phase_bins)
        self.win_bins = int(win_bins)
        self.cd_bins = int(cd_bins)
        self.Q = defaultdict(lambda: np.zeros(self.action_dim, dtype=np.float32))

    def _disc(self, s: np.ndarray):
        lp, lf, lick, win, phase, cd = s.tolist()
        p = int(np.round(lp * 14))
        f = int(np.round(lf * 14))
        lickb = int(lick > 0.5)
        winb = int(np.round(win * (self.win_bins - 1)))
        winb = clamp_int(winb, 0, self.win_bins - 1)
        phb = int(np.floor(phase * self.phase_bins))
        phb = clamp_int(phb, 0, self.phase_bins - 1)
        cdb = int(np.round(cd * (self.cd_bins - 1)))
        cdb = clamp_int(cdb, 0, self.cd_bins - 1)
        return (p, f, lickb, winb, phb, cdb)

    def act(self, s):
        key = self._disc(s)
        if np.random.rand() < self.eps:
            a = int(np.random.randint(self.action_dim))
        else:
            a = int(np.argmax(self.Q[key]))
        return a, {"key": key}

    def record_transition(self, s, a, r, s2, done, extra):
        k = extra["key"]
        k2 = self._disc(s2)
        q = self.Q[k]
        target = float(r) + self.gamma * (0.0 if done else float(np.max(self.Q[k2])))
        q[a] = (1 - self.alpha) * q[a] + self.alpha * target
        self.eps = max(self.eps_end, self.eps * self.eps_decay)

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        out = np.zeros(len(states), dtype=np.float32)
        for i, s in enumerate(states):
            out[i] = float(np.max(self.Q[self._disc(s)]))
        return out

# ============================================================
# Training + logging (now logs lr too; supports multi-seed SEM)
# ============================================================
def train_social_with_latents(
    env,
    agent: BaseAgent,
    episodes=1200,
    log_every=100,
    updates_per_step=4,
    early_stop_threshold=0.8,
    early_stop_window=100,
    compute_latents: bool = True,
):
    logs = []
    traces = []
    success_history = deque(maxlen=early_stop_window)

    for ep in range(1, episodes + 1):
        s = env.reset()
        done = False

        ep_reward = 0.0
        succ_bouts = 0
        n_bouts = max(1, env.n_bouts)
        obs_steps = 0
        obs_seen = 0
        eat_in_win = 0

        tr = defaultdict(list)

        while not done:
            a, extra = agent.act(s)
            s2, r, done, info = env.step(a)
            ep_reward += r

            if info["eat_valid"] == 1:
                succ_bouts += 1
            if info["observe"] == 1:
                obs_steps += 1
                if info["seen_lick"] == 1:
                    obs_seen += 1
            if info["eat"] == 1 and info["window_open"] == 1:
                eat_in_win += 1

            if compute_latents:
                tr["state"].append(s.copy())
                tr["next_state"].append(s2.copy())
                tr["reward"].append(float(r))
                tr["done"].append(float(done))

                tr["teacher_lick"].append(int(info["teacher_lick"]))
                tr["window_open"].append(int(info["window_open"]))
                tr["observe"].append(int(info["observe"]))
                tr["eat"].append(int(info["eat"]))
                tr["eat_valid"].append(int(info["eat_valid"]))
                tr["seen_lick"].append(int(info["seen_lick"]))
                tr["win_rem_est"].append(float(info["win_rem_est"]))
                tr["phase"].append(float(s[4]))

            agent.record_transition(s, a, r, s2, done, extra)

            if isinstance(agent, DQNAgent):
                agent.step_update(updates_per_step=updates_per_step)

            s = s2

        if hasattr(agent, "episode_update"):
            agent.episode_update()

        if compute_latents:
            for k in list(tr.keys()):
                tr[k] = np.array(tr[k])
            traces.append(tr)

        logs.append({
            "ep": ep,
            "bout_success_rate": float(succ_bouts / n_bouts),
            "mean_reward": float(ep_reward),
            "obs_rate": float(obs_steps / max(1, env.max_steps)),
            "obs_seen_lick_rate": float(obs_seen / max(1, env.max_steps)),
            "eat_in_window_rate": float(eat_in_win / max(1, env.max_steps)),
            "eps": float(getattr(agent, "eps", np.nan)),
            "familiarity": float(getattr(env, "familiarity", 0.0)),
            "p_detect_eff": float(getattr(env, "_effective_detect_prob")()) if getattr(env, "social_visibility", 1.0) > 0.0 else 0.0,
            "lr": float(agent.get_lr() if hasattr(agent, "get_lr") else 0.0),
        })

        success_history.append(logs[-1]["bout_success_rate"])

        if ep % log_every == 0:
            recent = logs[-log_every:]
            print(
                f"[ep {ep:4d}] "
                f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f} "
                f"obs={np.mean([x['obs_rate'] for x in recent]):.3f} "
                f"obsSeen={np.mean([x['obs_seen_lick_rate'] for x in recent]):.3f} "
                f"eatWin={np.mean([x['eat_in_window_rate'] for x in recent]):.3f} "
                f"R={np.mean([x['mean_reward'] for x in recent]):.3f} "
                f"eps={logs[-1]['eps']:.3f} "
                f"fam={logs[-1]['familiarity']:.2f} "
                f"lr={logs[-1]['lr']:.2e}"
            )

        if len(success_history) == early_stop_window and np.mean(success_history) > early_stop_threshold:
            print(f"Early stopping at episode {ep}: Avg success rate {np.mean(success_history):.3f} > {early_stop_threshold}")
            break

    return logs, traces


# ============================================================
# Condition builder (unchanged logic)
# ============================================================
def build_env_for_condition(args, teacher_pal: str, learner_profile: str, familiarity_mode: str):
    lick_mean = args.lick_mean_s
    if teacher_pal == "high":
        lick_mean = args.pal_high_lick_mean_s
    elif teacher_pal == "low":
        lick_mean = args.pal_low_lick_mean_s

    env_kwargs = dict(
        grid_size=args.grid_size,
        dt_s=args.dt_s,
        max_time_s=args.max_time_s,
        teacher_period_s=args.teacher_period_s,
        teacher_jitter_s=args.teacher_jitter_s,
        lick_mean_s=lick_mean,
        lick_dist=args.lick_dist,
        lick_cv=args.lick_cv,
        window_s=args.window_s,
        eat_cooldown_s=args.eat_cooldown_s,
        observe_cost=args.observe_cost,
        eat_cost=args.eat_cost,
        attend_bonus=args.attend_bonus,
        p_detect_per_obs=args.p_detect_per_obs,
        win_rem_noise_steps=args.win_rem_noise_steps,

        social_visibility=1.0,
        detect_memory_steps=None,
        familiarity_enabled=(familiarity_mode == "on"),
        familiarity_init=args.fam_init,
        fam_decay_ep=args.fam_decay_ep,
        fam_gain_obs=args.fam_gain_obs,
        fam_gain_detect=args.fam_gain_detect,
        p_detect_fam_boost=args.p_detect_fam_boost,
        noise_fam_reduction=args.noise_fam_reduction,
    )

    if learner_profile == "normal":
        pass
    elif learner_profile == "blind":
        env_kwargs["social_visibility"] = 0.0
    elif learner_profile == "autistic":
        env_kwargs["p_detect_per_obs"] = args.aut_p_detect_per_obs
        env_kwargs["win_rem_noise_steps"] = args.aut_win_rem_noise_steps
        env_kwargs["detect_memory_steps"] = args.aut_detect_memory_steps
        env_kwargs["observe_cost"] = args.aut_observe_cost
        if familiarity_mode == "on":
            env_kwargs["fam_gain_obs"] = args.aut_fam_gain_obs
            env_kwargs["fam_gain_detect"] = args.aut_fam_gain_detect
            env_kwargs["p_detect_fam_boost"] = args.aut_p_detect_fam_boost
            env_kwargs["noise_fam_reduction"] = args.aut_noise_fam_reduction
    else:
        raise ValueError(f"Unknown learner_profile={learner_profile}")

    return SocialLickEnv1D(**env_kwargs)

def build_agent(args, state_dim=6, action_dim=5, lr_override: Optional[float] = None):
    lr_use = float(args.lr if lr_override is None else lr_override)
    if args.algo == "dqn":
        return DQNAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            lr=lr_use,
            gamma=args.gamma,
            batch_size=args.batch_size,
            buffer_size=args.buffer_size,
            eps_start=args.eps_start,
            eps_end=args.eps_end,
            eps_decay=args.eps_decay,
            tau=args.tau,
            grad_clip=args.grad_clip,
            use_compile=args.use_compile,
        )
    elif args.algo == "ppo":
        return PPOAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            lr=lr_use,
            gamma=args.gamma,
            gae_lambda=args.gae_lambda,
            clip_ratio=args.clip_ratio,
            vf_coef=args.vf_coef,
            ent_coef=args.ent_coef,
            train_epochs=args.ppo_epochs,
            minibatch_size=args.minibatch_size,
            max_grad_norm=args.max_grad_norm,
        )
    elif args.algo == "tabular":
        return TabularQAgent(
            action_dim=action_dim,
            alpha=args.tab_alpha,
            gamma=args.gamma,
            eps_start=args.eps_start,
            eps_end=args.eps_end,
            eps_decay=args.eps_decay,
        )
    else:
        raise ValueError(f"Unknown algo: {args.algo}")

# ============================================================
# Run: multi-seed condition runs + SEM plots + significance
# ============================================================
def run_one(seed: int, args, teacher_pal: str, learner_profile: str, familiarity_mode: str, lr_override: Optional[float] = None):
    set_seed(seed)
    env = build_env_for_condition(args, teacher_pal=teacher_pal, learner_profile=learner_profile, familiarity_mode=familiarity_mode)
    agent = build_agent(args, state_dim=6, action_dim=5, lr_override=lr_override)

    logs, traces = train_social_with_latents(
        env, agent,
        episodes=args.episodes,
        log_every=max(50, args.episodes // 12),
        updates_per_step=args.updates_per_step,
        early_stop_threshold=args.early_stop_threshold,
        early_stop_window=args.early_stop_window,
        compute_latents=(not args.no_latents),
    )
    phot_by_model = None
    if not args.no_latents:
        phot_by_model = simulate_and_plot_latents(traces, agent, dt_s=env.dt_s, gamma=args.gamma,
                                                  save_dir=args.save_dir, ext=args.plot_ext, pdf_pages=None)
    return logs, traces, phot_by_model

def run_single_condition_multi_seed(args, teacher_pal: str, learner_profile: str, familiarity_mode: str, out_dir: str):
    ensure_dir(out_dir)

    pdf_pages = None
    if args.plot_ext.lower() == "pdf" and args.multipage_pdf:
        pdf_pages = PdfPages(os.path.join(out_dir, f"all_figures_{args.algo}_mean_sem.pdf"))

    runs_logs: List[List[Dict[str, float]]] = []
    for i in range(args.n_seeds):
        seed_i = args.seed + i * args.seed_stride
        if args.save_per_seed_runs:
            seed_dir = os.path.join(out_dir, f"seed_{seed_i}")
            ensure_dir(seed_dir)
        print(f"  - seed {seed_i}")
        set_seed(seed_i)
        env = build_env_for_condition(args, teacher_pal=teacher_pal, learner_profile=learner_profile, familiarity_mode=familiarity_mode)
        agent = build_agent(args, state_dim=6, action_dim=5)

        logs, traces = train_social_with_latents(
            env, agent,
            episodes=args.episodes,
            log_every=max(50, args.episodes // 12),
            updates_per_step=args.updates_per_step,
            early_stop_threshold=args.early_stop_threshold,
            early_stop_window=args.early_stop_window,
            compute_latents=False,  # default off for multi-seed; set --no_latents false if you truly want it
        )
        runs_logs.append(logs)

    if not args.no_plots:
        title_suffix = f" (T={teacher_pal}, L={learner_profile}, fam={familiarity_mode}, n={args.n_seeds})"
        plot_single_learning_panel_mean_sem(
            runs=runs_logs,
            episodes=args.episodes,
            save_dir=out_dir,
            ext=args.plot_ext,
            pdf_pages=pdf_pages,
            title_suffix=title_suffix,
            alpha_ema=args.alpha_ema,
        )

    if pdf_pages is not None:
        pdf_pages.close()

    return runs_logs

def run_comparison_suite_multi_seed(args):
    """
    Main upgrade:
      - runs each condition with n_seeds
      - plots mean ± SEM learning curves
      - tests blind vs normal (and other within-group comparisons) using permutation test on last_k window
    """
    base_dir = args.save_dir
    ensure_dir(base_dir)

    cmp_pdf = None
    if args.plot_ext.lower() == "pdf" and args.multipage_pdf:
        cmp_pdf = PdfPages(os.path.join(base_dir, f"comparison_suite_{args.algo}_mean_sem.pdf"))

    teacher_pals = ["high", "low"] if args.compare_teacher_pal else [args.teacher_pal]
    profiles = ["normal", "blind", "autistic"] if args.compare_profiles else [args.learner_profile]
    fam_modes = ["off", "on"] if args.compare_familiarity else [args.familiarity]

    # run all conditions
    results: Dict[str, List[List[Dict[str, float]]]] = {}
    meta: Dict[str, Tuple[str, str, str]] = {}

    cond_list = []
    for tp in teacher_pals:
        for fm in fam_modes:
            for pr in profiles:
                cond_list.append((tp, pr, fm))

    for (tp, pr, fm) in cond_list:
        label = f"T={tp}|L={pr}|F={fm}"
        out_dir = os.path.join(base_dir, label.replace("|", "__").replace("=", "_"))
        print("\n" + "=" * 80)
        print(f"Running condition: {label}  (n_seeds={args.n_seeds})")
        runs_logs = run_single_condition_multi_seed(args, tp, pr, fm, out_dir=out_dir)
        results[label] = runs_logs
        meta[label] = (tp, pr, fm)

    # overlay plots (mean±SEM)
    if not args.no_plots:
        plot_comparison_curves_mean_sem(results, metric="bout_success_rate", episodes=args.episodes,
                                        save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                                        alpha_ema=args.alpha_ema,
                                        title="Bout success rate (mean ± SEM)")
        plot_comparison_curves_mean_sem(results, metric="mean_reward", episodes=args.episodes,
                                        save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                                        alpha_ema=args.alpha_ema,
                                        title="Episode reward (mean ± SEM)")
        plot_comparison_curves_mean_sem(results, metric="obs_rate", episodes=args.episodes,
                                        save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                                        alpha_ema=args.alpha_ema,
                                        title="Observe rate (mean ± SEM)")
        plot_comparison_curves_mean_sem(results, metric="obs_seen_lick_rate", episodes=args.episodes,
                                        save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                                        alpha_ema=args.alpha_ema,
                                        title="OBS sees cue rate (mean ± SEM)")

    if cmp_pdf is not None:
        cmp_pdf.close()

    # stats: within each (teacher_pal, familiarity) group, compare profiles
    stats_path = os.path.join(base_dir, "significance_report.txt")
    with open(stats_path, "w", encoding="utf-8") as f:
        f.write("Permutation-test significance on LAST-K episode-window means.\n")
        f.write(f"metric_last_k={args.stats_last_k}, n_perm={args.stats_n_perm}, seed={args.seed}\n\n")

    groups = defaultdict(list)  # (tp,fm) -> labels
    for label, (tp, pr, fm) in meta.items():
        groups[(tp, fm)].append(label)

    for (tp, fm), labels in groups.items():
        # map profile->label if exists
        prof_to_label = {}
        for lab in labels:
            _, pr, _ = meta[lab]
            prof_to_label[pr] = lab

        # compare normal vs blind if both present (your requested case)
        if "normal" in prof_to_label and "blind" in prof_to_label:
            a_lab = prof_to_label["normal"]
            b_lab = prof_to_label["blind"]
            a_runs = results[a_lab]
            b_runs = results[b_lab]

            for metric in ["bout_success_rate", "mean_reward", "obs_seen_lick_rate"]:
                a_scores = last_k_window_scores(a_runs, metric, args.episodes, args.stats_last_k)
                b_scores = last_k_window_scores(b_runs, metric, args.episodes, args.stats_last_k)
                pval = permutation_test_mean_diff(a_scores, b_scores, n_perm=args.stats_n_perm, seed=args.seed)
                d = cohen_d(a_scores, b_scores)
                write_stats_report(stats_path, f"T={tp},F={fm}", metric, a_lab, b_lab, a_scores, b_scores, pval, d)

        # also compare normal vs autistic if both present
        if "normal" in prof_to_label and "autistic" in prof_to_label:
            a_lab = prof_to_label["normal"]
            b_lab = prof_to_label["autistic"]
            a_runs = results[a_lab]
            b_runs = results[b_lab]
            for metric in ["bout_success_rate", "mean_reward", "obs_seen_lick_rate"]:
                a_scores = last_k_window_scores(a_runs, metric, args.episodes, args.stats_last_k)
                b_scores = last_k_window_scores(b_runs, metric, args.episodes, args.stats_last_k)
                pval = permutation_test_mean_diff(a_scores, b_scores, n_perm=args.stats_n_perm, seed=args.seed)
                d = cohen_d(a_scores, b_scores)
                write_stats_report(stats_path, f"T={tp},F={fm}", metric, a_lab, b_lab, a_scores, b_scores, pval, d)

    print("\nSaved significance report:", stats_path)
    return results

# ============================================================
# Learning-rate sweep (mean ± SEM error bars)
# ============================================================
def parse_lr_sweep(s: str) -> List[float]:
    if s is None or s.strip() == "":
        return []
    out = []
    for part in s.split(","):
        part = part.strip()
        if part == "":
            continue
        out.append(float(part))
    return out

def run_lr_sweep(args):
    lrs = parse_lr_sweep(args.lr_sweep)
    if len(lrs) == 0:
        return

    base_dir = args.save_dir
    ensure_dir(base_dir)

    # fixed condition for sweep
    tp = args.teacher_pal
    pr = args.learner_profile
    fm = args.familiarity

    means = []
    sems = []
    raw_means = {}  # lr -> per-seed scores

    for lr in lrs:
        scores = []
        print("\n" + "=" * 80)
        print(f"LR sweep: lr={lr:g} (n_seeds={args.n_seeds})  condition: T={tp}, L={pr}, F={fm}")
        for i in range(args.n_seeds):
            seed_i = args.seed + i * args.seed_stride
            set_seed(seed_i)
            env = build_env_for_condition(args, teacher_pal=tp, learner_profile=pr, familiarity_mode=fm)
            agent = build_agent(args, state_dim=6, action_dim=5, lr_override=lr)
            logs, _tr = train_social_with_latents(
                env, agent,
                episodes=args.episodes,
                log_every=max(50, args.episodes // 12),
                updates_per_step=args.updates_per_step,
                early_stop_threshold=args.early_stop_threshold,
                early_stop_window=args.early_stop_window,
                compute_latents=False,
            )
            y = logs_to_array_with_ffill(logs, "bout_success_rate", args.episodes)
            scores.append(float(np.mean(y[-args.stats_last_k:])))
        scores = np.array(scores, dtype=np.float32)
        raw_means[lr] = scores
        m = float(np.mean(scores))
        s = float(np.std(scores, ddof=1) / math.sqrt(len(scores))) if len(scores) > 1 else 0.0
        means.append(m); sems.append(s)

    # plot: mean ± SEM over learning rate
    fig = plt.figure(figsize=(6.8, 4.5))
    x = np.array(lrs, dtype=np.float32)
    y = np.array(means, dtype=np.float32)
    e = np.array(sems, dtype=np.float32)
    plt.errorbar(x, y, yerr=e, fmt="-o", capsize=4)
    if args.lr_sweep_logx:
        plt.xscale("log")
    plt.xlabel("learning rate")
    plt.ylabel(f"mean bout success (last {args.stats_last_k} eps)")
    plt.title("LR sweep: mean ± SEM")
    plt.tight_layout()
    _save_fig(fig, os.path.join(base_dir, "lr_sweep_bout_success_mean_sem"), ext=args.plot_ext, pdf_pages=None)

    # also write scores
    out_txt = os.path.join(base_dir, "lr_sweep_scores.txt")
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write(f"LR sweep scores (metric=bout_success_rate, last_k={args.stats_last_k})\n")
        for lr in lrs:
            sc = raw_means[lr]
            f.write(f"lr={lr:g}  n={len(sc)}  mean={float(np.mean(sc)):.4f}  sem={float(np.std(sc, ddof=1)/math.sqrt(len(sc)) if len(sc)>1 else 0.0):.4f}  values={sc.tolist()}\n")
    print("Saved LR sweep plot + scores:", out_txt)

# ============================================================
# run_experiment
# ============================================================
def run_experiment(args):
    ensure_dir(args.save_dir)

    # optional LR sweep
    if args.lr_sweep is not None and args.lr_sweep.strip() != "":
        run_lr_sweep(args)

    # comparison suite (multi-condition, multi-seed, SEM + significance)
    if args.compare_suite:
        return run_comparison_suite_multi_seed(args)

    # otherwise single condition (optionally multi-seed)
    out_dir = os.path.join(args.save_dir, f"single_T_{args.teacher_pal}_L_{args.learner_profile}_F_{args.familiarity}")
    runs = run_single_condition_multi_seed(args, args.teacher_pal, args.learner_profile, args.familiarity, out_dir=out_dir)

    # if n_seeds>1, you already got mean±SEM curves.
    # if n_seeds==1, it's still fine (SEM=0) and you get the same plots.
    return runs

# ============================================================
# Args
# ============================================================
def build_argparser():
    p = argparse.ArgumentParser(description="Active observational learning + condition suite + SEM plots + significance + LR sweep")

    # algo
    p.add_argument("--algo", type=str, default="dqn", choices=["dqn", "ppo", "tabular"])

    # run
    p.add_argument("--episodes", type=int, default=1200)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--seed_stride", type=int, default=1000, help="seed_i = seed + i*seed_stride")
    p.add_argument("--n_seeds", type=int, default=5, help="number of replicate runs per condition for SEM/statistics")
    p.add_argument("--save_dir", type=str, default="plots")
    p.add_argument("--save_per_seed_runs", action="store_true", help="create subfolders per seed (for your own debugging)")
    p.add_argument("--no_plots", action="store_true")
    p.add_argument("--plot_ext", type=str, default="pdf", choices=["pdf", "png"])
    p.add_argument("--multipage_pdf", action="store_true")
    p.add_argument("--updates_per_step", type=int, default=4)
    p.add_argument("--alpha_ema", type=float, default=0.02, help="EMA alpha used before mean±SEM aggregation")
    p.add_argument("--no_latents", action="store_true", help="skip latent simulation (recommended for multi-seed runs)")

    # significance settings
    p.add_argument("--stats_last_k", type=int, default=200, help="compute condition stats on last_k episode-window mean")
    p.add_argument("--stats_n_perm", type=int, default=5000, help="permutation test permutations")

    # environment
    p.add_argument("--grid_size", type=int, default=15)
    p.add_argument("--dt_s", type=float, default=0.5)
    p.add_argument("--max_time_s", type=float, default=120.0)
    p.add_argument("--teacher_period_s", type=float, default=30.0)
    p.add_argument("--teacher_jitter_s", type=float, default=6.0)
    p.add_argument("--lick_mean_s", type=float, default=1.0)
    p.add_argument("--lick_dist", type=str, default="gamma", choices=["gamma", "lognormal"])
    p.add_argument("--lick_cv", type=float, default=0.8)
    p.add_argument("--window_s", type=float, default=3.0)
    p.add_argument("--eat_cooldown_s", type=float, default=2.0)
    p.add_argument("--observe_cost", type=float, default=0.002)
    p.add_argument("--eat_cost", type=float, default=0.005)
    p.add_argument("--attend_bonus", type=float, default=0.004)
    p.add_argument("--p_detect_per_obs", type=float, default=0.70)
    p.add_argument("--win_rem_noise_steps", type=int, default=1)

    # common learning
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--eps_start", type=float, default=1.0)
    p.add_argument("--eps_end", type=float, default=0.05)
    p.add_argument("--eps_decay", type=float, default=0.9995)

    # DQN
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--buffer_size", type=int, default=50000)
    p.add_argument("--tau", type=float, default=0.02)
    p.add_argument("--grad_clip", type=float, default=10.0)
    p.add_argument("--use_compile", action="store_true")

    # PPO
    p.add_argument("--gae_lambda", type=float, default=0.95)
    p.add_argument("--clip_ratio", type=float, default=0.2)
    p.add_argument("--vf_coef", type=float, default=0.5)
    p.add_argument("--ent_coef", type=float, default=0.01)
    p.add_argument("--ppo_epochs", type=int, default=4)
    p.add_argument("--minibatch_size", type=int, default=256)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    # Tabular Q
    p.add_argument("--tab_alpha", type=float, default=0.25)

    # early stopping
    p.add_argument("--early_stop_threshold", type=float, default=0.8)
    p.add_argument("--early_stop_window", type=int, default=100)

    # condition controls
    p.add_argument("--teacher_pal", type=str, default="default", choices=["default", "high", "low"])
    p.add_argument("--pal_high_lick_mean_s", type=float, default=1.4)
    p.add_argument("--pal_low_lick_mean_s", type=float, default=0.6)

    p.add_argument("--learner_profile", type=str, default="normal", choices=["normal", "blind", "autistic"])
    p.add_argument("--familiarity", type=str, default="off", choices=["off", "on"])

    # familiarity baseline (normal)
    p.add_argument("--fam_init", type=float, default=0.0)
    p.add_argument("--fam_decay_ep", type=float, default=0.00)
    p.add_argument("--fam_gain_obs", type=float, default=0.01)
    p.add_argument("--fam_gain_detect", type=float, default=0.04)
    p.add_argument("--p_detect_fam_boost", type=float, default=0.20)
    p.add_argument("--noise_fam_reduction", type=float, default=0.70)

    # autistic overrides
    p.add_argument("--aut_p_detect_per_obs", type=float, default=0.25)
    p.add_argument("--aut_win_rem_noise_steps", type=int, default=3)
    p.add_argument("--aut_detect_memory_steps", type=int, default=12)
    p.add_argument("--aut_observe_cost", type=float, default=0.004)

    p.add_argument("--aut_fam_gain_obs", type=float, default=0.006)
    p.add_argument("--aut_fam_gain_detect", type=float, default=0.020)
    p.add_argument("--aut_p_detect_fam_boost", type=float, default=0.12)
    p.add_argument("--aut_noise_fam_reduction", type=float, default=0.45)

    # comparison suite switches
    p.add_argument("--compare_suite", action="store_true")
    p.add_argument("--compare_profiles", action="store_true")
    p.add_argument("--compare_teacher_pal", action="store_true")
    p.add_argument("--compare_familiarity", action="store_true")

    # LR sweep
    p.add_argument("--lr_sweep", type=str, default="", help="comma-separated learning rates, e.g. '1e-4,3e-4,1e-3'")
    p.add_argument("--lr_sweep_logx", action="store_true", help="log-scale x-axis for lr sweep plot")

    return p

# if __name__ == "__main__":
#     parser = build_argparser()
#     args, _ = parser.parse_known_args()
#     run_experiment(args)


In [ ]:
parser = build_argparser()
args = parser.parse_args([])  # start from defaults

args.algo = "dqn"
args.episodes = 800
args.save_dir = "plots_cmp"
args.compare_suite = True
args.compare_profiles = True
args.compare_teacher_pal = False
args.compare_familiarity = False
args.multipage_pdf = True

results = run_experiment(args)


In [ ]:
parser = build_argparser()
args = parser.parse_args([])  # start from defaults

args.algo = "dqn"
args.episodes = 800
args.save_dir = "plots_cmp"
args.compare_suite = True
args.compare_profiles = False
args.compare_teacher_pal = False
args.compare_familiarity = True
args.multipage_pdf = True

results = run_experiment(args)


In [ ]:
parser = build_argparser()
args = parser.parse_args([])  # start from defaults

args.algo = "dqn"
args.episodes = 800
args.save_dir = "plots_cmp"
args.compare_suite = True
args.compare_profiles = False
args.compare_teacher_pal = True
args.compare_familiarity = False
args.multipage_pdf = True

results = run_experiment(args)

## fix other 2 models

In [ ]:
def main(cli_args=None):
    """
    cli_args:
      - None: parse from sys.argv (normal terminal usage)
      - list[str]: parse from that list (Jupyter / programmatic usage)
      - argparse.Namespace: use directly
    """
    parser = build_argparser()

    if cli_args is None:
        args, _ = parser.parse_known_args()
    elif isinstance(cli_args, argparse.Namespace):
        args = cli_args
    else:
        # Jupyter: pass a list like ["--compare_suite", "--algo", "dqn", ...]
        args, _ = parser.parse_known_args(cli_args)

    return run_experiment(args)


if __name__ == "__main__":
    main()

results = main([
    "--algo", "dqn",
    "--episodes", "800",
    "--save_dir", "plots_cmp2",
    "--compare_suite",
    "--compare_profiles",
    "--compare_familiarity",
    "--teacher_pal", "default",
    "--multipage_pdf",
])

    

## RNN

In [ ]:
from typing import Optional
import numpy as np
import random
from collections import deque, defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import argparse
import os
import math

# -------------------------
# Seed / utils
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def clamp_int(x: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, x))

# -------------------------
# Actions
# -------------------------
A_LEFT = 0
A_STAY = 1
A_RIGHT = 2
A_OBS = 3  # consumes time; reveals social timing info stochastically
A_EAT = 4

# ============================================================
# Environment (same logic as your current code)
# ============================================================
class SocialLickEnv1D:
    """
    Fast but detailed (paper-aligned):
      dt_s = 0.5s/step
      teacher cue bursts ("lick bouts") with stochastic duration (mean ~ 1s)
      reward window: 3s after last cue step
    Observation gating:
      - social signal (cue + noisy window remaining) only appears when action==OBS
      - detection during cue burst is probabilistic per OBS step
    """
    def __init__(
        self,
        grid_size=15,
        dt_s=0.5,
        max_time_s=120.0,
        teacher_period_s=30.0,
        teacher_jitter_s=6.0,
        lick_mean_s=1.0,
        lick_dist="gamma",  # "gamma" or "lognormal"
        lick_cv=0.8,        # for lognormal
        window_s=3.0,
        eat_cooldown_s=2.0,
        observe_cost=0.002,
        eat_cost=0.005,
        attend_bonus=0.004,
        p_detect_per_obs=0.70,
        win_rem_noise_steps=1,
    ):
        self.grid_size = int(grid_size)
        self.dt_s = float(dt_s)
        self.max_steps = int(round(max_time_s / self.dt_s))
        self.teacher_period_steps = int(round(teacher_period_s / self.dt_s))
        self.teacher_jitter_steps = int(round(teacher_jitter_s / self.dt_s))
        self.lick_mean_s = float(lick_mean_s)
        self.lick_dist = str(lick_dist)
        self.lick_cv = float(lick_cv)
        self.window_steps = int(round(window_s / self.dt_s))
        self.eat_cooldown_steps = int(round(eat_cooldown_s / self.dt_s))
        self.observe_cost = float(observe_cost)
        self.eat_cost = float(eat_cost)
        self.attend_bonus = float(attend_bonus)
        self.p_detect_per_obs = float(p_detect_per_obs)
        self.win_rem_noise_steps = int(win_rem_noise_steps)

        assert self.teacher_period_steps > self.window_steps + 1
        assert self.max_steps > self.teacher_period_steps + 5
        self.reset()

    def _sample_lick_duration_steps(self):
        if self.lick_dist == "gamma":
            shape = 2.0
            scale = self.lick_mean_s / shape
            dur_s = float(np.random.gamma(shape, scale))
        elif self.lick_dist == "lognormal":
            cv = max(1e-6, self.lick_cv)
            sigma2 = np.log(1.0 + cv**2)
            sigma = np.sqrt(sigma2)
            mu = np.log(self.lick_mean_s) - 0.5 * sigma2
            dur_s = float(np.random.lognormal(mean=mu, sigma=sigma))
        else:
            raise ValueError(f"Unknown lick_dist={self.lick_dist}")
        steps = int(np.round(dur_s / self.dt_s))
        return max(1, steps), dur_s

    def reset(self):
        self.t = 0
        self.teacher_pos = int(np.random.randint(0, self.grid_size))
        candidates = [i for i in range(self.grid_size) if i != self.teacher_pos]
        self.learner_food_pos = int(np.random.choice(candidates))
        self.learner_pos = int(np.random.randint(0, self.grid_size))

        self.eat_cd = 0
        self.last_lick_step = None
        self.detected_bout_id = None
        self.rewarded_bout_ids = set()

        # build cue-burst timeline
        self.bout_id_at_step = -np.ones(self.max_steps + 1, dtype=np.int32)
        self.bout_start_steps = []
        self.bout_end_steps = []
        self.bout_dur_s_list = []
        bout_id = 0
        k = 1
        while True:
            nominal = k * self.teacher_period_steps
            if nominal >= self.max_steps:
                break
            jitter = int(np.random.randint(-self.teacher_jitter_steps, self.teacher_jitter_steps + 1))
            start = clamp_int(nominal + jitter, 1, self.max_steps - 2)
            dur_steps, dur_s = self._sample_lick_duration_steps()
            end = clamp_int(start + dur_steps - 1, start, self.max_steps - 1)
            self.bout_id_at_step[start:end + 1] = bout_id
            self.bout_start_steps.append(start)
            self.bout_end_steps.append(end)
            self.bout_dur_s_list.append(dur_s)
            bout_id += 1
            k += 1
        self.n_bouts = int(bout_id)
        return self._get_obs(lick_sig=0.0, win_rem=0.0)

    def _teacher_lick_now(self, t: int):
        bid = int(self.bout_id_at_step[t]) if (0 <= t <= self.max_steps) else -1
        return (1, bid) if bid >= 0 else (0, -1)

    def _window_open(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        return 1 if (0 <= dt < self.window_steps) else 0

    def _window_remaining(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        rem = self.window_steps - dt
        return int(max(0, rem))

    def _phase_to_next_nominal(self, t: int) -> int:
        mod = t % self.teacher_period_steps
        return self.teacher_period_steps - mod

    def _get_obs(self, lick_sig: float, win_rem: float):
        lp = float(self.learner_pos) / float(self.grid_size - 1)
        lf = float(self.learner_food_pos) / float(self.grid_size - 1)
        phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period_steps)
        cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown_steps))
        # [pos, food_pos, gated cue, gated window remaining, phase clock, cooldown]
        return np.array([lp, lf, float(lick_sig), float(win_rem), phase, cdn], dtype=np.float32)

    def step(self, action: int):
        self.t += 1
        teacher_lick, bout_id = self._teacher_lick_now(self.t)
        if teacher_lick == 1:
            self.last_lick_step = self.t

        window_open = self._window_open(self.t)
        win_rem_true = self._window_remaining(self.t)

        if self.eat_cd > 0:
            self.eat_cd -= 1

        lick_sig = 0.0
        win_rem = 0.0
        did_observe = 0
        did_eat = 0
        seen_lick = 0

        # movement / observe / eat
        if action == A_LEFT:
            self.learner_pos -= 1
        elif action == A_RIGHT:
            self.learner_pos += 1
        elif action == A_OBS:
            did_observe = 1
            if teacher_lick == 1 and bout_id >= 0 and (np.random.rand() < self.p_detect_per_obs):
                seen_lick = 1
                self.detected_bout_id = int(bout_id)
            lick_sig = float(seen_lick)
            if win_rem_true > 0:
                noise = int(np.random.randint(-self.win_rem_noise_steps, self.win_rem_noise_steps + 1))
                est = clamp_int(win_rem_true + noise, 0, self.window_steps)
                win_rem = float(est) / float(max(1, self.window_steps))
        elif action == A_EAT:
            did_eat = 1
        # A_STAY does nothing

        self.learner_pos = clamp_int(self.learner_pos, 0, self.grid_size - 1)

        reward = 0.0
        reward -= self.observe_cost * float(did_observe)
        reward -= self.eat_cost * float(did_eat)

        # shaping: reward OBS during cue only if bout not yet detected
        if did_observe == 1 and teacher_lick == 1 and bout_id >= 0:
            if self.detected_bout_id != int(bout_id):
                reward += self.attend_bonus

        # eat allowed?
        eat_allowed = (did_eat == 1 and self.eat_cd == 0)
        if eat_allowed:
            self.eat_cd = self.eat_cooldown_steps

        # success logic (1 reward per bout)
        eat_valid = 0
        if eat_allowed and (self.learner_pos == self.learner_food_pos) and (window_open == 1) and (self.last_lick_step is not None):
            last_bid = int(self.bout_id_at_step[self.last_lick_step]) if self.last_lick_step <= self.max_steps else -1
            if (last_bid >= 0) and (self.detected_bout_id == last_bid) and (last_bid not in self.rewarded_bout_ids):
                eat_valid = 1
                self.rewarded_bout_ids.add(last_bid)
                reward += 1.0

        done = bool(self.t >= self.max_steps)
        info = {
            "t": self.t,
            "teacher_lick": int(teacher_lick),
            "bout_id": int(bout_id),
            "window_open": int(window_open),
            "win_rem_true": int(win_rem_true),
            "action": int(action),
            "observe": int(did_observe),
            "eat": int(did_eat),
            "eat_allowed": int(eat_allowed),
            "eat_valid": int(eat_valid),
            "learner_pos": int(self.learner_pos),
            "food_pos": int(self.learner_food_pos),
            "seen_lick": int(seen_lick),
            "win_rem_est": float(win_rem),
            "detected_bout_id": -1 if self.detected_bout_id is None else int(self.detected_bout_id),
            "rewarded_bouts": len(self.rewarded_bout_ids),
            "n_bouts": int(self.n_bouts),
            "dt_s": float(self.dt_s),
            "window_steps": int(self.window_steps),
        }
        obs = self._get_obs(lick_sig=lick_sig, win_rem=win_rem)
        return obs, reward, done, info


# ============================================================
# Plot saving (PDF by default)
# ============================================================
def _save_fig(fig, out_path_no_ext: str, ext: str, pdf_pages: Optional[PdfPages] = None):
    os.makedirs(os.path.dirname(out_path_no_ext), exist_ok=True)
    if ext.lower() == "pdf" and pdf_pages is not None:
        pdf_pages.savefig(fig, bbox_inches="tight")
    else:
        fig.savefig(f"{out_path_no_ext}.{ext}", bbox_inches="tight")

def ema(x, alpha=0.02):
    x = np.asarray(x, dtype=np.float32)
    y = np.zeros_like(x)
    m = 0.0
    for i in range(len(x)):
        m = (1 - alpha) * m + alpha * x[i]
        y[i] = m
    return y

def plot_learning_curves(logs, save_dir="plots", ext="pdf", pdf_pages=None):
    ep = np.array([x["ep"] for x in logs])
    succ = ema([x["bout_success_rate"] for x in logs])
    obs = ema([x["obs_rate"] for x in logs])
    obsSeen = ema([x["obs_seen_lick_rate"] for x in logs])
    eatWin = ema([x["eat_in_window_rate"] for x in logs])

    fig = plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(ep, succ)
    plt.title("EMA bout success rate")
    plt.xlabel("episode"); plt.ylabel("rate")

    plt.subplot(1, 3, 2)
    plt.plot(ep, obs, label="observe")
    plt.plot(ep, obsSeen, label="obs sees cue")
    plt.title("EMA observation metrics")
    plt.xlabel("episode"); plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(ep, eatWin)
    plt.title("EMA eat-in-window rate")
    plt.xlabel("episode"); plt.ylabel("rate")

    plt.tight_layout()
    _save_fig(fig, os.path.join(save_dir, "learning_curves"), ext=ext, pdf_pages=pdf_pages)
#     plt.close(fig)

# ============================================================
# Latent / “DA-like” simulation utilities (algorithm-agnostic)
# ============================================================
def calcium_filter(impulses, dt_s, tau_rise_s=0.2, tau_decay_s=1.2):
    impulses = np.asarray(impulses, dtype=np.float32)
    n = len(impulses)
    L = int(np.ceil(8 * tau_decay_s / dt_s))
    t = np.arange(L, dtype=np.float32) * dt_s
    k = (1 - np.exp(-t / max(1e-6, tau_rise_s))) * np.exp(-t / max(1e-6, tau_decay_s))
    k = k / (k.sum() + 1e-9)
    y = np.convolve(impulses, k, mode="full")[:n]
    return y.astype(np.float32)

def make_event_indices(tr):
    lick = np.array(tr["teacher_lick"], dtype=int)
    obs = np.array(tr["observe"], dtype=int)
    seen = np.array(tr["seen_lick"], dtype=int) if "seen_lick" in tr else None
    reward = np.array(tr["eat_valid"], dtype=int)

    lick_end = np.where((lick[:-1] == 1) & (lick[1:] == 0))[0] + 1
    obs_idx = np.where(obs == 1)[0]
    if seen is not None:
        obs_seen_lick_idx = np.where((obs == 1) & (seen == 1))[0]
    else:
        obs_seen_lick_idx = np.array([], dtype=int)
    rew_idx = np.where(reward == 1)[0]

    return {
        "lick_end": lick_end,
        "obs": obs_idx,
        "obs_seen_lick": obs_seen_lick_idx,
        "reward": rew_idx,
    }

def peri_event_average(signal, event_idx, pre_steps=10, post_steps=20):
    signal = np.asarray(signal, dtype=np.float32)
    out = []
    for e in event_idx:
        a = e - pre_steps
        b = e + post_steps + 1
        if a < 0 or b > len(signal):
            continue
        out.append(signal[a:b])
    if len(out) == 0:
        return None
    return np.mean(np.stack(out, axis=0), axis=0)

def gae_advantages(rewards, values, dones, gamma=0.99, lam=0.95):
    """GAE(λ) computed from r_t, V_t, done_t. values length T+1 (bootstrapped)."""
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        nonterminal = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * values[t+1] * nonterminal - values[t]
        gae = delta + gamma * lam * nonterminal * gae
        adv[t] = gae
    return adv

def simulate_latent_models_for_episode(tr, agent, gamma=0.99, ep_frac=0.0, gae_lam=0.95):
    """
    Candidate latent signal hypotheses (paper-aligned, but algorithm-agnostic):
      - reward_RPE (TD error)
      - gae_adv (advantage-like)
      - social_cue_RPE / social_value (ΔV on Observe due to revealing gated cue features)
      - action_salience
      - info_gain (surprise-scaled Observe-detect)
      - optional: policy_entropy (if agent exposes policy entropy)
    """
    r = np.array(tr["reward"], dtype=np.float32)
    obs = np.array(tr["observe"], dtype=int)
    eat = np.array(tr["eat"], dtype=int)
    seen_lick = np.array(tr["seen_lick"], dtype=int)
    phase = np.array(tr["phase"], dtype=np.float32)
    S = np.array(tr["state"], dtype=np.float32)
    S2 = np.array(tr["next_state"], dtype=np.float32)
    done = np.array(tr["done"], dtype=np.float32)

    V_s = agent.value_batch(S)
    V_s2 = agent.value_batch(S2)

    # TD error (reward_RPE)
    rpe = r + gamma * V_s2 * (1.0 - done) - V_s

    # GAE advantage-like signal
    V_boot = np.concatenate([V_s, V_s2[-1:]], axis=0)  # crude: last bootstrap
    adv = gae_advantages(r, V_boot[:len(r)+1], done, gamma=gamma, lam=gae_lam)

    # Observation-driven ΔV: V_post - V_prior (social features zeroed)
    S2_prior = S2.copy()
    S2_prior[:, 2] = 0.0  # lick_sig
    S2_prior[:, 3] = 0.0  # win_rem
    V_post = V_s2
    V_prior = agent.value_batch(S2_prior)
    deltaV = (V_post - V_prior) * obs.astype(np.float32)

    # Action salience
    sal = np.zeros(len(r), dtype=np.float32)
    sal += 1.0 * (obs == 1).astype(np.float32)
    sal += 0.6 * (eat == 1).astype(np.float32)

    # Info gain / surprise when OBS successfully detects cue
    sigma = 0.25 * (1.0 - ep_frac) + 0.10 * ep_frac
    p = np.exp(-(phase / max(1e-6, sigma)) ** 2)  # 0..1
    surprise = -np.log(p + 1e-6)
    info_gain = surprise * ((obs == 1) & (seen_lick == 1)).astype(np.float32)

    impulses = {
        "reward_RPE": 0.8 * rpe,
        "gae_adv": 0.7 * adv,
        "social_cue_RPE": 1.2 * deltaV,
        "social_value": 0.6 * deltaV,
        "action_salience": 0.3 * sal,
        "info_gain": 0.25 * info_gain,
    }

    # Optional policy entropy latent (actor-critic learners)
    if hasattr(agent, "policy_entropy_batch"):
        ent = agent.policy_entropy_batch(S)  # shape [T]
        if ent is not None:
            impulses["policy_entropy"] = 0.15 * ent.astype(np.float32)

    return impulses

def plot_da_peri_event(da_by_ep, traces, dt_s, model_name, event_key="reward",
                       pre_s=6, post_s=10, save_dir="plots", ext="pdf", pdf_pages=None):
    pre = int(round(pre_s / dt_s))
    post = int(round(post_s / dt_s))
    x = (np.arange(pre + post + 1) - pre) * dt_s

    n = len(traces)
    early_idx = range(0, max(1, n // 5))
    late_idx = range(max(0, n - n // 5), n)

    def mean_peri(idxs):
        mats = []
        for i in idxs:
            ev = make_event_indices(traces[i])[event_key]
            m = peri_event_average(da_by_ep[i], ev, pre_steps=pre, post_steps=post)
            if m is not None:
                mats.append(m)
        if len(mats) == 0:
            return None
        return np.mean(np.stack(mats, axis=0), axis=0)

    e = mean_peri(early_idx)
    l = mean_peri(late_idx)

    fig = plt.figure(figsize=(6, 4))
    if e is not None:
        plt.plot(x, e, label="early (first 20%)")
    if l is not None:
        plt.plot(x, l, label="late (last 20%)")
    plt.axvline(0, linestyle="--")
    plt.title(f"{model_name}: peri-{event_key}")
    plt.xlabel("time (s)")
    plt.ylabel("simulated signal")
    plt.legend()
    plt.tight_layout()

    out = os.path.join(save_dir, f"{model_name}_peri_{event_key}")
    _save_fig(fig, out, ext=ext, pdf_pages=pdf_pages)
#     plt.close(fig)

def simulate_and_plot_latents(traces, agent, dt_s, gamma=0.99,
                              save_dir="plots", ext="pdf", pdf_pages=None):
    impulses_by_model = defaultdict(list)
    n = len(traces)
    for i, tr in enumerate(traces):
        ep_frac = i / max(1, n - 1)
        imp = simulate_latent_models_for_episode(tr, agent, gamma=gamma, ep_frac=ep_frac)
        for k, v in imp.items():
            impulses_by_model[k].append(v)

    phot_by_model = {}
    for k, imps in impulses_by_model.items():
        phot_by_model[k] = [calcium_filter(x, dt_s=dt_s, tau_rise_s=0.2, tau_decay_s=1.2) for x in imps]

    for k, da_eps in phot_by_model.items():
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="lick_end",
                           pre_s=6, post_s=10, save_dir=save_dir, ext=ext, pdf_pages=pdf_pages)
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="reward",
                           pre_s=6, post_s=10, save_dir=save_dir, ext=ext, pdf_pages=pdf_pages)
        plot_da_peri_event(da_eps, traces, dt_s, model_name=k, event_key="obs_seen_lick",
                           pre_s=6, post_s=10, save_dir=save_dir, ext=ext, pdf_pages=pdf_pages)

    return phot_by_model


# ============================================================
# Agents (multiple learner models)
# ============================================================
class BaseAgent:
    def act(self, s: np.ndarray):
        """Return (action:int, extra:dict)."""
        raise NotImplementedError

    def record_transition(self, s, a, r, s2, done, extra):
        pass

    def step_update(self):
        return None

    def episode_update(self):
        return None

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        raise NotImplementedError


# -------------------------
# DQN (dueling + double + huber + soft target)
# -------------------------
class DuelingQNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.V = nn.Linear(hidden, 1)
        self.A = nn.Linear(hidden, action_dim)

    def forward(self, x):
        h = self.backbone(x)
        v = self.V(h)
        a = self.A(h)
        return v + (a - a.mean(dim=1, keepdim=True))

class TorchReplayBuffer:
    def __init__(self, capacity, state_dim, device):
        self.capacity = int(capacity)
        self.device = device
        self.state = torch.zeros((capacity, state_dim), dtype=torch.float32, device=device)
        self.action = torch.zeros(capacity, dtype=torch.int64, device=device)
        self.reward = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.next_state = torch.zeros((capacity, state_dim), dtype=torch.float32, device=device)
        self.done = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.idx = 0
        self.size = 0

    def push(self, s, a, r, s2, done):
        self.state[self.idx] = torch.tensor(s, dtype=torch.float32, device=self.device)
        self.action[self.idx] = int(a)
        self.reward[self.idx] = float(r)
        self.next_state[self.idx] = torch.tensor(s2, dtype=torch.float32, device=self.device)
        self.done[self.idx] = float(done)
        self.idx = (self.idx + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        indices = torch.randint(0, self.size, (batch_size,), device=self.device)
        return (
            self.state[indices],
            self.action[indices].unsqueeze(1),
            self.reward[indices],
            self.next_state[indices],
            self.done[indices],
        )

    def __len__(self):
        return self.size

class DQNAgent(BaseAgent):
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        batch_size=256,
        buffer_size=50000,
        eps_start=1.0,
        eps_end=0.05,
        eps_decay=0.9995,
        tau=0.02,
        grad_clip=10.0,
        device=None,
        use_compile=False,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.batch_size = int(batch_size)
        self.tau = float(tau)
        self.grad_clip = float(grad_clip)

        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.q = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt.load_state_dict(self.q.state_dict())

        if use_compile and hasattr(torch, "compile"):
            try:
                self.q = torch.compile(self.q)
            except Exception:
                pass

        self.opt = optim.Adam(self.q.parameters(), lr=float(lr))
        self.loss_fn = nn.SmoothL1Loss()
        self.rb = TorchReplayBuffer(buffer_size, state_dim, self.device)
        self.scaler = GradScaler(enabled=self.device.type == "cuda")

    def act(self, s):
        if np.random.rand() < self.eps:
            return int(np.random.randint(self.action_dim)), {}
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            qv = self.q(st)
        return int(torch.argmax(qv, dim=1).item()), {}

    def record_transition(self, s, a, r, s2, done, extra):
        self.rb.push(s, a, r, s2, done)

    def step_update(self, updates_per_step=1):
        if len(self.rb) < self.batch_size:
            self.eps = max(self.eps_end, self.eps * self.eps_decay)
            return None

        last_loss = None
        for _ in range(int(updates_per_step)):
            s, a, r, s2, d = self.rb.sample(self.batch_size)
            with autocast(enabled=self.device.type == "cuda"):
                q_sa = self.q(s).gather(1, a).squeeze(1)
                with torch.no_grad():
                    a_star = torch.argmax(self.q(s2), dim=1, keepdim=True)
                    q_next = self.qt(s2).gather(1, a_star).squeeze(1)
                    target = r + self.gamma * q_next * (1.0 - d)
                loss = self.loss_fn(q_sa, target)

            self.opt.zero_grad()
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.opt)
            nn.utils.clip_grad_norm_(self.q.parameters(), self.grad_clip)
            self.scaler.step(self.opt)
            self.scaler.update()

            with torch.no_grad():
                for p, pt in zip(self.q.parameters(), self.qt.parameters()):
                    pt.data.mul_(1.0 - self.tau).add_(self.tau * p.data)

            last_loss = float(loss.item())

        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        return last_loss

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            qv = self.q(st)
            v = torch.max(qv, dim=1).values
        return v.detach().cpu().numpy().astype(np.float32)

# -------------------------
# RNN-DQN (DRQN-style): GRU + dueling head + double + soft target
# -------------------------
class RecurrentDuelingQNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
        )
        self.gru = nn.GRU(hidden, hidden, batch_first=True)
        self.V = nn.Linear(hidden, 1)
        self.A = nn.Linear(hidden, action_dim)

    def forward(self, x, h0=None):
        """
        x: [B, T, state_dim]
        h0: [1, B, hidden] or None
        returns:
          q: [B, T, action_dim]
          hT: [1, B, hidden]
        """
        z = self.embed(x)                 # [B, T, H]
        out, hT = self.gru(z, h0)         # out: [B, T, H]
        v = self.V(out)                   # [B, T, 1]
        a = self.A(out)                   # [B, T, A]
        q = v + (a - a.mean(dim=-1, keepdim=True))
        return q, hT


class RecurrentReplayBuffer:
    """
    Stores whole episodes; samples random contiguous chunks for truncated BPTT.
    """
    def __init__(self, capacity_episodes: int = 2000):
        self.capacity_episodes = int(capacity_episodes)
        self.episodes = []  # list of dicts with keys: s,a,r,s2,d
        self._cur = {"s": [], "a": [], "r": [], "s2": [], "d": []}

    def push_step(self, s, a, r, s2, done):
        self._cur["s"].append(np.array(s, dtype=np.float32))
        self._cur["a"].append(int(a))
        self._cur["r"].append(float(r))
        self._cur["s2"].append(np.array(s2, dtype=np.float32))
        self._cur["d"].append(float(done))
        if done:
            self.end_episode()

    def end_episode(self):
        if len(self._cur["s"]) == 0:
            return
        ep = {k: np.array(v) for k, v in self._cur.items()}
        self.episodes.append(ep)
        if len(self.episodes) > self.capacity_episodes:
            self.episodes.pop(0)
        self._cur = {"s": [], "a": [], "r": [], "s2": [], "d": []}

    def __len__(self):
        return len(self.episodes)

    def sample_chunks(self, batch_size: int, seq_len: int):
        """
        returns numpy arrays:
          S  [B, L, D], A [B, L], R [B, L], S2 [B, L, D], Dn [B, L]
        """
        batch_size = int(batch_size)
        seq_len = int(seq_len)

        valid = [ep for ep in self.episodes if len(ep["s"]) >= seq_len]
        if len(valid) == 0:
            return None

        S = []
        A = []
        R = []
        S2 = []
        Dn = []

        for _ in range(batch_size):
            ep = random.choice(valid)
            T = len(ep["s"])
            start = random.randint(0, T - seq_len)
            end = start + seq_len
            S.append(ep["s"][start:end])
            A.append(ep["a"][start:end])
            R.append(ep["r"][start:end])
            S2.append(ep["s2"][start:end])
            Dn.append(ep["d"][start:end])

        return (
            np.stack(S, axis=0).astype(np.float32),
            np.stack(A, axis=0).astype(np.int64),
            np.stack(R, axis=0).astype(np.float32),
            np.stack(S2, axis=0).astype(np.float32),
            np.stack(Dn, axis=0).astype(np.float32),
        )


class RNNDQNAgent(BaseAgent):
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        eps_start=1.0,
        eps_end=0.05,
        eps_decay=0.9995,
        tau=0.02,
        grad_clip=10.0,
        rnn_hidden=128,
        seq_len=16,
        rnn_batch_size=64,
        buffer_episodes=2000,
        device=None,
        use_compile=False,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)

        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.tau = float(tau)
        self.grad_clip = float(grad_clip)

        self.seq_len = int(seq_len)
        self.rnn_batch_size = int(rnn_batch_size)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.q = RecurrentDuelingQNet(self.state_dim, self.action_dim, hidden=int(rnn_hidden)).to(self.device)
        self.qt = RecurrentDuelingQNet(self.state_dim, self.action_dim, hidden=int(rnn_hidden)).to(self.device)
        self.qt.load_state_dict(self.q.state_dict())

        if use_compile and hasattr(torch, "compile"):
            try:
                self.q = torch.compile(self.q)
            except Exception:
                pass

        self.opt = optim.Adam(self.q.parameters(), lr=float(lr))
        self.loss_fn = nn.SmoothL1Loss()
        self.scaler = GradScaler(enabled=self.device.type == "cuda")

        self.rb = RecurrentReplayBuffer(capacity_episodes=int(buffer_episodes))

        # hidden state for online acting
        self._h = None  # torch tensor [1,1,H] when alive

    def reset_episode(self):
        self._h = None

    def act(self, s):
        st = torch.tensor(s, dtype=torch.float32, device=self.device).view(1, 1, -1)
        with torch.no_grad():
            q_seq, hT = self.q(st, self._h)
        self._h = hT

        if np.random.rand() < self.eps:
            a = int(np.random.randint(self.action_dim))
        else:
            qv = q_seq[0, 0]  # [A]
            a = int(torch.argmax(qv).item())
        return a, {}

    def record_transition(self, s, a, r, s2, done, extra):
        self.rb.push_step(s, a, r, s2, done)

    def step_update(self, updates_per_step=1):
        if len(self.rb) < 5:
            self.eps = max(self.eps_end, self.eps * self.eps_decay)
            return None

        last_loss = None
        for _ in range(int(updates_per_step)):
            batch = self.rb.sample_chunks(self.rnn_batch_size, self.seq_len)
            if batch is None:
                break
            S, A, R, S2, Dn = batch

            ts = torch.tensor(S, dtype=torch.float32, device=self.device)    # [B,L,D]
            ta = torch.tensor(A, dtype=torch.int64, device=self.device)      # [B,L]
            tr = torch.tensor(R, dtype=torch.float32, device=self.device)    # [B,L]
            ts2 = torch.tensor(S2, dtype=torch.float32, device=self.device)  # [B,L,D]
            td = torch.tensor(Dn, dtype=torch.float32, device=self.device)   # [B,L]

            with autocast(enabled=self.device.type == "cuda"):
                q, _ = self.q(ts, None)   # [B,L,A]
                q_sa = q.gather(-1, ta.unsqueeze(-1)).squeeze(-1)  # [B,L]

                with torch.no_grad():
                    q2_online, _ = self.q(ts2, None)               # [B,L,A]
                    a_star = torch.argmax(q2_online, dim=-1, keepdim=True)  # [B,L,1]
                    q2_targ, _ = self.qt(ts2, None)                # [B,L,A]
                    q_next = q2_targ.gather(-1, a_star).squeeze(-1)         # [B,L]
                    target = tr + self.gamma * q_next * (1.0 - td)

                loss = self.loss_fn(q_sa, target)

            self.opt.zero_grad()
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.opt)
            nn.utils.clip_grad_norm_(self.q.parameters(), self.grad_clip)
            self.scaler.step(self.opt)
            self.scaler.update()

            with torch.no_grad():
                for p, pt in zip(self.q.parameters(), self.qt.parameters()):
                    pt.data.mul_(1.0 - self.tau).add_(self.tau * p.data)

            last_loss = float(loss.item())

        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        return last_loss

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        """
        For latent sims: treat the provided states as an ordered sequence and run GRU once.
        """
        if states.ndim != 2:
            states = np.asarray(states, dtype=np.float32).reshape(-1, self.state_dim)
        st = torch.tensor(states, dtype=torch.float32, device=self.device).unsqueeze(0)  # [1,T,D]
        with torch.no_grad():
            q, _ = self.q(st, None)  # [1,T,A]
            v = torch.max(q[0], dim=-1).values  # [T]
        return v.detach().cpu().numpy().astype(np.float32)

# -------------------------
# PPO (actor-critic)
# -------------------------
class ActorCriticNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.pi = nn.Linear(hidden, action_dim)
        self.v = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.body(x)
        logits = self.pi(h)
        value = self.v(h).squeeze(-1)
        return logits, value

class PPOAgent(BaseAgent):
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_ratio=0.2,
        vf_coef=0.5,
        ent_coef=0.01,
        train_epochs=4,
        minibatch_size=256,
        max_grad_norm=1.0,
        device=None,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.gae_lambda = float(gae_lambda)
        self.clip_ratio = float(clip_ratio)
        self.vf_coef = float(vf_coef)
        self.ent_coef = float(ent_coef)
        self.train_epochs = int(train_epochs)
        self.minibatch_size = int(minibatch_size)
        self.max_grad_norm = float(max_grad_norm)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.net = ActorCriticNet(self.state_dim, self.action_dim).to(self.device)
        self.opt = optim.Adam(self.net.parameters(), lr=float(lr))

        self.roll = []  # stores dicts per step

    def act(self, s):
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            logits, v = self.net(st)
            dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample()
            logp = dist.log_prob(a)
            ent = dist.entropy()
        return int(a.item()), {"logp": float(logp.item()), "v": float(v.item()), "ent": float(ent.item())}

    def record_transition(self, s, a, r, s2, done, extra):
        self.roll.append({
            "s": np.array(s, dtype=np.float32),
            "a": int(a),
            "r": float(r),
            "done": float(done),
            "logp": float(extra.get("logp", 0.0)),
            "v": float(extra.get("v", 0.0)),
        })

    def episode_update(self):
        if len(self.roll) < 8:
            self.roll.clear()
            return None

        S = np.stack([x["s"] for x in self.roll], axis=0).astype(np.float32)
        A = np.array([x["a"] for x in self.roll], dtype=np.int64)
        R = np.array([x["r"] for x in self.roll], dtype=np.float32)
        D = np.array([x["done"] for x in self.roll], dtype=np.float32)
        LOGP_OLD = np.array([x["logp"] for x in self.roll], dtype=np.float32)
        V = np.array([x["v"] for x in self.roll], dtype=np.float32)

        # bootstrap last value with current critic (on last state)
        with torch.no_grad():
            st_last = torch.tensor(S[-1], dtype=torch.float32, device=self.device).unsqueeze(0)
            _, v_last = self.net(st_last)
            v_last = float(v_last.item())
        V_ext = np.concatenate([V, np.array([v_last], dtype=np.float32)], axis=0)

        ADV = gae_advantages(R, V_ext, D, gamma=self.gamma, lam=self.gae_lambda)
        RET = ADV + V  # return target

        # normalize advantages
        ADV = (ADV - ADV.mean()) / (ADV.std() + 1e-8)

        # torch tensors
        ts = torch.tensor(S, dtype=torch.float32, device=self.device)
        ta = torch.tensor(A, dtype=torch.int64, device=self.device)
        told = torch.tensor(LOGP_OLD, dtype=torch.float32, device=self.device)
        tadv = torch.tensor(ADV, dtype=torch.float32, device=self.device)
        tret = torch.tensor(RET, dtype=torch.float32, device=self.device)

        n = len(S)
        idx = np.arange(n)
        info = {}

        for _ in range(self.train_epochs):
            np.random.shuffle(idx)
            for start in range(0, n, self.minibatch_size):
                mb = idx[start:start+self.minibatch_size]
                logits, vpred = self.net(ts[mb])
                dist = torch.distributions.Categorical(logits=logits)

                logp = dist.log_prob(ta[mb])
                ratio = torch.exp(logp - told[mb])

                clip = torch.clamp(ratio, 1.0 - self.clip_ratio, 1.0 + self.clip_ratio)
                pg_loss = -(torch.min(ratio * tadv[mb], clip * tadv[mb])).mean()

                v_loss = ((vpred - tret[mb]) ** 2).mean()
                ent = dist.entropy().mean()

                loss = pg_loss + self.vf_coef * v_loss - self.ent_coef * ent

                self.opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), self.max_grad_norm)
                self.opt.step()

                info = {
                    "pg_loss": float(pg_loss.item()),
                    "v_loss": float(v_loss.item()),
                    "ent": float(ent.item()),
                    "loss": float(loss.item()),
                }

        self.roll.clear()
        return info

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            _, v = self.net(st)
        return v.detach().cpu().numpy().astype(np.float32)

    def policy_entropy_batch(self, states: np.ndarray):
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            logits, _ = self.net(st)
            dist = torch.distributions.Categorical(logits=logits)
            ent = dist.entropy()
        return ent.detach().cpu().numpy().astype(np.float32)

# -------------------------
# Tabular Q-learning (light baseline)
# -------------------------
class TabularQAgent(BaseAgent):
    def __init__(self, action_dim=5, alpha=0.25, gamma=0.99, eps_start=1.0, eps_end=0.05, eps_decay=0.9995,
                 phase_bins=10, win_bins=7, cd_bins=6, device=None):
        self.action_dim = int(action_dim)
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.phase_bins = int(phase_bins)
        self.win_bins = int(win_bins)
        self.cd_bins = int(cd_bins)

        self.Q = defaultdict(lambda: np.zeros(self.action_dim, dtype=np.float32))
        self._last = None

    def _disc(self, s: np.ndarray):
        # s = [lp, lf, lick_sig, win_rem, phase, cdn]
        lp, lf, lick, win, phase, cd = s.tolist()
        # positions already normalized, discretize to 0..14-ish by rounding*14
        p = int(np.round(lp * 14))
        f = int(np.round(lf * 14))
        lickb = int(lick > 0.5)
        winb = int(np.round(win * (self.win_bins - 1)))
        winb = clamp_int(winb, 0, self.win_bins - 1)
        phb = int(np.floor(phase * self.phase_bins))
        phb = clamp_int(phb, 0, self.phase_bins - 1)
        cdb = int(np.round(cd * (self.cd_bins - 1)))
        cdb = clamp_int(cdb, 0, self.cd_bins - 1)
        return (p, f, lickb, winb, phb, cdb)

    def act(self, s):
        key = self._disc(s)
        if np.random.rand() < self.eps:
            a = int(np.random.randint(self.action_dim))
        else:
            a = int(np.argmax(self.Q[key]))
        return a, {"key": key}

    def record_transition(self, s, a, r, s2, done, extra):
        k = extra["key"]
        k2 = self._disc(s2)
        q = self.Q[k]
        target = float(r) + self.gamma * (0.0 if done else float(np.max(self.Q[k2])))
        q[a] = (1 - self.alpha) * q[a] + self.alpha * target

        self.eps = max(self.eps_end, self.eps * self.eps_decay)

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        out = np.zeros(len(states), dtype=np.float32)
        for i, s in enumerate(states):
            out[i] = float(np.max(self.Q[self._disc(s)]))
        return out


# ============================================================
# Training + logging (works for all agents above)
# ============================================================
def train_social_with_latents(env, agent: BaseAgent, episodes=1200, log_every=100,
                             updates_per_step=4, early_stop_threshold=0.8, early_stop_window=100):
    logs = []
    traces = []
    success_history = deque(maxlen=early_stop_window)

    for ep in range(1, episodes + 1):
        s = env.reset()
        if hasattr(agent, "reset_episode"):
            agent.reset_episode()

        done = False

        ep_reward = 0.0
        succ_bouts = 0
        n_bouts = max(1, env.n_bouts)
        obs_steps = 0
        obs_seen = 0
        eat_in_win = 0

        tr = defaultdict(list)

        while not done:
            a, extra = agent.act(s)
            s2, r, done, info = env.step(a)
            ep_reward += r

            if info["eat_valid"] == 1:
                succ_bouts += 1
            if info["observe"] == 1:
                obs_steps += 1
                if info["seen_lick"] == 1:
                    obs_seen += 1
            if info["eat"] == 1 and info["window_open"] == 1:
                eat_in_win += 1

            # trace for latent sims
            tr["state"].append(s.copy())
            tr["next_state"].append(s2.copy())
            tr["reward"].append(float(r))
            tr["done"].append(float(done))

            tr["teacher_lick"].append(int(info["teacher_lick"]))
            tr["window_open"].append(int(info["window_open"]))
            tr["observe"].append(int(info["observe"]))
            tr["eat"].append(int(info["eat"]))
            tr["eat_valid"].append(int(info["eat_valid"]))
            tr["seen_lick"].append(int(info["seen_lick"]))
            tr["win_rem_est"].append(float(info["win_rem_est"]))
            tr["phase"].append(float(s[4]))

            # learner updates
            agent.record_transition(s, a, r, s2, done, extra)

            # DQN / RNN-DQN step-updates (PPO ignores this)
            if isinstance(agent, (DQNAgent, RNNDQNAgent)):
                agent.step_update(updates_per_step=updates_per_step)

            s = s2

        if hasattr(agent, "episode_update"):
            agent.episode_update()

        for k in list(tr.keys()):
            tr[k] = np.array(tr[k])
        traces.append(tr)

        logs.append({
            "ep": ep,
            "bout_success_rate": float(succ_bouts / n_bouts),
            "mean_reward": float(ep_reward),
            "obs_rate": float(obs_steps / max(1, env.max_steps)),
            "obs_seen_lick_rate": float(obs_seen / max(1, env.max_steps)),
            "eat_in_window_rate": float(eat_in_win / max(1, env.max_steps)),
            "eps": float(getattr(agent, "eps", np.nan)),
        })

        success_history.append(logs[-1]["bout_success_rate"])

        if ep % log_every == 0:
            recent = logs[-log_every:]
            print(
                f"[ep {ep:4d}] "
                f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f} "
                f"obs={np.mean([x['obs_rate'] for x in recent]):.3f} "
                f"obsSeen={np.mean([x['obs_seen_lick_rate'] for x in recent]):.3f} "
                f"eatWin={np.mean([x['eat_in_window_rate'] for x in recent]):.3f} "
                f"R={np.mean([x['mean_reward'] for x in recent]):.3f} "
                f"eps={logs[-1]['eps']:.3f}"
            )

        if len(success_history) == early_stop_window and np.mean(success_history) > early_stop_threshold:
            print(f"Early stopping at episode {ep}: Avg success rate {np.mean(success_history):.3f} > {early_stop_threshold}")
            break

    return logs, traces


# ============================================================
# Run experiment
# ============================================================
def run_experiment(args):
    set_seed(args.seed)

    env = SocialLickEnv1D(
        grid_size=args.grid_size,
        dt_s=args.dt_s,
        max_time_s=args.max_time_s,
        teacher_period_s=args.teacher_period_s,
        teacher_jitter_s=args.teacher_jitter_s,
        lick_mean_s=args.lick_mean_s,
        lick_dist=args.lick_dist,
        lick_cv=args.lick_cv,
        window_s=args.window_s,
        eat_cooldown_s=args.eat_cooldown_s,
        observe_cost=args.observe_cost,
        eat_cost=args.eat_cost,
        attend_bonus=args.attend_bonus,
        p_detect_per_obs=args.p_detect_per_obs,
        win_rem_noise_steps=args.win_rem_noise_steps,
    )

    state_dim = 6
    action_dim = 5

    if args.algo == "dqn":
        agent = DQNAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            lr=args.lr,
            gamma=args.gamma,
            batch_size=args.batch_size,
            buffer_size=args.buffer_size,
            eps_start=args.eps_start,
            eps_end=args.eps_end,
            eps_decay=args.eps_decay,
            tau=args.tau,
            grad_clip=args.grad_clip,
            use_compile=args.use_compile,
        )
    elif args.algo == "rnn":
        agent = RNNDQNAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            lr=args.lr,
            gamma=args.gamma,
            eps_start=args.eps_start,
            eps_end=args.eps_end,
            eps_decay=args.eps_decay,
            tau=args.tau,
            grad_clip=args.grad_clip,
            rnn_hidden=args.rnn_hidden,
            seq_len=args.rnn_seq_len,
            rnn_batch_size=args.rnn_batch_size,
            buffer_episodes=args.rnn_buffer_episodes,
            use_compile=args.use_compile,
        )
    elif args.algo == "ppo":
        agent = PPOAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            lr=args.lr,
            gamma=args.gamma,
            gae_lambda=args.gae_lambda,
            clip_ratio=args.clip_ratio,
            vf_coef=args.vf_coef,
            ent_coef=args.ent_coef,
            train_epochs=args.ppo_epochs,
            minibatch_size=args.minibatch_size,
            max_grad_norm=args.max_grad_norm,
        )
    elif args.algo == "tabular":
        agent = TabularQAgent(
            action_dim=action_dim,
            alpha=args.tab_alpha,
            gamma=args.gamma,
            eps_start=args.eps_start,
            eps_end=args.eps_end,
            eps_decay=args.eps_decay,
        )
    else:
        raise ValueError(f"Unknown algo: {args.algo}")

    save_dir = args.save_dir
    os.makedirs(save_dir, exist_ok=True)

    pdf_pages = None
    if args.plot_ext.lower() == "pdf" and args.multipage_pdf:
        pdf_pages = PdfPages(os.path.join(save_dir, f"all_figures_{args.algo}.pdf"))

    logs, traces = train_social_with_latents(
        env, agent,
        episodes=args.episodes,
        log_every=max(50, args.episodes // 12),
        updates_per_step=args.updates_per_step,
        early_stop_threshold=args.early_stop_threshold,
        early_stop_window=args.early_stop_window,
    )

    if not args.no_plots:
        plot_learning_curves(logs, save_dir=save_dir, ext=args.plot_ext, pdf_pages=pdf_pages)
        simulate_and_plot_latents(traces, agent, dt_s=env.dt_s, gamma=args.gamma,
                                  save_dir=save_dir, ext=args.plot_ext, pdf_pages=pdf_pages)

    if pdf_pages is not None:
        pdf_pages.close()

    return logs, traces


def build_argparser():
    p = argparse.ArgumentParser(description="Active observational learning (multi-algo) + latent signal sims")

    # algo
    p.add_argument("--algo", type=str, default="rnn", choices=["dqn", "rnn", "ppo", "tabular"])

    # run
    p.add_argument("--episodes", type=int, default=1200)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--save_dir", type=str, default="plots")
    p.add_argument("--no_plots", action="store_true")
    p.add_argument("--plot_ext", type=str, default="pdf", choices=["pdf", "png"])
    p.add_argument("--multipage_pdf", action="store_true", help="If set + plot_ext=pdf, also writes one multi-page PDF.")
    p.add_argument("--updates_per_step", type=int, default=4)

    # environment
    p.add_argument("--grid_size", type=int, default=15)
    p.add_argument("--dt_s", type=float, default=0.5)
    p.add_argument("--max_time_s", type=float, default=120.0)
    p.add_argument("--teacher_period_s", type=float, default=30.0)
    p.add_argument("--teacher_jitter_s", type=float, default=6.0)
    p.add_argument("--lick_mean_s", type=float, default=1.0)
    p.add_argument("--lick_dist", type=str, default="gamma", choices=["gamma", "lognormal"])
    p.add_argument("--lick_cv", type=float, default=0.8)
    p.add_argument("--window_s", type=float, default=3.0)
    p.add_argument("--eat_cooldown_s", type=float, default=2.0)
    p.add_argument("--observe_cost", type=float, default=0.002)
    p.add_argument("--eat_cost", type=float, default=0.005)
    p.add_argument("--attend_bonus", type=float, default=0.004)
    p.add_argument("--p_detect_per_obs", type=float, default=0.70)
    p.add_argument("--win_rem_noise_steps", type=int, default=1)

    # common learning
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--eps_start", type=float, default=1.0)
    p.add_argument("--eps_end", type=float, default=0.05)
    p.add_argument("--eps_decay", type=float, default=0.9995)

    # DQN
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--buffer_size", type=int, default=50000)
    p.add_argument("--tau", type=float, default=0.02)
    p.add_argument("--grad_clip", type=float, default=10.0)
    p.add_argument("--use_compile", action="store_true")

    # RNN-DQN
    p.add_argument("--rnn_hidden", type=int, default=128)
    p.add_argument("--rnn_seq_len", type=int, default=16)
    p.add_argument("--rnn_batch_size", type=int, default=64)
    p.add_argument("--rnn_buffer_episodes", type=int, default=2000)

    # PPO
    p.add_argument("--gae_lambda", type=float, default=0.95)
    p.add_argument("--clip_ratio", type=float, default=0.2)
    p.add_argument("--vf_coef", type=float, default=0.5)
    p.add_argument("--ent_coef", type=float, default=0.01)
    p.add_argument("--ppo_epochs", type=int, default=4)
    p.add_argument("--minibatch_size", type=int, default=256)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    # Tabular Q
    p.add_argument("--tab_alpha", type=float, default=0.25)

    # early stopping
    p.add_argument("--early_stop_threshold", type=float, default=0.8)
    p.add_argument("--early_stop_window", type=int, default=100)

    return p


if __name__ == "__main__":
    parser = build_argparser()
    args, _ = parser.parse_known_args()
    run_experiment(args)


# New 12/30/2025

## Module

In [ ]:
from typing import Optional, Dict, Any, List, Tuple
import numpy as np
import random
from collections import deque, defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import argparse
import os
import math

# -------------------------
# Seed / utils
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def clamp_int(x: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, x))

def clamp_float(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, float(x)))

def ensure_dir(path: str):
    if path is None or path == "":
        return
    os.makedirs(path, exist_ok=True)

# -------------------------
# Actions
# -------------------------
A_LEFT = 0
A_STAY = 1
A_RIGHT = 2
A_OBS = 3  # consumes time; reveals social timing info stochastically
A_EAT = 4

# ============================================================
# Environment
# ============================================================
class SocialLickEnv1D:
    """
    Fast but detailed (paper-aligned):
      dt_s = 0.5s/step
      teacher cue bursts ("lick bouts") with stochastic duration (mean ~ 1s)
      reward window: 3s after last cue step

    Observation gating:
      - social signal (cue + noisy window remaining) only appears when action==OBS
      - detection during cue burst is probabilistic per OBS step

    Condition knobs:
      - social_visibility: 0.0 => "social blind" (OBS yields no social info)
      - detect_memory_steps: if set, detected_bout_id expires after N steps
      - familiarity_enabled: if True, a latent familiarity variable accumulates across episodes and
        increases effective detect prob and reduces window-remaining noise

    NEW (requested) familiarity effects (ONLY active when familiarity_enabled=True):
      - reduces observe_cost as familiarity increases (quicker/less costly observing)
      - boosts attend_bonus as familiarity increases (more quick observing)
      - adds movement shaping during open window after detection (go to eat after observing)
      - discourages OBS during open window after detection (stop observing; move/eat)
      - adds small "earlier eat" bonus on valid reward (shorter response latency)

    Empathy knob (separate from familiarity):
      - empathy_enabled: gives intrinsic reward when OBS successfully sees the lick cue
        (lets you compare empathy on/off without touching palatability).
    """
    def __init__(
        self,
        grid_size=15,
        dt_s=0.5,
        max_time_s=120.0,
        teacher_period_s=30.0,
        teacher_jitter_s=6.0,
        lick_mean_s=1.0,
        lick_dist="gamma",  # "gamma" or "lognormal"
        lick_cv=0.8,        # for lognormal
        window_s=3.0,
        eat_cooldown_s=2.0,
        observe_cost=0.002,
        eat_cost=0.005,
        attend_bonus=0.004,
        p_detect_per_obs=0.70,
        win_rem_noise_steps=1,

        # perception / cognition constraints
        social_visibility: float = 1.0,             # 1.0 normal, 0.0 blind
        detect_memory_steps: Optional[int] = None,  # None => perfect retention

        # familiarity dynamics
        familiarity_enabled: bool = False,
        familiarity_init: float = 0.0,              # persistent across episodes
        fam_decay_ep: float = 0.00,                 # applied at reset (episode boundary)
        fam_gain_obs: float = 0.00,                 # gain per OBS during teacher lick
        fam_gain_detect: float = 0.00,              # extra gain when OBS successfully detects lick
        p_detect_fam_boost: float = 0.00,           # added detect prob at fam=1
        noise_fam_reduction: float = 0.00,          # fraction reduction of noise_steps at fam=1

        # familiarity -> behavior shaping (ONLY when familiarity_enabled=True)
        fam_obs_cost_reduction: float = 0.50,       # at fam=1, OBS cost *= (1-0.50)
        fam_attend_bonus_boost: float = 0.50,       # at fam=1, attend_bonus *= (1+0.50)
        fam_move_shaping: float = 0.010,            # per-step reward * fam for moving closer to food (during detected window)
        fam_obs_during_window_penalty: float = 0.002,  # penalty * fam for OBS during detected open window
        fam_latency_bonus: float = 0.20,            # extra reward on valid eat: + fam * bonus * (remaining/window)

        # empathy (separate condition)
        empathy_enabled: bool = False,
        empathy_seen_reward: float = 0.05,          # intrinsic reward when OBS sees lick (seen_lick==1)
    ):
        self.grid_size = int(grid_size)
        self.dt_s = float(dt_s)
        self.max_steps = int(round(max_time_s / self.dt_s))
        self.teacher_period_steps = int(round(teacher_period_s / self.dt_s))
        self.teacher_jitter_steps = int(round(teacher_jitter_s / self.dt_s))
        self.lick_mean_s = float(lick_mean_s)
        self.lick_dist = str(lick_dist)
        self.lick_cv = float(lick_cv)
        self.window_steps = int(round(window_s / self.dt_s))
        self.eat_cooldown_steps = int(round(eat_cooldown_s / self.dt_s))
        self.observe_cost = float(observe_cost)
        self.eat_cost = float(eat_cost)
        self.attend_bonus = float(attend_bonus)
        self.p_detect_per_obs = float(p_detect_per_obs)
        self.win_rem_noise_steps = int(win_rem_noise_steps)

        self.social_visibility = clamp_float(social_visibility, 0.0, 1.0)
        self.detect_memory_steps = None if detect_memory_steps is None else int(detect_memory_steps)

        self.familiarity_enabled = bool(familiarity_enabled)
        self.familiarity = clamp_float(familiarity_init, 0.0, 1.0)
        self.fam_decay_ep = clamp_float(fam_decay_ep, 0.0, 1.0)
        self.fam_gain_obs = float(fam_gain_obs)
        self.fam_gain_detect = float(fam_gain_detect)
        self.p_detect_fam_boost = float(p_detect_fam_boost)
        self.noise_fam_reduction = clamp_float(noise_fam_reduction, 0.0, 1.0)

        # shaping params (only matter if familiarity_enabled)
        self.fam_obs_cost_reduction = clamp_float(fam_obs_cost_reduction, 0.0, 1.0)
        self.fam_attend_bonus_boost = max(0.0, float(fam_attend_bonus_boost))
        self.fam_move_shaping = float(fam_move_shaping)
        self.fam_obs_during_window_penalty = float(fam_obs_during_window_penalty)
        self.fam_latency_bonus = float(fam_latency_bonus)

        # empathy params
        self.empathy_enabled = bool(empathy_enabled)
        self.empathy_seen_reward = float(empathy_seen_reward)

        assert self.teacher_period_steps > self.window_steps + 1
        assert self.max_steps > self.teacher_period_steps + 5
        self.reset()

    def _sample_lick_duration_steps(self):
        if self.lick_dist == "gamma":
            shape = 2.0
            scale = self.lick_mean_s / shape
            dur_s = float(np.random.gamma(shape, scale))
        elif self.lick_dist == "lognormal":
            cv = max(1e-6, self.lick_cv)
            sigma2 = np.log(1.0 + cv**2)
            sigma = np.sqrt(sigma2)
            mu = np.log(self.lick_mean_s) - 0.5 * sigma2
            dur_s = float(np.random.lognormal(mean=mu, sigma=sigma))
        else:
            raise ValueError(f"Unknown lick_dist={self.lick_dist}")
        steps = int(np.round(dur_s / self.dt_s))
        return max(1, steps), dur_s

    def _effective_detect_prob(self) -> float:
        p = self.p_detect_per_obs
        if self.familiarity_enabled:
            p += self.p_detect_fam_boost * self.familiarity
        return clamp_float(p, 0.0, 1.0)

    def _effective_noise_steps(self) -> int:
        n = int(self.win_rem_noise_steps)
        if self.familiarity_enabled and n > 0:
            n = int(round(n * (1.0 - self.noise_fam_reduction * self.familiarity)))
        return max(0, n)

    def _effective_observe_cost(self) -> float:
        if not self.familiarity_enabled:
            return self.observe_cost
        # cost reduces as familiarity increases
        return float(self.observe_cost * (1.0 - self.fam_obs_cost_reduction * self.familiarity))

    def _effective_attend_bonus(self) -> float:
        if not self.familiarity_enabled:
            return self.attend_bonus
        return float(self.attend_bonus * (1.0 + self.fam_attend_bonus_boost * self.familiarity))

    def reset(self):
        # Episode boundary familiarity decay (persistent across episodes)
        if self.familiarity_enabled and self.fam_decay_ep > 0.0:
            self.familiarity = clamp_float(self.familiarity * (1.0 - self.fam_decay_ep), 0.0, 1.0)

        self.t = 0
        self.teacher_pos = int(np.random.randint(0, self.grid_size))
        candidates = [i for i in range(self.grid_size) if i != self.teacher_pos]
        self.learner_food_pos = int(np.random.choice(candidates))
        self.learner_pos = int(np.random.randint(0, self.grid_size))

        self.eat_cd = 0
        self.last_lick_step = None
        self.detected_bout_id = None
        self._detect_ttl = None
        self.rewarded_bout_ids = set()

        # build cue-burst timeline
        self.bout_id_at_step = -np.ones(self.max_steps + 1, dtype=np.int32)
        self.bout_start_steps = []
        self.bout_end_steps = []
        self.bout_dur_s_list = []
        bout_id = 0
        k = 1
        while True:
            nominal = k * self.teacher_period_steps
            if nominal >= self.max_steps:
                break
            jitter = int(np.random.randint(-self.teacher_jitter_steps, self.teacher_jitter_steps + 1))
            start = clamp_int(nominal + jitter, 1, self.max_steps - 2)
            dur_steps, dur_s = self._sample_lick_duration_steps()
            end = clamp_int(start + dur_steps - 1, start, self.max_steps - 1)
            self.bout_id_at_step[start:end + 1] = bout_id
            self.bout_start_steps.append(start)
            self.bout_end_steps.append(end)
            self.bout_dur_s_list.append(dur_s)
            bout_id += 1
            k += 1
        self.n_bouts = int(bout_id)
        return self._get_obs(lick_sig=0.0, win_rem=0.0)

    def _teacher_lick_now(self, t: int):
        bid = int(self.bout_id_at_step[t]) if (0 <= t <= self.max_steps) else -1
        return (1, bid) if bid >= 0 else (0, -1)

    def _window_open(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        return 1 if (0 <= dt < self.window_steps) else 0

    def _window_remaining(self, t: int) -> int:
        if self.last_lick_step is None:
            return 0
        dt = t - self.last_lick_step
        rem = self.window_steps - dt
        return int(max(0, rem))

    def _phase_to_next_nominal(self, t: int) -> int:
        mod = t % self.teacher_period_steps
        return self.teacher_period_steps - mod

    def _get_obs(self, lick_sig: float, win_rem: float):
        lp = float(self.learner_pos) / float(self.grid_size - 1)
        lf = float(self.learner_food_pos) / float(self.grid_size - 1)
        phase = float(self._phase_to_next_nominal(self.t)) / float(self.teacher_period_steps)
        cdn = float(self.eat_cd) / float(max(1, self.eat_cooldown_steps))
        # [pos, food_pos, gated cue, gated window remaining, phase clock, cooldown]
        return np.array([lp, lf, float(lick_sig), float(win_rem), phase, cdn], dtype=np.float32)

    def step(self, action: int):
        self.t += 1
        teacher_lick, bout_id = self._teacher_lick_now(self.t)
        if teacher_lick == 1:
            self.last_lick_step = self.t

        window_open = self._window_open(self.t)
        win_rem_true = self._window_remaining(self.t)

        if self.eat_cd > 0:
            self.eat_cd -= 1

        # optional memory decay for detected bout id
        if self.detect_memory_steps is not None and self._detect_ttl is not None:
            self._detect_ttl -= 1
            if self._detect_ttl <= 0:
                self._detect_ttl = None
                self.detected_bout_id = None

        lick_sig = 0.0
        win_rem = 0.0
        did_observe = 0
        did_eat = 0
        seen_lick = 0

        # distance for shaping (computed before move)
        dist_before = abs(int(self.learner_pos) - int(self.learner_food_pos))

        # movement / observe / eat
        if action == A_LEFT:
            self.learner_pos -= 1
        elif action == A_RIGHT:
            self.learner_pos += 1
        elif action == A_OBS:
            did_observe = 1

            # social blind => OBS reveals nothing
            if self.social_visibility > 0.0:
                p_det = self._effective_detect_prob()
                if teacher_lick == 1 and bout_id >= 0 and (np.random.rand() < p_det):
                    seen_lick = 1
                    self.detected_bout_id = int(bout_id)
                    if self.detect_memory_steps is not None:
                        self._detect_ttl = int(self.detect_memory_steps)

                lick_sig = float(seen_lick)

                if win_rem_true > 0:
                    noise_steps = self._effective_noise_steps()
                    noise = int(np.random.randint(-noise_steps, noise_steps + 1)) if noise_steps > 0 else 0
                    est = clamp_int(win_rem_true + noise, 0, self.window_steps)
                    win_rem = float(est) / float(max(1, self.window_steps))

                # familiarity update
                if self.familiarity_enabled and teacher_lick == 1 and bout_id >= 0:
                    self.familiarity = clamp_float(self.familiarity + self.fam_gain_obs, 0.0, 1.0)
                    if seen_lick == 1:
                        self.familiarity = clamp_float(self.familiarity + self.fam_gain_detect, 0.0, 1.0)

        elif action == A_EAT:
            did_eat = 1
        # A_STAY does nothing

        self.learner_pos = clamp_int(self.learner_pos, 0, self.grid_size - 1)
        dist_after = abs(int(self.learner_pos) - int(self.learner_food_pos))

        # base costs (familiarity can reduce OBS cost)
        reward = 0.0
        reward -= self._effective_observe_cost() * float(did_observe)
        reward -= self.eat_cost * float(did_eat)

        # shaping: reward OBS during cue only if bout not yet detected
        # (kept exactly as your template, except attend_bonus can be boosted by familiarity)
        if did_observe == 1 and teacher_lick == 1 and bout_id >= 0:
            if self.detected_bout_id != int(bout_id):
                reward += self._effective_attend_bonus()

        # empathy: intrinsic reward when OBS successfully sees lick
        if self.empathy_enabled and did_observe == 1 and seen_lick == 1:
            reward += self.empathy_seen_reward

        # eat allowed?
        eat_allowed = (did_eat == 1 and self.eat_cd == 0)
        if eat_allowed:
            self.eat_cd = self.eat_cooldown_steps

        # success logic (1 reward per bout)
        eat_valid = 0
        latency_steps = None
        last_bid = -1
        if self.last_lick_step is not None and self.last_lick_step <= self.max_steps:
            last_bid = int(self.bout_id_at_step[self.last_lick_step])

        if eat_allowed and (self.learner_pos == self.learner_food_pos) and (window_open == 1) and (self.last_lick_step is not None):
            if (last_bid >= 0) and (self.detected_bout_id == last_bid) and (last_bid not in self.rewarded_bout_ids):
                eat_valid = 1
                self.rewarded_bout_ids.add(last_bid)
                reward += 1.0
                latency_steps = int(self.t - int(self.last_lick_step))

                # familiarity: reward earlier eating (shorter latency)
                if self.familiarity_enabled and self.fam_latency_bonus > 0.0:
                    # earlier in window => larger win_rem_true
                    reward += float(self.familiarity * self.fam_latency_bonus * (float(win_rem_true) / float(max(1, self.window_steps))))

        # familiarity: push "go to eat after observing" ONLY when a detected bout window is open
        if self.familiarity_enabled and (last_bid >= 0) and (window_open == 1) and (self.detected_bout_id == last_bid) and (last_bid not in self.rewarded_bout_ids):
            fam = float(self.familiarity)
            if fam > 0.0:
                # reward moving closer to food (shorter response latency)
                if action in (A_LEFT, A_RIGHT) and self.fam_move_shaping != 0.0:
                    # dist_before - dist_after: +1 if moved closer, -1 if moved away
                    reward += float(self.fam_move_shaping * fam * float(dist_before - dist_after))
                # penalize continued observing during the open window (stop observing, act!)
                if did_observe == 1 and self.fam_obs_during_window_penalty > 0.0:
                    reward -= float(self.fam_obs_during_window_penalty * fam)

        done = bool(self.t >= self.max_steps)
        info = {
            "t": int(self.t),
            "teacher_lick": int(teacher_lick),
            "bout_id": int(bout_id),
            "window_open": int(window_open),
            "win_rem_true": int(win_rem_true),
            "action": int(action),
            "observe": int(did_observe),
            "eat": int(did_eat),
            "eat_allowed": int(eat_allowed),
            "eat_valid": int(eat_valid),
            "latency_steps": -1 if latency_steps is None else int(latency_steps),
            "last_lick_step": -1 if self.last_lick_step is None else int(self.last_lick_step),
            "learner_pos": int(self.learner_pos),
            "food_pos": int(self.learner_food_pos),
            "seen_lick": int(seen_lick),
            "win_rem_est": float(win_rem),
            "detected_bout_id": -1 if self.detected_bout_id is None else int(self.detected_bout_id),
            "rewarded_bouts": len(self.rewarded_bout_ids),
            "n_bouts": int(self.n_bouts),
            "dt_s": float(self.dt_s),
            "window_steps": int(self.window_steps),
            "social_visibility": float(self.social_visibility),
            "familiarity": float(self.familiarity),
            "p_detect_eff": float(self._effective_detect_prob()) if (self.social_visibility > 0.0) else 0.0,
            "obs_cost_eff": float(self._effective_observe_cost()),
            "attend_bonus_eff": float(self._effective_attend_bonus()),
            "empathy_enabled": int(self.empathy_enabled),
        }
        obs = self._get_obs(lick_sig=lick_sig, win_rem=win_rem)
        return obs, reward, done, info


# ============================================================
# Plot saving
# ============================================================
def _save_fig(fig, out_path_no_ext: str, ext: str, pdf_pages: Optional[PdfPages] = None):
    ensure_dir(os.path.dirname(out_path_no_ext))
    if ext.lower() == "pdf" and pdf_pages is not None:
        pdf_pages.savefig(fig, bbox_inches="tight")
    else:
        fig.savefig(f"{out_path_no_ext}.{ext}", bbox_inches="tight")

def ema_1d(y: np.ndarray, alpha: float = 0.02) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    out = np.zeros_like(y)
    m = 0.0
    for i in range(len(y)):
        if np.isnan(y[i]):
            out[i] = m
            continue
        m = (1 - alpha) * m + alpha * y[i]
        out[i] = m
    return out

def logs_to_array_with_ffill(logs: List[Dict[str, float]], metric: str, target_len: int) -> np.ndarray:
    y = np.full((target_len,), np.nan, dtype=np.float32)
    if len(logs) == 0:
        return y
    vals = [float(d.get(metric, np.nan)) for d in logs]
    L = min(len(vals), target_len)
    y[:L] = np.array(vals[:L], dtype=np.float32)
    if L < target_len and L > 0 and not np.isnan(y[L-1]):
        y[L:] = y[L-1]
    return y

def mean_sem(arr: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mean = np.nanmean(arr, axis=0)
    sem = np.zeros_like(mean, dtype=np.float32)
    for t in range(arr.shape[1]):
        col = arr[:, t]
        col = col[~np.isnan(col)]
        if len(col) <= 1:
            sem[t] = 0.0
        else:
            sem[t] = float(np.std(col, ddof=1) / math.sqrt(len(col)))
    return mean.astype(np.float32), sem.astype(np.float32)

def plot_metric_mean_sem(ep: np.ndarray, mean: np.ndarray, sem: np.ndarray, label: str, alpha_fill: float = 0.20):
    plt.plot(ep, mean, label=label)
    lo = mean - sem
    hi = mean + sem
    plt.fill_between(ep, lo, hi, alpha=alpha_fill)

def plot_comparison_curves_mean_sem(
    results_logs_by_seed: Dict[str, List[List[Dict[str, float]]]],
    metric: str,
    episodes: int,
    save_dir: str,
    ext: str = "pdf",
    pdf_pages: Optional[PdfPages] = None,
    alpha_ema: float = 0.02,
    title: Optional[str] = None,
):
    fig = plt.figure(figsize=(8, 4.8))
    ep = np.arange(1, episodes + 1, dtype=np.int32)

    for label, runs in results_logs_by_seed.items():
        arr = np.stack([logs_to_array_with_ffill(run, metric, episodes) for run in runs], axis=0)
        arr_s = np.stack([ema_1d(arr[i], alpha=alpha_ema) for i in range(arr.shape[0])], axis=0)
        m, s = mean_sem(arr_s)
        plot_metric_mean_sem(ep, m, s, label=label)

    plt.xlabel("episode")
    plt.ylabel(metric)
    plt.title(title or f"{metric}: mean ± SEM (across seeds)")
    plt.legend(fontsize=8)
    plt.tight_layout()
    _save_fig(fig, os.path.join(save_dir, f"compare_{metric}_mean_sem"), ext=ext, pdf_pages=pdf_pages)

def plot_single_learning_panel_mean_sem(
    runs: List[List[Dict[str, float]]],
    episodes: int,
    save_dir: str,
    ext: str,
    pdf_pages: Optional[PdfPages],
    title_suffix: str = "",
    alpha_ema: float = 0.02,
):
    ep = np.arange(1, episodes + 1, dtype=np.int32)

    def get_ms(metric):
        arr = np.stack([logs_to_array_with_ffill(run, metric, episodes) for run in runs], axis=0)
        arr_s = np.stack([ema_1d(arr[i], alpha=alpha_ema) for i in range(arr.shape[0])], axis=0)
        return mean_sem(arr_s)

    m_succ, s_succ = get_ms("bout_success_rate")
    m_obs, s_obs = get_ms("obs_rate")
    m_seen, s_seen = get_ms("obs_seen_lick_rate")
    m_eatw, s_eatw = get_ms("eat_in_window_rate")
    m_lat, s_lat = get_ms("mean_latency_s")
    m_fam, s_fam = get_ms("familiarity")
    m_lr, s_lr = get_ms("lr")

    fig = plt.figure(figsize=(14, 4))

    plt.subplot(1, 3, 1)
    plot_metric_mean_sem(ep, m_succ, s_succ, label="success")
    plt.title(f"EMA bout success{title_suffix}")
    plt.xlabel("episode"); plt.ylabel("rate")
    plt.legend(fontsize=8)

    plt.subplot(1, 3, 2)
    plot_metric_mean_sem(ep, m_obs, s_obs, label="observe")
    plot_metric_mean_sem(ep, m_seen, s_seen, label="obs sees cue")
    plt.title("EMA observation metrics")
    plt.xlabel("episode"); plt.legend(fontsize=8)

    plt.subplot(1, 3, 3)
    plot_metric_mean_sem(ep, m_eatw, s_eatw, label="eat in window")
    plot_metric_mean_sem(ep, m_lat, s_lat, label="latency (s)")
    if np.nanmax(m_fam) > 1e-6:
        plot_metric_mean_sem(ep, m_fam, s_fam, label="familiarity")
    if np.nanmax(m_lr) > 0:
        plot_metric_mean_sem(ep, m_lr, s_lr, label="lr")
    plt.title("EMA eating + latency + familiarity + lr")
    plt.xlabel("episode"); plt.legend(fontsize=8)

    plt.tight_layout()
    _save_fig(fig, os.path.join(save_dir, "learning_curves_mean_sem"), ext=ext, pdf_pages=pdf_pages)

# ============================================================
# Significance tests (no SciPy)
# ============================================================
def cohen_d(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    a = a[~np.isnan(a)]
    b = b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2:
        return float("nan")
    va = np.var(a, ddof=1)
    vb = np.var(b, ddof=1)
    sp = math.sqrt(((len(a)-1)*va + (len(b)-1)*vb) / max(1, (len(a)+len(b)-2)))
    if sp <= 1e-12:
        return 0.0
    return float((np.mean(a) - np.mean(b)) / sp)

def permutation_test_mean_diff(a: np.ndarray, b: np.ndarray, n_perm: int = 5000, seed: int = 0) -> float:
    rng = np.random.RandomState(seed)
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    if len(a) == 0 or len(b) == 0:
        return float("nan")
    obs = abs(np.mean(a) - np.mean(b))
    x = np.concatenate([a, b], axis=0)
    na = len(a)
    count = 0
    for _ in range(int(n_perm)):
        rng.shuffle(x)
        d = abs(np.mean(x[:na]) - np.mean(x[na:]))
        if d >= obs - 1e-12:
            count += 1
    return float((count + 1) / (n_perm + 1))

def last_k_window_scores(runs: List[List[Dict[str, float]]], metric: str, episodes: int, last_k: int) -> np.ndarray:
    last_k = int(min(last_k, episodes))
    scores = []
    for logs in runs:
        y = logs_to_array_with_ffill(logs, metric, episodes)
        tail = y[-last_k:]
        scores.append(float(np.nanmean(tail)))
    return np.array(scores, dtype=np.float32)

def write_stats_report(
    out_path: str,
    group_key: str,
    metric: str,
    a_label: str,
    b_label: str,
    a_scores: np.ndarray,
    b_scores: np.ndarray,
    pval: float,
    d: float,
):
    with open(out_path, "a", encoding="utf-8") as f:
        f.write(f"\n[{group_key}] metric={metric}\n")
        f.write(f"  {a_label}: n={len(a_scores)} mean={float(np.mean(a_scores)):.4f} std={float(np.std(a_scores, ddof=1) if len(a_scores)>1 else 0.0):.4f}\n")
        f.write(f"  {b_label}: n={len(b_scores)} mean={float(np.mean(b_scores)):.4f} std={float(np.std(b_scores, ddof=1) if len(b_scores)>1 else 0.0):.4f}\n")
        f.write(f"  perm-test p={pval:.6f}  Cohen_d={d:.4f}\n")

# ============================================================
# Latent / “DA-like” simulation (kept; optional)
# ============================================================
def calcium_filter(impulses, dt_s, tau_rise_s=0.2, tau_decay_s=1.2):
    impulses = np.asarray(impulses, dtype=np.float32)
    n = len(impulses)
    L = int(np.ceil(8 * tau_decay_s / dt_s))
    t = np.arange(L, dtype=np.float32) * dt_s
    k = (1 - np.exp(-t / max(1e-6, tau_rise_s))) * np.exp(-t / max(1e-6, tau_decay_s))
    k = k / (k.sum() + 1e-9)
    y = np.convolve(impulses, k, mode="full")[:n]
    return y.astype(np.float32)

def gae_advantages(rewards, values, dones, gamma=0.99, lam=0.95):
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        nonterminal = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * values[t+1] * nonterminal - values[t]
        gae = delta + gamma * lam * nonterminal * gae
        adv[t] = gae
    return adv

def simulate_latent_models_for_episode(tr, agent, gamma=0.99, ep_frac=0.0, gae_lam=0.95):
    r = np.array(tr["reward"], dtype=np.float32)
    obs = np.array(tr["observe"], dtype=int)
    eat = np.array(tr["eat"], dtype=int)
    seen_lick = np.array(tr["seen_lick"], dtype=int)
    phase = np.array(tr["phase"], dtype=np.float32)
    S = np.array(tr["state"], dtype=np.float32)
    S2 = np.array(tr["next_state"], dtype=np.float32)
    done = np.array(tr["done"], dtype=np.float32)

    V_s = agent.value_batch(S)
    V_s2 = agent.value_batch(S2)
    rpe = r + gamma * V_s2 * (1.0 - done) - V_s

    V_boot = np.concatenate([V_s, V_s2[-1:]], axis=0)
    adv = gae_advantages(r, V_boot[:len(r)+1], done, gamma=gamma, lam=gae_lam)

    S2_prior = S2.copy()
    S2_prior[:, 2] = 0.0
    S2_prior[:, 3] = 0.0
    V_post = V_s2
    V_prior = agent.value_batch(S2_prior)
    deltaV = (V_post - V_prior) * obs.astype(np.float32)

    sal = np.zeros(len(r), dtype=np.float32)
    sal += 1.0 * (obs == 1).astype(np.float32)
    sal += 0.6 * (eat == 1).astype(np.float32)

    sigma = 0.25 * (1.0 - ep_frac) + 0.10 * ep_frac
    p = np.exp(-(phase / max(1e-6, sigma)) ** 2)
    surprise = -np.log(p + 1e-6)
    info_gain = surprise * ((obs == 1) & (seen_lick == 1)).astype(np.float32)

    impulses = {
        "reward_RPE": 0.8 * rpe,
        "gae_adv": 0.7 * adv,
        "social_cue_RPE": 1.2 * deltaV,
        "social_value": 0.6 * deltaV,
        "action_salience": 0.3 * sal,
        "info_gain": 0.25 * info_gain,
    }

    if hasattr(agent, "policy_entropy_batch"):
        ent = agent.policy_entropy_batch(S)
        if ent is not None:
            impulses["policy_entropy"] = 0.15 * ent.astype(np.float32)

    return impulses

def simulate_and_plot_latents(traces, agent, dt_s, gamma=0.99,
                              save_dir="plots", ext="pdf", pdf_pages=None):
    impulses_by_model = defaultdict(list)
    n = len(traces)
    for i, tr in enumerate(traces):
        ep_frac = i / max(1, n - 1)
        imp = simulate_latent_models_for_episode(tr, agent, gamma=gamma, ep_frac=ep_frac)
        for k, v in imp.items():
            impulses_by_model[k].append(v)

    phot_by_model = {}
    for k, imps in impulses_by_model.items():
        phot_by_model[k] = [calcium_filter(x, dt_s=dt_s, tau_rise_s=0.2, tau_decay_s=1.2) for x in imps]
    return phot_by_model


# ============================================================
# Agents
# ============================================================
class BaseAgent:
    def act(self, s: np.ndarray):
        raise NotImplementedError
    def record_transition(self, s, a, r, s2, done, extra):
        pass
    def step_update(self):
        return None
    def episode_update(self):
        return None
    def value_batch(self, states: np.ndarray) -> np.ndarray:
        raise NotImplementedError
    def get_lr(self) -> float:
        return 0.0

# -------------------------
# DQN (dueling + double + huber + soft target)
# -------------------------
class DuelingQNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.V = nn.Linear(hidden, 1)
        self.A = nn.Linear(hidden, action_dim)

    def forward(self, x):
        h = self.backbone(x)
        v = self.V(h)
        a = self.A(h)
        return v + (a - a.mean(dim=1, keepdim=True))

class TorchReplayBuffer:
    def __init__(self, capacity, state_dim, device):
        self.capacity = int(capacity)
        self.device = device
        self.state = torch.zeros((capacity, state_dim), dtype=torch.float32, device=device)
        self.action = torch.zeros(capacity, dtype=torch.int64, device=device)
        self.reward = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.next_state = torch.zeros((capacity, state_dim), dtype=torch.float32, device=device)
        self.done = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.idx = 0
        self.size = 0

    def push(self, s, a, r, s2, done):
        self.state[self.idx] = torch.tensor(s, dtype=torch.float32, device=self.device)
        self.action[self.idx] = int(a)
        self.reward[self.idx] = float(r)
        self.next_state[self.idx] = torch.tensor(s2, dtype=torch.float32, device=self.device)
        self.done[self.idx] = float(done)
        self.idx = (self.idx + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        indices = torch.randint(0, self.size, (batch_size,), device=self.device)
        return (
            self.state[indices],
            self.action[indices].unsqueeze(1),
            self.reward[indices],
            self.next_state[indices],
            self.done[indices],
        )

    def __len__(self):
        return self.size

class DQNAgent(BaseAgent):
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        batch_size=256,
        buffer_size=50000,
        eps_start=1.0,
        eps_end=0.05,
        eps_decay=0.9995,
        tau=0.02,
        grad_clip=10.0,
        device=None,
        use_compile=False,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.batch_size = int(batch_size)
        self.tau = float(tau)
        self.grad_clip = float(grad_clip)

        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.q = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt = DuelingQNet(self.state_dim, self.action_dim).to(self.device)
        self.qt.load_state_dict(self.q.state_dict())

        if use_compile and hasattr(torch, "compile"):
            try:
                self.q = torch.compile(self.q)
            except Exception:
                pass

        self.opt = optim.Adam(self.q.parameters(), lr=float(lr))
        self.loss_fn = nn.SmoothL1Loss()
        self.rb = TorchReplayBuffer(buffer_size, state_dim, self.device)
        self.scaler = GradScaler(enabled=self.device.type == "cuda")

    def get_lr(self) -> float:
        return float(self.opt.param_groups[0]["lr"])

    def act(self, s):
        if np.random.rand() < self.eps:
            return int(np.random.randint(self.action_dim)), {}
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            qv = self.q(st)
        return int(torch.argmax(qv, dim=1).item()), {}

    def record_transition(self, s, a, r, s2, done, extra):
        self.rb.push(s, a, r, s2, done)

    def step_update(self, updates_per_step=1):
        if len(self.rb) < self.batch_size:
            self.eps = max(self.eps_end, self.eps * self.eps_decay)
            return None

        last_loss = None
        for _ in range(int(updates_per_step)):
            s, a, r, s2, d = self.rb.sample(self.batch_size)
            with autocast(enabled=self.device.type == "cuda"):
                q_sa = self.q(s).gather(1, a).squeeze(1)
                with torch.no_grad():
                    a_star = torch.argmax(self.q(s2), dim=1, keepdim=True)
                    q_next = self.qt(s2).gather(1, a_star).squeeze(1)
                    target = r + self.gamma * q_next * (1.0 - d)
                loss = self.loss_fn(q_sa, target)

            self.opt.zero_grad()
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.opt)
            nn.utils.clip_grad_norm_(self.q.parameters(), self.grad_clip)
            self.scaler.step(self.opt)
            self.scaler.update()

            with torch.no_grad():
                for p, pt in zip(self.q.parameters(), self.qt.parameters()):
                    pt.data.mul_(1.0 - self.tau).add_(self.tau * p.data)

            last_loss = float(loss.item())

        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        return last_loss

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            qv = self.q(st)
            v = torch.max(qv, dim=1).values
        return v.detach().cpu().numpy().astype(np.float32)

# -------------------------
# PPO (actor-critic)
# -------------------------
class ActorCriticNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.pi = nn.Linear(hidden, action_dim)
        self.v = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.body(x)
        logits = self.pi(h)
        value = self.v(h).squeeze(-1)
        return logits, value

class PPOAgent(BaseAgent):
    def __init__(
        self,
        state_dim,
        action_dim,
        lr=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_ratio=0.2,
        vf_coef=0.5,
        ent_coef=0.01,
        train_epochs=4,
        minibatch_size=256,
        max_grad_norm=1.0,
        device=None,
    ):
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.gamma = float(gamma)
        self.gae_lambda = float(gae_lambda)
        self.clip_ratio = float(clip_ratio)
        self.vf_coef = float(vf_coef)
        self.ent_coef = float(ent_coef)
        self.train_epochs = int(train_epochs)
        self.minibatch_size = int(minibatch_size)
        self.max_grad_norm = float(max_grad_norm)

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.net = ActorCriticNet(self.state_dim, self.action_dim).to(self.device)
        self.opt = optim.Adam(self.net.parameters(), lr=float(lr))
        self.roll = []

    def get_lr(self) -> float:
        return float(self.opt.param_groups[0]["lr"])

    def act(self, s):
        st = torch.tensor(s, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            logits, v = self.net(st)
            dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample()
            logp = dist.log_prob(a)
            ent = dist.entropy()
        return int(a.item()), {"logp": float(logp.item()), "v": float(v.item()), "ent": float(ent.item())}

    def record_transition(self, s, a, r, s2, done, extra):
        self.roll.append({
            "s": np.array(s, dtype=np.float32),
            "a": int(a),
            "r": float(r),
            "done": float(done),
            "logp": float(extra.get("logp", 0.0)),
            "v": float(extra.get("v", 0.0)),
        })

    def episode_update(self):
        if len(self.roll) < 8:
            self.roll.clear()
            return None

        S = np.stack([x["s"] for x in self.roll], axis=0).astype(np.float32)
        A = np.array([x["a"] for x in self.roll], dtype=np.int64)
        R = np.array([x["r"] for x in self.roll], dtype=np.float32)
        D = np.array([x["done"] for x in self.roll], dtype=np.float32)
        LOGP_OLD = np.array([x["logp"] for x in self.roll], dtype=np.float32)
        V = np.array([x["v"] for x in self.roll], dtype=np.float32)

        with torch.no_grad():
            st_last = torch.tensor(S[-1], dtype=torch.float32, device=self.device).unsqueeze(0)
            _, v_last = self.net(st_last)
            v_last = float(v_last.item())
        V_ext = np.concatenate([V, np.array([v_last], dtype=np.float32)], axis=0)

        ADV = gae_advantages(R, V_ext, D, gamma=self.gamma, lam=self.gae_lambda)
        RET = ADV + V
        ADV = (ADV - ADV.mean()) / (ADV.std() + 1e-8)

        ts = torch.tensor(S, dtype=torch.float32, device=self.device)
        ta = torch.tensor(A, dtype=torch.int64, device=self.device)
        told = torch.tensor(LOGP_OLD, dtype=torch.float32, device=self.device)
        tadv = torch.tensor(ADV, dtype=torch.float32, device=self.device)
        tret = torch.tensor(RET, dtype=torch.float32, device=self.device)

        n = len(S)
        idx = np.arange(n)
        info = {}

        for _ in range(self.train_epochs):
            np.random.shuffle(idx)
            for start in range(0, n, self.minibatch_size):
                mb = idx[start:start+self.minibatch_size]
                logits, vpred = self.net(ts[mb])
                dist = torch.distributions.Categorical(logits=logits)

                logp = dist.log_prob(ta[mb])
                ratio = torch.exp(logp - told[mb])

                clip = torch.clamp(ratio, 1.0 - self.clip_ratio, 1.0 + self.clip_ratio)
                pg_loss = -(torch.min(ratio * tadv[mb], clip * tadv[mb])).mean()

                v_loss = ((vpred - tret[mb]) ** 2).mean()
                ent = dist.entropy().mean()
                loss = pg_loss + self.vf_coef * v_loss - self.ent_coef * ent

                self.opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), self.max_grad_norm)
                self.opt.step()

                info = {
                    "pg_loss": float(pg_loss.item()),
                    "v_loss": float(v_loss.item()),
                    "ent": float(ent.item()),
                    "loss": float(loss.item()),
                }

        self.roll.clear()
        return info

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            _, v = self.net(st)
        return v.detach().cpu().numpy().astype(np.float32)

    def policy_entropy_batch(self, states: np.ndarray):
        st = torch.tensor(states, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            logits, _ = self.net(st)
            dist = torch.distributions.Categorical(logits=logits)
            ent = dist.entropy()
        return ent.detach().cpu().numpy().astype(np.float32)

# -------------------------
# Tabular Q-learning
# -------------------------
class TabularQAgent(BaseAgent):
    def __init__(self, action_dim=5, alpha=0.25, gamma=0.99, eps_start=1.0, eps_end=0.05, eps_decay=0.9995,
                 phase_bins=10, win_bins=7, cd_bins=6):
        self.action_dim = int(action_dim)
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.eps = float(eps_start)
        self.eps_end = float(eps_end)
        self.eps_decay = float(eps_decay)

        self.phase_bins = int(phase_bins)
        self.win_bins = int(win_bins)
        self.cd_bins = int(cd_bins)
        self.Q = defaultdict(lambda: np.zeros(self.action_dim, dtype=np.float32))

    def _disc(self, s: np.ndarray):
        lp, lf, lick, win, phase, cd = s.tolist()
        p = int(np.round(lp * 14))
        f = int(np.round(lf * 14))
        lickb = int(lick > 0.5)
        winb = int(np.round(win * (self.win_bins - 1)))
        winb = clamp_int(winb, 0, self.win_bins - 1)
        phb = int(np.floor(phase * self.phase_bins))
        phb = clamp_int(phb, 0, self.phase_bins - 1)
        cdb = int(np.round(cd * (self.cd_bins - 1)))
        cdb = clamp_int(cdb, 0, self.cd_bins - 1)
        return (p, f, lickb, winb, phb, cdb)

    def act(self, s):
        key = self._disc(s)
        if np.random.rand() < self.eps:
            a = int(np.random.randint(self.action_dim))
        else:
            a = int(np.argmax(self.Q[key]))
        return a, {"key": key}

    def record_transition(self, s, a, r, s2, done, extra):
        k = extra["key"]
        k2 = self._disc(s2)
        q = self.Q[k]
        target = float(r) + self.gamma * (0.0 if done else float(np.max(self.Q[k2])))
        q[a] = (1 - self.alpha) * q[a] + self.alpha * target
        self.eps = max(self.eps_end, self.eps * self.eps_decay)

    def value_batch(self, states: np.ndarray) -> np.ndarray:
        out = np.zeros(len(states), dtype=np.float32)
        for i, s in enumerate(states):
            out[i] = float(np.max(self.Q[self._disc(s)]))
        return out

# ============================================================
# Training + logging
# ============================================================
def train_social_with_latents(
    env,
    agent: BaseAgent,
    episodes=1200,
    log_every=100,
    updates_per_step=4,
    early_stop_threshold=0.8,
    early_stop_window=100,
    compute_latents: bool = True,
):
    logs = []
    traces = []
    success_history = deque(maxlen=early_stop_window)

    for ep in range(1, episodes + 1):
        s = env.reset()
        done = False

        ep_reward = 0.0
        succ_bouts = 0
        n_bouts = max(1, env.n_bouts)
        obs_steps = 0
        obs_seen = 0
        eat_in_win = 0
        latencies_steps = []

        tr = defaultdict(list)

        while not done:
            a, extra = agent.act(s)
            s2, r, done, info = env.step(a)
            ep_reward += r

            if info["eat_valid"] == 1:
                succ_bouts += 1
                if info.get("latency_steps", -1) >= 0:
                    latencies_steps.append(int(info["latency_steps"]))
            if info["observe"] == 1:
                obs_steps += 1
                if info["seen_lick"] == 1:
                    obs_seen += 1
            if info["eat"] == 1 and info["window_open"] == 1:
                eat_in_win += 1

            if compute_latents:
                tr["state"].append(s.copy())
                tr["next_state"].append(s2.copy())
                tr["reward"].append(float(r))
                tr["done"].append(float(done))

                tr["teacher_lick"].append(int(info["teacher_lick"]))
                tr["window_open"].append(int(info["window_open"]))
                tr["observe"].append(int(info["observe"]))
                tr["eat"].append(int(info["eat"]))
                tr["eat_valid"].append(int(info["eat_valid"]))
                tr["seen_lick"].append(int(info["seen_lick"]))
                tr["win_rem_est"].append(float(info["win_rem_est"]))
                tr["phase"].append(float(s[4]))

            agent.record_transition(s, a, r, s2, done, extra)

            if isinstance(agent, DQNAgent):
                agent.step_update(updates_per_step=updates_per_step)

            s = s2

        if hasattr(agent, "episode_update"):
            agent.episode_update()

        if compute_latents:
            for k in list(tr.keys()):
                tr[k] = np.array(tr[k])
            traces.append(tr)

        mean_latency_s = float(np.mean(latencies_steps) * env.dt_s) if len(latencies_steps) > 0 else np.nan

        logs.append({
            "ep": ep,
            "bout_success_rate": float(succ_bouts / n_bouts),
            "mean_reward": float(ep_reward),
            "obs_rate": float(obs_steps / max(1, env.max_steps)),
            "obs_seen_lick_rate": float(obs_seen / max(1, env.max_steps)),
            "eat_in_window_rate": float(eat_in_win / max(1, env.max_steps)),
            "mean_latency_s": float(mean_latency_s),

            "eps": float(getattr(agent, "eps", np.nan)),
            "familiarity": float(getattr(env, "familiarity", 0.0)),
            "p_detect_eff": float(getattr(env, "_effective_detect_prob")()) if getattr(env, "social_visibility", 1.0) > 0.0 else 0.0,
            "lr": float(agent.get_lr() if hasattr(agent, "get_lr") else 0.0),
        })

        success_history.append(logs[-1]["bout_success_rate"])

        if ep % log_every == 0:
            recent = logs[-log_every:]
            print(
                f"[ep {ep:4d}] "
                f"boutSucc={np.mean([x['bout_success_rate'] for x in recent]):.3f} "
                f"lat={np.nanmean([x['mean_latency_s'] for x in recent]):.3f}s "
                f"obs={np.mean([x['obs_rate'] for x in recent]):.3f} "
                f"obsSeen={np.mean([x['obs_seen_lick_rate'] for x in recent]):.3f} "
                f"eatWin={np.mean([x['eat_in_window_rate'] for x in recent]):.3f} "
                f"R={np.mean([x['mean_reward'] for x in recent]):.3f} "
                f"eps={logs[-1]['eps']:.3f} "
                f"fam={logs[-1]['familiarity']:.2f} "
                f"lr={logs[-1]['lr']:.2e}"
            )

        if len(success_history) == early_stop_window and np.mean(success_history) > early_stop_threshold:
            print(f"Early stopping at episode {ep}: Avg success rate {np.mean(success_history):.3f} > {early_stop_threshold}")
            break

    return logs, traces


# ============================================================
# Condition builder (teacher palatability UNCHANGED)
# ============================================================
def build_env_for_condition(args, teacher_pal: str, learner_profile: str, familiarity_mode: str, empathy_mode: str):
    lick_mean = args.lick_mean_s
    if teacher_pal == "high":
        lick_mean = args.pal_high_lick_mean_s
    elif teacher_pal == "low":
        lick_mean = args.pal_low_lick_mean_s
    elif teacher_pal == "default":
        lick_mean = args.lick_mean_s

    env_kwargs = dict(
        grid_size=args.grid_size,
        dt_s=args.dt_s,
        max_time_s=args.max_time_s,
        teacher_period_s=args.teacher_period_s,
        teacher_jitter_s=args.teacher_jitter_s,
        lick_mean_s=lick_mean,
        lick_dist=args.lick_dist,
        lick_cv=args.lick_cv,
        window_s=args.window_s,
        eat_cooldown_s=args.eat_cooldown_s,
        observe_cost=args.observe_cost,
        eat_cost=args.eat_cost,
        attend_bonus=args.attend_bonus,
        p_detect_per_obs=args.p_detect_per_obs,
        win_rem_noise_steps=args.win_rem_noise_steps,

        social_visibility=1.0,
        detect_memory_steps=None,

        familiarity_enabled=(familiarity_mode == "on"),
        familiarity_init=args.fam_init,
        fam_decay_ep=args.fam_decay_ep,
        fam_gain_obs=args.fam_gain_obs,
        fam_gain_detect=args.fam_gain_detect,
        p_detect_fam_boost=args.p_detect_fam_boost,
        noise_fam_reduction=args.noise_fam_reduction,

        # familiarity behavior shaping (requested)
        fam_obs_cost_reduction=args.fam_obs_cost_reduction,
        fam_attend_bonus_boost=args.fam_attend_bonus_boost,
        fam_move_shaping=args.fam_move_shaping,
        fam_obs_during_window_penalty=args.fam_obs_during_window_penalty,
        fam_latency_bonus=args.fam_latency_bonus,

        # empathy
        empathy_enabled=(empathy_mode == "on"),
        empathy_seen_reward=args.empathy_seen_reward,
    )

    if learner_profile == "normal":
        pass
    elif learner_profile == "blind":
        env_kwargs["social_visibility"] = 0.0
    elif learner_profile == "autistic":
        env_kwargs["p_detect_per_obs"] = args.aut_p_detect_per_obs
        env_kwargs["win_rem_noise_steps"] = args.aut_win_rem_noise_steps
        env_kwargs["detect_memory_steps"] = args.aut_detect_memory_steps
        env_kwargs["observe_cost"] = args.aut_observe_cost
        if familiarity_mode == "on":
            env_kwargs["fam_gain_obs"] = args.aut_fam_gain_obs
            env_kwargs["fam_gain_detect"] = args.aut_fam_gain_detect
            env_kwargs["p_detect_fam_boost"] = args.aut_p_detect_fam_boost
            env_kwargs["noise_fam_reduction"] = args.aut_noise_fam_reduction
    else:
        raise ValueError(f"Unknown learner_profile={learner_profile}")

    return SocialLickEnv1D(**env_kwargs)

def build_agent(args, state_dim=6, action_dim=5, lr_override: Optional[float] = None):
    lr_use = float(args.lr if lr_override is None else lr_override)
    if args.algo == "dqn":
        return DQNAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            lr=lr_use,
            gamma=args.gamma,
            batch_size=args.batch_size,
            buffer_size=args.buffer_size,
            eps_start=args.eps_start,
            eps_end=args.eps_end,
            eps_decay=args.eps_decay,
            tau=args.tau,
            grad_clip=args.grad_clip,
            use_compile=args.use_compile,
        )
    elif args.algo == "ppo":
        return PPOAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            lr=lr_use,
            gamma=args.gamma,
            gae_lambda=args.gae_lambda,
            clip_ratio=args.clip_ratio,
            vf_coef=args.vf_coef,
            ent_coef=args.ent_coef,
            train_epochs=args.ppo_epochs,
            minibatch_size=args.minibatch_size,
            max_grad_norm=args.max_grad_norm,
        )
    elif args.algo == "tabular":
        return TabularQAgent(
            action_dim=action_dim,
            alpha=args.tab_alpha,
            gamma=args.gamma,
            eps_start=args.eps_start,
            eps_end=args.eps_end,
            eps_decay=args.eps_decay,
        )
    else:
        raise ValueError(f"Unknown algo: {args.algo}")

# ============================================================
# Multi-seed runner
# ============================================================
def run_single_condition_multi_seed(args, teacher_pal: str, learner_profile: str, familiarity_mode: str, empathy_mode: str, out_dir: str):
    ensure_dir(out_dir)

    pdf_pages = None
    if args.plot_ext.lower() == "pdf" and args.multipage_pdf:
        pdf_pages = PdfPages(os.path.join(out_dir, f"all_figures_{args.algo}_mean_sem.pdf"))

    runs_logs: List[List[Dict[str, float]]] = []
    for i in range(args.n_seeds):
        seed_i = args.seed + i * args.seed_stride
        print(f"  - seed {seed_i}")
        set_seed(seed_i)
        env = build_env_for_condition(args, teacher_pal=teacher_pal, learner_profile=learner_profile,
                                      familiarity_mode=familiarity_mode, empathy_mode=empathy_mode)
        agent = build_agent(args, state_dim=6, action_dim=5)

        logs, _traces = train_social_with_latents(
            env, agent,
            episodes=args.episodes,
            log_every=max(50, args.episodes // 12),
            updates_per_step=args.updates_per_step,
            early_stop_threshold=args.early_stop_threshold,
            early_stop_window=args.early_stop_window,
            compute_latents=False,  # off for multi-seed
        )
        runs_logs.append(logs)

    if not args.no_plots:
        title_suffix = f" (T={teacher_pal}, L={learner_profile}, fam={familiarity_mode}, emp={empathy_mode}, n={args.n_seeds})"
        plot_single_learning_panel_mean_sem(
            runs=runs_logs,
            episodes=args.episodes,
            save_dir=out_dir,
            ext=args.plot_ext,
            pdf_pages=pdf_pages,
            title_suffix=title_suffix,
            alpha_ema=args.alpha_ema,
        )

    if pdf_pages is not None:
        pdf_pages.close()

    return runs_logs

# ============================================================
# Comparison suite (supports empathy + familiarity without touching teacher pal)
# ============================================================
def run_comparison_suite_multi_seed(args):
    base_dir = args.save_dir
    ensure_dir(base_dir)

    cmp_pdf = None
    if args.plot_ext.lower() == "pdf" and args.multipage_pdf:
        cmp_pdf = PdfPages(os.path.join(base_dir, f"comparison_suite_{args.algo}_mean_sem.pdf"))

    teacher_pals = ["high", "low"] if args.compare_teacher_pal else [args.teacher_pal]
    profiles = ["normal", "blind", "autistic"] if args.compare_profiles else [args.learner_profile]
    fam_modes = ["off", "on"] if args.compare_familiarity else [args.familiarity]
    emp_modes = ["off", "on"] if args.compare_empathy else [args.empathy]

    results: Dict[str, List[List[Dict[str, float]]]] = {}
    meta: Dict[str, Tuple[str, str, str, str]] = {}

    cond_list = []
    for tp in teacher_pals:
        for pr in profiles:
            for fm in fam_modes:
                for em in emp_modes:
                    cond_list.append((tp, pr, fm, em))

    for (tp, pr, fm, em) in cond_list:
        label = f"T={tp}|L={pr}|F={fm}|E={em}"
        out_dir = os.path.join(base_dir, label.replace("|", "__").replace("=", "_"))
        print("\n" + "=" * 80)
        print(f"Running condition: {label}  (n_seeds={args.n_seeds})")
        runs_logs = run_single_condition_multi_seed(args, tp, pr, fm, em, out_dir=out_dir)
        results[label] = runs_logs
        meta[label] = (tp, pr, fm, em)

    if not args.no_plots:
        plot_comparison_curves_mean_sem(results, metric="bout_success_rate", episodes=args.episodes,
                                        save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                                        alpha_ema=args.alpha_ema,
                                        title="Bout success rate (mean ± SEM)")
        plot_comparison_curves_mean_sem(results, metric="mean_reward", episodes=args.episodes,
                                        save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                                        alpha_ema=args.alpha_ema,
                                        title="Episode reward (mean ± SEM)")
        plot_comparison_curves_mean_sem(results, metric="obs_rate", episodes=args.episodes,
                                        save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                                        alpha_ema=args.alpha_ema,
                                        title="Observe rate (mean ± SEM)")
        plot_comparison_curves_mean_sem(results, metric="obs_seen_lick_rate", episodes=args.episodes,
                                        save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                                        alpha_ema=args.alpha_ema,
                                        title="OBS sees cue rate (mean ± SEM)")
        plot_comparison_curves_mean_sem(results, metric="mean_latency_s", episodes=args.episodes,
                                        save_dir=base_dir, ext=args.plot_ext, pdf_pages=cmp_pdf,
                                        alpha_ema=args.alpha_ema,
                                        title="Mean response latency (s) (mean ± SEM)")

    if cmp_pdf is not None:
        cmp_pdf.close()

    stats_path = os.path.join(base_dir, "significance_report.txt")
    with open(stats_path, "w", encoding="utf-8") as f:
        f.write("Permutation-test significance on LAST-K episode-window means.\n")
        f.write(f"metric_last_k={args.stats_last_k}, n_perm={args.stats_n_perm}, seed={args.seed}\n\n")

    # helper for pairwise tests
    def _pairwise(a_lab, b_lab, group_key):
        a_runs = results[a_lab]; b_runs = results[b_lab]
        for metric in ["bout_success_rate", "mean_reward", "obs_seen_lick_rate", "mean_latency_s"]:
            a_scores = last_k_window_scores(a_runs, metric, args.episodes, args.stats_last_k)
            b_scores = last_k_window_scores(b_runs, metric, args.episodes, args.stats_last_k)
            pval = permutation_test_mean_diff(a_scores, b_scores, n_perm=args.stats_n_perm, seed=args.seed)
            d = cohen_d(a_scores, b_scores)
            write_stats_report(stats_path, group_key, metric, a_lab, b_lab, a_scores, b_scores, pval, d)

    # Compare empathy on/off within each (T,L,F)
    if args.compare_empathy:
        groups = defaultdict(dict)  # (tp,pr,fm) -> {em: label}
        for label, (tp, pr, fm, em) in meta.items():
            groups[(tp, pr, fm)][em] = label
        for (tp, pr, fm), m in groups.items():
            if "off" in m and "on" in m:
                _pairwise(m["off"], m["on"], group_key=f"T={tp},L={pr},F={fm}  (E off vs on)")

    # Compare familiarity on/off within each (T,L,E)
    if args.compare_familiarity:
        groups = defaultdict(dict)  # (tp,pr,em) -> {fm: label}
        for label, (tp, pr, fm, em) in meta.items():
            groups[(tp, pr, em)][fm] = label
        for (tp, pr, em), m in groups.items():
            if "off" in m and "on" in m:
                _pairwise(m["off"], m["on"], group_key=f"T={tp},L={pr},E={em}  (F off vs on)")

    # Compare profiles (normal vs blind/autistic) within each (T,F,E)
    if args.compare_profiles:
        groups = defaultdict(dict)  # (tp,fm,em) -> {profile: label}
        for label, (tp, pr, fm, em) in meta.items():
            groups[(tp, fm, em)][pr] = label
        for (tp, fm, em), m in groups.items():
            if "normal" in m and "blind" in m:
                _pairwise(m["normal"], m["blind"], group_key=f"T={tp},F={fm},E={em}  (normal vs blind)")
            if "normal" in m and "autistic" in m:
                _pairwise(m["normal"], m["autistic"], group_key=f"T={tp},F={fm},E={em}  (normal vs autistic)")

    print("\nSaved significance report:", stats_path)
    return results

# ============================================================
# LR sweep (optional)
# ============================================================
def parse_lr_sweep(s: str) -> List[float]:
    if s is None or s.strip() == "":
        return []
    out = []
    for part in s.split(","):
        part = part.strip()
        if part == "":
            continue
        out.append(float(part))
    return out

def run_lr_sweep(args):
    lrs = parse_lr_sweep(args.lr_sweep)
    if len(lrs) == 0:
        return

    base_dir = args.save_dir
    ensure_dir(base_dir)

    tp = args.teacher_pal
    pr = args.learner_profile
    fm = args.familiarity
    em = args.empathy

    means = []
    sems = []
    raw_means = {}

    for lr in lrs:
        scores = []
        print("\n" + "=" * 80)
        print(f"LR sweep: lr={lr:g} (n_seeds={args.n_seeds})  condition: T={tp}, L={pr}, F={fm}, E={em}")
        for i in range(args.n_seeds):
            seed_i = args.seed + i * args.seed_stride
            set_seed(seed_i)
            env = build_env_for_condition(args, teacher_pal=tp, learner_profile=pr, familiarity_mode=fm, empathy_mode=em)
            agent = build_agent(args, state_dim=6, action_dim=5, lr_override=lr)
            logs, _tr = train_social_with_latents(
                env, agent,
                episodes=args.episodes,
                log_every=max(50, args.episodes // 12),
                updates_per_step=args.updates_per_step,
                early_stop_threshold=args.early_stop_threshold,
                early_stop_window=args.early_stop_window,
                compute_latents=False,
            )
            y = logs_to_array_with_ffill(logs, "bout_success_rate", args.episodes)
            scores.append(float(np.mean(y[-args.stats_last_k:])))
        scores = np.array(scores, dtype=np.float32)
        raw_means[lr] = scores
        m = float(np.mean(scores))
        s = float(np.std(scores, ddof=1) / math.sqrt(len(scores))) if len(scores) > 1 else 0.0
        means.append(m); sems.append(s)

    fig = plt.figure(figsize=(6.8, 4.5))
    x = np.array(lrs, dtype=np.float32)
    y = np.array(means, dtype=np.float32)
    e = np.array(sems, dtype=np.float32)
    plt.errorbar(x, y, yerr=e, fmt="-o", capsize=4)
    if args.lr_sweep_logx:
        plt.xscale("log")
    plt.xlabel("learning rate")
    plt.ylabel(f"mean bout success (last {args.stats_last_k} eps)")
    plt.title("LR sweep: mean ± SEM")
    plt.tight_layout()
    _save_fig(fig, os.path.join(base_dir, "lr_sweep_bout_success_mean_sem"), ext=args.plot_ext, pdf_pages=None)

    out_txt = os.path.join(base_dir, "lr_sweep_scores.txt")
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write(f"LR sweep scores (metric=bout_success_rate, last_k={args.stats_last_k})\n")
        for lr in lrs:
            sc = raw_means[lr]
            f.write(f"lr={lr:g}  n={len(sc)}  mean={float(np.mean(sc)):.4f}  sem={float(np.std(sc, ddof=1)/math.sqrt(len(sc)) if len(sc)>1 else 0.0):.4f}  values={sc.tolist()}\n")
    print("Saved LR sweep plot + scores:", out_txt)

# ============================================================
# run_experiment
# ============================================================
def run_experiment(args):
    ensure_dir(args.save_dir)

    if args.lr_sweep is not None and args.lr_sweep.strip() != "":
        run_lr_sweep(args)

    if args.compare_suite:
        return run_comparison_suite_multi_seed(args)

    out_dir = os.path.join(
        args.save_dir,
        f"single_T_{args.teacher_pal}_L_{args.learner_profile}_F_{args.familiarity}_E_{args.empathy}",
    )
    runs = run_single_condition_multi_seed(
        args, args.teacher_pal, args.learner_profile, args.familiarity, args.empathy, out_dir=out_dir
    )
    return runs

# ============================================================
# Args
# ============================================================
def build_argparser():
    p = argparse.ArgumentParser(description="Active observational learning + empathy/familiarity comparisons + SEM plots + significance + LR sweep")

    # algo
    p.add_argument("--algo", type=str, default="dqn", choices=["dqn", "ppo", "tabular"])

    # run
    p.add_argument("--episodes", type=int, default=1200)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--seed_stride", type=int, default=1000, help="seed_i = seed + i*seed_stride")
    p.add_argument("--n_seeds", type=int, default=5, help="replicate runs per condition for SEM/statistics")
    p.add_argument("--save_dir", type=str, default="plots")
    p.add_argument("--no_plots", action="store_true")
    p.add_argument("--plot_ext", type=str, default="pdf", choices=["pdf", "png"])
    p.add_argument("--multipage_pdf", action="store_true")
    p.add_argument("--updates_per_step", type=int, default=4)
    p.add_argument("--alpha_ema", type=float, default=0.02, help="EMA alpha used before mean±SEM aggregation")
    p.add_argument("--no_latents", action="store_true", help="skip latent simulation (kept; default multi-seed already skips)")

    # significance settings
    p.add_argument("--stats_last_k", type=int, default=200)
    p.add_argument("--stats_n_perm", type=int, default=5000)

    # environment
    p.add_argument("--grid_size", type=int, default=15)
    p.add_argument("--dt_s", type=float, default=0.5)
    p.add_argument("--max_time_s", type=float, default=120.0)
    p.add_argument("--teacher_period_s", type=float, default=30.0)
    p.add_argument("--teacher_jitter_s", type=float, default=6.0)
    p.add_argument("--lick_mean_s", type=float, default=1.0)
    p.add_argument("--lick_dist", type=str, default="gamma", choices=["gamma", "lognormal"])
    p.add_argument("--lick_cv", type=float, default=0.8)
    p.add_argument("--window_s", type=float, default=3.0)
    p.add_argument("--eat_cooldown_s", type=float, default=2.0)
    p.add_argument("--observe_cost", type=float, default=0.002)
    p.add_argument("--eat_cost", type=float, default=0.005)
    p.add_argument("--attend_bonus", type=float, default=0.004)
    p.add_argument("--p_detect_per_obs", type=float, default=0.70)
    p.add_argument("--win_rem_noise_steps", type=int, default=1)

    # common learning
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--eps_start", type=float, default=1.0)
    p.add_argument("--eps_end", type=float, default=0.05)
    p.add_argument("--eps_decay", type=float, default=0.9995)

    # DQN
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--buffer_size", type=int, default=50000)
    p.add_argument("--tau", type=float, default=0.02)
    p.add_argument("--grad_clip", type=float, default=10.0)
    p.add_argument("--use_compile", action="store_true")

    # PPO
    p.add_argument("--gae_lambda", type=float, default=0.95)
    p.add_argument("--clip_ratio", type=float, default=0.2)
    p.add_argument("--vf_coef", type=float, default=0.5)
    p.add_argument("--ent_coef", type=float, default=0.01)
    p.add_argument("--ppo_epochs", type=int, default=4)
    p.add_argument("--minibatch_size", type=int, default=256)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    # Tabular Q
    p.add_argument("--tab_alpha", type=float, default=0.25)

    # early stopping
    p.add_argument("--early_stop_threshold", type=float, default=0.8)
    p.add_argument("--early_stop_window", type=int, default=100)

    # teacher palatability (kept; DO NOT change unless you turn on compare_teacher_pal)
    p.add_argument("--teacher_pal", type=str, default="default", choices=["default", "high", "low"])
    p.add_argument("--pal_high_lick_mean_s", type=float, default=1.4)
    p.add_argument("--pal_low_lick_mean_s", type=float, default=0.6)

    # learner
    p.add_argument("--learner_profile", type=str, default="normal", choices=["normal", "blind", "autistic"])

    # familiarity toggle
    p.add_argument("--familiarity", type=str, default="off", choices=["off", "on"])

    # empathy toggle
    p.add_argument("--empathy", type=str, default="off", choices=["off", "on"])
    p.add_argument("--empathy_seen_reward", type=float, default=0.05)

    # familiarity baseline (normal)
    p.add_argument("--fam_init", type=float, default=0.0)
    p.add_argument("--fam_decay_ep", type=float, default=0.00)
    p.add_argument("--fam_gain_obs", type=float, default=0.01)
    p.add_argument("--fam_gain_detect", type=float, default=0.04)
    p.add_argument("--p_detect_fam_boost", type=float, default=0.20)
    p.add_argument("--noise_fam_reduction", type=float, default=0.70)

    # familiarity -> behavior shaping (requested)
    p.add_argument("--fam_obs_cost_reduction", type=float, default=0.50)
    p.add_argument("--fam_attend_bonus_boost", type=float, default=0.50)
    p.add_argument("--fam_move_shaping", type=float, default=0.010)
    p.add_argument("--fam_obs_during_window_penalty", type=float, default=0.002)
    p.add_argument("--fam_latency_bonus", type=float, default=0.20)

    # autistic overrides
    p.add_argument("--aut_p_detect_per_obs", type=float, default=0.25)
    p.add_argument("--aut_win_rem_noise_steps", type=int, default=3)
    p.add_argument("--aut_detect_memory_steps", type=int, default=12)
    p.add_argument("--aut_observe_cost", type=float, default=0.004)

    p.add_argument("--aut_fam_gain_obs", type=float, default=0.006)
    p.add_argument("--aut_fam_gain_detect", type=float, default=0.020)
    p.add_argument("--aut_p_detect_fam_boost", type=float, default=0.12)
    p.add_argument("--aut_noise_fam_reduction", type=float, default=0.45)

    # comparison suite switches
    p.add_argument("--compare_suite", action="store_true")
    p.add_argument("--compare_profiles", action="store_true")
    p.add_argument("--compare_teacher_pal", action="store_true")
    p.add_argument("--compare_familiarity", action="store_true")
    p.add_argument("--compare_empathy", action="store_true")

    # LR sweep
    p.add_argument("--lr_sweep", type=str, default="", help="comma-separated learning rates, e.g. '1e-4,3e-4,1e-3'")
    p.add_argument("--lr_sweep_logx", action="store_true")

    return p

# # ============================================================
# # Example usage (your style)
# # ============================================================
# if __name__ == "__main__":
#     parser = build_argparser()
#     args = parser.parse_args([])  # start from defaults

#     # --------- Recommended: compare JUST familiarity on/off (teacher pal unchanged) ---------
#     args.algo = "dqn"
#     args.episodes = 800
#     args.save_dir = "plots_cmp_fam"
#     args.compare_suite = True

#     args.compare_teacher_pal = False      # do NOT change palatability
#     args.compare_profiles = False
#     args.compare_empathy = False
#     args.compare_familiarity = True       # <-- only this factor

#     args.teacher_pal = "default"
#     args.learner_profile = "normal"
#     args.empathy = "off"                  # fixed
#     args.multipage_pdf = True

#     # If you instead want empathy-only comparison, use:
#     # args.save_dir = "plots_cmp_empathy"
#     # args.compare_familiarity = False
#     # args.compare_empathy = True
#     # args.familiarity = "off"            # fixed
#     #
#     # (Everything else stays the same.)

#     run_experiment(args)


## Familarity

In [ ]:
# Example usage (your style)
# ============================================================
if __name__ == "__main__":
    parser = build_argparser()
    args = parser.parse_args([])  # start from defaults

    # --------- Recommended: compare JUST familiarity on/off (teacher pal unchanged) ---------
    args.algo = "dqn"
    args.episodes = 800
    args.save_dir = "plots_cmp_fam"
    args.compare_suite = True

    args.compare_teacher_pal = False      # do NOT change palatability
    args.compare_profiles = False
    args.compare_empathy = False
    args.compare_familiarity = True       # <-- only this factor

    args.teacher_pal = "default"
    args.learner_profile = "normal"
    args.empathy = "off"                  # fixed
    args.multipage_pdf = True

    # If you instead want empathy-only comparison, use:
    # args.save_dir = "plots_cmp_empathy"
    # args.compare_familiarity = False
    # args.compare_empathy = True
    # args.familiarity = "off"            # fixed
    #
    # (Everything else stays the same.)

    run_experiment(args)

## Empathy

In [ ]:
# Example usage (your style)
# ============================================================
if __name__ == "__main__":
    parser = build_argparser()
    args = parser.parse_args([])  # start from defaults

    # --------- Recommended: compare JUST familiarity on/off (teacher pal unchanged) ---------
    args.algo = "dqn"
    args.episodes = 800
    args.save_dir = "plots_cmp_fam"
    args.compare_suite = True

    args.compare_teacher_pal = False      # do NOT change palatability
    args.compare_profiles = False
    args.compare_empathy = True 
    args.compare_familiarity = False       # <-- only this factor

    args.teacher_pal = "default"
    args.learner_profile = "normal"
    args.empathy = "off"                  # fixed
    args.multipage_pdf = True

    # If you instead want empathy-only comparison, use:
    # args.save_dir = "plots_cmp_empathy"
    # args.compare_familiarity = False
    # args.compare_empathy = True
    # args.familiarity = "off"            # fixed
    #
    # (Everything else stays the same.)

    run_experiment(args)

## Palatablity

In [ ]:
# Example usage (your style)
# ============================================================
if __name__ == "__main__":
    parser = build_argparser()
    args = parser.parse_args([])  # start from defaults

    # --------- Recommended: compare JUST familiarity on/off (teacher pal unchanged) ---------
    args.algo = "dqn"
    args.episodes = 800
    args.save_dir = "plots_cmp_fam"
    args.compare_suite = True

    args.compare_teacher_pal = True      # do NOT change palatability
    args.compare_profiles = False
    args.compare_empathy = False
    args.compare_familiarity = False       # <-- only this factor

    args.teacher_pal = "default"
    args.learner_profile = "normal"
    args.empathy = "off"                  # fixed
    args.multipage_pdf = True

    # If you instead want empathy-only comparison, use:
    # args.save_dir = "plots_cmp_empathy"
    # args.compare_familiarity = False
    # args.compare_empathy = True
    # args.familiarity = "off"            # fixed
    #
    # (Everything else stays the same.)

    run_experiment(args)

## Observing

In [ ]:
# Example usage (your style)
# ============================================================
if __name__ == "__main__":
    parser = build_argparser()
    args = parser.parse_args([])  # start from defaults

    # --------- Recommended: compare JUST familiarity on/off (teacher pal unchanged) ---------
    args.algo = "dqn"
    args.episodes = 800
    args.save_dir = "plots_cmp_fam"
    args.compare_suite = True

    args.compare_teacher_pal = False     # do NOT change palatability
    args.compare_profiles = True
    args.compare_empathy = False
    args.compare_familiarity = False       # <-- only this factor

    args.teacher_pal = "default"
    args.learner_profile = "normal"
    args.empathy = "on"                  # fixed
    args.multipage_pdf = True

    # If you instead want empathy-only comparison, use:
    # args.save_dir = "plots_cmp_empathy"
    # args.compare_familiarity = False
    # args.compare_empathy = True
    # args.familiarity = "off"            # fixed
    #
    # (Everything else stays the same.)

    run_experiment(args)